# Демографический советник — ML-05 V6.0.0 · FULL / REPLACE / REPORTS

**Полный автономный ноутбук для Google Colab.** Единственный запуск `Runtime → Run all`:

1. Фиксирует точную версию источников Stage I и проверяет SHA-256 всех **52** наборов.
2. Выполняет семь исследовательских модулей ML-5.1–ML-5.7, включая допустимые временные и пространственные проверки, прогнозы, сетевой анализ, сравнение с базовыми методами и численную диагностику.
3. Формирует научные таблицы, модели, полные прогнозы, evidence-архив, визуализации, QA и воспроизводимый исходный ноутбук **из встроенного снимка кода**, без ненадёжного Colab `get_ipynb`.
4. Автоматически скачивает компактную аналитику на компьютер пользователя. **Крупные доказательные массивы не скачиваются в браузер.**
5. Записывает крупный evidence частями в черновой Release приватного GitHub, проверяет SHA-256 и **одним коммитом заменяет только файлы в `stage2/ml05_education/`**. Устаревшие файлы исчезают из текущей версии сегмента. История Git, старые теги и другие разделы не переписываются.

**Научный принцип:** выполнение вычислительного кода не равнозначно доказанности первоначальных целевых переменных. Если данные не идентифицируют приём по организациям, закрепление выпускников или подтверждённую учебную площадку, результат получает статус `NOT_IDENTIFIABLE`, а не искусственный прогноз. При ошибке вычислений или локальной QA автоматическая замена рабочей версии блокируется. Исходные Stage I никогда не удаляются.


## 1. Единственная конфигурационная ячейка
Первый запуск: `MODE="SMOKE"` и `AUTO_PUBLISH_AFTER_FULL_PASS=False`. Для последующей валидации используйте `PILOT10`, затем `FULL`. Точные имена показателей для прогноза приёма можно утвердить только после первичного анализа словаря. Меняется только эта ячейка.

In [ ]:
# Единственная ячейка, в которой пользователь меняет параметры эксперимента.
from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass(frozen=True)
class ML5Config:
    VERSION: str = 'stage2-ml5-v6.0.0-full-replacement'
    MODE: str = 'FULL'             # SMOKE | PILOT10 | FULL; FULL по всем строкам без загрузки всего E04/E05 в RAM
    REPO: str = 'AntonDozhdikov/Demograf_sovetnik'
    SOURCE_BRANCH: str = 'main'
    INPUT_INDEX: str = 'handoff/07_v2/dataset_index.json'
    EXPECTED_INDEX_BLOB_SHA: str = 'd6625ccddc24f24a1ee451e33c48b736d3a57043'
    TARGET_BRANCH: str = 'ml05-education-20261102T160726Z'
    OUTPUT_PREFIX: str = 'stage2/ml05_education'
    WORKDIR: str = '/content/ml5_v6_0_0'
    FEATURE_PUBLICATION_LAG_YEARS: int = 1
    LABEL_PUBLICATION_LAG_YEARS: int = 1
    HORIZON: int = 1
    ALLOW_RETROSPECTIVE_ONLY: bool = True  # Все исторические данные доступны в текущем, а не историческом vintage.
    REQUIRE_ORIGINAL_E01_FOR_PREDICTION: bool = True
    APPROVE_UNAMBIGUOUS_INDICATORS: bool = False  # Показатель для прогноза утверждается явно, не эвристикой
    APPROVED_VO_ADMISSION_NAMES: tuple = ()  # Точные названия из аудита E02, проверенные по методике
    APPROVED_SPO_ADMISSION_NAMES: tuple = ()  # Точные названия из аудита E01/СПО
    STRICT_OD_COVERAGE_BREAK: bool = True   # При разрыве допускается только описательная диагностика / разведочный тест
    ALLOW_RETROSPECTIVE_TESTS: bool = True  # Не выдавать за operational historical backtest
    MIN_YEARS_FOR_TEMPORAL: int = 3
    MIN_TRAIN_ROWS: int = 100   # For regional ML-5.3 overridden by REGION_MIN_TRAIN_ROWS
    MIN_DEV_ROWS: int = 20
    MIN_TEST_ROWS: int = 20
    MIN_MUNICIPALITIES_E0E1: int = 35
    MAX_REMOTE_INPUT_GIB: float = 40.0
    MAX_READ_ROWS_SMOKE: int = 120_000
    BATCH_ROWS: int = 20_000
    DUCKDB_MEMORY_LIMIT: str = '512MB'
    DUCKDB_THREADS: int = 1
    REGION_MIN_TRAIN_ROWS: int = 30
    REGION_MIN_TEST_ROWS: int = 25
    CATALOG_FULL_DOWNLOAD: bool = True   # FULL: strict 52/52 sha256 verified, no hidden skip
    CATALOG_SCAN_ALL_YEARS: bool = True  # all recorded year labels, without pandas materialization
    STAGE1_DISK_RESERVE_GIB: float = 5.0
    EXTRA_CATALOG_PROFILE_MAX_GROUPS: int = 1500
    MAX_PANDAS_ROWS: int = 150_000
    PANEL_IN_MEMORY_MAX_ROWS: int = 15_000
    PANEL_PREVIEW_ROWS: int = 2_000
    EXPORT_PARQUET_CHUNK_ROWS: int = 10_000
    EXCEL_PREVIEW_ROWS: int = 3_000
    MIN_FREE_RAM_GIB: float = 2.0
    BOOTSTRAP_REPS: int = 399
    ML54_FULL_NO_STRATA_CAP: bool = True
    ML54_FINAL_SEEDS: int = 50
    MAX_COUNT_PREDICTION_MULTIPLIER: float = 10.0
    REQUIRE_UNIQUE_MODEL_ARTIFACTS: bool = True
    REQUIRE_SPATIAL_BASELINES: bool = True
    RANDOM_SEED: int = 20261104
    DEV_SEEDS: tuple = (11,23,37,41,53,67,79,83,97,101)
    FINAL_SEEDS: tuple = tuple(range(101,151))
    AUTO_PUBLISH_AFTER_FULL_PASS: bool = False  # Публикует только финальный проверенный этап V6.
    PUBLISH_RESULTS_TO_PRIVATE_GITHUB: bool = True   # Только отдельный закрытый Release + свой сегмент ветки.
    DOWNLOAD_COMPACT_REPORTS: bool = True              # Скачать отчёты ДО долгой загрузки в GitHub.
    GH_RELEASE_PART_MIB: int = 160                    # Ограничить размер одного Release asset.
    GH_REQUIRE_ASSET_DIGEST: bool = True
    REQUIRE_COMPLETED_COMPUTATIONAL_CONTRACT: bool = True
    PUBLISH_PARTIAL_RUN_WITH_EXPLICIT_FAIL_STATUS: bool = False
    STORE_DERIVED_RESTRICTED_DATA: bool = True  # Только внутри приватного репозитория, при допустимости лицензии.
    PRINT_MAX_ROWS: int = 12

CFG=ML5Config()
assert CFG.MODE in ('SMOKE','PILOT10','FULL')
assert CFG.HORIZON>=1 and CFG.FEATURE_PUBLICATION_LAG_YEARS>=0
assert CFG.LABEL_PUBLICATION_LAG_YEARS>=0
ROOT=Path(CFG.WORKDIR)
for sub in ('inputs','derived','figures','models','exports','logs'):(ROOT/sub).mkdir(parents=True,exist_ok=True)
print('ML-05',CFG.VERSION,'|',CFG.MODE,'| input: immutable Stage I | публикация:',CFG.AUTO_PUBLISH_AFTER_FULL_PASS)


In [ ]:
# V6 · неизменяемая копия исходного кода внутри ноутбука: Colab get_ipynb не нужен.
# Снимок создаётся на этапе сборки данного .ipynb; пользовательский GitHub-токен сюда не попадает.
EMBEDDED_SOURCE_B85 = "c-q{(YjYddl_>iC{ED`6k_MzE2vVYKgMsHlBqZUOBp3jc?V)(uO`t&{0s#aMNesm)*-m25u_vD7%+yTHOfqL?&aHc^?)@Okl1)8bHRlhIe_?*fU5~wY@7=u{B%S2ms_RNbboYMmwbx$nb?~2@3(c_G9bG6moBzpi4&eX%#vktwLU?;&FBslw_3!mAc=WB)!k6JfD`>TXx5Uf6a2&LPaR8qV4*AunKN&Vd(JVawz&ZY#<EO_jk6#~uee&@5(aA54k4`>4etPoJ@$=)O<7W>1czygBUVMD~3SK;S{^Q}_IIAl&`32|J)m%Q8cmDlb=laIViZkof%Ihnor857j)*AKOVtCGl=-0)I$NvmXzBqn$@;S7B{OI^GwEY@de+|PBFX7?j6L|gA$tTd|@sr~hCto@@`u&}5=q&ZS!FKNAMd$c$;lmMB`3OqF5TTZ+{w=in9W(%wa6YU}dgIPs=={g8e(S(*C+K!REZ9aimvfH)C(aS-9i2Rcu3^>=4h~l26*Tv2hgRYteTRo5>Y2V`HC!mCJ`Q$5rve`x7{Wubcn?p$q{iwuOEdEeSD_tf<LSvSor@P2=5Z>oV8mZRQ&b@rjcFdn^jnzJcVfx78LWK#^5hF1!bkAsDJ%tc^#}&^6jtjC-azPq_Vx)(1`9pIty{>={l~-qPCwrCa7xdi!mnW#k6@9h9a=kR=?FUc{P-yj_aQ-vT;r$F{IDQGPGTRP8Vi7<@#LGYX}7;U`P`$a(3l^?8v+MhiWkuJxA>7-ehlxP(i%U7R}L)HBiLeS7<ze$19?epz%=0ly*P4UI!9t<UdhcvXnDk)i|IW-`9xdn%LD+p2`@DOzyU%1PsOC59&Qzk@Z?uG0bXBRbe^Ds?WdO6hIc!yu-6P{pvi|Pzl0GyrX7IULT_T;@a<Rd7h3yCiPrfsj#o5>-yc(tygAPp27XDetfvPK98@BZqn&!qOF{#Cjq8sK`x5r&kwY*Hi+J&(hpYS)tMSr50ucJm@i(d(9YW{Bop9`T2K&A354B~y0?YW<;#9vRh}2MuLw_FO_8_?O3O}ck-@)T^#525G3KHZ_(N3ts0OG$!z&Uv+7X9Q41`2fszRcm=|A990JGmtY-p>hs)KQV2aIAQmju5bD;<3SEhfo3z$zxa^LYJRFLoftDN74KjVE%uH7>IV}b8VX1P@XARaOmXW-h9cx``Dr7dPs}S$D$T?!(bFLL>-Aa%DuSJ8Q+|2dsFev5iSLR2yV7q#Fw<VukppVbab9ZDCT3arq2;begR*f!Q<yRx(`Q)<K}1gy7>h^Y)zWMxYO^=et7XBO#I*IWIZB$@%ZF3KC@bNT#`=_vUoWif=${9*x{4EBpelCS%#OVFijpCv<!Xxy}-M$6nHIk#)K)TS9wIS3gH2OTU_+7c_0sI=jpV{Er$JgN~6`bOKx;7<pTSih7q30O?gG9_PYY2^T!d&R+tN4Dy(!A0iW1KdG-Y0NxTP$gAfqjF|Q5q<wMMfOwQslU8Gq0k%Vlx0$^ZI2@^i1M>S)G&rf~{tNOP*Fo&?HLgBPTN3>+G=p-Y7@GC}-yn!@(Kn^eI96GcwUsDqbmmy?(3}es5TE8&XMc&rhMG8M&@Wk-?`dHi|-w+;>7@aoZDPSsrGO*d#07QVT5S%=RIz!ZoHzESa#}Zk5LTkjADSyEcI3KQ68-8WEyw<2(uavH>ls|-VKa$IUL!_%QL2($E9v1VTaZzwB@C1AXQ;O|BPyU&j<qNT|5+a@v?themMd_keCl<y>;PMrbB8dY;m(^>c2E>&{pv}i9x;LIiq>J2yuQho5gunujs(v=`Ky1N3L|mSgNS&X{zuBIDGdu^8`U3}PGYh%LD}iHug@6e(>?eel@g%-h1z^y$F1T{hIR-M6j!c4NEI1#oR+r1g+ZXDq)pyIcFMNoI<ipZNqw243TwAHsZ~CR{jdIOLy4PPX)$7IUL3b2>m_uqBNmT?gNte+%QHYK2_$Ts?XP{E|7cBEL#0<`d>y?#iV=n(89~;6JFP#suL+s^W1x?D7m;wW8$hh6O+b3dBI*?Lvd35ms=@`1;9U>n1ygflo1%#$?9YnK>@XirqC`M%(5r@F}5dq+1!p{hJ{}kObbe)=a&0pbhH1;faDxfv{EeTKI{jfQ~3(0Txf%1p1y-ByL1K^w6SGM22GJiE40Q-~iU@|7|Yb%v+N&`oI<Q|a<FfDOk5s7Q3R=#kI`xmJpI@Ltm;>89lL#zp}_>p!aMljtT_V*l|OtTw|M&Zcm><#+EarDY_f>Eo}9LK5*g7I#*vn`9S!=vr$``=VM8L0O2uvcshCt)@!K;MAJSXk=!+MOMZ&BA|fm234%b*<ox#zUv*xJ*aP>~$Ar?h-FK(}t~@8HR&y&<yv&-q<x8z+<Uu<5_SW{l6bLbYPtajyOH=3rj6BgUZ*_V`iQlMGfP5d8x=<!~<UwqWqS~RZ$+nx}3i<TL!)twzO7SHAjXFuNqmYH}3bA`#;}pb?)@<&Mt?0{heXZ_DB7@;kehi6Ps1Nx=~vy``2ovwWXV?k-ebPixsS_t#82Xw6<J+UlrR8daZuDJ)3{izdJvR`!hnCyw?iv=YBfs$D1g>zg}Kyl$W^`|Jq9Rnh!*xYOZzl>ij~p*=jZCue9f{1ao0{WnnJ7e7Sk$t?f52Uu|6u7T(NXxg2Y@QL5c2Hxg3-0&zwWh%@u~`Kxnt`T54&)%=_DSN}ZLMs=fs(_62VuUFoW!AOkOjMaRvT6=f7Qd4zioBiGxpa*I#_;;`Rc|RYkalKq>Y}Cq(&X-CJfJ%R*bi@CkT&mRzPNz49cIM1FE2V4Ym2=g&S*=z69O2#iwY0KQeb29z8?|a3H>Gl`>{r)TJ}5Zb{eBn9<Jm%>KK?C&U?gsMXozT@#t612f+ZJ>2C)-5dHvEPXc8?VH*gmQ49lL2!}w1&DzIoUs~eTI(u!Zs&-vG@HQci0%2I=lwl?<Ibl2BwRhZ(&T50uK<;F&JqmDp`Ygxq=6cy<rrCI-Tl2c2)$+R2m8(I!^wS37`v?s`zzamk_k!1f7AzO#J?BA;TrRCL19Wj;<V=luO$CE)fgh6JqG(zJRenmrl%!DtFk(lmBL^vJMl~&Hrn@pM^DnZ;7$uV$jPg6!#U;h&fZEp7XUtr09A45%z8f>gzUG|rt>ssjsqN#G}-58A0b`izP92W6mk0rY;Y5Pa^3m;1m6{IobQWTYt*T7h3B^&pcrCy2;BWVDXJN=k7%JoJ)dCq<(xsfNtd_5w{hv2I}2#17ug08dM8IAkHP7{7^2hBU<a5RqHQk6AE3v@Ud<<<3St+XPsz~yA&My&+@)~fGCh$Nq<RlVLHI>T^>J0^+UWd|;i;jq(cg*|6`9}eXWgaWgfWEacjTSf=-dCM?hRZcCmy0KPSs;rk*Dve6H4k(~Jr)IVgD_(jZ)_t|wDEoYK-KbnEIPGpfz|vRXzPFVxEz=l$y2{mh%mWV3$I4$THI{D12AnLlys`A|@-=@I22%ULU#YBC8tS50n44d{=32`)ZsJhVFuQft)jH(U3n!L1wq}40ODokI%vmf~-&<R$mX?#J4L{Q?fRu9<=4Th?ozZT9k{$q#PP-GfJg3(OP$e)K-RTVM-PD&#Yd)M1d5y(RM9}R5_%-{(R@id(!(ixigYB>jROG$Rco(Q-ARK$GVC3utW9T>Nc79H$B#uM?Z_N42mHNA|zx8qrPM`q_3voy&zYnYG%RvI#yIxr-`v@O5YSoSPXjc~U78qJDtu2@8=6PDMKn9e93msvaVT0h$sFYR*F1%Zn@_Xh^CJO_xRjmO<Q>y)B1E`~=n;UEI#*VqI(o#wfmy?ANrmok@WgiGqb4};tYkI9(Z2&qit-~HfD|q?sw_`1?EL>seTdVq5zJVLDKILNP8blD(%jM|EELgT?sk*V&(C7s~Co7H0`bwo-i-tazzzmY|0NevB=_4Dk;+JZT%JtGx43XO_*4Kezg8^PE)ypfDwer+D06x`KnoERNh;qFW!xVs1#f;VMJ2&UeU-mA)>0O!g7A|{N-}K&m+k5M>_x79KTz)PaqeAsXlTFay%rNNfgc+#f%`MDjWA&y<B5Z^dT=b-{KpN00+&yA`U5LV;*@={Af;B(>4mE$VV<83u#~U`Z?&J7sm0P6-p7h4ejcXQy;a?3J09t;ciQ}IsY2nG|a1DR1$-pn6K!L^Lrfk>|dP<;*W5bh(6yR@l9YDqx`pNjH3|<oGA(s6MIsk@v^6ANMa5O09`8^XiFOW5q!!;R4A()Si<p!?ON*MqV5kmv6g=-2OT+KU{|6nM;XYDM}Z=Xt$3{ycbSp~wfeU`t-l|m7~*w#HC3V;^CO3RfSa4DVP4B^_Xl#zGDUmFeLfdGD&s%wBImZmi!!2>6Z$FR1s=D!DU;RA*QZmQDoua_!t5}?hE7zD?!k}8a3xeTxc;{a$Q`V@XH1D;9}bOwk$CMog9ESR<FQG{d~If|o5zaU8Vql9KOlD(5Ikj}}sDAS9djamh8UgC0@w+UF5uHPss=}AUA_PCY9u>;?8i2a}uC*u-{<a(}<cEB%^iP)<NX7runoC9|@y_(7ezf;Ux+nK@!8&zs>-xN`hl)+kpMc0Pb2GVvk+2)yadMIy$3ABKWxEr?M7c9VJ7~;FVehX+0_z~Vm=@kCz_IE~ZwvfS2W}%gAZtqU3Gt3NvA@Kbp5=(jEeW29+{++mxYA}Q?$hc(0;CeVZA(-;q2mFjn3=a+s3wCU8Z!!+H0X<6*`2z$Bg;k}JqrkPNq37%wq~_-mX67%qT5rAe=G%XmF!R<`G~*y;)e++q7DSTfK}guY6TD-jHArZ#j3Z`fp^qOGgw9y0jD-vWY5>m;{b|RBkT0vHUmrhb8<cNJX#N|ifhDbnxQ|#-%jOf$H4T;+Uyiz4OclH;TNlRHGjuyX$04DR@Yj(Ngc|scv`fUxY9lVMUMnvzBXcf<d2cPiA-{bA4zqvjs;@q;0XGDc*t>pxt+ZahS#9`l2A7-j;cX2Gw8AzJ@8b}#*&QFnvwpa@9Rela@_FYn*_dEeh;6q4>0iC#IX~}qwt@cNh9d@qxmk1=+$-YuTnp&>y#buCQHJZ~w%=O7m#r{knN2S2(I+<;H9H+Qo6Y7#VRtfa&%EWvXq4V|Q4SzC41?APNSb?IF!F&x838~Suf!{aVXGS!-9LOKV+@@pPEK##y=3`v$xSrY?KgqdX%#n{4kEB7qOaa|j*PXOPTOhb)KM-LU0ffR&}}ovTbCP+13+QOK4siL(z$c9TQP8GJKeCC5o6810Vp5FDjqlUTWTzWZfD%>^ukdlyE(u0Mv?a*23!m^9iS(KB~^Ji><=^Ut*Zt7#IpiKn2bed*BCboaOfDZMUhbF#(^g>)M9e*Vp}sjW>4E?Z!Q6g0ub*4hYHvcDs9J#z||5qabwS~l<3fDg<amy$W{<4<c^;iPO|4%cXNrQiNoud>Nmcu!*Dzq;yAvq>bBj!@^-V`&Yw$lizaOULc5KKlr8~b3-Fl4eg-LZWT@jc!f1U%iM9!Iv#R@5BoVghAvL8g8IJshl<|0Hzh;>@_QwPrt1KVWN8QvXrudX<ef;<d{rfSQZyiDF|08Z|0E_}Fq{Qu^^I=Mj{>LQ;!gl-Z?Kj^F{t$9N+$Q~z=ZyB@AA~*6L1mH$Amll_!DyFm7<u1x`W{Mvc0FgO={e2*U>_<C;k2N9)N|nAjPU?MbGtD5u7`BrVAyYlsE_2{@J7OKNmXV@#+#8IGvwjmzd!Hy!dQi7zuSc)2uq}NpG&w@;m~v7UJWMQu|@qk8Fq01Ks}CBS3jG;6>Ou?m&@zrwdL~KQl(rk9=L-Mj@vEBAD-Kr><#we1^@M2mXToz{sC|H131g~@Rj%ll<%Z@r?m~Q#1Bt9yYA=?aCE~Se1+ThPG>yBj~+f!U;Ni|0ZRheFdD-v^%KbD&d%<5XL}Fcs-GTWy75R5*9GIr=(`t=@AZdw?&FN)PtU#I9o@e-?7+gp`}g7DJ$``c3v7$!<xkHYhCiExqcQfRej443f7ZhB(I3zKX@47b1fInYT(j{m0)Eg9_Tb8*wZk_v>J4xT!FbT^kGZDya2R#8*BOmEy`AFb;LZ-AW|;B>?hkwf9IbC|4kok*S0mzU6H0UKP7jDNK=v7~4rNBp8i3<gg3X9<U%^?Y5xbr|5jim_K@(-No{QvwX*T=Fjx3)_p#vlQ&)5WH(O6Xa&yAxiMn1r-IQYNZPy3x-Ml1`>Rs42B6d<{XyG}lKy>@pp+KsbPQR&#_B`5!H3c0=}L*w+xEIPu%=nEo;p3#h+rkK(Ri#Wv&q?)HZJ^4*?m4Kc~nGgOGA*Wn(H*DTPkn3hP0Tt#L{~{;oxibi+gU$f{?eyUAbn)oV%(OZq{5}I;X2iD6P&2M~Q7qL~*1lGMXT1`&`{^~^Pe>wup0*#1y9o0*!N}<iJSn1w{|#Dz1PQyqTSh$eI#7>@0^x9u>GjVN?aGr2w*ehXU@`<opw8|`Ob1;E;`jlS<1cEn8%Mo(-gCiZ+?RM4U&P7??3aTz4Ut?`W-8)2r8x38{Rp80<uTdoHHJZNgkubcDb=9(WUOk{;cqvr(7Uu6z(om&!DAyTR(s*i{<utB6I?#^F>b@KRR?HFZ3rh541L1FvYlF|wFCFvdLMwN->Zc?$g=gJ#d0?ojXTY?aGcT_T>isOQ{97mVbJq~?GY@)IP^m#COv*R`q?BH0(VL8;uos8Mbs(=<7@b=1`;bT?-yBq<;Jzuw6b(*<>=y(#Q~=V06)D%(dc&K3b{SF$~&lhRrA9auvhqQWV^T%a@0t&y$q*f`C37uUgX2RA>l=&MMoh7U=TFJd?=9F(Jae+NeC?s`p`cHjlK+vPHJH^=l~-T8AEI015`M!t`|w1B?th2yf<(c`NMgnTPQui7Jx+T^?(#g2n*nSk-NyrB1j@4F;{d8d*NPxxL=5~4p35HzcACjeW6}%I9$^2cJ?~JSRbI|%c5&N1Ri=?TP`*=-X#%1@c<E_){s!bW)G(ghercf^}vTkg6Y7hiqPw&(Ri5Q^~)-55gGam9{h{Lx<pTQCaI}wG0oK#6Cd<(w8SpPmv{%*J0?UkvDU}1wn;WmKrmB~$<K&fiu^IR&WI+{ALVw!@u1Vn#I-~CUvM|*bnzaN)$VqN<4Mr<dF+{NPL)Vj{(+Fx35!s&Ml`HcrTFaR3#Yj=>`w*-C#4&QR7=W|__c5q`bK(0pfK&_$(O9p_@!j;a>+5-*Ns47ho#K!EZoqu?a?d`$<5k}1I3bQ?##M8!rWeP-_7ReER8bRlrE^7U{Fj}M6mV8;eFWW9BShSne3D%CPO^GMFbnYDe>Ok+?4iEj1UaoD7rY7LbT;T3}@7n2Ol$e5Br3|*%{GUTDx$c8VfWl#z?2!@AUlbeZa40U$WR-Q=2>A%u`lUwmuobnf(5(^Sb_c{^G@}`BRn^%1v9*@P0EKjGauiPO=;CR?wZW7$;j81f!7!6ZG1BU@8f#xCQzHywhmp?;uYDfZc0l5FO+M62kH@DJU@u<Gq~s=CfYf%-mtH=c^g$EvlDiXaopcluCz21I*ax=o_tm29YmbXW48H3#*0?ZRm~Y-&$$a^>9Avq`BA~7gm5M3O~Gy2L~Cqoo;_S=psOxA2TlBMnEvT2eh7tg_w~funY&-@8<0bi50qiVe8Pj(Yb~ezn)X8*Q27mDYAg;1-%}uMKhc-aH?wAhOHly)+uhxVc!Fm39i!qvq-S*qUl;fJcbj4rV3JD{*{y-pJBSJUtz}M3DGrT@p5FCseDdf0rIw4>KU#oQ9IIaMHu|C)52p1)p<ELA(ExTQ=z+&CTdv_^AttD>J8MbH!7>;B1(#Kd;MO2-0yXoI)X-ha~V$CIBf3r(4fyPxaj{=+gM}K3cydN-Dw)H2NU3Ygc-8_9?kNQ2`sp%S6{uh!a552ssa5<^+s`XYmw}T>cxXY{^g^A6FoN2>B#>{2@6%~^$mCnzl2Vrw6R=i&~tgUwAQFB(Z`=`lxq02R)(jjr^w2NO5r0Z;G?HTwNY9r<`-AXs79yvY+6$*FI8*H+6y(j>y;ZDH8H03l}bYmDe4z$Y*6X8OBRa<okO=kC~Z6$i6x`$DKC>;C>YU70&V^cd&Vd;QB>;tH(-xYs^k{(ha|A+;BgoBG2UA`LNUoS=x`4ZeL)a(QgoQ>_3ve*^e{IWH?z6UsLxz$CY#j4F@@*t0Y<^UJdL7TS-W0Mi0ol=bHQK`0!BP=p=rY4I0!F>(dA<BU(Xe-<BMTNJY+WuSM&L;!)!v`a4!7EJ(}&#E#$W%e3ju*ydftQTZA^?2hqyC-TnBKwIzbDo{NSY3B5y7i=iRo1E^;Z_}D4RnEiIA8)ncoCEn?7QOKZkZ1otYx=_7_HoHBl4M4W-W>NgmwqOthDY8O9@nxnP>}|J#LYqj2%$19C`S~jsFT#De?ZR&F<|YHuA8p&2hTYt5c)!)z0S+_?O2a*$^rfTtY((Wz4*2=~L6~uoMCt9gM5{=8g?>veX6W9&oy-4cI2Nuq40po&HYTt)+1rL0jnoMkv~uIV53j=^;K7kc?EVO~PvNi`#&=*8v_l_lKF`sDnn9&EzIw$?EJ065z)}|Ceo<c~>muEU<2&vJz*+nVe?wO;{``ykv_7b*gJawTZXkaNh64c9g~DV=q!sJ-@4>?0xd)PN)9nQ)Nr%y)2wCZ3AG0wzUTN@p(z^qAqvvkL=F|*&eY!e)gvd-^EmaIaQG1eWP4))Bxp^bF(EK~$zU1V4qscJzNwLn3gq~1U6k)VkrFj>6_G~(k`9y$?n!Dj%kZHB!>npAFh2QrPX>XN>z*iUrg7L2i`#@#BL(emmef&~Hc~Bmp6ube=&py-rPuPdyk>QkrG7-(+DsCaw?u1?7{x@T4Y^6ADwUM2k?Da;L^V=%6+MCU-Tni!3N_7F$27^uxzmH(geY6tusd%Q9EjZ)iAjr`>8}mjCQM?79SK-tR<BfC>>d)K=nY+fDpqFWlj6zY05rnM}FQvSMut9u5)Oa)NC72X>uAxZ=8m^yByHiQy<G0#=a~Dp;2w==33`>vxuzzn<L@h`{<P?1-MpQkbT>_H6sg}}lQ8{$R>^@x!i}-Jj#bW1?k&&36CR~gknb<`_O0DAHXSsD2jI)-D){{oyiDvkZCrxV5z{UOfXxwtW$p{v#Ln_H&xNjSYUA7S;b@CpXbie?$wN7KcVGYJ`Dsr7}zq#2dbS_0zwsNSRO_Ne!p9BHeFHdP2e_y1sxn8grPUWTnycE<6(B+!=cA5l1P6p0*QT)yA1p{$QJ9i5RY}mtHseeO&q6au_Yb81})0@ZtG~|5u6l@&B-PzL{N49T78X};zp#2O2gVzPX^vKlP8-;KzS=okQb$6O*f)PoOz(mG+sueSCYw`9RclN2B6S9i$=J)HJnmSM4h_J(;XYX2oikz7!n>IH-<T=J|q}@Zr*}xw;GQVRD8qFbg-tn-9!@D6n+zN0(%88*Pr9;&2U}LEg$`09#b$+N}N}xou#bQ1o-9WDzlx+z7<Qx!6#3Om=dML8ePrt+=S%pj#6Tmc5{9f(>N4sAT3)JlPM`0%I2C;jL<E8PlK&&c`%7ZI|jt`zz51Q}a^TWNtcwda5kl=P&ZF#s0tu}(yaxf0A523A0)@q5R9XGt6v<le*F*B~lSmKy2xdn$r-7)r4R&$g6;1p<+$n>k*&D>`~g-|H{#uAqPfbpTtDL8huJX`g>py(B3%c_ENIS1HeLKV^iB!cpYkZpqKf(6{KUc==-*#;a_qygndlf6tftENC`XU(vqk;`tjJ9Q>cq-9fbxWM1VS~|2WCk&W?KwnY`7zJW6Q+oc6s7&D;ERFLY0m1%WskRpHNXk|s9-ZLQEf8S3ctYF)uAe$4uAn>h*OC^^rVB}xtW?b$*TOBtRw9xExM&;hO2bX)`klNmzbLP+H$HGr?P=2M!0{K|!1URwgSHp|a&znSv7%Eei6C4_g|X54%0^v!qZtE4vMtSnPK;ehUM&)gvovC(i3UXr1uC?r!gF92&8(n*ADKaSHp2&88aB8#oUDo9HnI3tu{}sBB?|x;?4pDqnt<-HmrbRz($KdA=mW7b1@BPfm<$Gt)Yh*>FtjHelxUGu&c8qYyW@W|ZDHVoy4wlwO_xp+XAzs870c_dO&m`O9TJKPiztr;dmfsMbcSeHn~;_XLE$D>*pjjpDPco7;q`K<Ucp0ah8b{0>qY((n+}UjqeIT^y9F^YQxcilgOW^&iMVnm_=bNSrt(RxPO#JKqj^R`L$(+06_q&QkDwgJ_Jm$2Mb$g*kAtpIz|7f&>}C1`yJt~L36&I^s=0+JwrkD7#4YqkIrz&64qu{<fiy)h9xD?1!!$~BcFgAPz!;Qj!)4bn+33C94vL6+cRR38Gow9pXRx;-CI`TBaCm4De7fbeTdt?^52U|JvJhFj$rtO{Lm0@g-$T=KK-~v-3U`T$xZ~X=u_UypTO!_hqZkeA&`ynE9nCjo?N(6-$Q0WcG+(wjfs%a-hCC}6hIAqd8;Oa88_vn+=$jeE8X*sc*TOPexJW?>z?;MIb{LE)u%(JEiiM3xWfV1wBABunH^mx_g+~<<t=H^+$F7+w238cy4?O{YfBcWf|M&4fpx@?c)6gQMP>9i!&rd#!M-oL8`)l&S#DIoqh!`81G-o5z7GRP*XLr08w{NbnMJR$foU46h=FE?tbF;C!qG@z0>*T%clSh5M2s|6cRRCNiUF8?%;Xx^f&^3^-I(i1OuOI$gs#WWC;pPI5tl?QM*DE&=?QN`88XO>FD%AnK+$yj60*6Dr^7{?6oL$ylp?<AW7bCe<sc)24xFiaD;Kxr&erau4_a%gv%2!wT66-`9h61<&@9X|bb?M!rTH9Q0rQf`h$u2rkiF%{Du9Ya$eQRqAa2twk&~OZJ`{tJC2F)>$5^zDF&pbILjsr6BKK^^M9Qze%J74JW4LMXTMtl1_W!YE*<dgpob`)1jqKvtN_R+)`xoF(y58(ns1M7TFyy*6K@JT~O$vY!jf}60qE9IhqrVIj8#{Kd3J4X0svRBg_C4M}+{f?XUL?yq|ic<`UE8~>zP3|HluvYXC9*)FaNqQV}(=8Vy-sfQv@MZ?3Dw(_4gv<rs3lR%^)sU-dTVmM5xSg2*j3Uxy)X}B}EnC?|x?JgN>~v-q2hFj#ndkEPi>lTvRY~5&-C!_+j__iL!^TXc&dx3}9KrwXMa3XZXp5U16Q~tMQHHy%_zfjChKstc<gV$N*Rt)R_u2t&T`n9og8^UCR3e)_j8Qjyii&1w8h|)oTga-oDNak7w)^8&=Pqpaem5-MzR>RX#xw0;uhZQxlrZ#&Hwt>AnGuFRS=<Zm&)n;@#=C_%So*>J#X-<QEKqoBaNo(_zVMD4haztMo!H^Pvp^1daSQU-k?$$SDm_q}dpLVQ+k5C_4q*Qaxw-b?53|^YrAzJVI}qDHJ9{->$j`S93-Zkm#fu-$;s8?yu9jsl818g>g*S1Yi@PBj;}+(w!U~`(&JICi;RoSdxDakHZev`R;mi;a<Ya_PvxqhfZD70$95^O@p;Ujo$~e`-AEx#<KlkR<?FDF!=P-39xCIAE7`c6ck@D>eTiHY07L+Rx*c=|1pnArIBKR3~ejXO)=3%(N)AxskA6&h96^04a!ZaGDAZ%gq3?i47w!7ci0+_{!Dz>uul*4c(V{=eUjAzozOph6SIu&Cqb}FZ$7H=2=RXy6x0CrU3z=W><-V|$^vm<PQNvJ0w@wqpnYHb>K-`0=k3jzlu8%nEFga#8bhxo;WQbimFNfzxEr|~M1MWvfGes@hV08C@AQno@f++Z=OK%Mkk+CI_)-##o#H5mLq-OkalXxHf18%1ps1mU!mqSCRHEK(zj+LzeHmE1}xGNd&zHc@LuUMZ-v$?!vVvoL=pzcm$##8p^C`xLRz1yAC1_(iuMJ;)nSD@Gs1qdu-k$ZCNQbonT@-OU5TK8egd+W45cec^{U3qM>f{IGueLiTXWIf#a<Aw?7&=^uE?_`$WVo`e?KNb;${9~s+0M4EH!o!mn9^j(e0+qW%ZeX+~wcAnF%zS&EB$0vm_3jir52>gKsQAd+PpmyE#HVEuxLpd@Tas$^RlM8ecltve6)<cH&lvPN&t%tAzJDcs9cd9Z`T9H`}QpBZLPQe*k)>t4UX0=9ia`(_C+qmT3zMZOEi<>~N1$*fh&?%RZ1lH6`44u_|S{u7RhC*AU9_l$UQeEga<do40=y!1|ATt1#dx`t`gK((Me;CVLn09hp3XJ+#1Smr1iZmz51c4yq97)pkQNi`tbnn!Yn05oJxCyZ(-JAkMTj^`^Mo~7H*b_2xenf(y&)D<-J95bSl5-*$8oU>zzdMj^9eR4yf2QAs`JoML?ZcMxWZTADO)jNFr`iB=<s?$_Gqx&AlgY>9Y5VJ<{cFMSde977ae9R;(MUTx3cGC^qBi9sxluTts+};;Uu<}*81Pa_Pm+VnWY56vOCey!YNW<N6h7k6TA!hF5K8`DAAd#39msVE9vqP{=7?kbN{87mF(AhAqeU$93ZuIItz$-4ky$I0*wDdq!4kZvp_q}{z1>ci5lCz;?**N4<}KTzpKUpftLRkhAp{;k8CouZ#Z;+4I)Rl<bI1UqMB9n8(<gkeyrk$?@ID+h?;B0gejpgB6sKnYX1P>rTq~Cv!XZ!->ow?C1}3u;j*Axf@>)R%3&HkasN5=2SaMa;oz@XY>&Y0g!%Y31kPvDCPEZM+65NDRnP{LUeYvxSK>Hs7S9~p#d|(`)&&Y{R#6(WS1~{T508d5wHkJ5-GreF%%DX@k%U9elg#`bIQVUS(A2U$b)0}x`Wy~D5;Xtoa4DB&yV~#eOM|gW^A46`JK*Zb$@sH&Oj+tg~RhxB&3@8}sMI|+zYLNlBZ(KyAriPN8!AI&uDeRMKXsDLjX}X05;-N2X8Kn4<0f0s;v9OiF_Omv72fQ6=Y+0haVa@0-4_7nYAb}=)&1Q2LUno0mu$mFZD-l5Fp_WlbM2r&F87WdWGjwB8&!PjKP(6zrRoJ{?BS}gUC(1^Ik`7F9^$mdc3wy}SM7&HqrQ4xS<VRY5q2Kh`IEhOtEq|g3T3OS0=)mR3+GKC;7T{YQ8V^UAO*cQ6r~lw&=jX-GEssofi^@7bMpV1#cTNWLMqBgw`70j$v!Ln=JG*%Gy)Zz^+NeC2Tkz&{swx}F#VZb=aKB|vAc|4$8lCsIe+m>)R#zLw4Jz~n^Rx}nZG3gvwJg%stYojF<Azo@8Ie-e%E=H#x!z=tTQAPjVTg*bv7OHmDiGYE44n*ptd>(QKI096TTfn~0z%F-I;^m1K*?5a+^3$i#bOGWOm8d#5~HaIN~TvAdu<Lw-)t8f(FcLHhSn`@ds@rZBbS@>fK9(6W5ZY@Y1&;aU6i0A&SWB;<0;BqcA5j=kwh63_=~&^uvoz%bEqVJ{JnyfT@K#7ay58s?p#63F3TKVGDa-%e~h3^|548xzK=ENk2&%k&Q4_F)6!4<tBNuSeTbBfGMvq)BJ%RLPCaY_ZXU@*VU&R^iis?1YDw{AGSVX@ST=LoiR=$XE;fTb6fsU0VH6aale2E>ftFS3JZ;i{qIhOk7V^Q>=3DcBgm`8cF2RsE#GzLz(-T)+6cCi@I|c)lGDxk03@(%sNjO~yzXPQIWZNJ2?*Ls8C!OeR5d%I5LlF%8ka0P#w%@3}TV8XsLUm>jLLi7QYN&9Y*a~E$utRBg5xoW~LDU3L6YfKb*p5dm3!QtsXEjXHr^S7WJ<f^0X0J{r<@96=&Y6~uiLcT|<E9i5Atm@rnT^#z3l0Wmj7OP%dGd)JHhwK&Dupbg?rp%s6DykV6Me>T_eFPiJRXb+v$N=Z#3eB_((EuC^hdMqB{X5gc<7p6H1hQ_))NpFBR23d3y}R|I2=ix$bnm$jCbK4`nmLtz6RGE(1GqH9;N4&nz)(pB^V5(5B2QbUQ3MX66LEp)ll{Q84;(qH0aFSVoNA!6E42_nYp={`M2D3yF6F|4(A1kWVhe)Cd00G(GxRaJqmz980-mCe&lcg!4xx>^WN2bKI>h)c;_DQnhDn%lzbPHcGPHU5R5T4%{X%9xspBCAw+3CH&IBzSZzZlDA7}Cy&~jZ-yy^kCOZ%<U79<by>fcU0s_Gz?Y001ec|M?iNA8?_>WpT@i$tz)L9O5Y?(=<7R3d>GXJ)>kk7;a=i&dC;r~~rxeL)I7_BTSuZ2aDU5vV6ILORjyeQToyNLX3r#A`DGphWYcLhdsW%_7@9c#*X((AFe^m;fXx~t!l4%1>=Z#Ek1BxHd$4@JZ(%2wpX@Q=tWiYV&8kv{w56oz)u*{H2(l*N?LIfx5~x&EC$*|cL$;wf`}P2fyU5+qpFEd&>tSRvYmNfi#*%E=)#vpz!o6np3+^KMScy`5p|1r^(;0!<^ev_h&i{*h)xl|1{!;$?CFtd$!WO&_nI->dYpk4!7cM~X%<Qgv#21DOd}Vh12_($>I!a$<<i9Gcp>Xyp+eV~OOM9f)$|;^P4x;zM^T2FH8Cz~F#d5>E9+E3U_qNuz7I%cL&Mq<BGDVAvcQUZ_5K1#APNZziG^42o*-?u|0wZM>c@xwCNl!DZZ@#j1EAQ{-9E20N3vvu>K_?xxE;y}PyPj&=i`^)teBlB2?@Sz2PId;=I31@!ZAZnoju$)#9DCIAP+&RrlB&K&3Zy;=nZhUp+zDmQL63RcRAv)i&jnOLc>h`ReCMth+ZZydb+Ar5d<v=3Gt>k$*H2Zz>~W<)EZ)?~EH#f{fNYp>I@yvG!uV@&gzO<$AvJ&;@$gTyr#<49zW{lZz|jF=<#<umdqo-GZEJ6+|am?y;|$C{>l=?h2ZUZERH_^>21Y5Y2ofk@@t!Rt(F@hFaK8V!@m-ThAr!2OX@OWZ7%maXCSkF><;Q`QDQ>#HMT^(<9aS1XNTgcR@u8mV5jIwF564#h7uqTP*EH`1({S_6^rk>f4>A~wa?yb@ETaPoM2d`9^Ck(2@C;s`Ei`GBqb?QVaYL#x2iDH;>@P>tnFKHY9dmWBvL+SUjMQWpvXdV^pX?2U>jxNV~sPTS&HHsGU$!Ui|{M$yF>GOoBe6$t!3rYQ!Xb!1#l;8$WoL9=BsNGlo>--<G!#?>n#N5)`N)!VW;U)h2%iW^j(TzO{SnR8Y#K$jMrkx3eY%EzPy`b!a=k;k#<&=rH5HgYIZ=|iCmQW@b5pdj8DOIzYF;_E5b81(_$HxQSRJq#MPO^Ye*Y-p=^w2941F~uNhA?+MCXQN$GcLOV=Z-_KF7T>)A*^?V!%@$pvfbGIfMLz!rsf^e_BblsOp+{Z%B&UcXM17i6jYAxI{F^+r%tMM;LrWVSJDtgB-Xa)>Ek`u_q^pj^4!=>HQl(%c64hG#y^@<9Y3GopN;kpBq&7kfr4UlV5xGXbVz_02=66E*%Ase{6s0MNF=)g^vf1kPF;9)QuIAcMNrqcQ%Lublc7j*C_F4F65mdjKn|G(L!|y0m9**^;0P!O!KsJ^^E`mOK%HdWoQs0{{C43vXTZ<0ocDwe0Us*051mcU|LF1x;r4q=>a_vK7mZ+N4YQq%Nn|d~WHZ`#WiGcqS0h=~Obr?(erS)r95eso4$4&BOI#Z^ea3scejDVGeAzr;=^A<M8`VYoh(uzM38KaK`7EQ@}7|&q?SA?;o$;*GWv{t!}K2$}WZqyu4#D=k`+TRO$oi?RJ*piMYg|m@c20}q=0V0cjA>il{8LAC4cd}DOJ@G+r-r3R*go#m%_$2DwQ({jYQuG6c=m)TYu@*6Z+5WvAcILD2C^PhIVc#gO4MqP^@pss>8*bx8{Byd-DqYm3EZNZZWvHZ^`kg-m^V=B<3O{eTafV?tMD=trzbNd{QCeQi=TfYFM2NcK*mLh~Pd759Fm>I2^A5=obENYY!vd~chN&w|jl)Cr&uGL-WwVQNMwe)B`Q`Km&_FtFeWWeMJM)FpW9B5h<b_Y5L^+WmAA-F|cR51`5?#Z2$e!HM=|N6x|LL1I-E97p(HcXG86i#`X?hG<f^!swkr9@UE-V-q#)6ntPf=tHN}4JNT^q?njms#57BX1MLM))K1g&Q~h_M)x;kdaAD^9EB(`scTezl^4Nl{RpqRwcR@4i_^c(QIbRVc)y>qH5$5A52BK5WZ)hOv@+C`H_4Os4Aw-p=Q<$q@B;I61{v3O*q&1+Op}k}w(SlXN1KPfTmivi|+;pBAP3YMQuJ+_F%dhq8?(7Z}P;V&KWJ>t-!9I0p%QMjlD8(DL#L`!E@add&p1;zZgIWqc!GJe&5hY@KJ(6RURCg-?sMtf1Olkad=2IYaY&$9OQ}to}>Ay5?xK-D}hvQ^I9ozWp`lV5T#fG|Wo58I#;88f=`37F0^(I8vCpV`qr*#0OX$+=TZL+Fr#1Vj2XL3~_13w%jbt=eM-bEtXJ#Y!oK4$Id1L(WNWQ{yxU<iV&EnKLdLC_60dO_N_^B6VFfn3gcc1(~jw?G`T%8W<GIJu?V-tVz4QRTAKraqM|m9mD|#*Em}89F4ozIBecXnE%b)-W8xFZ(tpm!5H=`MF4u0M9QoQs3e0gDJw7F7c|2S>ORc22;Ug;#$ksBP1JF5`GCB%xz9_{fuSB{iXkD5{a&m1>_AERqjto1`XKF0UDCZM|46;?%>+2EbWn@%3Z<chIxQ&$KvOjb!**yErO%`cw6^kyA;;vO<bo^W9hfTZ6M=@g`VaH-h?Hnz~?F*L93U=<LgJSC3dB`5p{xBn-I7_E1ok=ob@*FgOgkih6Is!MFF%s8K>1+~Gw!|_D-0c?5H#H-<`uCZijn<)&MW@<^<4I0v9--S8=nOIZDqa@%PMM$ym~sxZxgRES-gFWHx8FDg9Zf?hvAa@OzA<<SK?jCydohKLJ}bsME4KgV7+?Q7xp5TAz;cUXoT2$VIr2Nl{{+kXo9SE3F1Z+HFJs<L?QE=N4!4R_N`I6?b54w3Yt$jhW3m|xy@@-OO>UzW>xqJ^Q82agYPC`Jc|9?&M2sl-M;Q@1mb6M4jf<;JCO;SQw#O6}=V>x~?-|I2{;;!ytb{)r^nFYqhq<mZ$#~$Kny^ZfuZ8!=!@!q~j9hPXi<U~rhKBua=)#Ta?sayCM92Gk!FbrYud4W%zS6a)O}>u_lch>HY0_h%J6xQFYC6o%<*0e9c*`IJ3N~DDe>1<OT?VJ&1w{MN)(E{GE2h3?I+JYgIgKOID(L=Uh@K1V5%^l1P*%=GH=|JCBW`&j)Na4#D3z~7d^!m&nEDwmotKDva_5E#Aq(S07==4~%F7-1Qy6D)nNEuzQFGh5Zw)yuI^0Z1JLyP?=Jlw-h*kJ4-VWktimZTqrEye7119j{-)Ys=R4i?%Z`hD}nARGvL~K49fvu$-=c3u{R=OPnhqv>gr3{H9y>74_b}c?VcwVRkotYSccR;lct+7?-Sg5KQEPDoni5NO+{4fQ@(AUlUl$l`OCyG6bvAM}VfFbAJZa5S$#R=?eFciN}>OWbb+eDYSChifB)vuHU`s88pfCA4^cUkzxogL7hQ)mT3G$vmamoRik#XvH)G&v<n7V>iR5s0~E5DKxlR3z_`^qBiHi7t*xx?RkOhmoBnmuX@dM(Rs9%c~^<5(~3sce|2KRWenr)pvIkc10d{83Wb5eZjrdR6;F$^xz-(gO2ML+$ha}-A#<>uBh3IF|<DVS+__D5m(Af4H%;+c<5ZORaYIu<>b$@w=X0XnINM8tz0U)&U-h@wK9-J^fGza5NNn)+jS_rqI1B(6yQT5-jdAo7$t%{Q?yCZk3uD^^&4jtdb>CBgC<6plyY!IXsnmkmP_@t;H`9SRSEztfuk!Hk8}u`i&<W}jC%>E3Z0bdj=wVrhAp_~mEeC1SW!G1h@-NA0;ZUiLZ9atV#KLZ8m^5}Z8;52HH$k+UW`!yodfL84BxsK6@3ak6{ELYtBv^)A3GRk*NbIXgj@6(A{mjG6qaWFgG92?lV4j!YvBlE)$0ok>u9bVc6~b6BhZ~y91{1YM_kQ4$3oYh&JRiBJF0CLH{)bZF}{O2`f$pI`+(``?^%)gU&j{p%k%PdXTa0n=O9}4k>>RW?BrSITKNBXFUfm+&S#=$WO{6F9z-Nf!JuzEMY|PrI?mH5Df*zEjF2cyvlVm#4rRk0x&DpDA<{w|9NdNkJx3Aig}@-(ncX&e5{zp><6%7~iZZTlt+Z<$Ii2igVF43^#VK1I2dbF+uDPHh;D>tlQW$WxQX{(LlXPjk^O%}n$^<%;N+jbh=PzSy&-_)_LPctYMSzy5dz&aUCBC!EB0pD)@GOK}80fjOcB5RUoO7oK!;zt%;mG_pZj16%vQh`pun!j<8HFBj_m@(nC|%=7Ao!T7eMV&COSJQe5+uQ+tH+nt8rGJT0}d<ddH~CGU<mPG6aacbx4(mAfwI%NeE}d1kM^Q-VCw4+9bqtHm`!$+=)_T$u>hEkrrh_0(eY>zUDq&5CQuBnXN+dEFgFhea1pkYGi;3PM%`NM9x=J^?df@SVeYLr-^vH)^YlFaYtCs&qTWaBAo+>}y01tu{m_W>DmiToo4PPRyD)E?6F?2&*>Bham#%n%BjTZ7i{Ly06(0!%EAxs!5~lG_m2<2}r1`0o?aB08x>kbYnh1L+^x|Zx8|LUb`6Zj&lSS*NL}<J~M>K`kJx2vIcqu#v_>&OG%Rnh+{}eQZrt?@D^8b?UeU)eAcagx2oDxVP@iiMC(xfoFGo3B6|2_E?(nIT%Z3Xt^Z^^BW1MWU@I9sTUx-0C9h2#Q;ZKf!bV=OoqFD7(1k+MySylIcgC7FF?*(CBQi@&}VF$|=0Cz&8Wq_~bE4|Wul5trd5&QS&9IJ$Tdrt;tAN^v6iT_9#h&hfth*!^C{kp75beSEB}rj@%cg3WV>g>u*zsSO{Q`P+mAoJh1PB8%W?HNpN9(y8FrD5<@X_)S##BYW^R^Z8~g$j|*D9=tzL#|}@@!aSWm^w(^T5zsG*gK|lzcai=<0qMxYNZNO1;Pt54=}+JpDU&_C7?olba3=7=xpdo2Ber%j490I^8~^tBe~+LFBdmzwdN{i12zn@BWoz3rQB^!yxVdhX*BaH@2bgtLy!)VBs$r-t&gLp!Hh`%vC*D?Tl^d0{(n{S(uonx=BrXo0BkCqqxc7?&y|oA=mJ~fmYRhvifljbo(aq1fQlRRtZmd<7D(fW-IlfxGQ7f(A{J>4CH*c#~UaGEDS3mGqO4s1R@yp8_96a1_l-^Hm>9T8QcH<)B5GZvP8TVUKYCH_?`FH?E(P--HRkg5w74FmW%1WK+jMDOQrBS(6cF)vZf4Bpr9lOk;G`2&;W2~dn$_j0G9l8`de-qf!)4J1}sFiP2>W$h5I9B{!g>k2@)s?9Go0S_k%laDMTCb+oTZrnxzAkT|W0BaT>P7>|f^t2z?p51P*Vmy__<L=6wdyx&rL{Umj<?DJ-Ho+(*Q)QWxv4NNh)RsU;g6aD5ZT_qBZ*ZEG%$pb*eJ9&!K^y=6i|{}#YPHPtA{z}GtEe*lyNE82NFz}D2(@6yxJKCt%>%AR2WhcHOhLHh@nD>0A0z|!g%#-RSFI6zgzwwAp=7Gp-3_?sFW@RI}1ljCW|H`ix!K`RPRNc)yUvr+pepmJ*9dhbHD>UL>bSmQf&#qI_vOK0ndf?Jt-9`6t3ZMYdF1h`rRgTb8ghC8|%)s51cutT3asH=qHRrFw#nrgvLneEbTPjlOT$O-JHOAKdUE`>cJC%5zo~{Kw;i(-@{ZLyiCwceW|pz2DCZiZP%KnC4;Nq+Rjxc<7!)EfR`Smp7>U&s#ml<;r(;5P+FpKoslB=T1FI7xwb)wfRN9vQe_2`YMqCvG8XTW>$sOFOJ8_DoAZ4XMf<+a48hojp)(qEx?P7uKGE9YiaL0T6FE@yO@mcsd<jeIZ>&Y*wIK=0xakp38&cjv3~Rs~k2e>`5kaSAUWti=N+QSVa`}y-OFK%E)F9$<58zrX6exJm<wD`geEZ6qt;@~!{M^FZZ@)bcU-I(XTW>Gq7h3T8a(K0UC1~-tL(9I#DTZi=HQ?>Eba9D!iDylbNrLT9$!&pY5QAZJSI`e`AxTh36sTk=oaaKCzD^Uc*!vvp?vE(*G9CE-9k;+S8{BrkkJi!DhW1)`6hv?A&C+S4`=N*6mlwwyU-)JYdvRPb7-nS!!vLn_{ovKJNq*ce(3o9+2i;-|F114Eis0oi#ic13tza6(ilL=Dlt2J@)h*`$cK@(IYT!!=C9Y&RU9aQm-2ODFAI~l(|3s~{@Z1<tU)z1SIw%GvodP1cTZ>K)20U~|K&tn~-F;_trvu>HiignXn<;oni^bjA6k??F;z~9x*J32oD^6Ey#D_8IaShRSB5qmos-`+`6r&!6_KQy`%hqW-C>8Pn6-SBd8%4$M9B7rjgvLvX1^sB|Q}uzHTggEhZE#$7smOanhE$9SWYeY~Wfh!2TAzZz1a87^L3lPg<Nl<%TL6?eLR8=cjy&18E|@^6KiL7eVU3qZ==0t#D&I-MiKvtlwB{_)n9oYC&=>?roIy7w-p<)64pz<0Wd(^LVD%J@6&df)Al?vQ2GH<MK6Sxk0!JaPIbvH%88J693VW$bl;V4_KlJv80=pG$bTYqUVp-m863futs#p|ujm&JKSew28(SSGZvW0@)A2Pg6_A<jwrkJ+qavu^|Gz0+TGm)|c(%OV2OOY(<YnE9FX_ORzXr^aRWUh^m1bT@NF2Q93K#cJsg+axA`S5-#Cht3Iga0pH%q}7)KdITsYIY`35mI)<?r{E_xVLFblHJ8F@(6EfNmTJtVcy0iOHCk`X$CRFxMbtWyPmuO5?4KA>7w4bcI_*RLYh4+w`p~q>^4RV;P3$eY!rbyJ}{=vPz3_rACfVb#vp77z*jjTlD(4mk*Ys4cUl9}D2zno%1jNo&rURc0T9N^Y>Ylp@(y6`D}n(4K4KMUASH!}6D=5ik~b-F`a3_of*uMxO$;s9CH%Z7V!96c9DrILs)Np<-*5W|8c~uGB1WUTi#pv+xqIV>(DZGU4yFo9%+Fb?6R*TcuCl~^A%yoQpGH<fvk6K|hK3Y@cy!^R84kJzr*j4`Mj)n|jy$8lp+olt2kEDML4%<QFWi!?u-Od&@?qLRcYlnPpb3w!!(QKM^(Sy^&XBkkBWRoa51e^#XE>lsRad<xmM(|Mod(<cnFQX5FH}-E86iIn)p%~t7wRZFXfO&?tqRzt6D|&-zrF7Zv`q{)l*}18N+h{X0zZ@vhXY^^Wr(`cO2e;}*U>*<tw92=m=KS=iAdT;?nE$7id;zIKJ|vP9arC5<CBqO1n@jYuFPuC3N?uVA6gZ-n8z5_PvZI>()e&lTyOdYrD=8(4HIr@V(Kr<En;mox(BmH%JvvPkZ5`g<ac;V+ZtTpm<2>^8Hb=Mkd{+luR79P7%i0glkuq20!nZ<bQb0n5z26sb82C;KZKLeY+?+<?M}BdhSL&a9FcL@-Oo)+qX>wym|~`!KxI%aof+5U?*}e2A#MT6=Q!uMxCZe>><*L{7upkayZh)Y*PL_%%Fq%S>qEtq)U}%J0-$ujQHl?iOe{tb2VjY$IR0w6akIKy1*+}?ItCw<8-8UO%2cjbfPy>RO3wmm6*Z-=fNNb^H<p}PlP@XpQ9@Xxk17FyD7x3|#BBhmMmH6<WyW46dI8u9u4mpB<mn*U-qxb);h`RH#>gj1a+qfk7UGZOxlJGs3Dv|U2F+oAG;&D2<z!%y98Uf2XaZ*7a(Qr;<I6~41`oYMDBMuX>&kdditxQP@^jn4_FHq|<#TD|uV6G|jtwHT$49KZreJj(>f>po;*7<v!LW9kH`ehhO)dX7WF245&&AZmO4&;HFr^fs5X_{amZ@~Urpp~!i02$q&@i#r946jhslQLdjgJd}aYom6=NzIM6chJ|V`;D=SRVrGX=OrV4sRAuF38IJnDUw9?|&3qE_PZImH?as;#;;a;81~4{)mKn7#D*wuf&w#^zBH5@s`OGX+%cAuH&OWvUYj??fI+Cx2|6KL$u2V(dll7a5jOM4)+2ekeV^$n3!FimTCF;uYnnT!frGcI6Z!ba5r1d&(9wJ7WMwKtp1NB|M(X&)ezB{WP*=xg+~9mZhRr4cx$!^u}CFMSZd&=*#c1ICmdUcd>=VGiGVY?p_t%W*-B`aB}}AfI0+c88|1BF>QOPLDDe|`^s0s^fEB4X&eNWzGF6SnUYhM!)|QLHc--n_g4$trGD+3PPD##&dcSV4x7`YyppYkm5$+Kpt42g33f>C9Y&#s1nw%-dx!5pQOZ7&%CUl6J--?*RnOaV04TbXe)b-^o2Yesnq9Uaw-#Pk1=wL8ErCH&d<L0jv3Q&?aY94PZijFkym<D0DV4U-Fb7yRjO8ayE95xFNGjk3GS>%9?EEh~Re)*h@Uyd3Vs@l-fHfp+0fne460xkvgU_cqU?D6tki8@Mj0TkHE@|in<e@GJR6YCgs$NSvFf+JtV)_-+lZA#<ONud>Ag<WiR1^^i?z@0GW!LFepSE~B6jQ47x5UKW2S2_%OBaYaQ4V#CV>;cZ$>M|}h{NwZa$ma`%(E!$*#7J$RjC)NIeC>o{3Xb+VB?}Hy5J*?elDcLtFRkzxP0LX5Y_dm4^r3b`Qs`m0juA5}10W7Z%-3m@xha;;f$}yz7bFN72nr)E3OoN8%6vngU&dlKs-r6<l$pR|8NJsG23EHxQYLI_MhrrxFRc@%*(N8|pl&n*BK-}U3E>T-U!%Wc3#n+TcsZGE6i)FRo^nmMMGOqf&In?N(q;V#ya3Lnj$o(G)#|%tM`VlU5q}!FcdP6LWDKW!aEfdzuQ1L>EOLP{$1f~S{i6UA9%}wYkrOfCAatAYX;_@>Rui?+TUUreEXGL0Ys9^>k`7ol?=f<*l8TrEYUt2&h$7VOLSQj_rg}z{wJT{3c&dk*fru=x%tU(5a=E@_nN91w{xvNfTw}I$@J$Lj!~46zWF%4}YX$_oy61y(ZKJjPhJ_T*cVrh^<nN*7=XrYkY&z5nJByYLTLS;+BZ-j2K=9VvjN=7Qdx2?~;Y9#@W*ly85-wy*lY)G(HLC~%Uqt<hNJ$cJ8*eZdZ=%b(wabWPJJV#M7b7_QD1Kz0^{5#>JcfuOIS?Y|?1O<X-+Ls9HAYe9a_T}~wsmx_b;UzZBCs)LmFYQ~85f$vvH|d7ArOq70HCaAe7Za6`)W(M5R7SR@?@hv(zl*i%vjSXijp+1Duc^<_SSK9kzJn}9<63}tDxjla_ZAWPSLORYTYj_uU6{yN)^d8%-=FV@!!zR{8=ZP`Ln`*eQ^8sd-#Km**D=5{ms_#%Ll@?oxD&U=v@8-W#^9(YL1dG9-n+>)JC!dJGg!O9`^O%KWB9!X{~(Ylo3+rUrO8b2f{_>5%=?)Z+c8wc=FK$a}LL^pv#L)5xgY2C|-i&h|?wc6*&)o@<4>Kc_l*}<5Ij-Q8C55h}C|O3}pcQU2@3Z;}<6nAJBzMB*V!^#+-3i=#(U<%L$n&-|zDW+I=jRRm;rGuM%t7s5XesO*Cxet$o1Z-cP=KK(YsM(?6i)n9JuM5Ym4|85V@}hGq-I!Y|NCon%DZ5IcVyfA@goSx-oW^U(tl9!w=MeL!IF(e2wghnDp#T+|oGud=E1^1mgcM<A(X1aK1Wq#Px{@TL&4$u*IPl8H$zzfB_OIrK*OjzlRybf<Eerwheu96Jgw#~C7}Umrh<i0&9UO$nJh?fpnLz(*NTD~A)n#GN~s3)Mzi-wnDGV77^P0z}^#^#ZCdBmzi7(4##W)g)FEJJo8^qpWF<C<ud#wIPSAwWD4oXn6w<<yh^Pp*e9(GbHT|OfaPkLAd$d<^@Z_3DuszR9aiEESDN_Vu<29$pd-LUbzopu|B<%Q7B4R8#l|f*k@dS41$DxO;F5Fp&#H3i|&RlN$Ora;6Lm}g7`rL)(55(Rvx?0PCj<MNj4e!<*!pj#*f%@=!I}CI^vW7%8QL`VZK#9>w+VC!a+KiP^5SquQ`1-r3a3V@go^_{t-!bUtxfT*f~X99%C=?T7;D=uK>A@%BJhp8Zs3A(#`VHyDod`cf&X2+S0a(B^7MNPc}+5ByrcuNjHLIUD^h&Ul4t7tSJY`jkQXOBf)_yWxv9DhV<vrOSXof7E{Oq(s2PTA2i2qfq=|&X(Zy+#B=2tlds_U&`v}PP&B=Fka`j}S(MdNP*CL<*%XP%B04D}d{=H5RFEKaVpI>efF~nC$t3ff`5u{VaWYnnPmKC1Yh+E}o|0&E$WnzERhjh??}>_4?_<=xEg)#LhLPa`FP^kN^5WeV(nh1%^$#~)q?}K;d}pH`B8g?YaZ(vA9f(9u7NFN>?IpGvg_K2t-e?sIX^VTd-ja+&vO)PJE7&<TBz(YpdWMg)+A3dYAB&nJyF7pirX;4<jrKcxHuAc<>`Ugkcwl(zM;3ReWk5F`(k1&q3Upwl*oZ_V2=2Erlh9rCGxT@DUO2>9U~(G`n-I4Z0*~M0{Y`1AIT;Qy-0$5!H-M7)@uanHHYA)*A{{e~(jPT<;o?CD@^POl`Gkbbs*$`L1yb$Ka1SjDgpB~EKMH!-e-uJ3QA%$ObrNW#fI=ApB1@wTaxTZ!CbI;&kCx$a(yl)2F4_IXJ^E2DQ>(1gbzLj}WTR3m*D;yZYN_@CIpg`a%C*Y%3VW@YO=DY~5$g8*;BL_AQeOLD)NgYZ$*aET<5p!20}A-nwUrN2)(&8E2;eL&<j`hesk(N(Qd^A$8jzqgKs%U%O369yH+Oq|pxI_cM7GiFULTgJk158PRn;3A3_E+sWh#3KSPx-k9Sey!drIiLs8#LSQ3ihXo8@xDym68SGE<eIMp^=P1Ak29iJ>H?Gc{GJdS(-xf#sgL-^TbMXrj{P&{tmAgT{am0FVmy;Fm3c4&AN0VMtQy`|TY4Cj382i)s|yQ~AZwE~X1TMYW3bQv@1k9-&8ZkmeW8G#;Yc(fhRMIu9HnKSetIZg#T(TppWD-{t(G_i}LNdy~DvKCl4jZxIZ_%*<T&ObA_uC={&0T_&BmE6G=2RA!7mVs9`8LE3$bYXH|RErl?V6)v^($D>3|%7j&mMw+$x!DtwJGcqN@bP6HLR}qrVX{vNei-f{r@k!Hs=vG%|7IH=y5{Z}uVyTsHRm$)A>t(p7)^42QJEshvFx_Lq<b<U9@+?X5KW4||c<hUMdc{XpSG&_XSAKYMQK%<ngvYjZuHAGogxg51r3l0xyR2gnGxf$M{Ts&65P=J%iI1+kasK`PGL2wPzWjI0JDIfMd8)-lcpjP8AaCQb_3mqnna#-;q;HAx+9z#iFouF*k&K5*gD)n4;xLv7n%-yklhqlfJmR~b-0XsgS}nr5AF=g~t~}CG+Gx=<Ofw%+E^#Ca!;I?Wa~Q;XrP`W~qb$kGP8+v?TDwtIq<TO;^7F+6R|y-eE?>1fDsr_gTInOZY^qjzkN6y-70BH+!C(biB%)N)mPCkvMEZ8CFZCd%D_lfm7k6?SSTJ95+*Vf-(?MA{Za|w!x3#m;`U)W=Lse0;7Ee3R7VIjH%+G|IzxTY62OZi>MpLns+{Y+hmVvF3qc|gr0jMh;IRquB8bL(i|3rv;J#BB~N04HlNVWu#whxtdi_+uC@Cb_%KA?y^)?q?C$~xwRY2}m1NFf)`%nKAD@*uSR((A2|@K~InIdk!VgcSwUahBHCYt>ujW&c(+k{tSLrPXrXQ^E?fBt^wAiykI(PK=7<$pCg%c1nm>aQr~YBtQ|F0uj|T!HDKCfO_(+@LvjlCg+<FC2Zwd!~OusscbG75hMa774LgHXi>L(a%SXE5>zgfOWI7d-2?6%UE&N$9VW+?*!D=|+LPE)3$}|A^2f&37F4uV=rJ|?uC%FB1Gj>{9I4N{S6WE~^;5!_VKXU->3Ob#EWpZ|?yo@66XIYuY{tbG33;AmMJF7F0jhAQ4icX!h|4BLTs?2-O|b0u0FUIj@IQ+h)f}xD>lTBN-<oi4BA98Gglq)iB55I0sU9&Mq<ygaeMH<VuN~P#v9o~dC3fo!M)7qqVy6lGLZ^kJ6V~yG6@l~dz+r<;K}`}%Aj>DH!GtmV1M&euOKL>ZYzHqYTafdq5$B>~^u2A|TkZKEcsqRSTsH0tXyZ<z`4^l8MUNyF*`0q!N0O*8QeqjZ%1F0KC!A#Z-^qL`dJr(p0{&k_t`#;fGp*cF`$_|Og<bvj`1hU&QRcC=``5x$N(IA{p|0WoK-hVX3}}agq~X4V;k^=i#_t5d{&AGEmSUbgWgTb~@hpeEekZfkGmV?*oQ>e<bA_`vs{}4u=(NWiyiujQJ$}Vn4R*mp(wWIy&8+%0*>5Lxg@0tb^(*;i_;&M3`;Tt7J_veYcdF6)dTFh^LP8od2a*YH1w5tpjXyr}9JXQipwN}sa(=;^ve&>km^|G7$$EIo1|BCbj;5apED{a!DOtR;Cp@dvk0`&ZI1ifXJKk`T5k{UZ{jB5+u`r;Nq83CRmt<2#bYfAk5ZU7HCtfl}KRq;#E)O=_vwTUI^|16*$Ms_5{3kaIv5E}T4wRQ=<OTDky=PG}`Ncgm6OkCUCc73=U|!!sGGf+YVuqWsQEwG{{ox*sGm}!n(uQR|wuWrQY$J*fd#i{U%;|R~y-;dPYGGhZ6SQN_q<yzEf!Y_0MO;Lx#$Al9iKcph66wkdIo;_?XSOEw7p#9sZ9vG+jk<;?PB^_MYu@R>MJ%LCv|l|LHdouVoU!}av{tlfq&Jn^e@lB<K`JYtr9qAp0%SHbe<7OD$Nj*3sr8V}0cFz#u7N17NT}PQbdqRTkUa<EZd(i_O4Da1yXTCl*8aE7F@Q}OozGvYE~7VRWvx*wiAaRNsl)yU80{S2muvF9nSMJB>~r05CJ)S2Z$Fcbn4?Jg#IbdM6|;qYnh<Z{g@<8P;2!&i`J~r$O?;(-0*mOY9LN7-Nd=vCm?Tq|<208pGHj4>Mu00~Pm=<Qn^4dsZXG2JmLMvz=CFT{_0pZ};i(%#q0W$z5fDJ^!=?liW8`AoTL1$TU`Qw(Y}h8ekIYGD?EgF*_W2-N+5@=DXMbe>*6WY4vt-k9wqa{0l;U`W&PEg9V}Qc3N$OP@O^(M#*s$G(8FsAXU0BP2Iup6No3Qk#whYEPI{JYK?kh725^?(#ieo7TPb#_nQmVy{MBtj2NPm9{9Q#)&_mmP%_JEbVI|mR}KA?{yStx;rLAz+0a|42Snki>@(-x=lbjl!p_s}U$4uHZhL<Y9iWq`PxhB1q=^S=cjQIVC3XukJwr!loW{#wa8{$X(H_%*yH$3ML{ff|$jO?rGUFfC~7&cLc?=?(*~PWCc;!2lzoc$;#wJUMGG8qii&L_n!7OT-GSe96ComT{h2T`tJj_!KFdVEITon3C*~vE}EIyVGOdKedbHiC-E4=1}oPtb1%+vmOiBM0W37u(OkiiK|fCnn@Cx7-zxFdgOq|FRi5}dv2DLsWM?&#wu2U4MSty{)p)qkEb$iGs??1%Jrh$g3a#>&0gZJk&qDU3NSRFPl5gLMT{cQ<dc`&2ks><lk8D1sOf^RZ4UWLS&8<u1(Xc#<V2@g5%Y8h?;UPh#gIu%T))F){1A+3P0oR|z@vn|W}o#nB$JE=V+rm%Dn_3-_qP@YFdyv|iHY}T<~$sJGOafcCel8LNKM)<U{fNB4fdAx++KLkzdKU)LD9Dv>^sXDf=6TJ0kT#)jkt2V{n0qtCblv|oqnG-2|){sE&F(X^u5RX)1cW5hOKGM&dSFJW-k=LNqbMd0&U$3dZ%{FhdXva3hiz`7-tv=QF1FzLktRw#r@>jvc&39Ii5#jTWG-uWHsteL5)0;g;2P@N?WkC@_Q}|s6!2~BcQRRy!zgKRVfZqwT1~&lAT6SrFSZig5PQv4KzsIq(uFZud`B+vXzERsTFNPv4*rSr`>oQc1{g8`_5d!eyHK2z>$EN?u~bSGNA%&sTIhX$9y*ZQ*N}|X^%5=*(se3`WONeU6Qwfyu1_ZDlwGY4z!nB$T|<4>E)0o%3Xcameen*7VF%~noN%tt@tc0sL-u)4Xx&`)yk!Jft09~>(!N8DS3A^;}#L|g@k#+OT^5NB14)~muE<M^_|F6sse0qn)Z_~MXXC|?HeO7B&kGYe$zB(lt(g-htR`vL^qOSrc6PZ$jFo>LTsH9twz`_u+av~`Bmx<7FALxj<1D-)JsF~WpEz<16pNS>O&IPb6}5he!d{xrM_ab3t{%JyviRYJ!wRD-V|}WDB5Y!B)*2ug?R}%6UEw#e0q|S)gRy(DOoKIkJIkx@>;p{@d!81oXmV~e!*e4Bh80CTVQaJt05KNq{YU#<(0Li269ZCP|q`#@s+YhA%)5}%<#V?)y4~+yx0}?wm=}`?C2-Il)+e@h<rhjrv<Mw`RzX@U?6fhO5B1@6C})j$tj|=^*OVzc&colU?7^eZA+htFfdWj!e=p=xtE&A$G!<TQ4YmB-Kz6$-*BIfassFUcUq*y^g6ws2Q%z>On7?zJLA1RyakGMObQln5RjM=Du`cH4{w%JC<B{QxXKEk8|R=eMbfF;=KWrPiQVnz(9kWD;y0g7Q0B_}eba@T*>lx|kgmt+lB}Pod{_u~3p&|s>hfuN)*6K8=|X5Mkhpqz=*USaML}vQq?q7OrzPbzKZ2)tm-3rUaUEX+98n~yzEyP>X%a>(9%NeZtr=?{-!y~0!DM8XK;Ni;75<#h2_12|s4qFSwMZBRNS_sa!r2O5BXWCeD5mH*KV5V*TQxo`Skw^H@aZ^W;mj+}G^Ny|Idjlrn~4LGL(i$bm*Q7kS929&tA_XYc%K`9g#)H7S;fw-X)CuA_6?aerJv2j#o*X5!XvOVOe$8^Zur%A8>>}{70CAXm|Cv1y1s!pNG4>l4|y-ZBSu~VEeg4^Rh-M^ttlY~eK`KQ9!{8OM7lUrH730%<mthqBtA8vYxvU*&=I~%%(I9JfTxzC1+);M!mn*0Fq^x;$QgMNX)LE4FfNqC-`PbmdhzbG118(24UNvYs14h{C`%ZVw3OAqTRMnohSa<gXHIpUY{LLPXa=B9m6j0fY80Sl7&ZuW*Q=GahW}orakIM7@S}7<1Z-~VI4lsQfH@kMKuaC~o0re!En`X^P5FgjWJ^v>-z#O@<)WQ@Z^t%QvOUCo3c8)2huYfWheUlexR*}6^jqfc8M?P`k1hd#Ixb#|;eEbhsZ7FcIL!i4WTj-CJ&{?$=UQs!Da9OFgqXn%BQp!#NYOCBP?WBqEQH}8`inCWXHbG<g%GDni6~}cmtPP9eU9|Eup0S<$l@1T*hZrGU*Mn$TCZf>EqcX-Ya;cgvJW9rmkj`Ptx&>dBxdLMvq>;SbEjTNu-3)k4}c@tglf2-X|+8*T(-1Q`k}ETg-V7`LhcUPTAA;u)s6+0X&0f$=~pZ}D3d%HE2vTEmji(irWgr;!ZJ>-05q1c0`qCKzzU+o$VG3dTxC+}1+5}!Au4V$A+?Z4<sRHaJ3V1zS>)xt<oaX)3b#-Na0}`vF%Sm^q{gVfBdaecFBkUm7pfcEQVBUUl$A0{*u9vb8O6t$q#-8>7&(Uqwz0{7-kIBCEnC)R)QNR0O9Wr`g3%o`m|UH7$DIl`kZTY7dp^0@!isG|Rlq(SxTiCgV!}L-G<Tw(tjD&o@Y}XRsoNq3BerEOKV@Ig2X3XZ)Zo3L__N<*gH(sM4YC{uAWly^n|J#jl9vhOd}>85M#iCTjCjAVg|LCPy}NzkQ0;nLE8#7TsXl^jajvJk5zU(+kyRc)H!bM1N?dG5tItd}`QZMcG7C5Gxfv7X%LFo3_FB^zUn8lpE8-N7IBQE~*>oC`EA2<bkswj5x$_Jx8lzdf#0{><zer=kwBa-!QPPSLMVKU3ymZ<_hs${+?h|HMVlajHs0!lmp$Vq*MMc?J8xr=?QELW;STG)idqaE&kQE*$6)P&l0GywB1nT)XKF4}R_WZXUR^oB{U2;*3?Xx{ACl(fQ0<q}={sV}#yHzA*LmU-|fy+ZgrQ;pxRJJ@tah2;ZI`JfJhW&*5G^0B30NJ6DC0kn)*)<h}G#H+K?N1Y6tJIzBrIaw9=N<|pPvKS0pOE#$k)wHa@coW<YxnqLLc7jMPq1u~qgPd2<=yS`p$J7#gq>XCkf6bbtGuH9dShex1IZ+2r8XrAFC$hI%GpP{LNVbK<92P^g`hi_v@q<Mxr}G_+R@_Z$soyOr@`sHRnp|+mz0r-Vw95z*zP*0=bIMPsF}8#<kFg-j!p-oQ{yK}5UxvPDX7y!>g5|FQZv%4Y=IW@%kk0!>%`AC#iBHWW3tPUEr5DFk_-tEDUQrz87TE@8nzHTifl||9YFq+Qeczn$ReYn#=Kf$!DdF2?P9|DU^A2Ep;0Uf_njYEj<c<n?xRsG;=kvviZKI<sqt{)kyIns3ZgAxemT=~#Oh|0lPA`anVFNu*c@LhI9YnxQ0I0}uk7Na7Zqd0VKnL)`6+8f_)*pK5JVem$|KAI$@No-q{wfI;$KFC$R5&3v;(2EQ#TFQk6+?<ljPCrKM``l6!3Lqc;DnfKbn@|G_lS<N?GT$@~N@{X=TrjgDPB}&@+*AGx8UCDwIwX+&Cqk8_hWC*tw*y#WRb<fnN)$0;a3f;$#^IUWhh?Cjbj=U#r;oLa1R&$bdm;W>tjjS#_XvE6|8Dh~no9Z1_#2LsU8tEAs8!4SjJrX9CKi$2?fJU`_ALu_nZh#kA*4^I0P0hNGqoLq9HHM&}Ec#!6;H&mQ~>f3lb8Dd{fhBbOM3=!FI|<7||vghy<JDor#irbkk@Vms#D=2X_4%z^6bFl#%Z-OgTTTujlQOo=)i3w*EB-;vW`F}Wck8f{$nA{dR+83zt!1XxMJk@Ve&UdSLw-npW)3WF29uh?Bqh&!7!cF<>>2tlZQv*{0X#Zpu<*J)*^wR4KXNo$Ct;?tUnc{`Y+Fc)WPJ5gE6YKIeq<&yFUzXXuw*VoHSmC{P%15B`h!GfyspngOGYLpJUMMpBmXRLz__V&UVJ)9)PVqhP$uZSOhWYRaJeELh_y&Wk~pD?#g`q~-YA1jxbs_56(C<9}yY>e0s2<36aaL(RB#tAGyeMQ*%wIEKvCR>0<@gNz>fb*FMxWJW;zs@<wzfXx&ZcN-fCOHeE$f5Bl0W47`@q=ZfPwpCX-_+XJvP;%wm`qK|o^16_7!Hbeg@Ds};M^r%#^s+Ra@%zI&O<UB-3`QI&;uuS@LWpcop-maxA;%#Jexa5(l6}WF8UHT1s-%+KU+wO?c_HluZxT8;Jo{5Krm5EOy9kvE!r7{bp~clOu;?_{n|!XUfm`*#obW2>yd5G*P4H)d7zjzi|aGcFVRf>`c$U2(Y#fUAI-*6-+VbMEn;$p6lUsGVuS>@swTxBIJ<*zNaCers669xIsL*w?p|IM@%urM^ydmSE`ZCJZT!#~A5(N&8a(_7Ra}<QoTi<fejUXc(>E|aE$!P>G-Gzq?f36Y22!iIX#o*^DNm7q+9Cs3bQwkNR5PnTcduJk{f87sx+DU{q%vuy^4iqJVD`e{PRM!^4-VA#8WVs%-F`(FKyM#I9`=)Q)POGT>`eNTk?46-4uHq}Msd!LYQ;%V)oplIeN5F?xqR>%OE=G=1u>Q=HqastG-g8}*L`am7{}n27oOYx-Yb8uz(J0)+rL+IyI~t})|vJ!b_`==p*bIg1E;rPAqfSBWSVfjR$VXGrc#3va}&dEqne(Ipb3UU*lY4TmmH}`(EbamrS7MgEt2F}?GE?}B2{vx1_^4Uhoi^}zcUo*Or2q2Lj`WXg!zK~D-JgegJvT5<kz_r2?9k3n3;<-^3uwalNvqZjE7%~Yd8ui@hQcu{+R4|If0==ROUy(Tsz1ANHP=5vbQid?;QWr@o(7p;5$JIe9LCM7WzPt)-O(e1Ko*O0^bTdV)pdZgS}&@V@&geFma<+TGb<t7Sk@yM5Xmo<0ge-iONul2-d*HZ)+{9nQ<9MRmc@0p5QumGgE0wN3;(A7^aLOOG+H$_}jrKEY_;k#;n^4hd|M^+(mwy+q={1U}76&j7M}0dla1r1D#t;J28SIAhjY@o^^33oGhJQW2^k@yzSaoR_g`xy(*Td6^@#4?;^6w3G2?-LH0Fq2_X(4GNE9Xgh<XVJMl@_A@<ZJ@o+CGp$hh!xzAFS`^erzdz-!FX5*tA4V##nPcu?=FHsXXPPNflOI0|4|3I-4pD?dR#4Mo)jIhDx(NVxdmfa`fXdJ&@Wb}Zi3WIQPD6!9(d;M=xAg9a!mBFyGl@`{L@w1*_Y0;CG?61E=pFuI=|MhnwA%&(|{$GDb=lv<QVliCOGO9iyzWOU^lgS<dPY`)MMggFTeS(-KN_B~kx%f`dSuIAk%BV_O0%AHtsdP(q2c7tClxtc19ltHT7~AW`GMZWL(*8EGZyX=obIf?)`m-othWNI$w^ZF&Yv}p1)G#qMG1DTJYOCetN@*>lW*d{hU@ObjjcY6AEKPhbY;^+k6BIF?Ir@FtXq3ITm@MVK&<EI}IB4)i=IHA*`LI};yk1(Vm$4G@-6Od7TOH!gqDa(k;(U#R;TQwWj>BQG?Pd=2vwR3MT3ep7(UrP`37FRq3CF_R*+TOuG!^XZ0J()BqqAo;=hrXOSQx>`!9}^2C~JO8E{Wr#>9+YPt_Qw7Po1yZ@!!5sFE<?Y#7Ce4`XBy^jI9{?xJ{1v7c61~;WfI+QT6j%j2<bQfZ2hS;A|(n2c{m)eYX5RK-_xhymzx)E8~L(2wDQTIZN)hY>uwfpxe!4En$%T@cw{hIyfC;{A#4~ES^3TSM-L7F0;5amJYXrmRP~>)#LX7dIb3C14F7sjq5xvWu)Z0qAU6E>fcyHh2FAXet&6YV>!i{EM@?uj?6(cl0!q-D()&)o<F3VM;}Kzbs_#q*tSwWAr|m*NZXfmIUP+IQ{*6)IK4(}bQNOr<l}_W53a}G^MS^Qzj1Lgg(mr>4D%ZW29<Xjq6a~{KXX*NEskSy#`~FaeDu35I+lWp+v)bVGemz~RD@QxK*e&CUboXrgle~#7{v@)LqYHoKTeS8aV)f~JyGVrYDz@Dtq>8K7fe=d_P4ZQyp8ej90(Jv>Liol<8PGNL6RWg>w_{=y6#eS{R2ElOe7py4;jrw&o(*d&3l)<E8c>4)qC@l0PT$`G$-jku^QP!Cv&}8TP-!5^-}F88|8)vvtC`Vl}Qio{CT~xoONxvIJp72LDI>?PI#w>Fh2inG3C|}7bffe+{-tvxz?M)+JcJ>jCYYt9u?=@b5)yuTb;QrF^S;5KMwZ>KCm`qq_pp&jI8Kh$)9UN2v;d(g~FA0Fj*uXPB~>0&I<sX<e;O+)v@3PiocMG@gpiLe%nsHZ;A*0{-FhieV7Bq%oeC<wjD8>SS5w;n?_}|-K5Ndv*t3_^eN}c0u16{58{THPEJifKBn1Xg!RPIcyf|9-HX_2JbbO~Ty-+8wpZy8owoU>PoJh&h`6U0dMc0CPLd--oa(Ti#?F_0dHnM9by9%T@uJ)?2vcEivKI~mJQ)$UJPp5eG+5K2%w)QXj8db%vNk}A8YP)Z3b<*>Y!3X+H0V1WDo%xtjs%YmbN}hhd~YC$?pI93kukZ~EIT$nCuc{ML|aMNp0nJ6sfw>7mVVKhyQ;){9K2I^pCmQvb1e?+^pO#yY{SGjQOhvs?Sz@RtJ%WoVrhR{!G{YF($PVLrn^)vt(5Ca<qV-DbMlCsZk1|FH%qmw=lVVp!@iFiwjaX$0H%0Mht``nIg)b#6~TiqJ1FGm&jMzr0@JnxrfnNA35z7I{fE=>YeEl#ws~6Y2hRDfljh&0n}EL5%;xwZj*Zauz-BXd7V)gg^Ju@pLbj3GjEnRJ%(D*Y3Z%u#UUIiA`*MDY>5s!>Ct8Z=R>2}(w7A3{s5u`3fH$gVQTZ@dwXXG9N!GOS2hH&$=qA}*$>r8JRx@##R@Uz6e_rhsg$1xrYAePHW9rX*$`q!m70w{+Ack_vhW%OAS*fg68qRz^pBKRlQb(OkQ!iO}Ug$T9N|=!Xn;aDmJJ2G-0U`0vY`cyY8xFg`D_2Yr7EvDbh~kp5Nj_y04um7o`Jrdg6v1~M<)IB+{6RaOO~t+p$*M#T`PVBe7%+`Q>`ASHyH&I{m(tq#8qz=_*<PcBw&Fi2MJQLs2~F!krhkDYIdk@xD9sD1zp<t@!Hcpv@X@i*El_eZE$vmQUQd;Z2;UU0=+Z^eA98lQ<&C9xm#?u(whs5~GD&_)OAWtt<3_DaP-esLXCdq(s494eNsHhJG(9M4TQRDpsG7yNtK1h3&-yfCw-t5~NNvKAT(gUIDzQpC%^YkB1ryH3EVU!;s^yrFrsFdTV1eiA8{tA>M#gweq4-~OAR*MCe8nZ8B4>hvdzZ}^pHuYr=bZftLl{Xij&r-CPFeG35iHcp#OX#IvK|rR6vHZ^xYUBLYub_MQCKgCCmH_%Ank5A9HG4o$9mB1t73^(y+mqO-Humg=!jeTPBGGwQ6yt2SPkn5jjPe^o8J)nT{hOrXhzom|6UHZ9X~~p%Iezt)3mgyN>-1>On61=hs@NpTP)-(hSt+e<nErsDDCcPMrn7aTEm#tPqC6VTauc!{m3RpMgsJ!*ZvC-8vaW8Rv8UM&0chvvxk^5&{SHc@}s!y`D4XM8vbHBt0{7r3#5BwHTU2^WF)|sE_re5(8!88=BF2n4)(~YEYGqUS9+1?gbLY|K2xbel*0?#qsCGTI?_Z~Gp>F`OJOFYkXbB}5q4u9CX=2rKqE&_wLzI-aPR+N@9mr0IMRF3|L>=m^;UKOX$XRpDBGaN6-b0+Uh{29%4;u-I|l>?Bvv2*V?c_umZ$u(n{2Lg;%t&zo65=VX3zQiRLQnv%eE|4o&5sRSI8$h&r5etcTW!hO7bSTl}*JWW_tR)pMG9{PYx0V=1Xth<^Xi$R#C*eGdV=najJO_o|T}vqdiKG*SuYHhOJmT@Fc|6MZ6k+I}I2^w$^YXTOe@hemv?X2MGDnO<KJ{5*JQNj4H0s>k0O=7(fw9T$J;^7{Dld;X*w3N-)B|Y-YYc=(A7d3A<{itxxnKl%Khr2<1EWAzYk*t+27VA#|f@%-bw(g(sr7a?txR!m!WK^%E0-WYWSC^u_rgP6W`#iH@_8I1rutVZpa>oBOGL1mT@C`%{D~PXKQwP?Q9()KGwR2+)pFKMWU&KT^N+mf%GLQ8yB6B>RIw8^*z|m0T;jR?>K8a(INf#yU!ht7f#<Olku??;Gukvr10nTP}eUcCM$t=8!%u+aMv9p{EaOb?qL~5UG!Hb5i?aa|W4Y9-)HWml30!AbijvGBqA!-qUO*JaZCdTSrOU7BTm7T2q%&M=kLGh*?s??^F^4%XjvOat64MD`viPoK&nbFQOt{k!bzq+V1KWh-ztI>RZ*do3));&Z;<SIMel6F$4m2JTeEtkz}QjMUjyCcvz~=0UgB~Mp$t3FCIM-gz;8_{KAHdt^iok)4d4aTnAJG4?BfvP<?a2G~lA*F(Qm?O;@MBnP`B`@`4VOk}0brrggUE`4hlOr1PmS7bief5oL&Gqpi+N)fo$#sn1;Q+4^TxDBv89XRXKJ<$m|%GZFV@Bo|Zp&zRw4gBCJiS+x`}fP`K9K&h^Bp%8?}K|W5d0OMf=0&%m*FDjVhKmJCA7HT|S)_(%Fw(pDw^l^uyiw0by1y1HJqC>M~Gr~*gELNY&r#)L3Y;NKL>PrS4)G4+Asi<zO{<MmUGk@8zcO0KrbPG)>sXo(4b4C?pJDWew%w!438yvG9tg9aCGZVdqSdO_P7Yas;l*&c6$>}6gO7C1E^2{gk@8HA{&IxRPaj+^Lsyo=X_yfWrzN`BvE5jloC#Q<KrypY$LXQsSk|(l`;K>tE&lsrzqvoJjLqwFpnh#7YfxLv}BG;ip4b<~Y?qIGUZ`CX8yiql*qu5z4R>TNd_*1wMz$hIs5-I^YgRBr-Epl;M4`B&cZX%)xc@jlCH`mvz+wWL03CpU~_0`(iN=_-W(g+5KOv)+;+=u05@pQ>}YJGHVb8RQAPtiaL)zd!g$av(`Dv2|dDyMQ3$3AnAuw&at=s_F!!x&gEcm#W{H#ntWrKW)XNt(Up=rroU-5WKJ_jBH$XRt<2alEq(N=D*k6=an1E`ti-3hzvZrOLSo%d?Kz8kvcm;EElYfG19SzoO%$e<d@AJC2Lxd+0Ra7@<8+?oZa4qj5Tl%a=nvo`p6bkp1Gac)gT;P5XZ~Cug?}cG8?!!A+(fOFm5L*(2e)$&_ijks#-&n+9S0zgc1u#P(4k2@^O>Blds~ZD2<nxZ(-=0#|NAJb{-NSYYn)X>kRz+_{>_IU+f2DMK<BKTkvU@yuy~Ums#g@dfnG)r`~5%Z{bf=kIl$WT$o7eyF^W*kUXXTfULlbzh)L&Anqw_QnvfZ|6E}OMj}vX5N^8+QXJPWs@C{@NX-!XhKWR7y|Sk=g8jv6P_QB+{LV=(ISFoj~#l()44SDbS~wdPGgIg>^&e(r&NTwzY=M<%Cj*L9gTx9kLX=8l-zL;luB@R@s49CWP!g&h#(&e$Nh(v@qY4YcN3|)ui~;I5kiJouH$Z}>q;y-sUnTAH<LjJg|o%-;^n!8#kos%NRfMXpQkfcMd#^EDjCXykh+o_bH+uC$ZI8j;2!%V`7ft|^avSVEAHVuL;VzTm$aHnbz~x_W=_wYG%0LFGMIXz*gz_^*{w#ND=vo5gkO?di|3OmfW8HxTA$I1;|K))VLUz<v{R=T@)X*+R^5o0RHA!N^ETGAA1|OE>EhBAz2D+V<1CmyP@3skhAIlafWa=ksKJUZso^$?)3ypo`u`x@)6tdZJ{pLPkSdWY#zt?ruK=(pk70Z}5N2F-lDS95TY|k}<I7iC7BNow^cfk1KepVPal7g%PUH#(O2`&o3k#Olg7~)XTP%m+Uce<X*+$ZjvqS<;$!_;BhPgf*7Icc?{9fG8qqq}~5dT_s0l!<$G|?Nhn!0k!0RqMuk-jHk-y!8uA6v?yLi}`(mCppJ;amj5O_p~;2W=W0javhl<Vb8KKfqVhb}sskVYy|uPZ*o1(Jgkpw!Vp2@$&K|1)GBDOs`ASW~bb==4gSjjxuN3GXA+m>AkFj@*ySHyO#p=YB|JEFNa%plB=D*r5wK3Garq-oLkRzT9*Iwfc!f+AhR6|EYrzs6}lP%sn$F9Ug=EVEFE*Vbe?y+bn3PaP*3obQ=<67HwkYAZq8i!j(h1lYVFg-4&AK9IoLvT4|HZX_6frDnS`k%zXvU@&knwZN0LyAJfU9+oFnNSp#=hB1hz)_kf3RVuViLWh>rT`YQT7wG>R;YCAuG`#gy5uLeyUVs*57p{OOm#vI#sfZ1!VB!sv|SQ7V7?c@y@}>>x)5$j2&zkJKvn2>n?%V}4q(v8;9ajt_qK<e#4Wk0<|3X-VCwM=7LG#Mv)r;Hq>)D`@;Xg45EhE)<LSIVqy=8s5Z*M}3_Ltl!)S>dFS8kbI$3x78g2he}F!q0S!;hDW`oFjF3O_eYt(F~cq%qP^~De2`E|_9p(J9|)*hwy2+6cr0@0Epv%UE}>Z1as<`R^mFvHf^DbgnSYs`53%uSx$0;L4m+$$*tWX+PK2tLXr-(RntJ`Z1E5@??5t#+{^0wF-Mgr&6xDE2C3eClIB|uvqSqIS&RkWKL)>0AlOLyfkJ7YtZ`R&%gh6Of6)-(orZTdvkiAnKpg;bGPf5kQgXh}nv`C@hX&@C}FFRl>oPAKx6sY^@x4OM9&6zjocn4h!$sinF{j7L<6l<1bLNErm!n9*`ZG}Ra`^o~z5&Ix@&6~V@=)E|&)yQ+^!wJ_|2-W4N4K$JWs2{-7U{WhHUoLubQMfuuFXM6UldQ|frxbi~msa;Lnd%AW5RFF4xCr62{GCiqeV3BorKomf2LFbjrQ!F-^k4EkesaI0Cui?>v|O#{%gSPGl&KycayB!06B{{4+;N_%yZE2Fopux-3ZmI?l+S>lPIEtXH98ah2%3p-gV5YkZD&@Q73Ub{xr>n81*1EuORBkVquR>N5dTn+sQnJ(a!%-v6Hs%|j~(u%vW>cCu2u<hC=DJEZds_59fr)BsULxLUL~)a$W(s;skGmZM*~l01uD7vkk*ghVAKFJ$UAV5U<pa&33#qmr|4(U#xR4>U+s^tT=t#bVmhgc^wIXm@d-?otndYp0nLv_&jOZajIeZ#sWa(B<<yXz)BP=C-YVYaiXWgFesiUWE=FT?py1yTS`#n|{jJT52&J9ouEO^+;ZasG=cnG2RFQD8ED|m?`U3HD;VuM^Q`Rxi4FWrks@3#!Q90X1=L8-$N4Hp;rz{0n4qrClKWA9%&i_S6HSYB&ROqD&f7XWm#?`#kZR5k{(yLcGFJIXc4`tjQyt;G7k61K;2X(Q9f9PE)p(oM@#N6IbQOFqth}cmMczgQa1ZL7Bf%bHl;4VHAPJaYHiV2v~p+U7DI;s+}>y+;(hb2~W$peCHWDM93RcQN{Qg}h2!Y{}WqI8%F<E@0c%?Bcw<U0xAs=*(AA!@tF_*D*{rpLm&>O%_n`%ZHBlo3?~^(iz-?Axad5h4L)C9pc8s!m-{B=ln4*wo4=7Nk~q+dt~{^o0h-zO__-^_BdEW_X(fSGwU|dM7?#2l(ToktrY9_N?o*>h8^Lf-iXT_x3Gs?-O%TMVu(Xj8VG9NkWmeZG1E2^MB>H@<0W2rn@tR6_zP=jGk01b>1(S<*>V?;q=2mKV(t*At8tfXI7OD$PtNZIN5Df*K5&wbt_E04J|v2ByQ+=IqL3r`%#*KMgJ{e*&Xie=4UGXq4213R8lc8u+MDTruC>DBPyj3vShci5A*xxsKD-EUUhaa(@r)VzM7Mu;pHM2*xv;D4VSTeIUV8Nh2QT#{WTtt`*|SvGR10=wxiPf8Pj0K0onaj2?<3p>mEh%PPLRL-DLwj)V$bpK>122>xZhfuGKH|`^p5lOXcD;j0qbj9<OZ7V79qZi>P%eP|gxGq7IcBz<+(_5!mc`rtZwfQY^l{ehqUqoZZo)oy<t58#QD<w%$Pd1xQI&`dL;>QKoLv#Ssl+7*v#s1U&IKMA<&K$mehMOdqUb(lUXtZ^5^e{_7F@I|{2FYY-*qv6HOcgPZpsr5ekW!9^~d4~c4im&uGqFJy)@3D~Z|hSRPW(Q>(`COyLLqeC;U(s*jMrwJ0!aZ+!ruK#dlu_n9_`T4V-uJlyX22HChHu5vrRM)Sr-nhAWb0-o<2Wy?>u;eOr^j)X)I__oyJ(D2eWTbsQ^UQoQcs=3*KN65eLTG32a2Wyojk<*&AY{Tfl0P$(3W5MI>Y`$%+4EYXL6Y2V_HG$B0Zr}N=H{C>HDZuI-$J(Ye&qev^DK%@BNXw1Tk`GQu-;fk^v&wVwOX#%dHPot4Q<rCtt)YNVIS>aSbCsH3??{F6YPvpl0&@cCR*2S;hQmtVVGoP>;M_a{_h3lofCbpJ6H^PxnlIQHij2>ud23lxCciE#)I#pdASV0KA)W8Vkm7hMSiENRCFvl)L<5qwY**~{>b@4Zeh!+)TV)Ix}BUA38u`~44NNm&hA!NaLfdHxuCP6UkudCL!cudc^pzM1ouZA^K5HeL+z;7>lO={9Y`ZLcQ}1gv3YHWM^C2GO767U`r+}5?yZXtpRLOi*W2#5B4cf#og1KPE`K-W1bd5*9r&PB-NY{C)#(tAW}Fn)9&_$TScjbHkbgK-urqY3;q+KBn7)6>suw7Vixe662xOG<&}P&yGMef!vWL2|19-t@<p2IZLz#h$ms6Qxv66G<Ps44+0^Y@NlK7-Db%_7N)pe(~;5!<#;jdQl{EI}ofgdYHyj`Ne|4DUw!_Q5Q4l7`;y&uShoTQZGd?KQ^A905BdyH8^_H-ThiF5Jh$Gr+JScf~25VNRj(hSMW<v-SQ5u-hy)Jhq!DQTqtZ&c@FBXggGVarq0Ho`njZI1otUv!bd%vWqK9Orf-r>eU-nTZHZJ`0eAl1%D_D4}GO7UrQX=<q`_!)ha6DKAB9)vKG^q3)VdP(C`L4<&rd$lOFDv-&dYs9;9M5q=DO&@Am^h8jVxVUqjZ4jg<xs5goTGh+wH&hlLa75Cl#k>%{{M$aWMRXqZ%S)3VxGarBxQQp(MFIo^_U?pi`h#)Joi)SWmoCSAshGZ-Fr}cK0npGMI7;s9kCx7S)o|yavM^;ec#$hDV(-oMZG$*GRcs^S~(|~}V6HbZt6Nw&?1Nr7Ls=nckE4>M#OUiG_5bjYXqi>&n;!o+cinyq*t>t3(9vk66urbnzjQnVfW0zM{Gv66Xd@~s@%!LsqyqbTjWk8ato}}{PQA#gtJ^e#hK2=xZR6A4ZGT*@Wp=!-aeUD?aMY4sD4EQbqt1k7$Sm3Wq&ZNHe<6N9El0lF<0KH2JBVQlR&rQ<!bIFLbFwq$?@mgNdK&eTl=G#3kR!lM16=v@O;&<``)4fX;bS*>;+T2`Uy%z0k-rT-cBed1&nNW<4cG1t_88rPuYcRs>1s+l>!i?a_YxekRG1z15QnX$DiKfUm=O+h0z$`R58?9z#g(e1v-Ij)ntwW2*&YZ`GgRa|0QvK%QNbYAeIm=Gpi!`(0BR1ioX7Up+^fs!}e#R!X2UOQ1m7rK!dxSTkXtiWbGBGgWl#IP{25cpF9LhJ&4IVp90C7pX>RzmpAco3W8ak&$NRm=kIwDVqE?nlLd|c<5Ah@t&w$DlyymF#biI~vzt6T9Y&8B(v>6GlE|8xJMUI34t8&WaNd6N=qsWg6|HnZtT`Q$!F&eARVC2#2rsvubo)0)V0|2sYx-;k;K9wTw&p!4_9blU8N_8Lya5u};1eZqR<F7hS7rNf*_-HoW;RBC6Jrcol5A3~QFgvKPxx~6;&;p4cF-i==9KT3p6T7dP{YH5@@-Chs2%tEmQWNM)p48~9_T+E<)iDd6Fd1V>(_VK%k=z(df_ZM@CfDF32ks6qC&^EMuTK{PJ6JpS&(XX>y5IQ5+!(cn4W1}n0V}6&^Rizy7)D;D%1qOOZ_g0!iM1$6H;YMez6mn3xaEvsMk?!v)t~3IzPSXd^V2X%c`yqaB+J1-`(8(~MiDP2g(RjsORy4ZVP@O<5v%HVKVDwaBw3>an+?gX{fgb5ldku!#3Ae~Xj>W2!pf;Zlds1j7Em$kKjig$Z7VA>c<!dS?x~@}MXot-j&=XfCKkeR)>*pqHF4W_U7|_Vq(lmP!^E;=4v!R~M%-j5)4&oP#(+z~1T$adX=*iuA&DI1$!qdOvTyY-~xBgK1rs~=zG_ZP4{{kfo`2Wvk3TWC5Ml2&j(x#5XAToAG5rrYB#0-JZ!N9@owG#;%3BrFM9B`subDy$uz`Z`|z7JPEnOLkX)aj9}MmXd|TObzWnZq-<R%M5_9o&RkrRb`NRWZM*y)bJ&po-tv++O|Z<_1<Fx?z4U9%eSZ9Omq(se@2`lXKp)yG^Q22I7$Ku987gWKzi#w_BVdT@Fh2>ujF}I^i*v^kUuVzOB<F+1-DTPVW;)(dW8<UWPtphITX#2hMqfH+6}%C`JsAG9y)-ZpOz{RU)+8S|wVwa|ezux_#tHgeU6Jc+lw>T}bP5e5;)%_cwE$-t^C+Me(8ad~U%pe^AUey`;y~i4aQX22<k<>QncEcH|v49*3<WQ$r;)DS_U<;i#*KuB(%a|3<xpHOTgJF{h()Ay`hrno#($yPeq_Zm;kOB2TuR#TgpH?ewfo)v-BkjimMVNBErgmNMw(pJYkCo5w&=@Z!`@ht=DQ&J!Y*!dZcKpfBQWuvWcVTXSDBuCvE?1Jvar27(JX?h@V;6asd!_IBNG0ke%jInA*x713v+d~~}x>Z1rwK>fShn>$-I;6UH1MVlLI??i8`?f~Bjom`9FTHV;K-l*+F8=Jck0tP{$72nk%pyNU<>EGO_?(D4I*vOO={2XQ<ExG6ZHJZ5`QU~-O6DB+OK=VfUHQWe@@c36wKUj7;eaxXW7>};SC=r+8yOER!da_InTzDlyhmN(-&9UUWq&k9NW-!D03TQ{con``LtP(O(#*&3*v#@3wG&M%JoN@-7T)KQ|B9E$1j4ovK8LC*eXZfm)w8a$G6cM2_3Xp#$OMn6$eDYvkSpyzP<r`YWe$6W8RQ~)z`1Dzh^CBLYJg0w!<^sxHNDh5Yyn4AAbnlXBo1&5DI3kK2f?N<5>@@?a;posW2BMHQo8&6u+Geu>-{YsV4#a7{&vS#(%1$1VUXq7Lg{_@E+i2i~9|{nEhcEba2I4G}QQ|P98G0eLDMF)8h43VSUUQsRhM}=HPtFyakAa`pCX6hvNhc9GZ@WT2A)N|}mAsC~5;HonxV*|5=S}0s=ZymEW+9;G9Xn8_|4uTtk*tQC65Jsh5-0VWG<Bt?CyCcx9PY8I1u0Wr(9ds0dK<2}bA-3FcL=-;9IcXbnNrn)>uwf?1A1Q?c8>?+0$XyXszVxqJTSIiWTq8k6;GJap-iL!X$q2C1b=ah^t&a|5ghR?DK{6U_1~os!a6y%H(K<l7T!hX_VM#_(00Ojic4F^zoQyBv-yU}0?_OS;lhPGx2%>>(-ht5p7}J!Zg9&?yG&_F&dzcy=y~UxBUp2WloedgcaqO5d!(F&VC^L@BNM9Bec)t)fo#}nCb;F|cBrCd+a1!@>c5ABF)~Onls#bKZ=v#*^OOk81XAHxg~;}xRibR5A_kTNU4iA#9f$gu>WfUxhFS{kF)KU8fQpg{+Zzmev$l~{MW}YHS2MMhsys3Nt_6jnLEA9t%IeR7`SKCb$Zsm%lRrH9pTH;n^OL{zgw6Fn(MjPONWqY1<c9`ctw1UrnIiaW<S>6l`0gJGgljaHY3Lhqf{2YRJ$J+A&E4y3n?FgJuIcRmdHBm2{FGG#6iBQz&Cm!3tLHJF^6~lD&&<eezJ6If*H5KiD=$P7drE%uQPMhq4J8|P?JFqU!Hpv;gB`^U2D8m){;m`COq3tx;ARpSCyw{LoK8J6lqvOReF=;K+Bcf<(h?8#SNSsH#QNNu#6`eg+*V0Rkk~T@9nyO+8!^JDdfpsUoHkVWeI*OvB@%1x<7FYsr6w~mO)hrhFlS5dPfy??0X-Xmv9RhzzIW7N9^-SUEh+zzTh3TqY2U?kPtAV!ed;F+D9r{$5?_w99`s5rg3xcIi*o4xhTsQ{rIYc=>S1u`inMhy*XkSB#EkYs3DIzlBwsejiUPDsmDKK$O_X1C=E240r8APwC^LDZeHJF2{Wo}$bE%76=W?`fo=LSJGuuQ-y(2Q*WwJd9Kr!jRK=w_`Df{Wi%Tu#;e#>D5@aZpf^SJb<F>yB0VbXvuQ^~})H!%bGG}<8{I>;5^PBw8!CPk<Gz|Q+I48@zeEK(A={V3y7@ki>Z(ZpmuJ=8iJW_X;Ly{Lx6i7<#zwfWZ7dL2q>`7Rq7as9V0A1CET{FMT?v)vsS88zK4>?Qb^zS6=O=B0_e;_1GHDAuzBztBpGoJOReq#dOB%BokN3zyo*skd{k8`gQVbXl(XliadkN+<ed37j=aj#9^$iTb-+rlVEHBSvD=lV?mI8g68~ZhJ><kk$~^gHrtsi*xwz;8rn9p=IgHWcR7d?nq#HQpt~MBD~uj+0dEp1|=g}$W1%zHHD<e0m-XCEHmVbGQ59dhv*1b4!7s&7wqBvEx84Xkn_nPaVjC*Jh@}fqp4@U)kEJXxTjG&l1SWom$MZU1flx8g%^k1K)MI#G%zc1a{s{YDoj}K8b|Q~yRySyU-IF-r<~9YqU=1`5&^)-F5nS%+plo)8GcQA%fKt$x9vk-R|fFGd*VJ-{5NFJddP{rIAzioLO1x8%!h=pGoau@{j@z(7P?uZS53)J&$cVf4=D#W3#yrBGK5j&I)TAEJGmb}VjC%WoJW`ifo+qkKfYOuBqJQ5<#V;Vw);-%a7VVzt<9~QYiy<6-Q2CN`OEgT3T?P}4VelL{_S*u0tD7~Qv&2<LP$bN{gOO!h;HAb-SB04vD{a>20`0#zx)LSG=8gXiN8^?bXkA%2Oeo2J{MKrd-`kP9`m(xdp&tDr>Fd_2xf$H<Oo+986iUX=cMLZ+mUZ=Z7Qm_Qro$<y}E^SJdH=0m}v2RGtpv>LS*uWpO_i^{fRup`;%D+F@pD9?0ep9y#IX1HL#M-Ig`buuy!4eU(Y6?OGPAseV|{_Gm-=^lN~6Vgxk*YV^a?m?Cz=_Ejryk3|6AZc3U(HtclW-?d6e}2`d4zBqvU3w%c7wU!rgavk6pSrLh`HIH3-Gb@vVF^H<5ktH$js$%c`?B&S!CNQeXbK5v(0I;*{O1s!8z(7=}zDnFB!5vApbzQt$1RCwAV`DJ{}n$Ec`4iH>cX##ZW_W;SE9$_YP9yrCdeBiYXBCiRzR?07si=ievj;TdASJ+)$YLttW;4EOmO8R2))>L{0XZeYQ1WMy|c7;)@{E7l+<T?T$TGqT&Fc(T7${39WQdt>f551{NK;2<_883S>EGl()7dVZYZ7<8!No%Ty+9C0Qve~-P8V%4q1{HT<=IN-8YT2F{*Ky%er};{A@$&g_9iRLy0x^*w{w)X2>sox_|11ES9!pnf!(z)xGEmpg*)AVHdF%|*cFok2|3#%f^2oWBxc$#bv=j%&2%XO$LD?bX3YC&lbx;U6e}S9<Gx*R|MB@8Z3Y|U?1Q4yx3ak&KWGnf|h4&eOa&qEqr+ED9C;x!R#&F6Wx`5?58;*W@|3<?@4)Os67??rH+FEMvK&CcPsd^u%WGCGFd(`hw1g{2r<e$Xiih-!LR+Nyu&eF8JMi7)__8(EY0N7^wG?VYLCrDz6U6wu~Pv*NOxFRh$wEKaK<DV9gXeJzItZ);3|7-8RXfcw&X#5wkiYzX2te6I~f7g-&dAm7e7-*L83phCf<yrndAU-d@(%gHw_44yTeEyJ<-7GDl4t>yX1DQ*biZ?!p(KoT>U1-N6;0f9n*l#fLnqzz^e6c8!NF=M0fv-lA=vuRNfEejeKlTim)x<l18?V{JNcmH0EpB^FZ+LK;P#u^U`XZA5IeArhl#k47?!oU;9+?%`qy+ROl4$zHW&3u#w!OKtBjSR|i&7mu+UJ2h5CK|6fA7f1@Tqq6^6xwmuM2m|HV9(75z0h6g!ENjRI(NPMDC=@%!z!o1CK~(yY@!y+MD>~dUbtu?Hx>b1|8!fAhT81A{va5;r{B%`X-_XV});LQeB&Gqia<V23IJdOrB&yUx?=H+E&V!^IoxU?K>&7zq2RY3u4YFx0%gqQa%)%o$Ag`ZPy&A8AlT(@AaZy7w*|+4@1v|GwZuA`R~4T5pf5Oiz@7v9tQAV_@{W0e&tmw&j3{qY{0Do1tbMPr$(m%oGiG!<HJC(XW_z9xeRRnVdyX3xSBCG{m>J*81%Jw*gW+_OuiR^SiW_0H;O#!;=!N?XKvZ+!=4>^M@cNdyx@%ocnX1IwDccC6U+nuTU<sj^6kyd-FaUeVSnDo9DhrT3gmmfAV+c0FA3BNhckMKjVYzWTkS4_IE)B?iy#5wlP;PZZn=y*hzCNWO=F(bQ2oGIw4p7pj1Pz8C5<2?QF7GjbWaM@Mf40wySfbJN=N-(w|}c3(eT}!QD`RZy^_Q;L5j48NC$p}iAlwu0CA!tT68eG?XD+C5F_IGci`t4ZmY@P`uOGe0O?;6F2EQvFNdiA1#uE&Pxx23eZnG!9XuiaO!b4{6tO=M^f2@<8q>XUHQHU@Do(5D+UC|fUctWzpI(HS+1>Q4dSLd87k#gAeRF%gy6XvY7kJk;*SCO~-2oxX`{~Z^O3~G4oc+-1xY*~;JhW@8>#Mua13b`3^2NC;78zRv`{INa!7U;;AZ#hP6guJXy^>%!u6KL<*!;*z!KLOqTRe524{%rk^{-Qs7jmh8bV%79b1ZJvl2<!MvLHp2d8CjYUt<*_6maC`NM}LlIrEBZnP;=6lB>MJ-q9F@S&#OLCuMpsIXX;=#?`zux3O}4ZT@BY$AcB{nsj?gBS!3vn<r>pKTMX{oqJw2@~?mOFG}a(F=8R!^1_#AoLFg{x<S))lrtfg;(bI=3;pt<U$g)u@p=&Wk;XT$_&^fylS-uroWaO11`9ULZFbuxeheJ79wWCl>sc73#x#BbGtex0^*i0O`GtiBFj>$5NWR@*VPRq(U0AHJ5A?QBXr<}K^hjQ}ii_r=#r|)-q!>5=G*3q%`_7T;ocN2&r=Xjbt~?&NI-{dyqph6aK+T7`rfG)039U8^3Nfwl&(l#3N>vhLS}xRXg+2i~S#-v~xj{PY>#MbuXsz~EZOuozqb2zEl@dK3@W{C$xZp3sql-Y6jpAnebg&}AEC}Cy2@_H~L7AvVUG1n>Sqx@D)+s_ipr<&}(3nuNOj~ea#!A{yOLVh_ofL3OjQ7sz=*%2!s-1MESih6b{uzFjCL)(9s{&aytfgJI;k2Bo+m%UO)t{G1oj*by%j$+UF6^J#IH7y>RT9VyK3Db{QT&HYYn~2DPf319+Yo1Vt!B#c3<MUhuF{9{GFfbpdxUgW%>lxxv!~F$N^=n2BwHHbwG%p6S;5BoZ4gPsrCKb#j|d%w#iC+rUom*vY&)7jliG#Tbjd&<17D>zW~mI3mW+(Z?Eb5P73sOs21mohuaGt{2f#@fq2wC-m0^hh<x*obtyqnP34sYkh{YS55yQkqG|3q=?DK<D!Sv`>anZP#u;_Yqb?xT14v-D=1$51UJjH-ixJvM}URk(Qp25-4Ok0NfORpWbITT%7r%XncJroQ`^zQcY<^!G)1bf&E3<u=&b@~%6Jpmld=F@4dxG~SQ6=M*oVFLHReiJi?V`U`L#tQ7F#!=)jbXQAW6((<C!K;-Q=RwT(sI{cRjFh)`>J5+f2o(VL6;1-9l$^lKXn-t?j2NR5rQ06@iw2v$6(^kgvK`~3!)o<TpON3Xe13lGa-~v3J}t{_rMy)ROeJAk)*BAkh8(<0tTBgUV6vKCehLbbrV>af`i<YHsmm|5Ed~~1B(|7yqtWb7z0y+KAMEUBxPq*%X!p>hn<sPcJnub#2D<!vU}?%dU}>_P1VKMH$n3&?ra5zvh1q?$GXzAxG5q{HVXL~c^Xvrw-)^|+01V<5IqnP>(ayc8U_^5Z=ew&+z(glr;yJG<*uy{u`!FdQ4Wt$Il2)^yL&3>2qSuQO4xWN}MSZbF!L1dWiHct<d6+H^>Yl@0mPoWLNq32E_u!^HXaXO*?6tei{r&(*Ctf1G<}e|x(*VdeNZ*H;0}#)jMyoNcAN3*$b13vB9}aF0UA#0?W~3P~wXXBbnYvci?F%=c-Rka+VSh}}Gzmg7DxKGXEY`O5SwYgAzH!Nu!ta9{X6IT}shy@G?u1;9To%QjvN7~lR9_YwTB=h%-yCvQ<a3h=J(EDp>2$i(cSj@90axS3X3mRgr0XL&+M~PaXp{yR=zYv_?eCMXC44N$Akxt;ABQkCbYoKWl+!UMfY7uJ%s_P5Yn|#<tQatYs9x-k+4Zo}&p)K<8(tc}at!oACAsk{7^YU}hv6}C5zuq?a)zI3uS}?Xj4&R!&Ee=1^2W-_D&cedWOerqvFUP7JS{XWVsxCy-kIm!oU7ujzV9o{xOLz(D&wd$Tvd>5W11}>td%<vJQ3K|`$!r_$muj;d|rXI_^{R<1>F#h0zyEVwmC_$iqFWuEzQio8Q*~X(0F0|8yy0f_B!if)wo!Hj}Fg!BB?vg4_!1z?b8S+i3jnX<oK+^M}f*vzVx#}w_os4p>qq>Ier?tOkJ?L^`5R8xr3FQ1sT>&Oi`X90{^#OTUo7c6cm3+TVLgf*O11Gp6%kLvVqQ2@i2z2K-I8CdxKM8@EujlcabPOQ^idYnwm6w&CzMpfgKzjy6e9U<y=)x<YE<9%CkH(W7-b-ZhlnDqbdYP#h5C7bg?{zAvLU~-wS`{b8HcR(htN>acY33$t}yf(_%d%o#N#oc=B@NxrI|a=)k9o<;BZ$3yX7?kX1`%<v}O9Js8~*8Hy`pNOf_4DFu}?4BO$b+dEAsxIKwd%6iCGiftnG+^QPk@EN)5^RAtm*%!nMQ#iwtZ&XY@vJ?t0fETth%OAw8TS5t-`&AHHYcuCEjLVm!xV#jh3P-Miivu=hbte9|R`T9(<hDm%E*_yHV@2qVe!S-4Ru9nbbotW4E%M(;kTh@E=4v9^@oB(~vswWQ)(ieh8AC|QS1OeiR5PH10i5u?cvLP|D#@@(pa81v99LQEe+8d>4Yq4HMAi6H{-afIWW$#Eq0?~a^R$KD59er%eIE6;8nziLl$!)2JPIzoj6R#i0Oq&oxZ9`P>(&<sAjtH?I#~d{6TZAf(=s2pX&y30C7waA#y!f9lp2w2SMQ9+WorPpF-FS+{b%)K;P>DQU>LB5-k`rP|AtF2IZ9NhxYWdOCdPwie?QhA<C#>CGtKV7&ECcvl1oIu%tkV>mRXdO92NylVKiMX)HXbOdV;=Yz-0sW8o-OJi0&cr2}f{&JV}y753Xo&+X?lo(AWxncbgreoI%F00m6{ZQ$=^vp}si4YkKMvbyzBf>rd{7NnFV~`{b0?*>^t6z{$m@7yU>&I0y9QY^=IEqqOy>pOEJ|{4_6^jKOf<m0v}fbUH8>#buRdKKtJ6yiV1fM!j~4PMm4e!zV$i_iUG@Yd+02Szp~Kh);zNfni$XXMtW!pC_}h{wA-Py;6ok<X7U-y}GPsQ!DMK3rZ0aW|djS_KsnAHrKd#R`l9de$<VP+P3$9Z?0}ks>av}Sj(P$JT^CSmpT*!)rqhE-l__%*AsR{sNU^mAbs_6X05rJ=PYyYa<n?Pf0fI<3K{)(UvjuhX}(n1T9GdySfcbJI%MoAgoO79_`i`y)W70gS_m#*Uh-DT3tm>8&vn(aH(e3|`9Vn`bQSR%nRGm}Cj>#|RAyAU9eJ*%itf=$YAvG+Yj0Pt?M9n#?yd_Bi^#yezE#~8;HoIxT&u2c-Q2NcLDO}WTgZzTlI;>Z9Hkk21O5z=X#^kQ09Bh5Jv56La=yIBG@PGQ*WS!Z22(A$mtB@zT`cJKF&1uYkjFV?G1Nk4Xi2mj4oSwoF~j7Vp1QND&90Zcnx+fu_TeUnAs)3T=4S=Hw3K29J;?R3{P8z9m_Po;bcYI%9*AW<jy(=*>&J;MOF)%EK`_hxulGUS>9NskmsTAcA@(_K&E-3$&&Ccp)N~mMzv*2#I2~f9lLUS#V2%p`Svi|99Y^B<``Wjfhr^>p@D1@u8!4<7Jm`mhK?JP)f<U@HG!eOej?pvl_*eBO_u<VKjh}P+0ukEt<<Bv$;^{9XM#qzfKj(N80ipOIN7!I4x!f?8(uHi)tnN+qUWR$eV_nW+vDG)RWxd6%l4HH4UGyBrUh7jz^X2|DldmwNXap{#c{|E@u^?|WLlQ&sb2{v%sztMs>f5td(mctj#8MhgtH)Gz>NQQ$73$?CBtM+8XvM%Y*(-56iYB>3Mu<0`?%IV<X~*T**-h`R;Bv4uh2PQi?>`Xw_sXc*O=9mYoVnU)G#C|pB^^a7u@FT*>|CpE(4A@XE}hhJ?^2Vd#0d4Jd+ozPlxp4;b@kKWgh~3+A9`1j`vH*>1WTAnN)(hF^~T<y-#aaNtC*@}|6s1wB)OcJd=5<b@fyY?@K-`_1q65wHaaWi6L>dD#-)GPEF#5BlX=eNjjmFGI-Sya7pKQNT%aAMAJ$MmbUr()cgR<rL_)l<iuH2C4HrgzdyNvUz&Xbt)s^ZFd9UTz8Qq~Zcy&)G?+gkuCc~fXoS&y1JMZ-HTa&oF9+shbecs&ZLmg4!(+4TeM{YjL>3ZUqbw-fu-`cLdwOacLpX*8H)fD=M*7;1ZIhv|9<(f#Nio7e#(sp$nn1!Kl<fDYcIEUxetCi<#3mM)ocMcSYh?f1UAyh#c*;=_N5;~X|B2G3x<ufhHq~nt1cIf+K-m?oPJ0Zp<7H;Rw)h%=wg)vHo-o>KN$l)&mhi~fnam&$yXl^dZL?jFsG>LJaFrm-Jy1#J*Oza$<XKcVjITqDB7z@f$M1T%3Liq?c#At9RjNq`<Qg=jDN$R|L+T&D*Vu}UNzZnFy07+jS`zr}p0mEODqBdCDmn`%9j5m-k{4|%{paPcd#22msqBcXSg|~bWriU0gQI^;QpOe!{?j#g7?Yb>oX|q_Ij6q<=#=3C`jDVMC#v*ui@#W@T+*&**7QwGe7<Oym07<z+Ky0Ld@06%*X>opOaSGZ1W_Tm$_^&B&`|~ttTs0#jxg`h~z2kh$pBTwL86f!~2Ykz<&Wv2gC>vOd3LJ(@Z;qqbDRfv%70QrU3~ZRj$Umk!IMVpLCy!omtay2$dAW7v<yW6CRvaj3dC7b0viHBgGCbkXKrtuc)I8`&Pq?ZnVb&|qVwoXmyKoj_bd-VP*-T(JqopXpNHPpD=UwOlS&VxfAudZ<nW#@M-p714BlKw}ErCSYDCOg$SP|fFU9Nb9b@r8pG(d<KC;}Ux7;=}GIb*-p05HB23Dr40!494%Z#6$k{H@E8>10VE;_1`c_V(ts8TX8~xTqUMmU|FKNM~Jo(6=u#Bj6%4Ryb-Fms!2hgJmjMagoumSV~$bd_+x<V{VfZZDQR`(YSjUi;yzIT4-413~(pmGLR+_P}a35H=HaITg25;Ug}{DpWF1?L4er7@TkYipddwIlCb2nbqP&1N@RZbr8_hgB1lQ9Gbl2RUWhm@lsGFD@!DdXqGP*8F=|n%kiof5@ye){3)r=q&QQ_mnU{Jj=6glCVs5zbo#^;XQ*Pik1mTo(QDi>Zxw(!KeGP?y4G%wkt?cdM-wR%C4W5_1+Qy1j9ibu~$FtWLPOR_!cr=*3F47C2qF_=F(bEDI6KMSA!B6-5H&qsJvd%QusfCQ!ROAZZKfWv-qIf+1d`B|xccN~kyx2ZdvGL2%vq$bn7=9nk5`Q1f9DqM9v=;7^It=0rE{TShCaDVKtxqtM0=(usxIhW53O>LHTp)?aKQS3BrUbVTbwdz2iD>@WQV9Em{tQE{baAG5x*1d!QF|{({De0X)fNj&m#(a{=0YzzU2|bq{*34W&T5_&K_F9^L=lMIbAt#(&xImD&5c5<`$BV5g*)|Ht$?RoBWlH<Tnr|vb%eGd5|{$1p2n%8K^8k2p$g{*q(zEs|G}bA2v!rZAgmPZ_}2u(#6e+qsFE}92O<2~tiSt`R>`|BHO^eak92f{h~Nz-LKj3?>;j5s$*>|Wx4Q^K`h+Jj{A$+KkA}+GOyz{9QTnKu=l!NlQzf0pz{mTNlWIbB_yeI0`I?bW#q~GIIWvj&P6c$ksC)q{PY#-?XAh1FuX)x{@4Qh(UV)Y+O5Oj2ChaELIp3;nS8u>XVE|E1<A=bQ_oBGh-S6&o$pl>Sw`)JXxeD7k+O2NifMt>Sbao;T^CUX5r1fbqD^X!&yJm@1ik$A0v(spED=`0=yAfjC#6tZM8J><XigB~sgPEgKL|Tx=S72-Zd);{ysc@b>m)TzVu@29$+p4Q;5uM_VT`~~3_Jwcgh$7$&j*_@-2LEK|)*U}}8N;WR77gA)x{%76^ZZ4PYD5JXs}*6oQbIfoa{9#EYoeaY72AMDZ^A5>Jp-z{w1^iwDR%7YGuc8Yi)xjkqQo4-^ltVM(aeKOIhJyp)ELnhsNg~uJh2}nA`3;ONoAkiH0=3CF-PgP!>)XDucNs>?k8PTz2Uvx9JLP6v{GW^G}6kT6`4a6?66;QxNY^C-9v$~MubO7#DkG82;+2NC|QyfC`p=RcE}bW;h2TSaa+dAR-7`d4knKsNHYxVcrfg?Fpoxg0adR`f7J%whyB)REtO8Tya>f=5&$Pvd<(FuwD7{b{3heWS-ymd@U2a6b0zT1WGD*14yx(LWAvAo)h*6nn!n<z?J@>NZo;hgyXa)m-w!;67tA$DnBs1naBYk8goK~1YYCl4WY`FM_BZ)*po)OZw+4r@mL*pWSD*&zkIl{=0V9TSS5j<Xuh~MathjsRC~?LtFbl!oSvmsT4R>ng1;fd{R$erIqXu<x!7P(!=^!Mx&FvdTVS0FdwYIimJYK)KVf^0Ru5Q4BZtv<(?P18R?YuhdU;%ZDz+VCFVGMB-*u+TrY2VTg$0CjkBqiB!{QF4NNuu(TRu|#kJ8)sh5=fw6+1{y1Q6~~WHrFIgc9|9qo75xLYf-%C@JPnni8XU|3q6+UQxVJ}3={=_g>R4fCfaGyv08wzU?iFu4~QqZ+lQTq{y|8TZpEkb1nao$d*=vKMrGX)x=R#mJn%Tf4F%98$<ZNWKTFIWF(ip|;~E;;>2N;VgCk6%O$;+8_oh`0DNvSgh6ppU<OR!;_=g2j1InO{&IhCJ3E3!7y_z^XAg{KAre9H)H*XSagZag?$=9r;^SC*z+fnW0-1rP(^8jgiUOI$BHNx!SxbTs`0Adrvn$x{LNB6q2gJf=FU)|f!to@J6m*Q7CS1z4P`~Px@^9~_6tc2JWsJxu^fiP=7AY531&YPmueukP^*2f2&Rp1`u{%U4`KiDv_Pd`<v_aWT#sHN_=;=q#_wS-xFA0dc8fBJE%-u?FJ-*B1?RQ?7YqYPsKmA>L@kNS;8?gid<x4jRwUTco84h9Kges8U>nE>BV-?yB&Rsk0){9)?FBWXg&Bv^#b{^iqOr`Z}m=W(O{<jHp>Z@0FyD^eJsb>cHZ>%GfJxM*(pjIA4@(obKo$>G&jtF?Ez+<6|8gRM3qXm3PuU%i2R@sodXZh-qdVM1N>ne=310}Z{VY(t-j<(ZeO;~R-8*x2#E$s~knkG>=HXOV{D14>lF=~C`eDwSW5H{Ca!Mk3%uE1Y-zo3wGU)XHXHWmo5%nUo2MY!6>+99-?~t=jhLI!XmwYt@bLjz*<01zbY$+{iG4h>BX1fCN!~g!2~o2f#bLFH60#xxM;RD5X~;$Q;m&L*V`^f;e2;Twh<^O(iXi%#ZeZgFTe1@$YWGGpM7aV{!q$Fvvu`=03M6aL%Kxn^)IXh0%Sj%6Uq5j8929oqo7hy$b6-{WE0_jffWpd8+Pf$J?hHzlS0)$a4J%;hEaa!y!fojevE)_qZc4lDXH!@H^lY(R|Px@6iSce30r7#`}AiPGUGb>_y#<nOJcjO|6t*9rH27`+IO9ce*E-6ttTRd71DdtE@S3X<7<kX-0!%ihPqE1k7z5^(ovlilMtnL~-U3>bk{yy*5n#*i9&+sf8LS$`zP$vW_oAZXw!H{z~mF@&qGg*Er$$B=T)NhRcH72H;{RTEno=%O3i>j&7sUi-wLOiL?a5FXAso-DJQagNgK)_07E)$L%C)$H%+_pcm0jt+uigh&LVdMHAa<b)&i_-s)A1<7AA$bV)QE^tvt7!=ST!3sr81z}9=o(GVG-W$MOhA9{_BxI#;U$q;CEeb*T|h6O~VC}0OxYH!#57UCJ;ypFB4)R3OiZqR(9MC)Dj5WueIQSfI1KV>ACHyDoRfET~$5n~BAJ-Og(b!&VuQiSM@>dqS>oB)tlFrVc?GdU<&RHLZp(5tOPZ`R(~3G32DCeZRq206JFIa8^y<Vk{J<}l3Dguu8&)pR(0UP(f}M4VCK(sDE&bUG>zj5*nP+CB|7tl70wfoqOUDm!Q33;r6joJ`1ri1*{?lr2n4KY({OUefpEMNK&HbR~Qx2>yd*j}YM>sw`vQ&^2MXD%=&f^JC&pxJJ62Klk32DYOQoW~-+opQB?)v~hDC)(zM}&FV?N7E>cAFV!J1l4j<0@yMAoM|`9wFt$GC67*E$#z5)m<TSc<m{X@pC=(qIqNGb6Kg7~Uh832cXqlm{-Jb0v2YXl(1J^@X&!i1@W?)u-sGV-74b8<Imp2<K016bhXlt7KM~6vR*FaX~vMK0P`m7{k30N^fP`X@Sz11knypg45|FmGk(f~<uGyub&(15IYV#TUL<NJ!!_|YoC78s2aG>{bva@rBcII?D)pB0NU477rCCx!Wz129hC_7W`)zp1c#kC;T{4u4CBNh@8k^9cgC#*8B+^>XCM5$QK5o#_XhZp?S1KVt_1T$H>$bAmEG?l2yApgyp-&I!lR*0I1vEa1)?s@92yegg9tig`VuITfK`o!+21&ifQ==TGrJCB%J$b?~ETeRXGNb>l_^OO~IHT8#W8Z^Ut@6O(HWP@K4M8k`R8p`M2QA=KX-jhd%Wpp6ZM+~eh;@os3mwDN8?lUrE0n-Hakriwq(e-7Q)#Q2%mabhHQ{Pk+huk;Zwt{?c@>pR-7pH#QBU%(laEk)F6Fbq%YSZgB~^1sgc60Oik3VqD<*zDuK&Ap^>IyWpLm@qf3iX`ui(xO!8!i7aFn1hPSeN~^!ni+L!UOo}O$=Stv_IjE1G1_otw$RgJ@oZA+;M7A8r0KuJ1Equ^nR61aILtXI^g<%bo|1mQ75N`Us!Buf=X^0^yyxeEIh;oxuViIzh!26Qqo`frc9Hss6NirH^kql}8~O>K)30JSu!HFz=VTO1$EKp%`qu6{(e>@>y6tPz;iFqh5I>u}Gztc9<W9dw_;O)F4)26+e^N{xoH`|~0-5@+SYl`KLeZU!k7_KwwrF;Ibz|q|_3NwGR*@eRmPit!Y|tKLoUw|K>Z#8yG?wWZ6ZlZQpVK5s+mRyd)C)%6UsInIf8^M=X<;Q;@ModcbWRRsTE<S@Xf8~<>Ha9)4mjl`Ca-QpyJ(B07z><BRYpQO^RZBmT~%UVj2b~SGAo1<j{4xKdoh9VzLs`G*t)3rYw|-^=5%f4J3gjR+qt=gy7USkpYV>T`z)X?-%+)m`F4a5%78|`PPiBUih>Y7BA;574%$#`-6toW2d0|iK9YfNxHh3ddHOMXVX9C@nRWLAJzJL*MwD(6AOrf@Ipwo+if3m$G7eA9nZXLOQ27$xP;aOBdbsGA>h^JH6r%$J|H|{yKF&WVWX(q)$#6WX7t9X7);=!&NPA`<jI@cIb3v|G1GGhSn>}p6tUrXF<$ow9fI0^QPj?`)aFjP%)(mguW=A=FLVyxZwfbxshy>N1nRw`C`)Jr>u<C-Dr&}kd4aqVx=eM)9wz?}gKXDBS-|=rqmHbhf5}eO~pFMDNp?vR+$H~~ycJ5NYu9HA#tGqI-n<14$a)2N)==x8s3EhmdQ5><9XdLa9a^!P&M><Com>}w-L+BnXYy~8O$cYglBm)jSeAXGj`gr`kxOoc(kA^2l)96}xhv&<$j7RjLUxEJwq%eYK@L&F0xOb>O#Zxhr743(M5kqYH@c`r}R-#4hm{wcyU885~*Y(x6Yb%6IUwg|b{1Qy=fFJ?LNmgp1P=$A9qb<O^MAe%stGn4?D8?LBil1S-v=fiIz^&9V+eU2{Hg~bqMXXjQKtVb(n9X|nvmrL3z{L|63{A0kRMrNc4jc<2d37{JldM&T7u55w4l9@w#>^vpy@(fI*m4cMQ`8)bw+vOfeeI3aw=76`D-V!)ow(U$j6NX!=v=BA_BO71H`dlU-r=yDP&e@)qX6%rJbg&`{To*oJ?$$}sOdOs7^juof`c&XmoUkd*F~Oq>|KSwYh24<6nNV)nO6p*_3lZxpMFot>b(u&i+#0=UK!2ab?oelYPDc~9hjBd4E4J&;s0U33(el}pvglo1|ICpX0Q8xjQ$-pcC}5&XzKL_`*Y~GMx>z~L#IZpOcs{$IibI0_?D7umdR_K{41HNV16SYhI_G=zz7|Bd7z%q$|oi7I;y&x(1qKuI)~__gKi{&^Z}w+OPh;Gy8C?|z}g}YYkVp*xwWy14myfpaEl#Kje>Z*FOWPihWdcyQ?yK5Y|rjNOfw*7L&LAZBehWe<8Ka|Ck4I^$hc6fc&%P{I7F2;`4Wj>Ti8+pvU=QYStleJXwyu%6}TUhZjT&~NJov(3uN41h_Afj-9CVKB5A}ZZpGbWY!1C7ORjBT&s@igB9zsmsKDVqv_-;&QaO+l8J0^+fz;SXB1=^_9)%0#QrU&Oj1`r*9u|b@@8U8&mzDAx@k_jOw86!)FN)cLbh=|6!`s1W(FqBMN3+!(o|XnfI4iL8<>3Sli+)f1NcbGdZ(u39<Ln1bU3v1h(4D{|8pxH)#bvkZMJrK)qkYgE#)Y{BH*eT1_2FcM;X<VuHnq}0Zh0yJ7s?Mp)+5NwQ0XsOW})%=mCFtw1tcGag9;xyKPP-1>NeFvupfE_YN>GgT3PTBvKu%by<V#~NKjb?zeaef;Hv=WSub3O=Uxd$_*Xd(=G)z2VY*9Fk|W5BL3)HCu$Z%!c!UeAjfuts3_K|`1NZ^Qq8tnd3Git2C)6kYECzG0l!8}Exiykv7adPqw+eN50CXpj8lIrpKny!Pf%EA>%GUHlbeP12fP0I{^{u$qJs1qy6`10Sqv8d)TF1lQQBsJ<%{eGi7!`|irx)~}0Z=jI))4b&+%yx&snZOUF-Ek2!a^x9RcF{tvABFv#!;aSG%2L}2%p22!&lfER!}OJ7wjsEMOQQfo#&%|VGlZ+x8`x!{BwLiCt4WggekzbN<yW=<GJBan(a26;zGF;#5gl?u{ckoy$BTVf=;cle0wma?$j45jX?g9)8u{#6S4c}qA{Wr1#AKu5bJz_8p`_|tG*Z-J&-Nt?0;r}cy4jP#b7^Mk}j(xtQY1M7fS*Bf3a~9eq1R9SK!YJ+TuA$4^YieQpy6XPH=+QCgX!aJM`D)u3o>fGk4Vwe%5Spq?2fk{|N@<dxn0guoJojYkz5}3{uM&nOQjE3-DuK{&@K;pRhnpa8r=0rA9y$t-;sOxyh7b8~xZ0O<J*k1Q$E8L+M$9i@HrMzzG3nVn3E%$Jojt^Z=Mt*Z!rYYYpN~6k^E}zTe=6ieyFYh-N8$!8zzqwh)(|97s;>%<dzKN0=TdIApNNmh}SYZV_e<T~4LwGBJ+CEwL;{ff1bJ(NUko@+d6xbHWd4-q!skLnf2C8ZrDXdvA*h#NF*$w`q6k^WT!wD1A!kQ#wC^%)?ooOz4|T=>I(3CoT$)EGJv+k$X&rh|K0C!)A+o(}`%~8pDo72T$*U#-3i#5$f`|b-g91P)NlhtO>n!(Y;+dWoVXt;L>+_KcEQI!D}||w@%45g`_Y{nf2pqj+SuK;1-nkiM5l97jpPO2PP46)^S^{8%%jHcL~==-8a<Igwtv5YgA#<qTzF35b#1Ph-Q8e;zg8y6lEoyQBxcR!MM6wny0}BVjtvHD$f@T=J3cm<J1|YCoSq0BZnGt#M6XVofyt+TpNBU7&pNYUdK+rX4pb}3ypXl^@ZRlcd3h;5<~!N&9hq$yF(zn{g|kbqziNc0@@^`FAvapCB)pX6v2YdV4TA|(6~AdX>qs)kl{Bp+z4Fu1egwt(^r<@;*aoWJ#}L-{BC@w<}4cz-*Cz$q)c0@Ro|@MKx~93f2aA<qIFz4seJq7u1Am>-=JfQf=xh8Cz;c~CG{sGb_lEtX}hC`57;Iky(pmt>6H1+lLz*Sy*(a<so|VwKQ){$zderi*SIX{+l;B1t>aWliD$4_VhTGvb=t`%$Pf-+1A>Wwv7y<LoPww&BwDO&a_B&5Q-z`MI&`cGRMAgR`x=jsW5STM#1FQ_1}(r7wAuT?P8Z$0R}X>Y!4N59K)in%`i!RK7mMe}GL|CR6L$!ej}gaz)ZW3Lqe8LRDAF{$G9{r?2s#z$9hd6I0y<Jc_c#0T8-sDJkM6{y!U0qcTU9tThCax1gAoG!(;~VUz`dV^`V8Rm`@uE3<#y9%3wjp<xUWV@9AQ9paxmz%!_v|WlTwhbYUC))8iWwZdot(&{}lSY<h_xP6XNV9a#RJ_!U1qGr^67RNOA%@*Tn#!dnRm6;7@OL`)$9%RL7ST$*({i0q+8GlL>fAEBIea|4YE`a)W8^tpB;o{*=4y<@`xeo2$l)vk?m&73sZt3lVw3g{4bN0p7wWt%R2m$q`y?4-P40G!6@-MHhkScqbKK6TTbjK`~H1wxQoTh1<9FA7RfbSL^R{$Nv9flU<o%ldavjx~>%Vg?<!EMST)3E|zf;qF&rQj+1bK_ggz2jt|16fQ$&}GLLf?mdZ=h4>?G&QLlNp*KUR&o1o#ib<pj#qoj$eK`j9F+yh}|d>Vo*)Q^4!EK0a={+<0FU>hT6=vQ)C-IRSKHNuc|lBk??s6Q!qNU0NLSow@<#Q!An|2|j=kXqmf_S3T{xQbFr<rmB4{{S1Cqg$S3Yr7O?im{$qwJx{(o-j|M;nbmQ*G~txHQ|u_5G|_}!^$UY49i~gc+hPpS=*ho*R5`^?XYPKjfGAHM^!@+pfKHp@b6S#hfVYhOWiHl4%|8#p<lW%VV0hK8`mdn((juf%s>LNC#zh_tzDg`X;x~<QU$spah4uC?0^!9nec-U>33!I8ik{+-(1^W-CA3%ZNIpK%O3uX<>xxU6G!T&E(C}?Le3}EwXHX*(L%IU+o-Os{uJF&t}gnMtBFrwQw#?^5n#b^yJB{t^;&f!s=~jUJG;?B`RxcNaP#U;ZTl^#oZT&HKBqLIX?XXKVE=GTXu^)kIGcz+13bn!9wv3MIT{(QSMi*UNL8Bjrp|51p?xb(qMj<I&_VE^`tVLmv<O=Uee?OW32$xj%m|xc)~H-E0|U<7$&Ay$0rzxp1EFX~#05YXFtPx(F8ngizH}2O8DX1VdIbN)wwNnrD~;{W`sSN8k;iuNN_pan<#VaMMPz6@81I8zD4L_94y+N2LF8aRv1ZAOqMA=JoPuwSg{qJ7**J(3*jL1RfDRlgMEd<bU+4#KkGiO`j0h>2zn(@WkZXw~sA<RwukZiH4+LFL5a6zz*}+Nd(tKg$+vma`Sg84Q-JySxdSGkh+z}qR78U8*J}j~W0l9)XS&89PeKGatq7SElVqG<F7mvE*yEK6*A3?r~r$gmJP~owlP4V#P5XVw>`-Z_AxjK2;=TQvZZ7aRAEY3>jX}aaNZFQ*pyw1zr&>qm5H?jjwOVI9VOd(c8PJ&}3DCw7S4>d}#AcqOe(Az{ZG=gz#Lqvw@=W3@m(yg8oc9qULPP)BNL_>yoQ;nW+r07qVpJxpRK6S2}N~4TUvRatgbgeShkD{R#nqJupn~v)OOES#u<j8dw-%MfH*-kFb)Jw9|<EFk1mBa<ZwvF``gF^En|24m;jmhdJuR!x+>C)1LLUT@h+AQYMm8;dA+S=+y%}H9Ge)8mNwQ4|Cu+6FaVlQuB?ByDOQs1|T0(xWs5xL6-Oitk_)7>z)5J>-5xK=jT)-Y~vcXhp%Srha;HUM8ze_T<s7JyyhNFs96)xxq(Pu-Q;b(pFZzi|NxA_t-twS~=aJ2KW)uU&?}npMBKvH6pYb5xvq$&0$x10yNjgc|g)UQvhkNdMA}WMLS?L<M^E$D~=2vFR1Llrl(N-klVxUP<`?yN0|NcF~WV*OU6>_Q;nNy5CN!&uttu=V|uZ&IyCyjMK09uyQvjXauu*60nCa%wljFfS53NnIIwsf4g?WUF4D|0u|n<Zmj;4t8#2HH!i>oXU#UD&U`K;%A_9ioM(O1VF@bv(XVHD;k<>}0^jX0cNGd>%!%?=(ct9n5BInVBo{}Ua5eh-zCIIqzVZ2d4}u)(SFtm&UWJ){L2+VVVO|A^r$Z_62(X`WJUsm+;T?UaVLeg&_d~*U_*Mp#E_;-xffMR|F7q@<u&DIY&oT3krG44&4fdKnkH9o`H+QRRS&DDNcxxgi2B;?fGJ?FG6Qi4w3d48v9bC<EM8SL1i>>PR?oP2%UeeIOtpr>kio{X76SWSG`nM8sw^4FhvtU}ERJ)?j85Jxpr$fFN%FtkO9&G2Tp65skEx38D&wsX14#Toz3+}hjhY2lMh2pZ2+oa>`S9Q|F18Pr&ft+l?H~`THRzBjbS1_)3+tUNh6c$66b^Vx;5#Sp)#MvaajDO2`8)8=Ett<nmNaWB9_hm36Vw^MaTSo<s?{+d^@^Bt3Bo{FtC!SocnI}%0rDCeP%EnjbWbDF9m5>si!xabrqpwK%=Q9{AbY?NsN8})oh{hUbSvkpR@xOr<Go^Tp{N!L7A1FdwY3<cxgQt=^0f;Z<@n9X1R)^X|v`^}d2@Ox=dK}861X!_gh*(3t?jAbsE-qau{45#tOKrFYk^;?6K=JU1#3U$NKOx>FX?43o7Xs`e+Hj!nrGxmS-QACIAoa?^6^CUA6>wq4zr$0l5YXR!Y3|*Z0*o&yYR^jaEP{t;(VbCTN{;plqjz7b|3B5apEl>-FVDRiH7?3B0k0N8RySrTFV_;jxU`60(9FTVIAQn?&d*utXM;VMAK$Uvk&yEhAt1}<`x#m(H4vBz6PA)u@Ce}%)gM?zi7tzza2|i-m77}xr&f!&LZFtb*Q?id(YY9#a^};<VbId1iV18#z?ckF?S+>dpP@nnPN?`Ae-#|p4fS&vTKW(EHmm)Zawt2Qkd6`M)NFGI2Re7Fu#T4vZ?0&&wzgSa@dKQLi#~?gwNRtdZ=Jd<M%rT3@ry>NF3G&Y+qG+(+bfO~b>|=*I)$J!R@a_1+|mnCNFTcf+@+oHR-*x^qm1|{`xQLeq_yNR1xhXfHqWnQ)Jy7h3q*j5t!&C`QF_W5aii!C8RW!&@+MJmcsbSPnha#WnB~tahM6K}tvR(5OsQKH49cYpCoh;Ha(UI1oHEjx@f%Px^3S?wrbtUhpkm~msq&|Il+u|{jVL?xFA-JjTVQE1fz{VpBqN&x&niOXMV~TCbf{=+DOzZwV#MRn!~&wrjD?_#3`B!BD_F1XZm(WD$AK|-zw7vDC&CePtSb`m_R}w4l8n*)8F6}^K@9f13m0HUa)O8R{F3!FeFh5ZGXwROU%B-1($do2xqy0KDS1!+PC)ZM(x7+|y@L?)zL1!`l$u-v$4ha69>`pKH@f3Dj`rp-xer3?G9(@YsHOv7Snv?Bm$opa)6;y8AE3;q?QlZrC25Yb{XHZ|F9PxUP(YOkkq=*``P$4Lva~1;5u+77{n&&bMts-LNkC*gVL~Yq0_6h+nUx^K_c2G@0|N6C!stf~&imvavGekzYBOnMm=hu^59F}`SNTfM8^|bu^v%ZzEGUtY?_)76PD1QQPd|CV@V<MmUTMADSz36J@V+-UR(A<5X9sDVg(hXqYT&_maqV%YzVX<l+(@Cw7&3oJ+mfcz#P@yj-v#89z*A<J<m}d#8w?c#4+R9Xy%8UYRmPUT5Xl0+($4{x_?9~LfB_;o8|0@J7`3sG53$Y+@HBh_FEmBOPHpWvPn(h50VQ0_D;se-?*d)<DzmH2DW;XAWR^h>=sTY)LAyStzQLP{0O`~EB!oqEb<IC>JJMTUh-(d3ow==?d>KG-3<NNkN>bQ*|N6l{Xt4uA4+JxO4|j`&t&E^Ul=$-z69Ss=j|klxfxDr4Jmn@TkwEoC1=2@!7IP7a5aSow9#g+xzhkJyeL{k|6H}@-S8l}0=?tQ(ucBLogFt+7@j~HrZn3;j##mZ)8S=!QO|22tha^{wntixlE-h&QO8f|_UJMq?i&qTamWXm{glU8qJm`-2X+cg%-Nesvq=Qt$X*2@8a6@~QX|*m#*$8=<%ksHk4KmM#ic>s~geo~(uf|1(K8ZxD*D$>=tsUj`*1n-_7>Ha8p#yl1Yt_21A4ngSLzvYaZu8pPt3>uFw+(;v`a-#!_vOZBbp0k_X$wqk_>@kU_&CQt@LDKd=R^9S-MpCBtiY6c3(j~IGff6O13HpT?pVpuhk(3yA~G&T)tzXQX^!eVRaY12JdHfuD^*OxdudGTJ1rmr9!IbNetZ);PuVB5s-VGh0Y90!ieya)C)Chq#3Hmtrw|ut#yO#SpLa$KvZ@IY!n7**Afa04oCMz@=o^BzY1b<fMBpsw!V-Ctu_$KE{a9yRDCGj)*Aw_9B&#0^ZTlm*+cKn;(dw+NvCKGo=JG8^rxArx1^iD0=g_S}lsNBPd0>O@8Edq?x^kl?v;tA>$2Y5MexNHBRHK&WBJG2y-^P)^zx2h4T6@uHfd9?!cM3(Xcbzq$22MIEl5+-ht-5=46TR^!jETBp4~TnakLKl6rILI2SY+9}xr@0|uCD?Gu2F-^r$A1kgy<lcVEsLINN#9t%Hlpw8T>bY%5*6I?4%hO14f@~Yt^0I)oUA+u5V*^70L8Q?M8KX^{pDsakNvtZVv<dqm3cw@-?QKhG0)Mk4Q|lG2;H7Vm5d*0>Ez>-|(Rzan*+Pqhw$2ksUy*BamMeE75x@u(Y9jGa(WRB~W}%+AR(5rf4|8C4m3tF9F3s1g>V4aYfVITlOwvQ7{A|^&S;@lwn8hQX)mAy3>~g`*+SIp&{8z!Xr#4%wCmKdX+y3!e9P{Oatu0t8ksN%kIkR&KBqVa8E#g@7xp6cOC!!Uj76ay^}sX5yGW0pJI|1mkEjI!7x!3dH)-}Re6`$ALz3AKwM<q_8)(P@*f5lBg6k0sU<#ycb_xHKHXhUKV*^M9?6(rn)!85M_@_qC>#U|uK2U3|1T%dK><P4iQVvIb{IoUi{3kG0zW;*ara5N2XuG4N|Z(4N_)+A#OX3dr<3Ff|81uX0et%9z(;Wkih$_!G2G4z{@IL1uxD*y@fllqXNDI3iMWt&X|lic&kPx))k7QyZDujsKo7SRYLmq*7K}L!{FIl^+ricKtE)F|Zr<D>L171zvZsr+BG`0lo9wT6Tk%iL?Maw(!_YU{L1S!MSKZhZm4^32OJ~a}p}VlHPzCfLM{Pg?a;nNe#Q`gf#EE#Aqn%(~g6l}7smm}EX1g7?Bbp&Rkak+nRCj=x%a|9_M%<*##~2P&7}cdB0q2EP33xGK-Z@PLkxhBV*_dQk$eNjmyk~mrRHauSmCScSm80o*w3Hn}0W*W*tZ1)ZO1(*#3QRH3<S}$6P_j?*ZfD*_UMqZ!YBcuLeN&`7F0+gJ{t3*D&RcStI$6`YJ+m^oy>O*`<<(c>b19QwEg{)P&C6f2<bemA)j1*=)@P(LqXb$ak>$UB@b{emQ%mmlTVn~;M-Mpd8EH*zCm<(;g{5q}OI@RWWD>VL{iVoQN5|?T0>l;yB~)K4WP1g6fl0kS63Wak;qPAyRTk<f$s1110A;#=D6+nh;Pprxe)D?z+Ob(xgY-SRGf~br0_YYS|7*Sw32WbDD1Jledvf;$D~exkw_a{9F1`Fbiekg*$KQ*AWQY`Bku)9p2F)<_$~(2UY8z<fzPVQ05qaZe?mU^O=Yf@aZP^nn!>?!xHGS}hf-s;};S)bVY|JCzR&0Llhf_9pPBuAME7)4UWhfEK<Y(9utP3DzWj<z$AML;U$h&BJKNZLwTZ#0k(9m5Pn9i{9gaLb5y|J;0q7dqrF~j`z)$R4_?&kJ8`1_4_wl;U)sO{9~t@vi==GEQm&YPw}+H$^!>+x2sysmB^Aj8Q?&O$%qeMAv8#3Hypu<eDJj^~>WH3Mx2=r^KD`W5!(%JMUQUuApqD`iwra*|c(%UQ0M+4%M#y4^fQ!)FAt`xqlAhr|L43#v#b-jsjr!MKJ=c<I$wSu8-!>%v0$a@G}c_x4~8s`iH6{dmwn7+@|o*#c?Dl|ADdNrt{9Qb}%&^5hE_dzIMoeg?9f%~-gPzM}M*#EMAbs1a!!qd!btXyqV_T9z5%Wo~Dg$#o*#%htE|JoT~TR75Cg!ilO`TozCFw5NM+JH7nA_}VKnEra|R9;I1jV3Eho$jB|z>P8^w8NV^GIDIe_p2lg3-Vr9ygDGTB2-UduYR`DJXFF^i>j781_l%S~NBVo(p!rF6+Tros(K3xppn|m-)~{Gx9`*OJCXS#h!5L@*#}~m<b|bSE6*jCvCZtUwN(~Tr9$c<Xt!7B*&he-KwU=54gKjHUGZUE8BwWInPP^i5_a;=f7Z?*ewX)kZl{As{R<nN6xX{|GpEfwHOF*`j_q)SFGuSIOCd{(fY_~Zp59WX#OH!6*_c@M7$jG&W+c83>B;oC1PFfiJzWCZgx$NAV^yk0P)`Y1tHXnA>jt@KR_7Sab@dExqX^{{caWr}@EEg{>2vAe_jXwhI{p*hR#bQowru_L(6Am0V@Y^67L0v(qyy%91Naro*!&hI%57XHcSk&&io^OWj1WAcfOLdQ9d7+M)x3ezJ96zm6_z7>#Y_cTy*i4mVUGFj*n+}r3oJWcoM>g`u#Hrp_e6z40nCTQX!#gt2mbmw+E33Ma%YzhdL2&7m<S6h6G(-oil(DzApOiBzG4PejBlWBzSbpps^K|UNV5PER4+Sai-}MUQyAD%XtORG;-CfM*G8r8l{b6whgeI87Gg4D1|6%!{lgyfLxwKPep>ctYrja8}k~12IQV*p$k?b0bv`D!XI1mw~$&rj}?(HT-agi3~qKAo!k!li*26{qY^qR@sppyg5#}Yi|1fiarrl&wR$OVQ0%1+VPZ-?Jo0^&wyVyv@T!S*j=2T1!a_0QSwGNJdC*?P`homk)2>e}Y+LfKb2gV?PTP21(N8`lf-i;-D_(V`0dkP4gFP_&n-8!MaZWQoZf(C%S#bZQ3JQbMMrHNar{J7@e$!J-jao0g)c;(~O+)L6WUJYt}~mRbNaVXVU&#dIm749as#3Gkto3qG6tO0q~Qv;!9`$Mt(WI<4euW?yW4i2lE!Gf)K2R5nf|zowih5u>3XnjR;D?K+L5&k`I$3`=ULvCiE&+|;$SSYm6|snZS1Ul!9MvF$Aw7VDHhG6#Y^K550nu~(yi2+kmZXA@=+Hg`;Z2)=OZz35B4hKoLi0+1LLpA<_`gbMj6!c>^s8#y@_(vjq}NwyrseY<m5>Cz_d28X!&Vgg2j!(G75Ts+JDoQ^iKfN5L7t@t!##5wP<QsBf5`lL33SJ1PVn}(moVs3sijVj<>>n`_0Po<km*!STv>4QEZO%9)T5R6$?rsT4pd!~B?dlqWh^2par+)8AX`ds9M=Hd@ZdB+<mCdkFG|KOS^)YX1(1dM+c;pKPYVjOY~7jKE(+WOY!HaZ5#eNo{&fWrKxd+iDZBT=<`tX#^d6_+hFmeh%<RN7BJ36wVpDzDf%_byx1*dxZd^YFMwd#9`q1k#?BB&U(dSr|H@4P>36i1L}tmT{zCjHwRh7ToxDvw#8RUecm&o1o<phTepm+1U2<M4mIfZ`>e-m>G>0XKNAj-DfJfzPeFe6V=oyoPT|7>2jn!Ze;nx%Yr|2)`Ev~smwD^s}Gdg!?smoSm67UTS;E0;YesRtJUnDTiks~Mt>o9UYGEL=;kqL95oRv6mzNM^c^WSVIabuy*u34**r-R)IbrosOUoaEZwvPeuoxJ)z?2;3d3^xGYr!Y^N8kT4r3UeC(PCzkughXW{`g)Pq=w$`J_r^SbRa3>gvwU&Dx}R6<}hI62Bt+qR?j=t;$na5v6XPbUVZKjf!8+4+1ahm4#QPb2+}$q(h%p`;=dhi4($?lPI_h^X_$;dn0y_^pc}JkWa=(<JdndPUJ6b5<v7tV>+c@jFeNH5|9#mrm>xTdGWmdxawkR2<jeonTU7dRA@=7iyDGXx8;!sd%%wD#p#@L7z$d+Xj8edh_urxuqVgezKg9et})35%3LY*anL;JCSkca+f|aLX~l0r<jM;TJ!jOHY9WgINqo3R`6C^D59Su~2*&L3T<3|9=lBDOT%&@(T>aHka~hHQlg^2V8Jk<MkY;q(TlNp0RLg}6xuI~*X>`R18gd$Ku5RDlKMs&KHg!H?of^5}e$LD)tI(v0`-6;jO^R`TCZocHFg{Di4^>35Q1NmydSj@@fEsRTe!}+@YAK2Z6Ki&+U@)*|T=)}B9;pGYvb?{?PqPp!Zq!FZzjf|b^x#9yv^5hyFKf1NU!>uT-YNAcpDi0A#yn)^7mfzc(`hL*y4cSjSQu#zqfIx=x3A)s3AF7K>^Fn`z5LTjX4h9=od8FPkFRcS?(XbvSGS_=+SX30Rmj&z<*W<z{l(Y@bAps@qk(2TZ5B)I(O^jEp(-o-)Oz+Z_3XLou_Un8!61qImPW%mr_X$X`Lr~9ldGRhV04d$@A*#B@)kR;4gS&BBi&uLATmrwu)({DNG)C6*}U#&PZGDju^H(jgdgi$)$P@t&5a3}rhTpj=1^Rta&dg6)q{EVD{}sYQYz5C!`Hm_@};90-u~ILb9TiSF0{It#LTxD^i|})eg-p*G`@Jku>s?Me}&Fj0fjWvZ*r6LpcmmC)v0%1g5U4H)Top$?VklgU%{@z41gNm5yLS`PF5HQ2c<gqu2DJ~wn<bZVId^1)!ItLJhY3i=H(W}NWOQitT6c$&<pB5TbS4liM`=q5`=e|oSLT}vg=2V6PxrdbukbR_!ZCVhtl2D;ZL{h`T0B(Zt*!Qh};hF6im$n=5h3RS*V*_<UW4-mojvd{XXG8U+Dhbnxot!-*xi(DI!Clj&0~T$MOtF$>2r1C*@#X7~V<Jk{I}Xk-XfUK7A(cl@hq8F=v=B9nIq!tqF1ohNCvRsU=sLB_a4))@G4~oWNqEE0vk0Z}bb?u0>MQ8Lh3Zue#2R61*iNO&gM(`5ry_Cdg5TN-tK}s2&Gdk{WXvExR8OxzRlH?g}r?`xLx>mye>UBBqa|tJWjw@apcblI91!c0BT$qt*f9&v}A!QqI!eppRLI^^}TvIqhkexFau^Z?4Eg{UFWsCW0g2#b3aAt(6zpDo%%&j$ekN@PSpJ2BZ=PD)VreeO<qx_-g{{1A6i?`#C;H?djM?WFj?|j2KNk0-GE*v~4RAW;vBoNXR*tWbmHhT&gZ8Nm2)+gtt4jbqo)<R`<!o;69Tg%}IpbxQVJ*@ETAxW_QP=^-+8R99-LMe7oxAEe7mN6z##5za_x+)x&nDG#reP@rPQegp?r!h!=S&Wh7SH-$czKbcHVE>24<!3M-3Ck#j#Tlr!;y9AIP>4P*?P_1ux#I~y~rtn0JR0z-mlXE%>7rMpT|rkfkvwVlm1jB$IXR^0}k4JcLhMlHI!U8}z7R#RwH>JQ>fXjnjZz0H+5v?x9O0F6<L<;6>I|CPi^yeF_Q*x;O=z!X|Y8LhNP6ZY77NEhw{`G~2&vVj0UA-aOyR8#Ne@6JQ$ELxYuhv;C{O6KKac?ZqW;h^7rpOeHkTcbge;NTbk^@G31IWPzw$!XG)BwZEhx}+1hN4c>AI9*>dxf19fDlP|J1BIFc+VyKRbUppRFHWGTixqa&zo+{V{N0n^&BOBk{>krT0E=c+lN=0L9Est<ZhV;nhrmsBpJrORk$+5m|CUybKG6fTABcJ+lO~4o@zcK{mQFdS!y^Zo-M&Fv*FFen6%jB^q+LK!3Cr@aq(^(F3Y1!<msLfI62#DrRf}b*l3XM_3O@qDTb(rwQry|CZttQBK0QP%Y7l!7<5hxMZ}g(^E9J9iv4gm&;G)mdFr5VESQxEwNf|=c-pIrOKbK)6JA2Z0p(YN*D_%S_QDw{%_%O9hl*^5?$#bf}3*E0f$T;>UO}iD4kVk#dy~RQiyKsR=n`hpbondfPjy%C=zbPpb%p`3CEqqV`yiT#*WVj37{tiw#vp4E;88H}7KL(EZ)B-TlK|T4OSwshPZhB0F3x6O2k^!-V<nx{gqGBwGPbj4J5eu336z;N=4|8<Wi2Wt6vwP_DX;8u`wlu%lri9`!@qhe{z7Bu<4az{??e6%%JAxg%*WEuF93=$WVR#8TRC?|i`n?_cgI6XGo>9uafIDGv{;jRe`JJsz;Wa_GHqEP?n@7T7R-%L9-y9igV2C|n<Z4HpXBrPZf%SegVcrBmoKsVWPL*g3L=!u3U-~r43B(QT-HK1$_ng|2>U3H&n#Q2&l}CJpqU30g4Ea_ts#RrC*oDbC*+oS82*gQPAEoCZ4NA4D%X&K^w|7#44@ct!hg}dF7@uL3POuW$hU-$AbSj@o=QC3if?7ipPkyKK7$U??XoJWB2hGfW;Q6JW4Z8h8=iQg?j8gcev+#~KZD;Sklsj{iSe;81%Nc3{6U98wT~gG2Kw+1k5TELq#r0?>E&S~>!utAxkJM*^(UkJK>1ZiAkDI&K*EWA5xQ~ka8ova>j9zY5iAMis7O}=rK9y}8FJ}^P9Bl6;nSRAdk28>TNYn1Jc#bQehn!0LV(yBH0>BvO5+9M)z)M;OFl*?Nz}B(A(~NI$$d1~l>J0)CH%CZ4*uQ@wt9`j*$~Bz5p8eNok37UJC!imm{sONrsVikVECT6Zs(uOoeyNAjq4%1I^VBXwDJNz3+t&8xjqU1s1pgwcuuU?C!_X?><nV?=bWq2Pi2Wwxqz#9PyRN%%*c3G<P7TqcEKQ0^f(qRE(V_2Xa1{4w$0K;~D8|V29%iT|l(uV|8!M}rmZQ2BZEi%hx2xB7qg8lz4XqM{-lAY#<NyUwvY%~<8&G^tBTZBU!_>z8dO3gx=0N7nA^{uG9K(V7fWK@v4|);qra>Q=GoDDZ`3n~oEsNKn--5E7;O^WLA2bu;vMKcm`}+l#9)C#4@DbTLnukdU>oSa6T}*OGN+IN|`409eV2z8@96(tsqe;Mv>NtXLys-+!os%IxD&}3=qHB>cA%0-|oOVBFYilFAL3i(nD;o}{)nHzKm;H=oWu>-rZF_Z#fSul|uC0PNBdjD6g;Y@ihr)ubB_FbKL&PFc0IddOkl}h6uJ!>Jr<5hj-sZ|5e<Q|t7-w(=P~Z9VBR^m(#Bf+FKCAxnvVXp?=Zu5_jR&SqT;S4vYJ2^fDv8vsH$z>8)BtlJeSnSGu+%hyR07kiXoXtC%?-C^_k8Q0Yd0ganaPddRrXkD^ixo~Cx3@pnz;pm2=tAJx&K1QF`ts|TA>L@<&thX!(H})*khj(@{q#XqhPr-gbcO`T~YeYOMPHHPgYeWJ~Zbz*W{Hq&wr{NnAw1N9nyE@?`0&7@O4tc=|f?dxXVA~rF~E0G7wM5r<u>CWPNzGg<BNxp2T@N_AzVLT#|8C+n5m=acZ9YgBdNvEcv(MBz+*g=e1^)8kd4A*$DatYJEi{fx(Koq7bEJh6F~AT>5m95REK9@>c4`@yHAjq*>|NvJt>RJ&t?C$kKf`i49v4ag*N)7OYmjJYOp>rL6jAtTQ%1yo9nrAP~^_yp^NYt(B`1V>z|9Jy4_wcHeO~zD+YFfrG#HC}aay{cDpi%XqP8q(In;D-raU=?BF%zFubbeK4lCH!;)3jqS~wTObmxuUEI<5dpK)<h7+|Bv=N~`n^EG!M!A=wK9qLpmCgRY>x>s<O31I`XEiV<(2}{xrf2^2B(MUMuzrcRJvcDTUeaCw6N@L*LD$6kvvd#1>eVjvhHE@L&y9U^CoT<3bb_2JJ6;>QCz<4R~)?x%tA}5(2`y#r+#bBptI?$>ALABe~{b;NxF~HXlmp-enC1NBsd>R8;uCcS!TFw(x!RhL_K0UHc=~dHk(6RhM3_jKlxiF7$QOu%$B$j`OrPoF-q<tB@>ZU7J96V8W#xy+yu+TB*?S3W3vVOqfot*UTo;q9*@Ji&*C~<jBQl>qcr4_vY?9nQGI_@GK;ea->}O@z${Zx4SbMF1mYu2AQ<fZEN+b>BAoDZALEwsdE7o~aY2672)Nf1<%E7De5TmX8>Ix0369YnZhVRieD{464>)0Oq@)Bn6fAGeh0GA`5*9=?>Hs9o9uBzR3y=!&Ai@MG(IFXBrdFmMieWrL!2v1?DOeSMi1$WanExq36mJxxvWq4cqVB_#jZjS!N8`a5W-+2%I&^`7r00kVPNfk!9}!%8tx<Q#f-i1=j9N(0t|Qx-CZM3_psFsF;RZ)z5Vhij+zAOLe8-i4IpkOMiB&W*J3PDZEa}me%)JwK>K%Kxi!}7!kG#$-2Z5BXyeApNC+wW(IM}Bk+wWP91tnV-n>0Ncs-_oyFw*UOn|JT@12J(EXD0C(_HxdCPN!4tY?QN&*3Kt#KN|-lc^C4(o+sUQ#f@Ngm#|z-UWVO_9<!qN3*nmiKq!L+ylV;v`EVIMop8DaeHNMYywmWBl)Lvw&o1>$;W-?JyzEArokjQ^F$H(!-G7%|TsfCT>ZI`<yEc3QV-(Xw>dglRcQAw4f@4#BrL5J)ddY{M%P!u7iIT#ViYJ`Q?vmIm8CTv3Bpdv7;E~exBX)P7Oh7946|DKPD2UKA(llu}y9j__G9YmJA>J8}RLer*&;%$nXL7C(O@P>_?fztQ`%Q$$6Msgvl^ZpIrnkP10GzA*lflM!cZ@J*%Eke-FaX+3?h?X%EJM=mB*n$lpCY4o%;=RZZ)wp}beDee_hdM{gWuVa&`#CD*K%qfOm=wQ5Bn#jmF^8p3SDp{H6}9>jIPuj$D{Gluoou<T{G%-0U5T%jWn_pzidvipT1ts@gCSOHQT_#xA!S(1#8nIbBX~!%!dCI1rC>hMa%TU;lg@S;`Gqbpo4y_@Zn&X6!v9?9JLmBsy`!whoae+(p{3Z8DFGU6@nX>WN~@usl$HXcl*@tYkk5=z~qwd=*1!2V<TKj`lH#uRoItP;PbzJ26(UC9kGk_K1PnEzY7^8o>OS`@G!yp@ej1bpO$)qqvK{5eINM=2BV~v*saeLqhq<kh*l}Y+-Mq3N!+Hmk|VgeyTo<O(ur(VJe=L+pgRoj^nq_tXc(#~3l#(V!!~6sPH+aYq~$y#sYo5of|S>#5dwuM5ZoNc#0Nx4_Df=A9fwag0{Z|pJ5G<rZoh?QlzmvmE@o86xhY`r%-}J}_5Og1!k-h%X4`>-lj!hpb%8iom)|saSWyrKZSFKS!h@qW5OS7DP$S3#+Di^XY-k?;!>+7&7Sdm@m)YE|cO<_y?7+c6S5SdMa&RFzkRGzqQw^S|D*Mg8yArgNfb1zMn!}o)cyF+epP#!%K}%8|CCze3J4GggXI{EOljp<YCW!j8kEuiDq623Ry-T0V6?VI6!rMKBWr^N+V^!mG#4H8rdof(AjCks<ThblBLVr;(8FgHk_5xp>6hfU(k6v1IZo$p916S4se#+k96juQq!#CD`E(YmEOgj+EWN7J`<H*6Ol-rj}rMtFba1IkzPbB}TGy`9|n*53@?9mrtgxWbe%f!Yh{$xU<8U<w5m5G(x7oPW*D5$6lO=$jfO?OsLgTX|axzt@7BlPs&6t@#<{kQzNL0oZobQ?>i!Q(4=v>%Ue4@S3y+-f)_iPf9vj5MiAo)~AHyd$2uW_{_*`n9E>Si7J+^3o~39xhfgE>?Tp9)do(a9+h-gm9=L*SV^yUYT2H1evD`m4@p!Itflgb@eqHf%<#TrGb!rGPiZq84V8U^w6*I08fsO(;xk;*=jXMLN+`3(P?p>{`mPR5nuRMC%iMkipd=>8l~#eb3l#bv(w^iwLRx~*n5_9VCoR$rc#q@Z7KD9#tlVhW}-Fs%K5DMe=sG!j1JkE1ArtN3SVOTD+;jm^s%T7xaws~0L0qAWzYroF=J<WWww&~1Ubm50`?W(iJ{8w1)mFL#jm7xV|pM0y`Ufja=n@)Y`<LbY@t`U;c;vLY2+UZ#QKkDgr7*K+**0Qwy-Sim8jbPE2V+?S|wrNG#rNOl+11WknI5?W=#1Rpa9A~vg}##9YvM=`pH90?nY@PJ{gBd#ehoQtjj-2DH&A9fMTh#-tw>)3!<LV;MVwXkosl6P7(z|nPR=yR<~fUA+d~V<=TQf(UPjklEz(?V(`RHbbWJM#V1a#Q@IvK$Ac&Vi8;|~%>YfEgyOq;-KTN6lXD(+_eVmuWew+WFg)rBHy%IRq~B<ifFH!OZuhh`=p7yQlmElqyDztqoOz<JQXa>40j(kgP@>cgy4nGeki<1_=0)8-G@}*-3M4Gt1b`ITYEHP_zU}eM^w{f(Gcn=u%xr9&|29IFTekWlb;LO@AictQlJi|Mvof=?3LvGPGmfyD0IKqyUw-*re!uXMsIO}9D_Rv*B@>xZV-?M3Cz1AzEsDO2)Cq|r>(ytp_Ek!ZMvVTca0koG<ev&7&l3?$s>2eKeTSl_F&ehpI1Cu4&w}LZl{3gtPp7k>O5(K?2denN>L;{8PM{p`{P9h5Ecsenxe>D?VY+6%>iY5`WnO5aVXS-Q5?qZB;{ptZHNdH^Iop=1WZ#u%3Fk(B;rjuo(H%U3Lca$mB%IsPpbwknXz(!fyKy^prR|k1o>|nunw=q(XzhbVd^DlDRe}Z)9d_y;*umb(Nt~1zm*a}yN+B>OKj@XGneV^r$+$Ny=|Bc|(bo8!Vv`=Td}ku&7t-MT%wv1=pN;f1l+uz4M=`EZ>c1eQVcYCunbrIJ)z29I@d;E?x)^wMubcrJ?Sve*@SPW7B28LR2?Sn7Oyq=%&ar~z{Y$J2hXYL%RS8w0SN5EO)D0*|j@v;$kZ+^ryjSq+WnX<&E(Y*LCHbOK+|Mv^GHBSQ0M9KzL7m?4c!J*M?GB<Zukc4SfWmS-JPQ-=)QnrZ;E!RaP>B;bzd(WBrq6ieE1<2x54I5St<dNWj~d~8xl}Cf7jv2mD(R#!8STm5?FW;QjCRigQ&f0nAr~PLf+ulOO!0N#(E?V|Zu)_oAnmCO!=S0c5-#ub&Q}0uvPK=$4DW6m>(Tb^CTI>jiGq?XkS~C4mWD9X0nB8lv(o0;!*FY3W9KS2f9zjvLwBQv=w|dt^yIws<H13<bKo!Xe@m_7-jIg}`^&(QzC!1(%D>Ko_Aql6k;*a@<>B9j%x48%oz7RuukVQ#?7P>G(fiDlL4<o;!TU=a?}@#5KSV!*voYpP=cfN{A;Rd+R+i^USdr=I`Z|xw?CXTbnV@vMmw$Nq&oBSy%iq8JgS_obQr-{uW#^`^5d3?CQS2%C=&5rJj^nvhV#h35c{j<Wm6_Y(4uN`}Fij*^lIIz&zpD2r<H~TZjmcu#&x{TDPUw~MvhY-Vr9Ki~vjKv7>`CT7&cD;BEJ#Da`*;?8>Tl6wCs;P{q&B$F(#%oBKc;bp+Zm2JNg8$ZnOaUFD3g#*5(2Z5>F&HABe@r5-zJ%4+vo=Dr5zE-K0=RPGMyFc!jM}<4Jq{;#^8UhM^z`us^286ZA^uw=+jj&>Q^%Ul;f>&P<}?y-(TCQ`YBul-)X%Yqi>cyPaJA=Y}@Sil0)P^$mx@FQX1|DCyrs<N3u@I5#p>*)Pz7H{eg&Ta^+fTu@AY19m(a%?Wp>}qx;npE9uhRRyVtvNekF$gQgPcV`Rt%o$}Ho1iUos<w510gLY-Ubure<m;X@mFv44fIzKab9iMVY`X_?$RuAXr^i2l^rbdSO?;@CzQyf)lr^bG%18F6um;`J`W({EKi39BGsVBC@=OP~Sr?e^Z;&RAa{Uu2Q26VTB*PkUBG6+hFg6FYDbeZIh1gR(%nO4Fg`AdOSA+l?Pl=%q_G38xoK5#*-OPM;*c$Gj6v-59RiKovTMz17o*3IZ>6ktgRG?Wp$uW0?_gwfZ?hkc$5n>KMWsD_hT`A-h|Cn&UZgf_@oZzdb4R#l2P^;1Dk`~`u3Jdx7?CI5le9>G0XOWKM2A`&SDnp0AX$a47$tqxAw&^eJ4)EUJkz#Ib4IcY{2d0u|~hN(<j?>4TrZ?sykLuE=fRSc3M0BwY;If*G*wHs6L*`Q5;^(4jwZ0wki^Y|20@n#3%NHDUm8G9J)H!fxKp29VuzP?;v)~Qe5gy7-CCAo}L1eZkTnv=7Y9sL6>la0!t2u(n6Fx{`yKRoV?kBGp5Of7n*i9#d@M8ZXXuF(Q*2S;`+pfKqgGd8n2{H5CZ^2##iWh~WpYO5P}DAM+@J2+@`$0GY0)z;K=whBUgNvWsE&yxODq7+4=K{qbN4+?&5d2MBT8`B}w(|*y7+#;|KMuUgr5Np*Cb7-OYj^I}_!4v~mt&~$ePjrS`JysrdWMDZIhhL~%P%~;(#w8G^3*Sc!4Z_V+sP)q2=+l1Q^WpY98j!D@ZFKCnqbn?Gf(91Rl%hIF{s_0Yi7*F-{`SW1)>1v0mTjVqr>vEG^{tiLE3moL>}_XT<GKs;ky@rvvWhV@Ni|@GFmZcuREAL=<dPEtBF}3^eFWY4PJJiZU9W9!Zf)E{jGsRF%(ePU=28%`D9$ycyWku|e$SkGMJ}Zlo^D_8$)?nDm(C&yjuVuW1T8etdg#JC6*$Gz%w^v+X_f~8B;__CPcp!tzOrypEp6<s?`->7kto{2soGOv?Q%I6dc?5-d3C#9+g`z(tMe(y2oI50cn%Lb*w<RZpSWJ4D8?1oP&svQgiVfO4{q$^i8mjV%Vkf+rjzi{Ii`{A<?hNhu}No123Yu*@rgDKmG-$=+qp}tf3q<;DmZ^Gmf}-rWBSH840B$x_HgL?Fk7qjr5(@qp#)2{?VW;|ec;u$y?eE-rMs{VJ@cp3q|)o=v>C6PKc!yZ-d!u$ISj>W_14OrmGvD<NT=(@g;sbwd0t~k?4Bj*4rKUkq+VaE!<}!vU?)e2Fk6`g)RwtNRLXHL;vZ7fJGZws)=)sS(1Z`i(4_GF)GmAfZhfn6ji|S>?ZHy^){)%@Hj=l4@YXo0_Eh1xLNAN@8+Hwpo9W&9(5nRV!8Mq6^8wPqTE|Tcnd$do<Dr5uM$?pwLMvq8%zlZrO1g8>B9a5ooi6xS9$fK@ez1pDz#-hpOYL~lJR0<210brnn@;3=>?N*!084g#xwcJ7g&8(}V~6Q2dZ1O>YP%9la>(~U(G_Og7N}turgs%jx$99IbmoXf!&ym8X9l*jF+58fn;+6}lmxXhItPpu!PyPTH~y7q{Gc0kF*v-@g?`{b7O!L)xQpJ-hG)G5-GJA;!tD)2NB1^sTR+^b?*s^uzE!6~-TUk9o#kQ{5e%hX34<>8KUtw-5I~K9S}r&@a<#ne+#4PN6?^j_S;fl1emJMm&_P_S3-mV3pn>$KDi>1#7D(a>y^6Ab72XT-uk^!yzWVT6IJtkr3GUK~$SfvL@r;<O;7{qy<;Tjns~q_+zrHHKH|S4CIw|1}>n3esHP8SA5<>#RvKjE=D((D5rut#yw^3}4z1U_@FAz0=c3v{Uqz5NQ2<L72Rrm1v?E#!1jqxNROEIYkE)7>`g-{#D3@VgNdHfK~mk{z1C<T$chh-`mb-2_HM%cnL151GN4A8PI8^*w)3_39B^zdefx`$U_-6MS&RBskbdWD;blg4X|Vvoc1yJvV{GK20I6&Qj3Df)%iLeOuH54vy(Me-||=GlOoI-QW-;i%KH^%``%<S8|%wCmfC)p_4My6H};KgAEF0y^x+tbO`XsQ)OLqCW~h%E%(h%M5jZMOl?OO4-MeV3r`5t7hS&R1{PYk4A$Lh>PX`Q+w@gfNjN4P}VZWg}tfGa)iC?r&5qU(vnZ7_LE{!#oap9<!<A!Lb^Eg^)PicI_k71g-XU-VuTLUkzwxlkQ$9tjPxj$dW};pA>bo%>n>YSlxlr`wC5@$eo(+Vip96!SJg!MHBx)MCB7$xUZeoKZ5H}U$10uoQ~4uY=nuvbt%S%RZ3&YCtg)R-eo!qc5>Heyx1t`LDaWH&2VaLPKi&$q>gYB{Rxl*`*{LP?+juqh^U`SPm=%Ys-T}>vBF<cF#IytV0^(mqGU}DG4<Dz@E%wQ+j|Nc@j-Sj?fP@PQI{?}!%lE6Ch>{GtbHbRnF2fk$Cv<z!A;<fnDxjS+8(T2+yUX=;xTvmfyq|wbU8s`H^tu9!;bz0dwAbjySKBfEM+)lunC3N(5gc$5<JHx}upwv(Y<vy0#On??L{t;%gygZjvW@gPl(Cs5+uMy^2X0Xm9b%G$Qi>Of>QSSA7#AwRg1sA1(1b}gwkV4E96L5j9b98WO$X(47APRS1M$)1zKowTug4d@?JzRHh4t5K<wSv{24!AE+8`CiV;L?d7A1XC7{`Y=U7=rI@QZt$xyru&tfMG>YUF-uo9D9ejMZP)8W7aW#%pQv<?OS=E^|-M@*c(QcqBxip6flW2m$?o@B|$aY<y>rHf~3oySG+Xw(p{5cjwMl{SI2E-B^o;Tw9QE5{SSz#N0<p8nLUMCevn+XP8YcW|;;R>H*cV(LEc>+<0TGo!LR`iV+43e#ver@^64DsSUB2P46gGp#%t#m0%Y|@2~9K-PqlUC@E)kW2uG!N2$Qo3n3%a;Hq>Ru1eAE+S<zMO5J4{Od@v#Se686;=FaZQ2gZ8Z}8(!$zdrYoFy=>1l}`o(9n;j<Dfmh4E)?;hJ)1^_##7uQgbu~1^_hl&Tuqn!d(FwU3&x24ndbcI37>>@fePP?WLuw?=LNp>=X9~k*1<3Jxsm*1|Irw`A^C1GnIj#lZTZ!WH`;<GqYL5QyC-%-%djOs3a2IW&S&bsD39jlGm#l>F$m^_f&#=W?s9MMlKVHGt$Tpr{s_?cHtPo6k^nHhlj!a_$(YA;+dBsE#fuN=!R$ouh7^Xb6|-|dBz4FmW~=@?(<1-Z1><{nN}p$1`T@{H|}@O<Z&3c5*Sp>)ThS4({>qSqYN0zLI3dLr5wS{K-}1lO0>vh8q9(i5xe1)SsQ+&Rfz*~l%6f;^??)A4r07xhmLg3MSg%na>AqZ@2Sg(+;q+)(-9@b^_l0$v7ofnQN3YKVk{*sBAKX9uQFt@Urm<IRlO)28{>0ssvq3xMu(_b<Irnxxz&f|K>fyRLH~7JMhj<;oSBosFp^tAWX$3jMzigx(5#5ZJrdFos;zJe<JkT|`_k15B2A`81lE?1lpSvu^pTj;$|%0hm|4!(st*fdcWGxAiP?B<kh!qFwz;ykvIJkQt-x79AcAmQZ0&BQj$1={xe+DFi5MecN3tbLaF*Oe{qfhyPb6cgJQ}z>J2Nm77wP5i8~`}aL=-2U_5tYnhw&Dypuk&Yz>GlP6H^Yya|+oIZq7hINh!c1BhDuw4N@pMT44sJJ3NVrkB=8Kn5Sj@7AHzNaGD-p!W4>$l5E6C?zG9&`XX0{mzY8vr&F0P_vcemA#W|OZFnkyaIVt_9Tb+-aTCM95k}|qjI&|_Z;T$jjUVi+ZBVE(`Q+%v*=`(QTrLP7F9G9IH%`kxdQ*n+Ti4A+`VG3ZySiiG1;HA{Mxt5$gAaL$Q68*{XQ5x~%e&%<#0Q;(ykN$Fy|Uvl^To6gF*~O)v8f3oo8?8!0$Yq_srw<^ta|izA?6TkIB=XM$8n?0Mi6|^`q?f%jQ8Ra=3PBxn^ydwP$|BJh76;50@E(f(&bdddYwbcM7?~%<vxS>MXh5D1j8tRLQYv3qbQm(WwhXlY2(FtVuOD7jMsS*Z4Tc^zqAph9>5A8+&^eE?}u<q5u}yZm<S5zl6%CkA*Rtcr^6XNr6euUMOVk>D(M%8Xkf*+UuU*5GGV<`LToxYGikFUH@4e2jPk1HCvvhU_0i=VO6+eysgAO{+Ng0RV+Zs{XpTGCvFJ`$Yof|sNRN}V2UeqtRUH2&>~_-k3jE1Ykurw6Cf72D!y`hYMY2xUie8d2Hl9YL4fmB%jO)7ZZEn@)(Cn$FSI}v*2=~ep<Wm*g-$dOxcC2ekAtY&&YHWAHX7_z+FS@-tW4cCRYq{MqGmclwZcvNKIHdBEd#EjM(*c702)8M6LWsGa5)LSi*RbH)6Un=M4pS_o1iS**%qX2BKARKgtE&9z_?R>r^zR;NU*i(@G&gg<_*Q-hq$-KVATM9P_?CKB_UOHe3k_8U_Tj@>yoFk4EP=><(zG7@ON3-vbC292pvTiCdXG^j^?y;<&B}vRoYrU#;AYS|Ebo`hN;2L*r3G(3T7m_R1c6Xw5szD0M{$3O*tHdge3rx0@YIncKSkzRRj^1+a3e_1^RsdsziXU({#bJzt0FY|G}z<EWAt55yvbaJ&Qy-6o0Kx+MiQ+s5#pzBSc0dj(m$kBzL|vl%y<CyX%ILkV1I^I@1=G@C>Po+FAF=g=_F+RTA9&7C1!K4P>hZE;HBEy=I(Yz43u$4y-8?H7^rzX#C1oGLe-R6e1#&R-ROZaY1t=E?iPZ@a)C0X^EkgugjS)X&{6R{GU+Xp1Hx#m-@%bYnYy!8gMSI^-wl4EQ~*vcoeIeUgL5GmmtqdH=y$yp0$4@`fYWC0$7f@zg;SKh<`FviJiJyaZ9x+N&A~}y)Ipf+>`Kt;+9SB>N9T(NjWH$ak3)DTy}Lwo1&nwWCZYVFEGQZ`(SVmq=Qhg?RJ}PxN^k-fGQr+V`rdx=0%#!;TmW$nW-C5Rfb3WEYyA(vNHC!iGLRAwBq4>BP|gNR$Olc3M=!4wgiy#|OH(SrAoCn-1+zA|tjkrshf8LE>H@?vr?emiD^QTI0yNQ)e+8?$;!A9SDaBDtK%<FN$3rpoImiQqy#SS12TfI;Gw1o0=`aVf;-Ec5B-kO^D#L5V#nI44FW5_3wm-|L=eI$)YYfzbArHJjAy%LiYd8!Pe+P<$AbbS{w+YHpqu)H6HnbN5K}f$TJk=M3%}6X_CUVBqEvf-yV2}b#I=2~Z^*9yyBf0k(KL%85R=%s^YzsKZM*A+FE*In)!#$`kGx;?vGQYaF4_|b;<hz((xZzii8snn~zrmqApE8&j!40eBg7%F>rOZFv^;+Qnmj<cQu}Ef-rg=bOs_j(+rDukLLKPC3F9l}Y@nq0$ON$crPAhJPvg9N-PGDc5@^|^7#PB#v(uVE|syM5)Td;dbX%))qYK;@dwr8)2@v4@~l|)<895Gv%xi~{%E<`JBgHAbX+F=@k2Z}W@ct~_H@HcE-35^r_TlJl-jqT0)68T?DoSbQrYuMiN4`Zau8y=l$fi;QV_@Te(01RqRlWUg(0=l$B^K_)3-0=vqP=PWd3SE@p(Mg9md;IFk`f~k)Js(aA^nFAi#0X8u)CThQqv7)KRX&^93BH4&`tAq5)?U9KF~pY=&3Qe*rxE^3$!uY05#LT?`y!FDwy5JjzaBWi;mmJDYqh#xH8t}+48lYdz_`pmCqspb^}hT)xBR%0ez%gm>%+pCQ0-B@0$Ozccno8g_4u5-1vv`NR1_m~@{;w<(}<>44L)efqI1I-AwI%0ghW-~JIL(8c%+1wARr?WIv8@G%{j0Up1%CLs>l}Aj|_|;z)USV5}Rk-!pES%s29*cL@;1DJkCk|C3$y6V*V2%Jw!+z1sH+J_WxbbN}3L8)Z?^`QcVS0xVDg<5=x<Li2>tI9n*6*?|zs*fh`u;G@=r9(FvKM21g=R5R7XYu>zyj2;?Efhx_X9abB5iB;{QepeCuTp1;+w1#N5+WV&%;6y!+ex8K$VR2N;sV_fkL<38q^Nlm$V6gTgWtKPw2a>SV|WS59XqYh=k2qJd!6Qih^Ly^$(jzE(iVbF3fhG!Gsgl0R~-h@*a&tst`V~flvHc4r}EZOX>5Hl=2l)pOlJ9;F64GMW2>}XVW4+z|D3gKaX2BNf}B6RIUS`v#TDN5aLRVvj1FHQ!kvygt?o2%?EvJMqa&gM|>xx2Bo^4A;d4rEVyOtq8SWstliEz>R48Lh<sX;E@kxoMu5OBCJ~5scB%pxXkKr$Bkd<3@LqWvCj(plYJ6c~ZdJ0p1_^wooYGYghb_IkKEA_r+4&C$BpEbrherIvA)_DDG9?*$*l=ivMLH49kvT9E6WXNqC|dRf60!8#_V`Z6Si`xU;ez-Q8GS-q;15%W-zJT1JujsAiYD3@1fysZ#41hkFHor@pndf~im(^%1ycU!RxGxYI8T2Gb6pznQh1DFqe$B{&qf5p`kK65q;PXsM@LtvA+>xBlir)fO{wP37DKoiQm`^j}ogDBM;~Ts70;&2lL%TDp_M<V+5Y^R`ri>#_)SeM?q4)fk>tUHpKm6P?H}2H}H3`$VbuI*z9r6}BzBreOgYOL%j#`fALrrp6-CNwVSSS;y=qFXYyVs$L{XN%+5kh9+8_oZ%%y9A*)mbsf}3Hn-}_^`+I?mZpBJEJb%~+jsr*Om~79nD0r0e&Rj=sU&>_$*(9cauVm|9BHtErj{gcNsG>dSLzH{L~+p_@|?I5HaT;OfHUMB03!*87?n51&;$I10&OU`fL#q^FK#rCyz#Jsqz%H<n*oB0rH1X!gmQHGl3LK#oR(@AOh&;ZR(IzM*s%K;V|>^^OD?CpsC&2kBNLxhWMG7u(6ZS6oSn4itE&^d2y>;16jzi*doBems(JmWf3Uf_u~ox^!Oxnj?N0x&i})s|m<kM|w1~PI#e_7E2AyWSC?cZbZfgz>D4r~$I_Zi@8fVNBK12_2+>9{#`!U{MJI#JPK{p0Q!yg{>{5<Cr@}}ia+}SX5HL#sqTLl(%+C}{|sb#^Q%;B)t<uH_)*NK|(N#;U1n`T9(bwFLb;?vD|KkQA6MIwLX%dlKm`g3#k>dNupF|l0X9>#0bF-<bqI_{(5Pai0)E<T+ySu5MyyV=(e+`#H0X9}a+wUyP~t@<psQ+1b<b8yA?{3~KCs_-YB=J9E<6j3&G6rl<H!Et9q@ey#W3}Hf)Fp@s(EOX9up_viMU5P{$k<tsSU~Xs8vxIfKU#wm@23bGF#o?UMrqqQRr8|x%Ca#<&;;ICHj3#4D%Pa`V<@25D6h+0XrKf}J#fx|9>-DYL>dIdWL#}BRtlg{EcD6G@qf>aX3-eKY3P;J9+^)vVXk#}cW0eE*Hfn<^()|{YZv{SW9UI?@TM~Y;>IZuTC@GnZVs7DX%-Iwe<E3F|TkpQpdbctE?ge3If0GGpYNEe>Ln(a(WBLmnFqNz&p9#3uCluTNf|3otBqNMSEchvzt^ULr0U9ptzq)b5C7|pJL6G!RV7*E1=HJ4Le*h(fv26Jn@zM**8E_hfo(f#<Dd27|pS@v#(1qrW);sO^=Ia53-n(A)1U5kn6xYK>U)qi&ojaL~1ZZgNu|kd_R{^Lf6qIqmsoXN*-Agj>Jpqz<F#u6)93^yKe))(wUCaR76Ko<1x>(r?pD^63u#7&=Mi;$z9dz?spvu*^R@Un4J6RZ`Xiz!19+Ab3(&i=Vygb%I-w!B;wQL^(sVb7AV0?TKpMpN!=~2cM`-<ImR*DuCh&rxhH9n6e6n=8p%D>>Q#wl_Hd10Tl{Q+j;Lra$t!UD6r{Qz6Tr@1Wk{o2-g7A*WWqu?MU#(Y$XD&7$(E1+q42yYMSmgf^w%rgne9Ekj5+<<<`+d1ltF}x8m*BSz){_4S~(T7`$=Izy%U<4C#G8{ypogxsL$nvKaR4V2iC$Omq+?RqQgM#GF&|s00o#8+y!^+2DJAwRDfN~m$T>VactC$7@^aSwSk3k8Fy0C$|2AH4zgEiOsizyI5?R!zhWdR52<s|+o-9+oRz<L24;2F>3Pp;Y$$Ws)j#>?hUTvw6{m44p(&92RcGn9Z-$Po?VJb(*-K}}}jr=uWhtY>*+7dB+E=z-4P=^fj>S*v|8<Lix6d|iJ#jezmUz@Sqr-rB&7c(?=dr7#64vKZ#=LbE3blL*@aBoy46V1vcu0%&3JsF1sSi-E5*34Tl||9^b-f52B?rZG{mjwwiuw4XA-7)>C?;CiqS+z4(4?--DbIBJvLFigpT5}tR___z)C^MYR@Qm95(ZT8}#c(Ziehg;qK0xiS-VKlgJA0r9_rFBq}C@4f8Jb=r!PZzhSbzmI~@Qp%2`O#l(*LOU@Qb3Foh~RPkowZy3kNyf8qCM*4lanJ{CgZST4`}*H(3(X8Gn5@#sD{h8qMfzPqN4^3C&R=n*cV80gWC_bNjwVMcn#03RsuAVjJ)DRQxqe6DMUqlm*Lk!qgvkv&I}R+&6U-i`j%JVjbiQiutMuhZ0FFiFXc)VkGvO_>0|BIc0qhK2Mck1S$--OzgLOp-jNMhUfJGRS$Dv~7H5Gsg?=&cjT|!r`7^c56)0wExfd4&Ayg!WS|9-lbtL_~>ymYH-E~4TG*jxDC4xE-B~Tp8I^vEgI0Y!7LXi@<#p*o*@{-(#AY@3lV63%jo=mf9ah&rK_XVp<))qpliX7l*OlG_Zav>-7sR(b#hsY!1e!pTzDf=SR*m`ar>k<9tKp~7$p}_VJBD`3`rF-W?_&Z4*SgqfKyZZ;I6t2O4l4;xdaI+3y?5@LId~16JUc+zfF;H~Bj7zh1r?xI{^BS%etZ<3CdF#WZA^!P((NwM^Ch~3gr=AC5m>!__%GC_$+=~d(S%hh2;o6N03sWg=bZF-coT}H5EZ(IBOioEDSY!KGHBa5|Ko7kXkg@DPV-#EC=Ef+d;Yx6d(?Vr7+~O2J8D3-QOT?06EQAdOxmVk)W?A5C&|QW2DRA8g2-NQm`slRwDhr4$R$TPvuid<#q$79waC0kVI$<#Ee(Yg*lOEgLL1HbarGv3t-eizeE#=Nt4Z0M;^n$D!D5>ix_HLFIU|Dp#L`@-}Mlt8yTc;G#$><mn#SpXZxOL_wSqvFXPwiKIR%!BbCe97HW*4ha1juDZO9#!GcFjd6j5N$Wt+<QeR{v-3QzkSvNc(Z@={A1!tjGl}R`*u@hr26V_2nwMjyQN8=eIGUZ2Dn0_Q*O47uj?yjFh<j;&*b|>}Z{%Wo>Sn!na)A*xs(<j2(7BGYB71J?DY8C*oH4$XVmLpNTzvvJfTU;JjZGJV{%@aG<Wq4X3FPJOF{OH%$B4Jti(hxAv&fWMg=xR1W+DIL2BCd{46w9TValIjPJeQg{qfb=cQ|c4$B1j;k7syoj>p5t%_<`CZt%)-~Pu1~1~sCT(1<8Le5rJ^QYtr>MObHTOgQPpJ*ksoyC0QAB%d{Z175Mx>J6&raNh)Am26^>*C`XEybKo3l^p?f%D@_0#DGa8iRlP)xtgbiVYaFAihhN}sU_l4~BvKX{xX44RiGSPs12F&scB=?`L0Ib1Nkcg#+Fcw_*(crx}9B(73w*+gvz+DC4-#C~u;j)zW!Nh)tT3%R5<zTh10+Y>@IQ#Tj-?9E7Xq4W3d;J{3>Ai?dEq#WEsd|%YW5m-toz?m-^i4V#WDj<RdIQkC$QU9Q}w8Qd!cP+a-M*@K=OWY9hT9t8WGYGzCViV#>rtw5&DxAaxg(l1;r2O`z<41~(HLF;dk`yJt6jx?7lQ7pVMp(sR*y?16hAAOq;sm!C8|2ask*l;&ucCmtY8fTPu3e-ZyRobol{)<~EIsn~%<(eMR48Wj1c**JR-euxmBan2oCxgZ6=O$EfMg5EYp3&eex9YuC^yKAjr7+CCgP6@s`DsR?J7kn;i<;%0>+qTXG+pm7a8z{Te344c<uP1j>n?Ol(C2B`G^r1MB;U~ai}+?U9>5LFEbkII55{<NB72wyNakhZJH?RzNv0X1ztRCg7VPq5*+gs(!2qaG96C_bAlnwYj}*LLmY#0$KVYE4<N5S8uSQ!dXDYFEw5glH>eY8i5F2_sHBnR2}S>FbfOM#Fp~s+Dk!^%%hAWnRl|clLL+oQdM})qeG)ptLmB9eI5>u_Iq{Zi>v-`!kmoNSuZFWU@$_KyA05PMa-hefhI#DbOvL4R%kczbU|Hj!4nZ#$zHaBgWm9vdJb(S-re^40mIao)5G|9Vsuf(oN0t^oDaTDeGATuDYb*Pepz(%#0$#)K92Ly$_`$=Hz71w}_8fH%kNE3!y*c(I(~%Q?pDZ^pt{3r<q$#Tmi!v?bVqmV<)V<iyCG@=$ak%OsVu|Wxh4@gOUcrk8{@KHSO7>g|NIm*Z4Y%Zuu)l_9>6>JbQ;%dI@kxyk9!_1yux-X=0o1{d{VNHeAXU+6rFMl7c-*_G8mwxg1_PJmJ}19cg-&~(Q=j<jP2L7b70*)8WrY{gzUf`3`c1!>`nSI*zcrL=khO_gE!iK?y8*XP{L`d}@U!cJ9fvEI5YWVxQ2e5GNVUPCkn7fSS3Sfg8$=I}&O-FW4MsGCa0{e-UoJR+izH_ggz0#6{t;B9FcppN)~C5j9Str+V}>e2et3_Q2C%1-IQb5<SNF4{k0UMo*a!;uN1%D2qZElfBIL7dHF`sYDu%(v6?pkzWoH~R)WB^Cpa%GVMksRXj*=JXnxuh)3a2kEBi=D94SXhXX<o#5CCmuibkUo{d{jdR=z2}0Zp+^2(ZKpm!QMa@4dYh#49bS)t?O6KGbStAcPjs^-W5STD_X62uMFPO+~y~V!1GeY^^#c<G4?TvG|WU_6j^Ja8449y(~mpdE-3fO{RVf5p=ka;|EI@dpR<&4cxJq2WzQfT<RD*)+C7D3><o9NoQ1K{r^pBj0+e(JdS|YZ(xHtMRjS{X@Kl|0U&^UJxhSP;6^;-(XBLrD8$}&j%Shd02Jf7fq#BXo1LamUV`BXgjBz;Cb94{G*>ue=MOkAcueY-3T2VqWCzlTA`eX;Ub|IuX19LD$d|ao+ftm>EAi#3_t;F}Fw8*PS#WzlvCKL^H7?xd7r-m3~f<E@0>oUd;sZYtK-0<xg<47rN^o0ls{?eU7PkH-B<fjNAy$b*N11G~%0k3`~Aj8vPp_huXzJ9ZH{o3`$`~{<||3^vO!M*?cF9?ACr<Y$b$bgD>{sIFTenKqRPZNIXzk_~r)a*|P9Gx$X90ctQU8XSXY-|B4^b>*b4qv;>1uy?Efobr>8DL9+xO%KFwIvfYA>-cr@*m6dCGX|`#yhX~_S=Y^@b=pYA?Aw=63j(67=Y+4J)-qb5N>KjG!zMrQQ@D_Ks}cz3dk(3e)vmPVEBuqPtWMG{--w_DSoruEWdkDUU<z&@daFEghcTTh0s6ONz$qPVvJr;=w9YJlV&6L47$U#K`>6iw$Rq!W5be!U=cb!Y7#&G>eIkAm&9cB3BzR&kLy!fmtP`g169mW5MT2%&jmmR7O!Xuh{1@K!8gQm#9jZ4s0YN3O~f)QGZ&1EUr2`07HpOB4Uf2(J}FsDODNt_3td20<cu#<G->1e^6Q*<cucy#6V<lrHMm#Rc6PTjkv{}6?;X6U_=zbQ)F{J@teeCZFyRML$8rs4^wju4H~jYONlP0wn2Du&!D%B3{3u$lp%>G0n2OAIvqP-4Qu2eXGs+taMDb}wisy1mP3%ah0b~?uJ~S<CT4soYIw0+DRv`2R;;4l=?edY-F56m;UF~9_(ywo{P2#d<Fy1+%nvK}3S!}>*{>4bAU7QvpXHhifnSpu3jv5a`<E>;POJk$j{^__I@uP|+NeV3?ba+;crSvM4mc!&3wZ4@7FtfPiQM4&)-^&%2QE-NNF)dByT^6ZB*u%pY6jf+3;7d>UOd*Op9+cx`hp8x=9}G*>=pA$pj|azNS%8do(Q2D>73v6Hu^*}0K@K}daNPyX1p^fh1pY$+BBj|qMhtnrm=Up_6kGvbgeb~EqsV5fTgf>=vhtKw7RUEv3_Ua-OJAz?d@_JyTU@~kDVb`(;uDSe{=R`#f-xg5mWkd$tASy9RSQYy$72NZIqnU|n1@xG47weJ%?aW@(F-@m%}z(sEo1=f%wONnh)!{DA^G`D8il!Jc*U+{y185J8K=2B`8Ctr%@yXFrB(B0vP-8o>7FkWCZ$(Gl!ASbwB6dSfSP6~Q;t#8g;gUX(2R1}YLg!sH*cB~aVg72N{_KMC(~?-<_^!c&TGG0F*2HU6npG1n$M;z_uBKO;RT0ly6N61++o1ielMx%pH|Kk)4rI>yci&H+_Al1NLV&4x(xr>GcL{eT6r@V^JlnqBv%f;0P>}e?+i}_B;PMDX?JjFM6W*0zGj#hNv_DEd9{w&Fg`WF$R(&Uqm1S+8)bB11GN$Z+M>Udja{0lH)K3ThQ0dW%rp_Agh+zU7dgLHnyj0HBTzBMfgQ#KC-G!3X>{{oZnKpe*EpCk5tkYP;Rds`7{ggrECw3JmI#}%ACnwl7mL-N-d*E%e0ODgXM@=%6-2I>LQNUwZ&F6`>4!-Y+NR9p-LoXXMPC7PxB}*MvpJ=?5}kcZ#nK2Ljnnok4jwvd*lDG#R(42sx~=<D%)f-AA+0J&Ku^^VT>X}NHt;{z@d<2mjPS)^R+<qOkFK0V`?XI>P3uL0J@F2kb{y5E?8EOg-?{ltd~p4O_Tk@_1X&1+ST{5(O6DoH-E0(7G;uv+W9K(W;t68>IT>r8S#TWkw16|>89nxbUi!k@{^6=<-fv$14&ibaEj$K(?N3zeH%uzdr^!zlz=O^5B5{w9Ii4x@a0O+{ROBz*B;juwnd6<f_3n-M^^rM*b^j*jY}}tiE<+O9Frkb5%K}<lVszWWGKSr&K){qPLTBT}K{t|BlT4g*^(0g1UIkFn^<3<X1cF%_3(ptfYh-5Mt?ks4Z_U8CgH~_Ac0e160k4p-Bk??oPbcsNJ(@(3E3zyHBtur(m{DV?GiKg{RJ>2G^ZKNlu6zwB*hNPQUkgIgtn`jTZf(UX791s{hvH+p_7JPO0PCt%YhH=75{%~DC2@3g!1#SCJVg{GZryNNIEPmwY2X61BD0}=>0QaOpx2%hM=!&vzO%bkpM?=2!aFGz@+Rm&)y3G6jO1XzFnaQBkb%mnXAU{)s>vf~j%5aYoX#*e<7%Ca6v`^A+fe$VwIVF0Px#f%UH_jj8~JkZ4(@_5#(9i<RDKU5s5-94^wj4b3#_20k0pH4whMgH*8qGpr_B@4iy7#g4yv+%zf2}tfX!G0k)~_N_$i~ZR52+)9rLGjy|ynU--Dv=7&c9`hK?f*T3MmO%B|7!{_Dodddj<j407BtF;{x?LeYWL51H34N=Y{+G)yzmy5D9$i_VfQlHp@Ao6EjKd1zZ#Lm{3p2k1Q)mUz;_)KbuVDejbZk+$H%fzsJHHNAt;CInEWL-%m*Plbc>K!I}jHn>M%e&c{iQL0N3)9Q;?@-NA5v=!0eOTn!O0z)~(V>IO}j#o4{NHQx==II6=?F^8Za9!&+&o~;1NNCy8KmfQgW)-{8Z6hYIb!Qe>Tie=k=yNo8Z4nnb%pRraS{ph;F1SHsG?Dkc@cWoXz(Ctc(0guq2?h-5d{Yb8)?p_WSmU!*pp`Yf8iFLbNa<O!YI;?Y*m#E8@&;dwJ=$^9dV3=S?;iI!lfB3)-d=WnYqgE1^%#^1#<=Nra|FLF@$I}0*Q<L%6`GshfBX6s`Dy>{`77+0mC_*-4MFb6Y))x=sS>laF-aGon9d@XHo_5!1UD&B@!)bJX9=X%TxHxb^fVVDNyKe$@Atxma`CO?cUQMBMHikFsqM!1!g;+ibMw46Ls^c6Jf(HXyVlAL!#pnXOsBA8oRu>&kL^Hn^ulF*I2yELvh6jx$$^X)HC32fyc>>M_y;xk2@Kx2kbQ!Ve2&L@^)s?gePZI@evCE}`1)5L3mDq3$ZPWFkI9(zbGDjbEZWCxY!T6>pKzqztDpKo6Or|KkrT2WyE6KXT<?Wz{LDR=JxsxC?a%Cix->aF-T<|J2x9of$RLf!hitC+MLOYKKjZh=c@l|<YGr*HFYg;$(RO{Uw!X8n6#AQ68@E~cfd>+REdi9~;~GC047#GqtYH?nM_)i=r%1kuHzs2|NHuorI&aUG8i>Oaem_qwL@ikbgjp(+@59AF{`P(Rn4|~FgCS7U6VOYU2L*XAxJxDum^Q&?h&dmDIk0r95h+yNwDHkb(mIg^0#u!48>7Hrai*f5j7j3+CvceIgVH9C{B1NEs>YP0S-8|c?ul&rG%T+{geZ<(GLn$bCEdm(tjCwbCHMi82}ph!$GrxUio?r;CTzSyv2=)6h$)L6Hu~;*?OttVwT8JdHvQ2|AQ>V3Ye^w#1Tu(NQ$nSfUu3pb(J&Hb8x&P$8)0a0p$5$l%%W6>iu)-`lj4A#jVL@z9fp2po^h08m(e;5JLA^<|I;=(`*3EWzC$-RnW>bkK4b#0-A%9-6yH1EAR7~1`S#|#3jE)KUSOn+=J5z)Jmy4hO?w0f`{=ldMP-1TVO;TO;$b%o>&5r*3;lkC4AzAr^%A>iJ{(2~^^x}U^eQhvr<<w-ts@xol~Yw(4QA^^bI|L-WG9;Ntk^D`5L=$r&bk;4D<M38WiMxmWu12M__zyKiM_nM(r)7bb2-&=L8z+&a)<^{D>B?8T`M%AxStcHfu8DcQ_!9l;C+Ot@`LRTs){SU;V~LMDP$@>JPUm`Ir+t@gxdvw3~L$2WqX1cfTPwnejLG^1&W1d3;$DSbcaWcaK2nB7mNGF{Fav5oe8d;y(r;ipA3-G8}=!~N?~%VV6L%$$}9YA7*Yk2_8rz+P9=FIwS~QmX^H%+ZMFU$@sypHn0G#hm09Pne<Z1hv3<&S0B2#8u$bFCx#}D|hxMVe-siNNsn<hMHGV=Mmpt8;wzAG|FUr^xE#9AXG=nIYV@T(>kJ=i)eQx_M&TUWp&z4e}FhPs`mJm4Sva~U?y=ZMFt2}fdFq!G2Y-;-4`lNMHW3P)`ly;nIJc@ay{3^=2JbvhCXsW1>v{2D%q&S=POQf3JDMWU<H?nHWcdZahKb)8ZfwR_@#W$Z<b8dkdr1O_9&&@oa)8A=_W`1?kR(%qVTFqyt&#y)(W7G{>Di`w?9@2%RdFi8JLQ5}?z_Vc{AH7ChKCQA#QPI=ixu=#*td@+#G_iL968o7%*A$mXCWpO2|DM5LFRyJ=>X?nc9tJIUX$bWw8JUyD2-K2EBBXVy`ABfhYgF|t8XdMjU8W^`oVhJZ>zSf?R?wgPx_ANf3j6ME$5PhINHH(MM0uJpyiyU3FV)sIck?4gC8UZP77;(HMvi#w%b{%kURe>{GsAuIOGW-N6J*6DiR)*<sKDLeasGhkUzu{zzv9bqGvb)s_`%NFhKK{@D?;EcZ`9dWN>~ABf_4W%(3*|oG3MjY4xa_DVundI0%UjUYnvNedU&igUB+a&pe!6UW)iRp#Lc@z6Dw=9qHf(LC3o`c`bI>OiGZ3>fz!EWT^F>hc-eHF`2n*j1iytUa(&GW+H(w9=ZT;jA^~fAP)K)hhByr#!}0YgsBfIWm?WsQWR1`(keo~)YdZ%qzTbs4G8&A>b0R~k(PfWr18;6|6SO6)6hUnJ+=7ue9E>M$>a>ozO|YjBuA&)Vf=rBqS20iFQ<bd#g`k0Zf&B{7`P7s*{?yYsX+${b%RfRfzm!Se*>R51A?mzF$}$ExVsP@eO#ySMIHBk;7nU!JK8lx*+cSOv!YYfNDTA5GUBh8a0G#?O{8eyaMk>4f>{Rh&ZMu;M)tj7gA|BT>kUjI4>=K4C=m_v6Z_2o(mm$B4hRw+=v@nWs;@c%?V&}n3J-AihdB0v?_n5n)QZDC~lsTI3h2_KpL0iv}E_{eY$>E-_Em{Np8$R&LapH(uQ?6U9qQk`a9Kud2sgJD!*Q%SIOUS)ywCj*xvaHySH*E`SbVwsYuWP+Zlr<ypIu&6pm}<~oa?*j$bt5I$^-J>MAxwXg7;HWhVY7#*1yjtVHWJgUVxWY|D@G@lu>0I0r0P*;P^8F^;Xs`)&tIRb%+Fmz?YlFMkdUJ@!E|A~S22)&IuRLq@LmXui$lA)j`zZM#K*&uzK@c@d(Spd6Nc^heix>{U*e<|#VzlN^p883RJeU~+-`T{ZOk9C<m`ADrXDHuIn@Z{GHEc;xbsmQmga*IG;q)(H&kpx)7b*N5kv=JKR$&)91lxW9(FvSEXT3EM2ac-kM0k6hMcQWLx!a<!Vj_b!+|?qU6{Y{n8T8cno6B<t20J~eEKmqezegBFfiEM<*u!_7&V*Wo@<lx{<Q()KcLQ`rL@_<0`Dfh`+$0f208N=M|}N#v?;?QF8|gCSoZz>Q2wF#4R+Umu)gvBy2))o)yn7N>Cf9!MaF<V7=^<#YP`9AEqkK<GV+aw{b32SCO9YV5HB|LFi)OQ8p1IaibZD|<dw4^Me0EpWQhoZ!5962>nl3P6P`7B=oOk{;KRHg3_uBVCRJkpE@h_Aw!>#o3Q;4&FlkO!xC&;IKDu8Oqll{jCY3A3pmE}J=JnG*&nTist4FQOTeLWGgI@Gy+0S!p)OfMg^rePP#?M$4|9s2&b5@53t#GL^xrL*$6_X}D7!@dXlnhj1<(+cSiie;CUk^xP(uskBtW>@cz>#|}P;o6e!8Jkxy0AG}ndO`;keqx0dN}IP*@PD5aej!5a-r0M^t!hlDX;>RH97<-kX7iR?K-)Y^<-l-2`zU9FBtS@^w4XM$LJyZLGin0({OmOz&gKBLMj^QhT#&Wi0r`Fj?oMZ8vynYZdT}V0OjIFY5KuNKfXJd)Cp2zR5*f}K*pN>{r=!#KlD3?eYm~(#Q+CmKRbz9ZXw>})d+9sy07b6xfF~CU04vI-yJ^~WmzUUOa$Lvy>n|VS#o{+uG6ECh_G_A9AIp?TzD900gLYh=BzTGuaESjA0s$LMJ!Fw2gA9_MNZ&MCnAnH`X1pw<nN0Q;S+i5r>@Ko5Sw<|+QdWJUUeWkF0k%qSyfmE92?DUXNW3xP!2$$DmtIjj|Q54=)=&NP|_P__y-pRxpIUFj)UV#^oR<|@fPY{LhNGzh37l_A{Yq6!PgtU<|)7yv5N|B@fr)^<ZmU3N0U7z8TK<|GVczp{`4ab7p0m{s={_ta%n{|(=>+;0DRpY*&FKq$X@pIQyh^Cx;3BdCiU|5XlZL>dmDD%65(cN>}IvFY266}y?t^>hooq4wnq}WkX4V%$9U*;G9Eg-jJBh**|%JrxCv8ASC6&6Avu3oZ_U5XT8gv0R<Ru6q(K7R1FiYAfHyg|W92Bpw42X{w03EySDhk&TDXUzuNC=M=rvf}XG7P+y}V^$PN(6OAu^-8_0{EQV|T|~9TDCWGna_r4>=SL5gtb6_^>W2hX2EIVI7{4@9I1og-3`qvVsaDsOC09<^wTe)4U;5%Iw8RBboBG9QLNVS<>c#v^zp@)ZE=zUB&^guWhbwe7GjTNyyi-Ro{Wnv^{mt35~JgTcMqjnjX9>;Hin@3d_5vxmU0-jKGX1M}w9?Xr}dNpDU(cWg+cXS*~v{ZLMsQR|WZMjfGQN&GM{BcS^0>g{^QrLeufpdU?TX#pC9vGsN|<h+ifgJsTU_x-FeJi&}%2ae45va+X-VuS&olWfWHqlta^(A^i^4d}X>Zw2gw(R=d4Z-_qH{YSA6^TxYu?7IIw?Rx>W>iMUbKiIw`8iPt3d9AY*?us51_kFcyUR85TT7{2MAEvob}?*QiKKGFU3z4#PuP?${ynqOmLvGDZci5bPQUf+3tW9tVN5l`P0cBXKzsF<RH3V;tK3P{K{9^qq@s}YVVL9t-GCtq{PTv+8NpqmdKqR{;ak84@fUUjarA0!_u)qOWfMkp%VGb+WvQ7pW;pBy6s)@}%63cBxoC*qp=u?hmYGe8VA@rBDR9QCtnhfcjl<s3LLrFJwtA<@ZDW}x>m{oJfE$vGc@@DVBJPL1(s0E+xch>)v|QLoVlHHRxGnIjR%udOyF`Q!Y1@xAaGCjy`{)^GF!{!cN3Rl1+@_8WmiWMiW#wIUMe2ORCd7a^ZwABj^?v?lAsl+BMc&JGmPzOZ<pQh6}W7>l{Av5>|kdrWvh#CS|=5&a{lL$(Jeii(9oI^#AdY6vb$U(Qf=f6yaz&>;t=(#f{8v3`36lVCTtK2)^~Ptuj(cQe!>1y@9Z3l{V4`c{2=WA$Er8I_l7(Z*J^v2zzhHHI%K23=!~+k*N=gaR~54iHlf(3X!|NM8!5C~jSk-)*$Z7X&K$or1bSi1FtL-a~-e&nSWF>z6MOX9!W@ULX+A%g3*NrlI!JkND&_uYOJF)R;K+G#}IGPkDGqf70>j(5Hp|;m1x;``_iEp-X@w9fOIJ!vvN#JjkwRfsx0^M_)mCe^y|@Upv|<)4>plyX3w6Lk@8F7Xt4baqJQLh@g>P{p8he6ka{zBnjvx@8uJZP?Ub^VGB&K@W+VeC%~8xQi&jcpii&<j=+kcks9LuXNZQ*fajtb>gfxDURQJr*tu^6U_1`|Uu2JsCS&0W)Sv3Y&V7l)^*KECm|te(Csj6a3_PX`ycaZZ0!cpb(4$ZO{J(qkGtAOMpQYzcx5G!D;czqT{g*VxYJ64L00PI0b@_{L0ZyvYz8<%l@3h|pd_V89s$-AQ%)XtsKJkgS14VCOqdW2{Z*>}n{Q+X}8gU?Z;y&UCBu;jOn>;;ZCmP1VEE1Yr6Yj*lPIFE~5n)ym!vMpND(pjq6k#F<vMjbPhn#=2>P_pc+_=wq$yi$fmu0??`8q8{p5_AQCxJZT8sZEgT0yR|mD?+rd=(+EsLaH&Y$E5=fvr?<q=+M1f-BI<GU)HyKB||8-N6B+g{dr_%;!^JSSN5%2s0md<6Pl8PNXaRL10}i%BNBgQQ^4J=FnjIg7t{2x{x0wcP9Onz>R1);3njVg!VR(M_dWj;wcl1Bq1Sm{+fXVE{p4ZOBiEVbk^AS^s(>FF!sG^W1rYOr(%>*m1qr>C}r{s=iqSdm&YnI$7)IyZveau%0fL?6e=zyxGqdsEJv9d23GpYf@oQOMqQNcEiCB*?HIpyq$~6q!-9Y!Kc!29Lw&XI$qT+A$Ao$4WB<x&CLZ}a`$6SK@xLsDVL7|SRt{*=YV(&IN4cXc24r`Yg}>yNB>*ktt0+MJCMie_@P3IxE)LELsZOM!;R9Z`GQh`;4KHyeFhn~ns1+h?4Zbq1aZapbdtp3@@4DQ{7uhgTaP5|)7QPHobU8kN6`6uef4XRxmUBnbjS!M_*;n!E+|JiT1+hVDZ^G>nI)wK<R8T}hb*D=vbk)9T22_0T01ci=j~E(6ebl>jVayM$9BUfhOW|y2;Oy9<EMRtm0Fb!C_@D$<NTVpg0|}5q_R|@l3i!r-bKhj`WpT_>wQpr@Qt=Sgwx69bt!lmhh9tKvmCe{(vJwLK4GAEFIPht?DDJDCyPnNXtCltDA#9>tJ2&Gp+16N=3T$|U8MD<r-^dAfjL|IAo|thvwWuH^L-x<J+X>^9p9L&ZUjkS`UCII0dspgA;@-HB2h&bx@_&vT;A_)!fVr_8DIizL9(CQ32rM74z!fi&%nL<O=$WiI@f7oN%&2=xk#>r3B9{%{7IL-l>|XKVgQbMz555vASA4<wLc$|UJ*kxUulO?3AgA?$aiLv`PdnoYUYckPE{H@|Tz{H-yb7Z~j=e3o@^^Z19g*w`d^cUVDHvms>Y<Gco(Lb_8IwAE(Y$D$8#AT@X2p>fEyg5&;p#3|)5VGxQ<;$CpVI=HHM<%fbtPD13Hif~jx!_2S}e&RDbf|lkiR_h1OmsPWzU=R>f0GEVnTbJF-6(?vnF(_#j~#t+ixMQJB>Lc!HcHkaVKkGdlOzv%of_3xV-`Gsi6-wS(8F6XM|(Sk~^*jCOcSz#wrz9L2LiDv&YpToDT00tLtKk<hX9zdCBDF3IF;T<*GazBs)8|e3n-(nPvXkMQ7%lzjn!4ndh%v@SE9!vu2`maU`AB>l?e<QEhvBV`-(97dvHVsmm_sYYgWzs0;$$E1kbi`#{TKR|=dI>=z=DcZLZk$?!ACtXcbVx%7CwglURhoJy5<Plp<+H<>(t+0~g6_!m4erfeW*EMMBsow-#R!W}c%d4#7eUAS+!(&|<3%rtuKj4&e1)B!PdE&DRs)l9}u9S24-etuW7tsR^$pOK!=ou55Be-=-MIq{l2(y5)4Mwv5O8k`*?C@9rd(`tP(uvMk9pI01PmDJj~pI$g>z>C?Fo>PH)Irt1Fz-@%l)}r0`r%LjZLKDcVM({CJ6D)@`Fc)ah5&POiw4hK{W+XvFoB{$26tjxhj|LZ-9(r~GMl=Whfs_~eLP1bWS|(`Fm}#x(p?hm`SQ6SC_V*p8e3gPshTM=UGOsb78??P<x6$d583ul%80ik^|MxboZf|Y~Fa+4LnUDurJ4NT`TIr9E+wBf<RV8iJ(I+{5n2a7`KU1Wf?}9v-rN;FEV*m(Vu%xa8Gt_I)9mk9NgX6=a1i{^sRoQG{w&=+t1ar4|c3!ZXgjdmxG4Qg{Wn!?bQPBCp)Liqz7{xtA|MeQ6S_~TmhJgHF{^e=C8jWV7@owuv(K3~Ci8$?3?)#@0i13u0*~GmR!20VqMrVP?0W1_BK~WMI$)E+X{9b=c5#Gb2zr;WYMS3qHIg&6ID{4alDtsy8Qi%JSVp=dmoc=}s`ZN6UFCwzzxr(@1uHVBVenP!^E<qJUti~rIum@SgKGRzA@^g0C!)t#=$2>Iizf;7}GmMMjUcLH-6%!+{9RBj^Qwrmtp?URdk3vO$E<yo5lO2A;!5rmx-fhP18_m}V=CGq2aOon5{1V1D%(ygYlAc4)aU{MF-<il4&pgWT|AvB+piEtgeK%_fQt-_awJM&b3auuwK);kMVOJ4I1E%s*n7pSipI&vX*Q+=^$O=bOZBJ;z^!Do4T0F>7ZN0X&Qr|Z0#5$Q@o5@4l$8>+}9ZyK$Be`}!`S$Ey0hKjiCD}g1;7CLsZ<0W0?@QaX1u-T*h7YI$weKD0;wXj&^}R$l=l=#~BuH&@6S2+ka1EsRuF?v?2(Bj_Dh6XFSp3p4I}}cadkvoRty6y#Leute6s|Zt{>jMfr(tH(J_2_V365SecB33k>wELn*BX-F(^!Du`ifxG(_Kw@g|a%npV^1%8<;c^hxTQ{ulMTbxL3adafCoKU)W6_Rx&uON7e$in%z4kTB}jkwP<lPAC8=pHkv2b75pdlYW$$kJ1uHUL&j`e<ej2Sbt@OIRLXC^{f^b)`K%7l=lA%#d>{!?q{SwxbL&qWN}4{e;LAinJ;NFKioy<|EGJu~iu^>zV}Td+wI2A0MLa`U`IO|?6Pjl9ZhHUkPk!go68Q7WrzNK+cUTc{!^29+ZTNZ*e#x-56-$l&nbYajh7t-s2~32?d-+e$fuEDaea^v@N>sf1ThEZi&rsNC<0FNB62J*U-@hURXlhb^Cqi*a<o|-T1N}IlXxwKM_xUHm8v4}xfU9^4)jy}<Wc%ir12w<Gui-?8a()fd__rQO!KchG%?#Dn*m4+rxYr~ln#dD{05Gx0IhLIfip8gS^-)`wEGlrysH|ayri}ldw|tiHX!ooV4l8>wZ#wHu?{t5U8@6u*L^xWuU(86}b5u1S4(IoV^WSn+!au{EgI#`%frMOtv1%{2-zU?LOD#S9u#~<QY^`5mG$u!G{t6c3uP?IPrc3BexL{I<38-&_V?^rbi#qp%kP#<@$H*3@-s!nZv5<O)v==#*jE&_8A*8kf#XiYd7*Kr+v)KFMs?J~TOMf}kEf*YJZYsX+Q=P5m-JRR38}Fwo*6V=N0i$a<c`X`&RvM3GmDh|ag<L|WWz0?}R%dI2cpMrbtc`)@@_?2ota?F`?x=EeYvayVZ4IMK(PL(dHPn+i;=+2+5dxz4&C{`2BA0fmCNNXCRhxq8b!JZab=P3X)VowDuH7zYezjcxdOjqPj-;$9n$Gf%6o79gG0<7@IVtFm)y5@JE$(wEw&q2Tf~mz3Dh2!J9LCR&LpfiFz;pa8Nrq9U!x;a+H4<UOc;E$p#<hr#1}gZ~5fdt`Qh6Iq8`b*K#`?zE%2Hw`PsW$-YD<{UQ`_FS9o2T1S4>N{HoX6YIV~dHLBy9+EV_{ck)dLwj%<7RhSq7d=<%o!h9ik?&cAXE?FI$Z*H`V8_FcMjXDLH;n-eMp>$P9g{Sz;l#E^vgZ8as#HDNDvb144=mzRFnlTPx4`b^7@)D){y93l@@H-TUGm~&`eKC{$;@u(T~8sl*&WwaK>>4;jgrD$W!hocPSG<UHL`G)^Ydc#nT>R$Rhur&)+B{tGeZeYcfWV023(_tt(m(HeiFakSjHbqj^hcemzPK~i$4;TXyLia>$Gd#?Z&6!<NqeubH-MBsRGiZQ|PW>1wEDDRP86Sebl(d1P_>_{-{D2e{5WjuaX=YHA|4oAheXN@#?A85D+OpI-{faIDImuDh;*~2Owz^-OS;*tMs%_m_S>N_;MaLX7rl7L#sA*CnY37c!paK}Z5h~#9I8kL5-5H9O0FKB|iS}aC_A5Iza~Ao_-;%8PTQ(Rz#dJ5*QVAm;pk+kfs8LKVlsz(FeJ*}|A+Rj@Y}0bs{^jLA;C1DnUjF;be}4Ion#kDPsxRAgRp`WAUD{oR6L}edIyMP6cxMCiHYvWxqOpRRphq@qTRYnhMJCJO*LGO^g!H#?Y#&kiI~zN-)oO6AnQJ)-@DXcNvI#|a;%89k7bJF{Walm6Kk3fd#JCAMpXT?^puzpFiB#GJNl{EPt^Ua7S~?A%YNlX16~E-Pt0&&_ar6H2Es?E8n~F@n-Vnnr$B3`nG_U{+vhGO2?-J-Pd~gL#Gp3lo=U;(sRl>uv@T0%-fAm*ZFjQ+;RD`|sWdQ#o2=5j8M3+oLP0_~tiofqW>9gC2>zhgE_J|vD9vt@xbnlFG$8bWE^jEHS4=I=|HDwtsPL}}Vp^$;0ZOl>pD0*<t#KF*@;dz=R*h|p_I}0Z?KfMNpJgj(K9Z<Fc>^0pdv?d|yGHkU%aj)#=pu{@A6KcDIMf_GBt_!rm{?(QtxWWXhoA(G`KS!bRH7h!xp=ev}+zB1f>C=&B_SB!LuX$|QVsFJQVzAy_UG+A$1Yx@V4dS({9Ch_hG4&I_nNcG=oMC>-iuCedbDG4UrKShvpJwc4-7+Qjt1wfd1oxb1u)VWPOGcEjWV&a=0$*(db+HW$-g}IX2{4c)AcR)@UDZ+B_WGA$@vXrW->uzSS-+#&`(CB^+GC##ju?Igx{v#V>~gHZ2<7KFN$F2XVrx2)Ja?aEOi)tIxtm4&<^N>I5f!&qt6$w(+pb@IzqVNyp2^?B>HLo`e{Xps(fNVa)}I@PnuGk!IJVu(z?@ob67{)N6Y8GB+Pyo4+O2J|!{(AbND~kJ%6M=4hg&-Ze0g)@{rXnw^<cipCE%!J6|uXPQT0`n4=c9|<`czYP^L0*kf%^kSF5*HK+)W}^y;ky^TD;?daw}O$X~LE=8dt36w~c<be%KKf^;+2fs(<Ey1KHqvg6H{%jId+TPjd(T?KBtwH<#yOqFG2#Ujv%sYvSU(UC)obcXyI=Ch0@uzldtr(ZrDs*5CU2={taNRM(8I4Te~dv8x(9`?g<RN}wsM%5GHK5Vcn;;;O7UdRo-r~>gtiJ0HO-%}vpJ{JWCm-yDOIOC5x)uYxF6;jBJ5xdSU#PB{S`)(d0%KerIjc1n29E}ZMfvV|Wkt1@&mmgx+8L38ARhb!8B`&I+G2c^rGP-M2LP^k=47y<@p1T>0<X=Ur`S4i!a#@|;*5pPez~Um@NtD*qiV&N@1g6VgnKNI|{#6+F1b=`8Ul5l6yMu18YGHh2M|nJT#ES%Sp73Qpl2{)g)eOyxhNkfH3vZ~$3ox#axdnBwAJZ0w^Co^3E?{$tS;v!42mm3K{;vZCS<e(x^v3w>o<EF^V1zI~pWuX2_7AEs)5Rw%L02q286*Yvj=Pf%2oJQ6jtkBw1uthRne0MA*~b4Ezc2Ku4aL36enz*+7Xd4ABAMHv`wx%eQ8I3@tS1c&j4pL2g>o_QSMhv|FmQ5XJz|NEq)!`bYxr5}E*EWjPQ8a*@7<L<cW0>Bum5ltqx4~$7qcc?vycf=ut?g$P+*dH&v=!h5(N(ae(2Xf*j$A(2N!T;5=m`u>~1a95o79x@7l46fPf>k-5C+GGWsieaAbDIBo#!hGmC)VrJ;1BsTidKZ`7Z7H?8oas^w`%+jlCA%afYrQ-09&{QitXD+EeU(kK+w9S+9nAI61B@Q%HGSc-;i_5?iYoLg6Q5vD4`Sov{lIg9WZIZ1K}@v{^M!7j^vX_XUC2*aFdUvip0xdzwPABFsz%+q$BkE0UY0y@n&I%teL<L@9i<~?##WI-IG6{UNOnTa7^0^D-?3!b?@qA+IUAr27xS^`~FG5Bwe&s#ogpVI^~HqWQ92FROFI6Q>A!!zn(r1+vUXizzn(5K)Dgq|m&P;q_8AerTp5T;t;kHRhuAB{>n9%;3-aCnY@liH>rSrA(0(+dHzYvWn)Dbj&WUdpeABY<&|ro-&Hbb;f*{zbS-$ocl1LS67|{d<0y0y}@outyl^iQrd+d&PJ~(pLk6!wW*mxA(&GbYaN`WCRhLwHi-oF3%WmOlvXu1>J^m4EK{jU96ZrX2>Qy=Xo$vYY`XBud@<*E;8jR+dw>`J|~axLU^Q&UiAw8;)=)CrkrI+5{JDgnxz8zz9entCt^4iQk@Gf?@x{(UGcRV{Kr;%?tZw57=zoo6XED=IG7y8AP)R$+rP`AFUSm^kX=wg)amgSnr1W@v6w-IGyDPyp$PX*8-7ei#n(p2Rhr285S}Xm)KNvQ?@BZVaW?2;?k1=#<H>RBjIsq3kQl9TlKUX?uLU=<@_%?)i<-!3k8M2eD?f<B?d1}~aKZhK6LrOBqQnT>M7?Jvhl2${z=0T9*}hv}&H#HHqfa1`^OSO-B$HYU_Vnr8cb7&NsK#-tV*^HRgGk$9e|4_5uU#pK6UmK2EpD3^DhVBfa3Mdli_){6XTOO2yNLTn9>8B$c~|_Ljt|R)&ftzOD4yXvAzQXSrP$>cnW9EqPAJ;e4x`re4q=jLMbPLF{*O05xplt7z|AoKTcAMt+C`mpjFO1Z($z=26rv=eT{#Z-?laX)V7w=AFT_P(!9<7t?b^yJ)THSK80Xa(HIE{s2eKOoZ-`7S#*@o#Al=tDmKCk&9_3hd&p1e)60#z3lA>mt)Q^6y`kp_Dn@4@rgHe}7za?e?)z@xeYq1;pFe=BdITm6+Vi)~X`HA4c3W$(%3w8{$-9^exYK#Tdl0wvD<X&@!b8+<#>Px$nv-JMT&fSP?Oq9dKWjnF?X7APJ$||gsjV*E{t<+ZaV@`CD@MXI=fQ~-9c8Exy*^xu<UG<Vzgt{A#tA3E4qrg!`Ix%NSPPnICCZIfB@x>gR_>`ewJk1gTk0lNk=LW}g8$?Jtj16z;r89MITzj{2<9dAKf|)um{{horC=Ng*o;Y;XXAa6yKF?9d`PHl6aJW7POQ}<x%J~|zF0@RZe<M9tD!u{-7o&@RNxH#TI!3Kb08sJ`RqSV`eETAGe<EcPIaCP&Qw4BYG1KsuG|{+1z{paCcKAQCb&VeXHnMGQRNAf9LgmJ5LZ@Cw=u|os1(F9Y*>22CfBjaB$`UN~?io_o!ug6@H9i;&?)wN?FLw}%k{qm3Ba4T~YUQL!K+>lf`B9JAIW9uelhB5r2ONZ=vY=9vYs`KQH0gB^Eh}4W#{eO-vJH~UhQd$Twn=DhHYP_dUvp>7I34z7^-?6b-x?7v70fv(8HBjYIA!Z8uaI*8@Op828bDvEZ)r{sba>2Ukh+Llf{7Fdy@SCx6g}Dd8}SF5gJCK<{lcEFfw^O13Stg{=8T~1(L9DEH+sVu>HJ`=>y&)^IjC!d!ZnJU%xy%CxPY5NQk1pqp~o}SJ3t>(LCk{NzsmlrAaRPz>)f<qHX_l2bTD=RvltUA<=}V-wO+tauY2o-QR(|cYs(8qeh|w?e582_jTsW}z~IbPa!s%&ml!7udV$SLrpEu6zU3DwT7OV_J2)v>Ej#f3>&D7DbaQumW&KXU*~1yyi~i07!AUT8BK|0T51lQ!`!ZuM2Dt2nX^;OFf`R|ecMh{b@imH-PsQC#nV@-|2#0_$L-4K*OZ_;`B8v+pwF9%d(IDCWfI0qA?zI{5-=dQXl&zM)jD=$WLo#rnWsAfKI-wrTl6`RwQj#sGa;S2!LJlh-x6>$CvARal?)nd+D38_EEN)&Dmu}ak%WQVS$-_bPuyKawsR%^*1W^o;EH8*o0_Xz#e?Az+!!beV+}haKL4Kf3xEXD32QFxxJf_C{!4gw$301{-M?b24ZS)Xkc8VzXtx^fsoQmYjlU2H2c+U8J*W2i$w%3oD$4QeliyjQ@91N_d_^g|A<h53V%&EV^++*O^r$&bl>PXQxS6pn~E+j(Z#rMK@rl8tIqy8aoKxVjvmDX+?cPIF5VYX>b<}+z(f`zphNuuG#xG@?v&I-}V94s{#DRHVqa$}A0Wzls}2bKy$nu!5wXd#JrC%v4fLH;p0C5%V7g!c@@Acuhw&XR&?A6;C<-^D<LQnDgloT2!KG147VFVXLgT2i4vmF}JFQx{H{Wskcav~Zb*G=2ed-ZDEF|2#vDxJX2$oRIKdxk}?HjlT((4|CHLXbuC}Q2BCkT4m_892--QwM=^n1k@v+?dRwuO_+s<B`9JZX;4EuQFriA`}x6f1GW_b07A7vsXV`+Q3~OA#rNjRGjQ^lQUvNbg6`&2^zO|!Qjv7CS9_q8S4!nz{%!n=$T`Vc)*tjgibn&k;(H+`Nf)mcl1i^sX7$Ktl=+^s%r`QK-DA*RN8_mBS4{yGB)=W_RSb+j&n}SJO{Xjn6fgP$5zR~{a`txXv!aT5{jNAK)#>yNUE@=@Y`W@~hm0i@LivK!4;3<-(i1V`RZxLp%C7*YB26c+0-XR%n}w($$-MBQD0DUBffMz>+i$~E=G8$oFH{4HJWQ|Cq8Y>pnn4XIhOWHgwI~bWt&zbNjfochB*7cDmhM6orZXmjsl*e-3>$LJZ#tJwzMLXZ={3gpL;35T7JuW4A96frL(*$cV7yI)yoiGlxvsMVil`MLph%X$$mRG$;4aIIC34s!oCRft7v%&A8I)&Q8weI6Zkb^!vewx=MDXxhedq4Ra$>tTEneUCmX6})eGd-&5i)l^IL0c4eNf^<b<YA)_Ij{_L0_C4H6~s|+>J4-r*+&UI?GtNpe=GrypE0FCdAs)qYIj71dO;JcN@sZ)Q>0dp^Jz1;XzL`;Y4wN9QR-tb!?G{N0eOK;iGdiRwn8(bBnk}sjToMJ=tE&e4Ps)eZ*0lEPslm*L-9}pC*r>{40iG#Q16-r2{_wt&P=Hxb;UnD{FNd83k7_SL?DC!i0sXA<i^peb?LETwA@mev61kvFOQB+!=W?7jlk8hlhb`hw%vJzlYjEbL<U~XuCy5VFqbK#ITYpEr(c~WezCsJ?*ASXRXNFcRg}VJYpu5X%;{{=b6j`Zon|_-2zvMEUCCe@kK@R#6OYaE<<!mG9G$RnoF3w2oo*~x$h>SuFhhR^^%-K&R>fvAod*&h?O;F(6y@}x}M3@ZjFwN;);6viM!LlHDt%9pHZV(9G<cdB%C4a;b9L!4f^rnL8tE~Db=<fwjsm<jB^9!acY}LoP^BHQd^~CaR)XG(li@qrk~o1MF(T8YL%>O{|yVJ^I5xjGrOCnF2KZN2oQOxF5aXr*2@^NS@Bb1kOeuExM#`2d52I861W9sJ5GQF_U)F8GGp`8($N)b7_p-`+&iUGPO+4g?d{z<6>^$U2#x8f0qF^9e?}q)h=0ba=dgjmzg=5fS^ZFAY$V@{7*qZG-7N&al9--JIrzOzV3Qm}`utEMBjt3Nq_)41{oS+dLnj)Y+d5ZgJVx3eNy;p1n^UC~9V!yPR25^MsPGEO;Wqq%qJXu7F*&&i!;@oUFi>CRkjmmN5LcZfbqke+B8NzAu{2fHWWx};z?;dz`fjX*@WQn3)l426%9au3Ux2(GuA$$%o+ZUIhaYL!ZAJ!Zp1w*fD_2)h0^PZ_c6F&SxrIs*tPkPjjb|9a8@@U^C4aaI*>HC^pPkoDLy_IXb&D!plgVtL6pJd=6{N!%YbwErSWA)vkY!6g$gc(&W1omFE!uI#mDgMTZzPv<ZZn*rbiBVJY?oR=bd^O3ahetbrz{4F#hgGzjoe`b_t|mdgadHmZX_96F!J%ZSICmhMrf2CtLCUQxp7j(TEu^qW(d#3)4Y19OGa*vvmSY2i3HJ3RS6(l_QCrw7QB9<?AP+-md`mw4c>dSjG2svbxm5;&iBNVky_~9$DVBFWF!Yd{E7<``FiHd``YAk>3h{{`_399gTgpU!j(|YM~Qu2jJ$T*7px=whMVyzKZB#nq;#&BRy`0_2AV+UP0eOwE7HN5D6s4fQMYEZg40!coM0_SFf@lqAZc4JQo9zTM`co=85x4sja`mrLoj9dv<e3R%M+A;@QBa=>RVeITiZ%7(o){bS{vFNI@@D5VtZ+YNpY8~7GyRs=YC?PFT&kI<#p`O9{*UJxdtsXk{5JdcqDTf&R)0y&Gufoc)93D_YmhqJZgi!&1-XcV`&!^1sJg9+D?swAXVwvlaz{%w7M-KJF!v`|7cbY`@~Q%w=W+aX>2BKmo}QscsOa$4afv9e}rH;9LJ8`xL4mo{EI*ahNE#{BIF>(wC<!11)t$e$IV869I-Kh4;$J!eT+Mg^cY?ohI_TuU8H=0+Awl}CsE%`rbeEw(`a3UBmsz+#5QBB8+R|%k7($ci4-0hBBQJ(d|{a2LKu(JHBi*L2xCp?{%9cN{s}UV4b}fkNe;U$7|<JOWwHK%#2ZPCnJYB#VckUBotzWIy?;`pk}N5y13A`X;2$0L;C2mq)M#*mXjO6C!hGGqC~CCm7)FC<)!6xAShj0}QJF8#UxTi*gD*OOgY-Vg8|d+%d!i0L^?R#b8V)Ga0V<`3rH@_`)}oy_HNNRnkk=GOQxiH6aZVr>vdE%<Cdhma&<5pXJRFSf%Ww>4sn{>3!;0@+_rz}GT#zI}*&M=;J!nr;a8dG&Oai6ENa}_$#^UsHPkcsR!4}&;<~a1jWA&O^f`+ypd^|=oF$V;k(SJ+cYNL4{Lr3L^c*2}0hYJ_<L8pbx1A4|8lTTPSV>G}RhHAD{1W}vp6p7JRf%pXlWvVa?jK%#l*g2LmW?LZ?tD=Y6S)oIMOIYZXI74Twd7rUzrl<8MAa;sh#X%a*FH*64mFm9hngqH{7@f_bz`eV%wer^+>yC#6Z)<zcYD9NZh|Y>{CEuFhHYN1<LIuTrSDD1_nko`0YR{Z6pfQ*o_mb3=!!78V9U*n=&Q=Zn-P(A6+imZa6tBs1eDnO#kFvA_t`0e89JS#ceum2AV;R;<T!XG_p3NzDhHgjD-P;C<PCkQ<pwEe<>0Q3WBs$&rkYumC)<%aNl+459gRV%din@(MsnO%`XIEsq1F@aFt!-yub`fcTZfo&~u}Y(jjnN?ZByCQ#y;g%W{q70|0<OTdj;A_X$(I&b5qzdepcJhWKkUw#x?Gw>`^LNPwpz^#q)A|w#BWO8%YQ}yMFxYw^hWRALe<W`GP8OED?py|wTQ!mb4EYYQynDu!!yPVO{Jf6vezH^YnNny`M;@9L^>zzpf8Q24Mquc(~7wAGd(VfqxPT51Zge*8FPPS=e0=x7lf0I@Yv7;7_{H30U<Q`7x)jRS$~FkXF+ya#@A^8rpNr{GX+#BI9L9NUx^I;ubH_8N9&ieS^TAh=5>?CFky`o0`?nwHfT5y1^>U{E?8M+oe6WP{~L7Zv8W2i=W8h8S6X_+7mo3N!}%od#+6nxzV@2=Bp_4E3$Va3s{~cdIr4=`-?2<{4A93GliuXDI*r5rU`!rG$_+c=3|0>mfl=pl=ByHJPtkxebg;JT>@6m+cO|sqR}6mV$XHT)@v#P`nUhhnATDJ;tRXlie*~>Rsx2+mH+O35OLaquQKq&C8DPiZUT=@NVD~eOYy0Uy3$uvgcS30xw12T}uWriUD$GioLQ`Z`5L(Pe9VD3=dq<6v*i&d`9xn@z%J&^u)d}nG8)%ogj%dLM)s}xDz(~=0y)lVv-xx~UH`N|m9Nm$c4C~3keb9!6Nv;pm)eAGJPTULE6OFpJJv5(GOuDvpIax$5Z6h&QB>_(f4wcSE8QEY`fxom1=k1!?%FNW0k;1MDBrUNT2e{xLH6ocGF4c{}eP?lp$Bhx*-*l)Ytx4*7P0#VZ9{WK#yDapVi=d^<V$>r+xm2_uzO`{pjCI{EUb&5K=@;$fm04FQ8PcUkta*g7xh<jrqS+Riu6{9#L5>XYw2~tu1utoyWz^JKIBSylR*c{SP=_7ggAl7Mi8&5LYk}++dmlLbr$GTg9v(H`!Kl%19+h0X?oB(9d%bXPl)%JkUgS~QgJADn%y+;Xa;<>Gfjr8H(HTwk9jCm;ZYOY#*Vyf5PI-+jL>}m%<;B`v|H1mk`|Gw)p=}5^*5e*sSA~lK_eoLr8TEShd$_#AN2f}_7!PY+_n8Bjl(XeOAo44)<5KL-(Y<Y)*rA2LkVn%Q8{=}CP751jv7f5Ub;-9~^>VQ0Xd+w}Qxw+Y@Da_b6y3$FSrT4Cm?D+ws7FW4Jc)W+MGHiW@r1{=e56Gr;062k3YwWP)ndCi$FCHfAOUn&w38n=I9d)4s8&|x_VbZYG8H26Ud{?uIE5nWR15|N_68h4T15YGdIU0FM3hqFO^?Ha^>Sg`eXs0_Zb8GWScNRBJQAebTm5hDIelYw*51O&bv-Qj8@IOWTlWxUptg0Vz9VUgjqAI-yQfwx%|niV#^hPq5@AYYS4k`aFz84rsQcGxGI-O`Z}gt=?X-&RTNOKJYWF~l(nq*3+^I(^Ze7dN(A_2;Y3L9Q3Vg#6kAgGPb5G|vU%r%)KS3gmddY&c9O=Hq#b5`gsEU_`yE?Hn)C&$jwt9&0Re2R@5B3^EbV3K_GIKkE`?xM}<tI@INrm>JobEtJKagL=O{I2%0iQH(4lq()ThFdCB`&F0y##FH^m``(lo|4Qn)BPTq}e|3b9V&xLIel<o%&Yj(BsU7n587@UM9myr^<GFwRIhEn@#cZ#$ELm<`tr=hnIi!k$DYJ?+~X2&IMe<E$;wsdpJSI{SJbmW6-b|SZp0;Sc|(DL#U5SM#rVY{2uO9xC-0udE0@Px=^Oe5cf*qWt7%?RO^M(W_c5fQ@x3!3~yrEYhWK;Rrj4hx;>=BNromb%wJuY&%6mh7Z&E_5|h(JQ*4bup4Jafn{FQL=9Cx)wTMdT-I<C=hKDz{5F4A$_Ivep-5;1#=4=Pw;JfszsBzlfwW?0GiiqqBv%K_Lt8@pwvsrAcW>E}hm>g%NtLKLrxSExA=PlrdGs5dfWXaB0<bi*ce{*}&K`ebt`_z~*Po4sSv!CcJG+nz|@~(ID>P>nte|9fzQE%L6+<3S0?)=4ohbm-o6;LmXy#I_q_8Cv|&(e6S*|&cOPdwIPg(MNzlUJV!&Nle!OT&bVV2-aobr^9GZt|HY*&CTq{22kIeHx$z9hq1dja9>i`~rc}o-(|rz}<dEG^xrwNfyU19h~CAuuWK>pNJMeV~%TR@pH5Y3)Ims-mpdZ+Wf)wR(bxNH_7LHZ)JP8wn}8)zr}fj<}+mVUldR_p(lMQ=u8t**?PGWB($<$@aRca5<xRd_@K|wP4Mzdi_-gl!S{bc(gvhcGeEc{cBo`?5Z|FQ<kjC30OwP($N$`6;(x*p3bYWD>)_{$atXi+`-JA@b8<vHmAF;}G-Z(GA{yUMUj0-zbayAEQFF5aN8&iiu4uy{R%$nTo$eWu8OA?u+&lKR8<_v)w>yV$(w5*q<IYE*9u+F(U;!jX;}nix5Gx`O!L5xTRX7(Ap@9c9zqps;n`}&uglUu?9s5`5!7F}g*gv#YYV5%TFXd6UaR#fTP+Y{Xuy}aAfTabkVJEEIC<g}zgHuotn@31FL@)XLG^f#6CEs4TgMm&`KPN&WD6QybL&cvmKob9%cH=kXAbO_QS;^z|8%e!{+)?DEQY5YD4O1wj9Qv%eZs_^1gv5|YWcVa)v9VV`mxO1QChL0Q-LAtq0evfKOJTs7exV*^ExsF!J8b2&!mCr6UYf1NLGXtnAySs3$zX!9ER?T!o&toIB_)U<qAevswC7NeW~kJn-O~U`HkPwWCM+W{IPg?1hoN5Ee!!s^jLnpyGi)3l7PK*dyK;)A6fSQm3lNzmn72TaaR17<nB+qxC^J4So-k3faO#3U!4>rw1)O)))7zRx<d?(kLgW{}?HO~pkNP#Io#I7I&6oB}z(GMF3{woU+uq7@kmZ{ql0_hX?_F$3*_peb>Z*cgpTeTvd`5UE9rxka=KTUjZHc)nJ1T?!mqszuJ4<rNJ)d|mBMugU%HZQxE%_*{sh~bBBXfmO12al<d2ZlxsL@Ha*f|Uury)tj<AY&$Fc}ww_zqcm7Z<Uy_)FnFR1o?HI1<!L*vk}G1_d-u9rnP+X$g1Fq%-NpA>svBljwoR=;izi;Wp|t70-mzlM%Tk%5qBi`3z*3grSzf2nmMw*L27#&WoZSoOT-ri0_5usjq&Cl2y8AiSE%4&g29CVjsDTL2J0=;S1=D@9uFoKExAY&wu%QBu|(9v&k!2^>|JS7I!=Iv45dsTZ7zZyh=&;ggW1^aY4YrO!M}RyJ3?~^F9KM!D(J-oOWQRo%+FOz$$ubm5RloPGi4Zi3URY7;-;-GuEOR?IqVd3H4Y<J|Y&ICyZB{5);jn62Yh0qJiE}5)lLJ$F6tHC#DD(Y}!dmtk{gKqOPJr;m%KGaHXDHX4_qdZ3k0EyG{;5x<me8&b0{JT{u2qZ^8>~NjD=(>7?F{kqnfhN&`N6;Boo`j&3XFlmVOY_MCx0NOlT%AJGOx?fW_mVHW%kbP>ZBknX+Uzx*dfL9c*SY3CEa4C>*Z`9WdCgE6wkq8OZ>P2$r@SSSaT`C@>7HwVaA75amA8z$2m4m#x4J!-%bh!JGSpSovK;!DP0yqGGyB>!@z3}7hz$F#9AbV=gLexZr0|1nKEu@Lx%boXC-MnxDMlG&^16ve?<rXq<oCZ`$(N(&L4DjDW!4pfx0LnO$QJ!>8GVUHPVqkL|W__HZNJ)UsWdX3Xmxaw&LKkvy$!Vv%6FZH`<u^CSal?5bxhx2+A_YWsWzO`$OpPZZb^x}KB65Qau3&-S9;k3lz)I>k8BcBDQC9V2ti?7YAkG2AFb>P*L?wC)FIY;{eS|C16g2w{y5su2awy6Fc%J`aeJ<HO7hb<m#$$G=v$>-*>dhAbK57#0cM~{*wV^>U$1fjMD`GF#EQKsX-*LfmP`_(uhke%BFEAe2@e9Jx|9vI{k=|kqho>fiief{St$IFB8fr`Ffgb#aW?M!a;mRTf&J!E!ngyRdXGGDrC7EE>Nj=={dde8Reiz#_<StYSCApkN4fa8*iix0!nwd>Bc4ZEYzn9M!AI*-oDK_B*`cqw?;X~8A`VIbGwKWQ8*pLwBI3CbCAjSIbkg^FIL3j1xE@#S-avBMyK$*8AsdW6GUs05(94@RMXIBKB56u#s7DnX+=gmb2JbE*JtOh{U&Z>XVTm?c44q{$3edq~vmgy9h4LT+5-e7VZZF9zX@ABfWo6=9CrH2}+LK&18RO95xCV_&CjT^bm6zD8VJG^NCK!o?gS>c3pQ9PO4)Ik^+}V+8wY2|*)K?1dGkI2E8?-i9x2B)+!r&cU_D%?lcd=S$wp{|!?66S}AT<ke5D;7}E03YVcfoypzf1GM5uGJDhZ2#?oKp`m2<eZsUqq;EVU-lZ2rrv5cwYMv;T#UygqjBS3iCfbE}o2?tIX5)2Cv;s?<w6ubM5M$nc#MA`liZCs?^$#T78xBsOhICGlM+bVq?X$od29HT7AJbX+GhW)pLDj?hy^lIW^f_t~WurmQBLx?f;;|<lYaTW9aIE?VOZ8P@t>li3{j2`k>Oxds-d!U9N3^zbXN&)Wb-cT}vt2svj!!kej&e{s%Mm}8pn=*u>l-`uTN@icn6>UwXL!~>&<a?qt*_j!Z||fE7#;Uv*Y-PYINwV@#+2|3v!=1?6Vgw?j)Yt-&&c4Z6lO9Fpj+ZsQhkOs3rpcDNIr}<p^OyD>Eet3=X53hlxbw7`Ld+@!?rMB=<dq;&UX07SMakKhh^c_2D88uWAdER<g(%JjKoe+`29(5K$RTA$bH16(l2n`!r0}(hzRA;MunZk2xquylO)fnXmIdjkeZP!od}Q`-N~6~pbwpFP#{`wcs%C%L=X5C&8!Mzqy|b)A5sHj+=2~*m==QI{sx2RoPF@KILrgzO?vLD+$Q^bclm0?z>BhN(6EnaGATAKz=OMWY;<VcfZM|v%({d`r0>tO92RhV;c}wPPzm0SlNm6)m!(X#L&n*1DRk|UG6bSGp~*381njSVa_ek(@T*6PA_^h-LKuXVpnUl|kbHlXNhr+P_n2><o;cdS0_pe*N<{FlU~kg5BW-l4h2%hQ?`dD}mk2<Y;}LnW;_%XL_H#Lttzv|cNXGEcO;5m<$yXeD2W;Wys56{&PU28LCYVLlM*;Kbz2)hJgQ|?*D!YejvlD8^v1b{48V02*dt&^&A8Kt=@ZzrG*_`h#!~J!t&9Z#0%3^Mv=HQ3Mz{tW^`jZ>ieMjo3xmw-WMD&c3I2neakK_!u_nP~~MHxAXpA<OfM`EHf0qMnC0$Y=J!Demihr9KiXnW~yeXZsPZT}H2@bl86=6RKqvXqI3QlOp+6Y;BGU*)WuY>q>0UHDP{DDkw)VbBk$D!ooUoWAg=h#}$>bxQc`7&Ab`i9p+q2QFiD#|?x$;kx9V)cqt{{Kw%VDsjRRiIz5(P&$emtpXCc7teF~52zRnh@wBf7e0cXo##M;Ut}oMaLBSjX<*cIyjgZ|67UcD=En7@wd(6Gr!rhq$m%ktaJRxFC}8qOfcLY2jGl;?R?Hi*;!OR=5Xf$$chG8hCskCqO0DDGa9lVE;{KS<$Z@mN5ne=rDIa3YBw4x;XxNU!nuUZJH&4>JldR4uI7}kOR%z?>HJ)K7GYQdTOuJocOtB*eXCRM)WKtQg=agcaX|66k%1BX|QaRzz&&h%GT%%4G5j06c-{;ba`cz)M690L^aR2I4E5?JPW#siJ$ZMLWGLw}rC6zdIi2PD`J-n*L^hkbRPZdG7;!-Z3!6+^<{a#_@7sy2VF{iYBtXOM4UaQqV#)$HdSxfpD@8lm32SWk!Q1pXl0OhPqgZrOF7CWv3d~-0yLU$2<_3AUf_$Gx)F0Dq!9_4lKv3TaA>O|Wq(>8<%7-?r*tNslQu|)m*37tkS^s%Aat0(LMY^C9#ui9udc(}r+VO7r~6~Rk*SCb|k3Tbk30)JtIndz7q<K|J^YeaB0KZrw@M(p!lOQv+#n1hcjIFm#HsqW2}=NI%#1--gnvw3eF%20LB7Y_%1iYHgQ5W;h1G<w6o^YrgEfB(E_TB~tZjc%4y8(GZ-ty>)Ulm(G~RBswngjbM&RO^_1S8(nTKP9rBYFA^Jspov5<{j~bB$Olyr`%)8f5(VI*CfA4MH8ClehjBJ`sP}uDo9nd-N3e{I}@)8SRKv#IhUm@OHbPurX7#PMb&#m7alGnIM)Pl;0Y7NFJvBlcD}SY5&oAjOsI8_MLdJ?1_4T?d6hEC)nC8*q@>lf$Q0Wo;Azhd8>JnJL+5BL1+zdStd_ndlAGPQ(GREGDKjJDTCl_ruHz~_jS+GDvMpmCjxbH4_5T(3t<7y*XPV#VSG1B!H6S&?i$pmA2iby1NJ0%I(j+CvVYpo&*(70s0E~-7S>aY}Cyuk887H31Oe&LgCf@zHQ$<UrP1%zCVgCUA7v`7jdpo!8(;%qWH8T}UYMegj^trtE_jz~Xt-WFUUcW%2Mijo5>U*cseX}I1eH{~rJ&lmk?UWQxL!L~qimZiF*BK2Ieix{2m6QssJ5SSu&gH~4kc^cmF%)`7OE`6(tLXA;XO1D#FpNVb*kBAc4I$2UzlHT@rXng8GBp*MZz`fW9=4~w-RS3p+;GQ08SriMW&|C1mSou829D5`tX%(8(^qR9+aP?hLC};jLRl2+p<X9HAR?|+NbD=M;zdfe#vJmp$snhLpt7=c^kYg3Ae??$gh<ZV&fM-6&I1kph4aNqBIQWX#w&>lbc#{VMSEM+-%ApZ4IAAGaoKGAAL-;qxRlpyHbfa4{9FRtWJV-hrwPSRTK$99XbK1zg}n}9Hem;sBrJ(M`kyg9PWxCe-1}>y!GCX2k)Xf_3Xc|jEBwZU<(UA6P;bwKyTjvRnu5RzpaC8J#LvOB5Hdr@%PVkrYOnb)g$^q6)g(5<n`wH{ukd1*u;6N`vAs=5VYK80`tN%A@-(RF5BjB@UXr|(;Jrxth3P0ARfHoXy-as(2rN7bH#;p2R<^ktlY1>Jhu@$1(Ul)wMed9N8|LwK`ad$99t`3Ky+&8PjIz!yq6poWP>;jRN0pck!U=~9J{4uGZU@@x7V)&RJ)n)3-@iQNF^9YnQAY_GQ5uw#scDtG&HpaE5xqe)=E~&=;~b`^iPG?likD`JKAf)R_CbV{G&O?1`e%fcg6~E&?Gw*AD8gImOP9ui(aO1c{(T#^jQ&BgSD8CjfzmXC5jx-)YG<m)b122dbBqj&iJ4PKA=cne_+`JF8C#Uxv3q5M1lF&JGJWB#J!fB}5Q7qt?TnLrSr4(D|Bfm$_E1Lo#u%`ORx&l<eGiSjFU9956Yji|0NrcMwRUbYUW%!R-^||E2D%xVEv)>8SDr(B_z!(EC#L;;3F(Sg+$Hk5hMB#qm38Y?f(mm|=Qwzhs$5e6>VTrSlo&rd!?GeK;zj;`&{vKsdOYkW(!&Y6*@(A%p`*`Es6S6XG7|(lHQ1ZnlX3f>813NKBGytcS++OrbkeAp1hOVQvWWr(XPV^kH}934PafH$q$~H>Gp4%JXj*oc0xIU|&hfc1e&R0r{&dMK_3?tzM0TGiT#Y?A{pj@LbM$%EgneUpCk-62DT$8#;id(7qz}yttxs6N8*CQ(f(%#~s(j+<dcQUMN^IPY{b85<>*@gfiba*+^mnJfKmC*U>#2w203nm4nwir{Z7QfuJ5k<weZav3vfQ^{b4^(JgG)GqJEEY$z5C7E{ulGAAJC_B_T7hc<Qd0K*;Dd5OU;7e<k%m5#=Dui1o%E@l#L$utnyoY<|ryPua>=UM!AUP=DTc|JUnD<MtC6*WZ6DP0VK-TeyL~{tom=bSMjs>sDd8|nkM*MC1J8)t!WPG;_0q1hu4tfW*ZF)SWqnrFf#0IMTAqdFVl^a^v>BJj9@?={e|Lk|6bxwBr%e#YPe)i(w)JHm<F=fpa5;lLIyqknULYc2n~BmiI_GRpR)NaEqHpp60>pXVQkkBG&pxO#3y7SdXDSv5@||RRk)7+W!T5HG=$%#sR4$LBAx$?fa(OM(d<p$&QQdFjw3ePqRz%xj7j<$fC`(qoqen+9FfguV%q;Qr-xqSL9Cp`edz$d;0dQ$T4eVpi>p6Lce1&?4_e&zP(V>r%Oeyv&>6!H^Uf^wu-)8hcHx9H+rVK!78-6*h(xN_Kfru`NiGXIH5lSVH1QW`5&RhT<P#rRSEo<NI+Do;jn+Nt@t;yTqaTjwCz%2wH8o4A@-e4~Drv{;<TIMw+m!bFfkQV(h#K8a%pl+l-j?ZAR*=IsKS0j?XWwT+FYWdaQ?~k86p>N@=*IpTcjAenXhJ+S_F{$=ZBNZuT+}&Y|G{jYvO4M31k=-FomwF$aO^;r1wDpX3v~E<<n}f(Iq}|gBRI*e=ks70c|X(_oG!No+lutb)9Gy784iHMAkpX}n;A6an0=Jlv=g-jl7^gRHO%>v0fCHT;Cr9@`rqPvcuLvJj|~In9Bbd#zLFI^^nDUZ!by=%-6!`@V1VefumohHdd+^VKW!D;1oaScyj9&nd;c4QMys>S#`$9oCsZ_(<8MFXf5=vNHs4TD4LQ%?lEjB$joy*N_&J?%VZ(V{7ea+hX^`iY)?w2?B>l(Q`jnZJBj|D|TogzLjsCqpr>}h+sJkP~jl(G|wwEOl)0(8lY*xX(FzM7yOEuE$6NQQ@H<3F*+DrkfX6LXIcs!9tPR)E*DsdUN%_JofFQrCqQVS>2nrtMPUY6~uw+6jNo4fCG2A_7@if;95LM%qugvXM{A>#tWKn){SJBY9UyV<9Jw)u(h2|+9HEVJ|?L2V>SoP6^;G$Wc_qw$Een$xFFiSq)qotRD!LGGExek^!A>ZKPpzy*z<8Y0jm)z4C^&Un=Cu_k2FHMQQP`tflC)v1na;x0ep)A)6IB8>k0nL(>g_bGvZ3G*&ekJVaWZL`|ctnI@&4kYgMxs&t7jN9tWin<~>u}S#CIMIL2C>*(*(CW_+`Sf>N+6OtQ@<=^;!KhCyQJ@#pl2Q<&=zxyEpWtvlXG{XrDkLKPm_#|EvYA5%FL(A+q|wr|J-ji)7*90$BK?=U)R(}MAZu_s(9f4q9wg)y7#Qgccy11;yHAu%t%H<>hoNwy6N;IS(S4EyQXEX(tiK2nh>FU{fNh_6?q;el<YridsL9I&)JWl$i^zXOlQ7J)p_M5Dm3hw4;*1iV!!|uSi)L6EWra@&x3z$==>?-_<Fy(|?0L++husdy;{C)O@Ch9&LMAx<LJgR-EX5oXsPoMMcA1Wp9!!M#Glyp$2`u}uR+@x5am98`_dX(~_)IHVTD1I>G#9TDy)z_Hd|%Lx@X$AUvX^krDRYK0ix82Xau1G){(s3|&EJcK;t4Sc%gppT`wmvIB!N2+_Z+GrQAo&9FsnUt);P)mR%vT5eshTFS%YRf;f<h!uT2F+#l9x%(Xn;%HSG~uu%99~T1Xlxaw(Ry9i#obr|vi1Yv3}UvV!t#WWv0iv1a9iH2Qs%0o^=O+YTE%!UJ(6%z#c*SXRsmV_)9kjdBwFvBmEd#!412hf$DN#(0O90HNj?b{-TYN2N>q{!;J+;bC<4Y(JtLuD3Y}+A~|L^wl`s{X2z)z$wb@3^B*X>FF<$!~8y-ZY3%EdKDnC`K4R8>TBb4{YP55BNG0ejO>~mU8ov&2tPaA7^i#Ar6#U{VS6G)%W#amkQvCrJsW=r8%zQZz|z8mhR{Os)PMRcerL*9P<W2VVgSAV=X)PHiMk@>%}FHShnQh*0sR=Z$#E?l%y>*n^wBs+l0^0|*wU;^gybDhqF>A%KTGQ}WEo0{{?7sz@j~?k2X?GOX_%C4Ae8VX?J}|KqTU0Wh145o@Q+3Ce9i<zDQk~y&SFxbB<(b3)&Ag(#5>MZuhKbD#MvlC{F@Kv=sd7?$MT>B26toxZKEiS^Lc+(*&}pe-W##ypID)<h=nKyz$`S?Q~3q;4fuhyBs#DJg7#RYm@@1M9auj9^m*^gkSEUU^RPx)u~E@+ytLdUhhzv-JGMeNT+f)wt&8WY%LtPdC>$E$!vy4o^2YD@5I><MewIr3EV*$lst75fSx5_$<W$IsX2EdjbmfOG;)D+hSUot#^z0u`Pfd~kzDs#H<g1J{lw)y3gbS#i9$!Bdag%Jw(odUR3JTNxA?E5^IS~R|JG&Kf)nn;_e&H<&wj$|=;>IRPoZID`!aOfHg??4uq7X37g!Z?SuPVTpjDRAX5fN7{MoBAXX22y6TpU}T53U`JJvyqHy0gT#Nh9f7Pjyt@IeNuxgRTPs&#BUrQe?TS`zpRnXBe&H?JNy1c7W?bX~?8`<{Td@GI*tSXSEh{lw0MDQ|<)ZI&Y2{7)o#{I*$36)1@bMj`iKy(ljwoqEu)b|D4%RIZX}-5$#<PhQAcU(~%WWe@wZ<!br~yDE)|tGFz9P3((66@$F+KWX86mQ<kyD6jAN>?F8<}fza3^B1_-=1KUl3R1)2Niu5TJ4)${QmFoQ@m|KBI!M<<hk!e2sPAO?`%fT`cs0s~PnpmgL9KXo9JU|6<Djw?Oq~u{`xdB~pr%3dYeSgy~H(Xw+EyP&m4PllMN|^#=u%Alky{rt!oq3JpSGY?=MC7_cL~Ee4C>@Rq&+KU2VsYduS~Pt``a!pH#wc^$5+^#B_U~;sF+dal!RbIza?r#AfA`Kx4&w7u>U2Twl~u+=qoODIhw#B>NRK%^x7^<w4w@~>+r-c4Y`&qwExAE-<>K_z%$185XL3gN#2x3f%ml>3BA9qsUtDr3GS(YiTpW?!ecm`(H%K-eA)6oLo}wEs6hM?qF(~pqy$HoaMM(?_tZODfc{7h}WxyU^juiiebt@*F&y>VVMnuUqQGi28rAJdyI)$L$Q&SGQ6;UQ>?>5^>B}C8!4k``hS$yvS9JW4!jdrEr)1xpIsjGrTITfcL4>}ZyI&3xis&>5n+pyPC&kqmj!1IuvTj~i^R)&g*rD|<=40ABmkdP2q3({aa$gJ$?+`?sr_lo>talCoib`u(b>NkZ}xkwV@1F2PrJNRDs5`A-RBkb9uA^B6c&gyb;=?3=qFXZ7SYmq;6;Ux4Pa4uZE1iido3BypA4ZK>Kg8z5%y`N?Z^?7Sp(ybK}r{fW2z&FwlJ{1SjQ#POK@{el5qEhfsq$G<O81#2t@6ifCM7Nh9JM19yy5H;!`|&Pe=Cq7$Klp$DYPH)M_Ig~4V7UDdKlPupHUhhlr-&iXTsQeMTXTEiq!^+B!>@a?rQAYg!Eb#})HtbWnM#}#sA36LLDbBDB%oRs`TCo8=PW349wx6oCmME*<?*>PfqtQKrBaIwUz4%|2UJ9Na-nv25{t?x;|4=;i&?`_$j$bu@G0@tNMPXGVgos{0NV-s{VPI^e-{qqKa)m!62PExNZ>5tp|^AXkV!z-m=z|N3Klz~S$;}pl@pDnN@1rq1ePuU`YHLhXhG%zD~u>m4pReCt*kW?fGDau)m>FtY4X4V*!t!l?KSpUJIGG)*jPV3;aS1V)XY^27KTnNmq+EAu^A+HIzPVMLC1s(^-6L#3^4tlzxXH2MYQM3zMm7aRnfFQ$T_~xy|9fIj(!@rQ7<Pn{0wG$hV75%;Nbq*^YoePYg1Py(W{AdU|r`Z8!A9k>paA$Qv-#9%W!34dS>D>ViBq&low3rKBb%Xfk7a*b5Nr|SDt4ebBX-3E<UckJ?B4bgUT_T0=cz()^dUDPmd=#w3eu*a&Ab~1zf@-abNPsJ99xYLAZ$8rHgZt6hhXu=sL<~t9lL3_%f+~rm^|0<Q{g+YReT@1a|gG@fdH&A6cntQ<E6plA>w`EY-2DYJ=I+%)zlMH?tq<E9mq$bazeqxQZTq;TaKcQ0guFTO41nW4ko<CM@r(+No4lyJh_|WI1)Y#Hg6!RkT{WNTLHaIigH(Y|rpx!EAlcUJd2)%NCiYnC4%}yfq(=6a!?sFYo`uH@`$QF$%u>K#06s3tE=_F_j8QbN&-CjK4#mWLy>C7sR>JYzd^6?^JTvXlg&LF9%%j!xakCqytX8=pJK;^m&GfD0~nS&I3*m!Krn_`f|KbyS`Xmt1ajTE%<^~<to=6g08M!;Q{lk8*g#C2<l^@PN<M5=+gdn7+GRrarEn6>t$qRBsKbINhr);2RZc&z>X%>Hm%GQ#~riVYP7Q>!amkk>bGjQ*W%^H>g{iRR2BaQ2WM(JJN(i@yjr_ey}eeS_YHmS;ofXy$52YjxBqP%7mnI{($9`tzrB2CEndA@{Z_t=OXqn;AvNh+|28M`BRi@TasBp}6vHb#<E<_)*4N_dorU_^w?3xCV`3<6f3NXf?_<8(z|_FQd|rHn&|}zph`}B`EQbY}{?@lLfuyBF2EDs$YB_$F>#gz^h>l7Uz~>~zK2^zWia(SC*nx?e*{hL_s%~R9uyHN2G)|>ZNCays2rmgnx;Fa5j>E$0xKvi`j9_QDD5X{_zdA`kM@k$7tt2@8J-G-3edk>f37@m1VFA29H&RPV2VhZkl-z~l<kCWElm%!TVa%k$52p6+_s=rITM3xkJJ|&H?Hx&OA8lJ_v$!6c4csHm(P?WqnPQ7`<J)roE9|-7jaAplV(nHOyc6EkoF$O!Rp>_ADRL=g*7suSWgQe!*AY`}%ja~7%>5PS_Z%<Ece{n()%MxDF}7FNvr^v=7SZn^UXuv|7fH4Q3AHW5CEIEZx3K_f(t8;6hrmYqeJVE@Yz~{Pf$<Ac?WHz6XC!E{^(7BVe)2(n-1(*3Yqg)OsSO%=kA;BHvFH{{<Rj`AviPANr%h^PWl6>VJvI5R3SWvJE7wGLi5LIus~`&ncR@|f1C64;8k*pT3p99H_k*?GRM9xy6@r$G>WyG`RF}*}+0~~=vD>}!%3sRq(q^B)G5eE<r%Vc%M??bsV_X5nj&QkqHQ-3QKq6Gi0vDd^gLWXv6};UEr&u59DswJy_UX}FFyCo4HUk9YCH!bMBD)Fc#!p@v1Hzl~Fhs?+1E(A?b2FX1dU&t`aE=p17s*#b`|U=zqMC8UE&4aNx07v2DX}9G@GMZe34sj4t0nwLKO2}sjY=9x9gEu|9JA-faZABP_u_7Fixm1gVR)Ff?M=o`DXEIj1)QBVa{}IpsWgsm%ZD)CG9Gm9CGE;?tFs9Qtx)99nRspK?`pSG#XD@6Mt+SzA;?)yb;0Rb@yFA%QqCIxNi~A>Y)BY!FbDrzd3UcH|9rPO*c)yFvmC_5_3Fe=8xwC$P5da{cr-H=9fpzBLuygI6_4RE)aLK3)Yo1w>q9}5w*em@nEL@uZ!Tb%^I)^p!O~`-vo!nNSiN3RAz&pmV#NL7&Q9~75SE~g0I3%}`@>`cmppQM%NwQCOF+Sg^;M(U4Y7j4cHG`HTT0PXswtKeypP8EPP=l&-MRIxlEe;boEzO6pXrSY6;C5`<~PP#c5Af_w3R~QQYl+w!20im)@=@GrV)F+&5VkN0sT3SPy749ErGAM{XNEsVS@ggAVrq-sX!_aG!r`en5>y^vqOW7t|!b2f92G2CCvoXojFyoSNwqKgl=e(vX2|{=ZRzqaRVZOpF0LkCM9CMs66TtUvs1$BNnW?k${mpIU9tyrdj`?T}^|Rtzv^UhH}cB_;Y51CW;&wIGj$5`Wg<lP~>XtcMFa@r$F=J$ana?mD=)Bq#k2>OmmP5AlJWg>^D%27%uK6YW$vfaNlU$NPj{7_CoC^>-vi}^Z>k|*g%;L?9N0ui6st)mKtLXY*D4GrQftz?GuK5^6KFk65XLYC!>(-^hKP5WRwDJvjf>_L|g}|w#`53Z*})cHHt}8<st{^7b=yU;@||ruhIo&91IJFS-xJYu26mQR9Jmzz)xZ{cgAXM(6kxftghaSus%ISXv0qJQlc4rse`&hI|Ln=FZ9)$kMJcys%UYAL&@jq$0U*`+xTiu`EpK9$80CJjJZ`Wp57f@!dwJM1<b<?A5Zh3Q1~ClnB-Jek1?*K$kj(-@j^-M;^1|`&W}Pj@2jk=efCo8dU})|i*<r!C3&Zo7K&J~0wZkX-!YXd<)^)d{d5K6M~3Zt?aqU?elF!pQyUDFp=Ie&G2atXO4pXws*AM0vMSA%UJ~rn630`g)MJsO)#uNStMl`<<+Uo(z0_+uGnw~N%1x*}#4wccc9BxLVgt%}tG2dMpI<F1wH{VBE$-dZMB@HB|7F9{i{Y92`-Y(q_t({5H<&=&iwLJJt(Gfo#a(*g7pgOdeMDI41sOW#eRta1Fa=`c1$<W8@4yY}w3}On;sl(tm9==SeybKuyW7xe0@ZFv?3Gxx!|rgi)fDwvoyHKjPrQ8R+G2h6W^Ex%kyoVsFpJiXaYVx_^Yok;=2oK}Y$gFR53FSvG<Fb_5k$LyU!KDjc;%sy67g}B273vjQzij@@;1tSb`6}vaTJz6xut{9rYKfTp8%2dD^iMYuGpf=@Cd#c@);ZMPO;>%4Z6?}w2@|ew-OEqI}<<jRN7kX0No1EQUy$-97ZH35EF~|0Pq7h%V2Ukxf((D;jL?P*=j7hNRi!(dY0djmik~8TH;oMiND~NQ$dQ3fv13!T^39>@};YSC6|A=vvcL@E9a{UcDWQ_hLVD2M={UMh%rD$k`cpcy=cKsW!ze}^J8t3_8&8pRDI}1@YA<j3~c6VGWDaGt(m{37+Ctoy;_Z{w-=a5aP=y@egAgGszoCTrI~o26N!?gz>XsdR+6r=iB3f)m+1ebaDbi$uU6eWq9Z4k&g8kVp@P-V33-Df<y6T%3n12#@fVlof$hxStj+)3s$(<2ETXNwL;-3e%_jJX=)vB@N|>p)16nLXITm!gl(EVlM*%QC3TXfkpKt^%XoSe~Lev{TL8G|%@OrhrNady)Xwxdg6b_5t6ehUbc*}8r!-zqv%RzVvUWm98KNMr<p6~ke=Z7MRvTV0phl4_2>L?FnmB=UwoE(G>^!c?K3NoM&#Mc&=u8~|@%#6g`Uu3BmRhE@C<X<A3;8T!>U*V*iR;z&QT1v-Xs1Va5<_WJDh*f#7BsFS_o?-O_;UZKQmu~QYB5+qKvoqOOM9hZIfCT_Gt$L#tMoK?}x!~^vTwfR=%j@rGtcSY1*#=khkr(JY8u&=wDdx^)W$CqX<|3`@w_&N*mR4TRnNp99RXd#y5`7e(;?6zEXQCvQOh5TEBV}Zvj<tEN-C<JWn;s3z1;QCGMC~nCyjq>F-o6c+MC~KkZIY4VzpK4IdRlq2T9}%Nmsgfh+#Ru8Rx4Pk>1ne-=Im7};f<B0JIgS+>f)lhDToUl!z>q;?qGDq{L-D<YoVE_-EsaT?=CNmF-dHNBabmrrY(9F={NIdQk%L0|C=3SHrK1S>Wi=A3(y|sa%IF+V!X7~g@yWB{cg>gm)l4Ftd#Dq1W4bcq%d+;$1nc4GIKTCwNYg6Ru@6W2CaY8+UwS`GE6e)n|*3=(L121UO=JI#}favwz3pP@Z436@n4IzwHnZJ#<V9r&Y-L_<}?i5Xg%yV`{=W51gH=Oj)ohEIvjeYGOGv4jE!pO{2B#e_%!xv3py5UpC{I*MWCMh!TzvM`UB-FqAVKhH9>J`2O2cjJDWSV>#$+6K0w>+*I^E;nk8s`m8qB$mH|@l?M{H^8kF`2FqFXZ$+EFw$vY}u3F>P#27(HTC~_g<daU%qAg@*pu{>CJEj}A`kPqmBS?#?;**s6|5&U)9JfT^hF<3|`<F=|O4=wV(1G$u(KT#3#ZwXGvZJWdX@T(SQuON9_t>SSRaY3w18ZsmxwjN_bA#zp2!UJ<MQQ*=88S~$k>tri%77MNr1sMV*sLd0TS<P9ME4F!IZRO6|O_(nTOLAAuZ|WlGFIif-LA<Q^<0;w?D>RQk4_R01A<OYG;t$1wse`34NsaW1A5+snJL<QpE3d~(%eC8h)_ki;=Pxs+wy<F4AUavRauvT3SC@fE0ac4{R7tSHk9NLH@nKg=L_v~#770aOmbe>WcXxq%EZnNEqLn(nUA+Yt@#mRFtIOF|S@3-|hMG#rt>voA>N)kh)%o|WDjbegr1xue7CFS`#)Lu}+b%gqwo56~TG1m6LpwNR^NV*Da0OLIC8y$Tsh#K{O4-`C@DZ@ap5Ea;UB81a=J<N8ihly7BSDw<T9_x6Mj~O}NVy;zRafdYA9qnd$<4q?W5T+0$&>{1_`_R^6SJjhPS8ya=m-w(fY>~3pcv8r<@5IMT_ns8f_t#YVmlF7bC`isU9Yd)!n2&7nEI`>wbyBGCGjS(E3EX`Y!}wUh1%V?y1I0og-dB!MIn+%0ZeBV&o4-HwY%Yl?4j@T12qa3rSguLyjd7yj5DIFLbuF&xIx`oIYps5XPz~z0R;e&3rXoh{q~JjEN8M>Te+*$t(&heFRk5#M>%8^1IOoA=EEpA7oMtg!LP7JQpdvT#UHJXmN0LWxmMILX%lHA+>5ue12AC3L2D9G-N@&Au-^*PDfiirQDVd5QgwkZ+&-VrUV`8nNL4w7-vq5F8N+uxT|AQokF`zU$IxT>D`P)VMfct{&dC?_*WcI>)e=IYF!h{Fi>DGoqN6ojUv*|4<#8ZN&g~U9FQ!8&+MAwc;!ZMA08}-KY_lWGtGe{+u-|QNVS;9*hsA%}+oWlPKl|C{SOE~w4pSp&3cksuHWi5v#cvR~F@tkh2K}U}6`!lqr8{dNQNyuDzY|rM686rPy)ateC|ad-RBQp&|IJ3nz@42MV_cx1*GaIi)^96uz>Of;FaGzR{j7fxR`LhEVf*Jl_yHNt@zxfdg&SkAmD;O!>J%FZR2wA$)`;nYJ)e;MR#4ZbXT}-`sQ7J4X;`iQv}RpD*EasQGx5~a6#nlFR2HO2$w4_i4#-jU1(2gew2MeANe+som?V}s9)*}BAW<A<#SsW;Fv4R4I-nx~b#Cb|Q&siY{s~k$vhp@%h;)LNH&)_VAUJJq3f=kfPIQzX8UjZ-&pf1t{ezW)i(5j*O{}(*iz&iZQjNNEDOIk|8{*4jsmnSBCX4L9G9hqGqU=4<bCntUluLm=*2!LBaV|eN+)s1`>jpvI1lg%r4kQEg@6Ew@98fi1c4B<aVWZC^Cr)QMYJg-ocVBZ7X0TGdRT>$_%w}8)>OEJ`IfI&1-a_()eH3_|i|1KWbT|3pnt`nGwAh{NmA$P0%a46l{b(2PSzY0b1Zju7=14!Fmoz#UaREEbL=B|8$~lk>`P9aH+Qy?a>~eUuT1g>17)Qj2a50~MXeYEu7oDyb96F|kqH3XTN+UYe<_zN7i_<U50ag^(@028i3WV36eGk=|pM6h9U*vmok&MbLjAQ2m6M>uZ6@z4?oV#{R+N)I}mx7rxi8+~!V#NgZOUjVfaMUXEboFMnl#SzB4_oN!jo(DahrP<arpgSu9uAVNy*8Dr-KZGPk7sfCN2PXHexcF*Tk!!n0@+*F)@rwwQ4<{fDj8;4II)5Uxe=#%)(6QKjaw_7KZ<mAwC~6og#Wb;{5rHncOAR{=cOD*Ifqy64<Fr#{uxZNHl^=J#4f*nm@gY-e55K>oUjfoC<zs0uHi)@&ZPQWCr|}=K78nqui6Kg*o-PwP3PWq!BVRJ5thub9Jne+Ar-+%Y25)!C6b`<%qb;BS>&%)=ae3E;+AX^OgNESvgz3tJwg?%R$pGLJhD8eHPU9qKnRESMlH0$PUIA)u7vL;FEBEn2Z6Mig_Gd96;b%}K;Hmpa>fnGX!!6tJF!jo&UCZqbBQzrO4I18_>ycKYNT$CKhVPp;qXv?hwk14?O?E(GzJz-h2VlPyub?$&B_&0RPy`DBv+rHG{7$u#PvIZWA)gGuygFJr@|0UPKAmjwu`|&r)0x4V&6*zpX~f-YIAn_^5(gMPfTG44gqm8b!dg)S_lG?Z|Nt3ix<~w^EYqT=f#zB@nV^BG_2xTHvWZRpS)n4ov&SnY_X^)cqLmgi$F%GaQWv3;z>|8(h$B8=URSfp!ZEtJOW_&%mOhIpnn2G!~R&A!E5|J&nt3ZYV1-2PiY3aI*|JnK^EQ?V@WV9M3C4l$)e;gt7Xeytlz2w!=t6u&_|NvLg19|@w`UkMVQ!=#~FlW+!-s6wG^EG2N`JIamb%V8buK^2pCT9Nb(t3j!T?8*LWl*KSwn`G2j;#b`w5IEyzl3d9gZQqg6N-<fDwF#M~tfLXDGvMJE#eHGu^4o@Q7APwBa4C_v37=sIw|@H-Ko4fzN`p?R88P~Q`XYR*q)ic0}oPvpgSN(y+Gn8mLY@|ozu&xG@tJggiH5`m+C7jFcih;f}HCus_sQ5pI`bg1V#b9yCdB@K`*0@+UL_ZNJk6mk5u2XTqG+`DvMEO@Y&CR~lF<cHJK+vkj$y8>(Y2d2N}K(^>>NyLzdts};8P0{Tm`*<D^PI{US2_2MlAXz4E$<cGqxT>P5-OV`OM`jV39nju?yWw<=SEjBs8qn)s5l-hK+hL^w?JZocFV`09w`;*<pnS){jr!WnJJ$lae~2Xlzmma!el1v7dhIq^o^2*e=-v}QMh1XHP<fW|5k*o5>{sdI1Gtv>cppS{iP^<}Z}%EI1HJO_pfSKwGVuB?M*jzr4duvz3!Q|soVOG7(yIyqQ7~-xk``v1Aj-`f$=0BcjzX@BhJsA6tAm64rJzP>x9Il5qI%8yiR39!3F}_6(>$2yZ*{uKwl#QjpC*p7d=zy05vUe}y@+(_R&z57HsLF-UWt?`Dhe1A6>+evq{3cjKTw6H1L;y1o(}N8Z#l_+^a4swFotT6){j(+VXp-Z(45rQ-M%^O3=+SX=xWkxdqG}Zy|aiWUdraOMIVWDjB;<`b6qV+Rr$#Yv9xazozQ#@8q_!CWezS>0`@bPbF<HYSbgOU;uP-WIy2}1r@Rk%uy8~POjqj>uVIK&{VvG5G2fFBlla=F9z9lg5ex_6kRDoGmGMREk~1sJ*ssM4aAVPD4RgZ;1N1H~RM)B_W~5=gZ1DUS?o2iIS&#{w&F4*?c{aLDMm-5{)Yii2T6MK1Nu{6*+{ZG}#r&Dl7+(`HKT)87C^T4j8`k=zMz=x)FJN?&1dN!#M*FEJj0ArkGtJRkFxD1;JmoHwpvW=5za;$64^iIme1?guxlnlpr<@;F0}aNUbQTqp6(ZVVE?W4NTokKRuR&WAxX&;1!qWZr0^m~uTFp`U{ctXTXXnAEZc`doWnHrCa!VU{6v+)Yo10PYsXKpp1-rBYwolnkE0wA8dDmCz4?gSz&cd}zB+xhN>5p<Vrl9kwCy8K^Y(_X6cHsleX17|od6v-D36>uBOg&5G!WdB17jg$ur|rlpM<UpYR_y#Y?}kOsmgQ}<r$~nViEgq}^yrW&kpn1kp<>&@JzkCxRFw(}2su5<z~rf#0t$%e$aey4(PotcB)1w}EG@>7k=r2S^!A%=Dk5kynJ~pK=<Ec8J;r<?PdY<|Gy*uYtt1W9V97CvLv<fBLdS<ffQ2>7q*tHU4#f^$;zAm1K#x=`(fLgZ`t+9el7sE$ZsKufeP4U!K@#jW8!ahE3i1|+Y|Wh=E>X~I_nW(IO1cDDvDIuR!6w|7dm!DdB+8v1G`0pq5dI|i6aft2N~PRNF&R8xB7y*x<xJNwcP{SX3zak5>ot>C&T%9iUHx~ahJ*v+VO>!!{q_}sVx<5#*xQ;-8+w+GZ{W|1`@Jn6f`I+;8y?&Y^)5<xp4U@rv2mX3=|!gbEQ4smV3ur=uo8F3DDn@wG+0U|vk*o08oI8DMM<<!Vcoi6JI10DsVh^kNc5%f=iw!X4AT>P41JN@8?;`YncA>#IH^=I=th-rzcn>GasTQB2DDCumt-EcGJxt1u?mjxz&NYx^`De_2!r1(gy^k?|2WiuU%F7?Qpz)vIEX<VpBy2iE1QDGdhto7pPYVt`fsQI_w<hl=k_?r8N^!MHyg1^Z*rU&<n27mQ)LH8aXCianKzK!MwF2(bgTX7L}7cZ1_oP_ipww4M{q#8o<<I@s2y}$&8_C3g%lBEH+P54{vHUXlOUVQH3BUUL8aigfVIvh+^2y=mJj>v35L<{AtzD!&b+o3ffPBRg(Z%APWsuf54%!%ggotl8LrL{Di+HJR)BdX|KQMN@>cdx>WU95GzxX^q#z+TcbhsBHTi4$m9qPONRV9gRO}7-(7+`~CU--k>@d5Cd5JgXQKx4&j3Ygbv5=oa<?B!(7^qUq<sr{RJ=f)2DUq7jb-_P3-gwEVZ?@rNg=IQ$(&c`mX$PxF6BP~a>#^5qC4^dI*Q+zN)42lQu`n408qOg@&(gsDCpx@{jrspml_W5DiY`F3_~cw{<)hF(4@$W0->-lWI2qnroQ>~ZjWuoM6tsADrUXsGIVbwSIY&iA@q{)7(<6Ci55!}e63Zt*BysYN8Im9~4?BsFr{KVb;(+GWa}6kkWz!=H7=h<=6-<F1*Yt4CFHKsWaVbN^GpwC-Vtumoah{QZYNkgu&MF}KG`3~|G8Dgbo?#feH@lT72M+NG-v>)26g2_U-FYTqo$%rQexvuWlERe80p}5>AC~1WK>ZM4VHDI(s$T^!Llmt4l2|exx(4l0BtGyGHw?N*D;;OKnR{yj@pJe~W>kj+Dkv%*dEG}Jt@J3D9;7?AOKkUZ07pVNZHU;ys4i#6Wu%6Hw=n@N|835i)=?c<sF<*9$r|bIz4#~nQJ6MH=t=iA64Ye`LJBYa7&tFDh^`~3c&;rVN9Jb=I}FP?uoIb$)ZZI#HSQ;kffTSMBj^_ryr_x-ad{ecRR4w;o22_&C3779bf$NdJ=EH%C|zU<%LA({`SD7GAdvpyP$-TWF~^ZXC;czx#4JgZQOJ~b?9#}PC0mMJAy-g8X!Mc*J%*gg5NsxJ`4HS?A0&)m*u}?lIx$m+ta2n{(zyRn<#OHzzPb`52w23Xk)PI=Z4Am*XV9<o2{ESdC}U0(^iaaq6%&!R1|CMJyR{noo7;_`TkfXuL)od-7b7KSiA3~wZ7qph+g@Zz6c(?y*{{V96~8iE!``-yOY~{ho}#BMeyiD~lTc7cp$aGA^)<M1vDO1Gg(YYfo|X5aJA9ck4YOmFY0&#1Ynflgq0o&A2Gy0&>+n5V0}0TRK2UY-`mkQzdJx=}!`_Db9$41Vz<Scbp%<og^m2Mv0zN2}Iiz8oN4TBHOZN+^@lHixY!5ioE!d)zvq8_pQ$qOS%*Bf{Q{|E7Zo79fhyDspy8%@ozSX=IIZk)-8y%;R6N#`Mk3xmaIwyB{0#8)o*O7C1l=qF3{C%bAox|YtPtf*v`Ho15V>|sVaD?BU{@dvvPCtSF{qgj-<uJ0Y#?L_=Q|u;Hu;#IenajgLqWKWM6wPI)HSF)@ctrJ@h=X2$NLI&6mLg-oZl|*g2Xc$7#NsceY;!!2^psHzeYsHj$C4u2nCfaBYJajyokpP}lsBvT^25O%-nwQVWR-_%Vt8j3o9%m%JkP!@KNM8U^n`L4&@x&W$<zXu7_L5|y#-r)!}h&?&}n0@_vSdV;UO6#_cxnuU{L7G6DB1-TxL?X*#veJWUt<!f+q*}>;}4%7i-m3$}(AtZ`H3A`Hp3rsMA;fcI0b+TLHNz_h#a#0fqXLm*p6`E0qwP3~r>0B>>+UgYYX%LRk&@`wfdI&gqutaoh1rtlT`x(MyB}>LiphvrnC$NeKntjrbShGQscb8;2=kotgueyX{H2+kS5=eaSkY8C5VAq_mYZTe&c2`b2m#%bUbTFaV`M-TAEklwPz$tXvO)R1Ld))VwrZp1Qn!=!s+RR`m8wVjrHKEBnoMLC-IbKut6+=6!>3EdwI!Elgjjv~$vkU{y=~B)L~;N7KIi6RfG=e{-<kt#nhM%s4?rW)5>QdT5ssx!J8`jL|?Ns$GCM+Gw^>sBMp^oxBGG$TTgFPAk|P?o{y8$>D^1aD@p%q&p~{QQKVz26QCkpuXhBqKH61It2wh+r`DXTHEnA?^$VJqsPNF5+Mx6EY9hPP7r1RKFi%Fjc(obW=Lvku`XA0!x(Y#+$sz*x!3I+&|FX7<46=9TsaDhWV|@7a^OWfoP0!=bVTY8vkb!3Ognti{%s`In2GE8{<5chTESdqY_-lyvymoKJ2@5JVS~tbiq(`^n<?+`@sp!hj5ZND&ia7TuuF;2?%u+|v)fp{s0$UFAeTqP`bcqAj3mP9lWL6>AhzwK{7jYgd^_#gb3Bp^@0L-4Z3eoU)_=`AIohwaH6v>?jH~tDOj?M5K>NYsLn)}X10tR1tRWK{O8vcL*avB2>GtC5c+t6S*w<*INMni51M}J1E4jdctUbqysk7a#p#It1&LHVC%-*AhI#T6^L3Z;*bzRyCC#ARk<RjDX;iQ-dBRHRpeJsWeUkgkF@u)#BIJ5!A-6^vW<xbg}anz0W;S8lGF3F(X;vAeiCz``;*J$B6e^@zlVaOEe)f!<Fq!+<eqQ4uQ5C#$9&q%hzY$LuS)QZVLncTYqO*kmHJ;K(8(ZqDolf_wn87dcwu!<K<<q&4B7-BK<$B(S#XdM}ubly_=oc2k-TUo8Gu9i58w?GHVI#0YQlN>N3{j7)RH^VaL`w2#s6)sOj(?GC%fD2?Bt>7?XCymi*K1UC>TAjWV++y)kZhWCKGj*jrI-F8P#*?C*(Y=MiN@(*0-{{b9ijY%3?6p9;+ZznJ{qp1_KkSzn7Xs10C(+6PH%%esACA6#qYHofIH2gKf!I0@F=G7DU!yeVYoNr*x?`{`_preC4+>_v41r<sK;|9o!D#{d3eV@+oiMTXunWH<@i>eyj=r+voTAHDr_OVXMx2}qoSf-$XY&nyiAZLiJd3T|x-Rw?`;Tcl5!<t5{%#Seb@L4cLX-2_Uq=}U+ZDzDo&kS2MDc=66z$&?J-m>jho}<4IW_%gQg~q;QaHk9#fexL_#?#7y_=W9IR#YkT%p}O*7eCL^~T8pa#K;U!njHEbAuUeDBI09d@OJi6u_-L3dC8ft^lKrudP&X&)>AI%(Ovxz$-`&=x+sn&y~%rmKQ40+sztW_$r6@EY@$_T+77WRpC8Q`UOp$u+nv5GRd8wutc=xVojFK;lRqf;0y|ZQ55WT3)-+W4rNe6kR20CL9nXUM}Z@cE;*p`y#zGx?m+Ce(4b;{zwc%XONqDPGE@p`Pzm)AUny{1`75z!w^7;%0gat7t#e;nXGSDgdM}vWET2PB+^u9`sA^dh-hD*S99Etxu4b#byEiB?kcXrUW)KVTVE>wAh7(oloqJW4=k23T<ZaXk+0t;8bACiOq`V)@!NI84!N8J!F?Gm=2o=;$iee%J`&X+0dOMw@o$m)raE^NqaHZ(T^!G5Vw%OQicPOUTNkW<n;E(%oY+KtHIf{8fDj5kl99J?bWz2P!b2U?No9xE8DX&&1{Yi`3G5D3L368}`mF4`YN+5s@`dC@HITsbvOMcCu4-76(R&Pp&H;H{GjK^iAC349&liK&qKbCTg&6fS!x@=gJ#nG<pgm<jbl2)+rg?6%?;!0>PO7&{oZDZN7->`{9rY{pZ?}?bV821|^Z@v7Asa;fM2!76`rs_N2IOFo;89i%}g`&N5c!w1M3Qx^H(#BNv6Npn`*@&N(*y$5Ep(7)7!aokfa~f&Mz!59p0ad$+ba0=^cdVLF5<M_GbM6QT%k{6KAJB1-6Y>yBg&1Z;Aa>H9*@e~gEbew-UtxDb$#2{m_RxMV`I&=Wl2C!qIi$u^kB4#<Idzebp=3GwJ^yl65Ik;=FJ>-p^uLs2_we-c()JJNOBWNPDdI_0PhxAN4Do;$$4Zd#+;V((-B*1E5j9ll8A<3PH?nm`#E-nNQoX(wE4Y^wmQA*QQ>})o-{xJy(bwb9yKMQJwEwNgWxTf67>E(OPv#$Cg}yNC02ZjWVhf6FFSOXCTo`g0rYV{d=g6(Ga7Wn4t@@1>`kNG2ET=*#&eexfa!UuT{(*8aafF7yANy?bc`RESk@HgSoh#c{&Hr+W{8{AC-`)w?p*QR&Iq&Fj6M4ggpLpAwm=tbr#i;n(o4^%$!HQ(@zQHjq+*smUy!1_rzOBMN)ct^UpaDTN0gl7CN5!42b5<;$BXjG7vXoI4;&feCV9Lw=Aonr4LP4Me%CXG}N2Uf<aFKn)7Xzw=NT%JOr{Ea+ouHjOfI065RJ^Dk3_4iF7A*`?^cjTEyE7#xIf3e8?KN!u#xk{$u+5Y!BW&5UO#%Nq+C)a*5P8NUdFijM_zc4#iVu^LYCj2@HTSa~>eon=+VJA6>MzWD%PYj`Hsm`&-*mXKKD8k=UM|?nm9I5@Nzr?s5wgmW&SjKfbQ7%iwS=8K6BrK^1k;Lsj>8gmG)8tmV%hL#Tr_LO#?A$lWC-%#eYB(>`o&o_1k0@hAZ1&qoQK7c_DrLHFfJ$SfRDLcQIyTu(bbEN1^D-c&vA%zA<U=BqXUK0XyR(qGffggkj0OXXkbi`BU?FDX`@pszffOMh(WIzdT{77ZuL88XI8O)dSVmkVQ|QX*ELCY4R5(-{2F}GsTnS#*!?O%(LCVJ&1M^;xvc`_B-xO!vrr7aAFL8~21n?yu&DiIpe`#zGyYP_E%Z(vS{kVM<14dg#CI|F4$w$G<%C{@ssHa2sti&&+?61_q&`2#@r$jj^S->a>K=Ifz0bjRaKa@3JS;PqMrrfv6-juLiI=t$&VK_TN`x5@thse~m<!=aJ81mLbbiWu<D__PI6XCW^~x1ESakBreD2843=@LC3q3h3jy?!<y5-1~zq=VBi9gJ_c1%deBG0y|%m@AIkgLr05xvseM|v_{<+7Q@I$v%@jU)bNS(d=}1wQ_g8CkwSjqP9Oa36$Hay{-SRJ^N<T%+-7EKRK(JQ5B1FhC-qjX!+vddZ-P$;@F6=)HUx{#SpiVn^k5Yn{C($l`q#JImpDWQ2L-QT(-~qHJ@J6ca6*v2yYILv@0gQo@GHzoQW37FF`tL+X44IAfj3QD)$$msT;)747lI6tB0WHWODGX{0AEv8aHPDdhA^8_DLFF=o{gjY5UB<;Qudk&X69+Xz_-*7gz-3~$|8U1RAGg+sK*cUj`xW=Rp}Yms|2S|}S$&ec~x=X>WfS;Un#?oz?Z=d{PM*>z)=uztygX0+X6lq#%1?g_?x^82IIz&V%Sd_&&NH&b!dZaJpBaLxXneG$;-WM3IiuiRtkz-kdK(V>!2NIMypDKZk%VPt%H;^}m=v;7dMlbCoog|`&UJEm^T4+7$3$?%7zU{#x~f>$PA35>Z4S{i#5G|UIL-3e<})C~odJaSACrQrIIFuVH07N{tjNvi{D3>`*7$|6V~3V`o7(OOjsJtrt`ZmR>qPW)5|6oqwAh5CY}`G&w7715X^O}e;oY2Au{I#7`+Zqv@zAQ?>bfeJVFr6|ThUW65PXvJ^a-B4CUIs|LX2o>u}k`MK4&uSX&Mw(RCtp(8&g;{%6;fycMMZ%GxQeWDr@Z&RGzI1UKBi+4?d}ptdP$6F5#<%M8wIemWVHpNp>cuzjL#wZxg0kPd>x<jS)2z#9JMFw$TgI6?w#tf-GrtsYO9FHHRIne7hY5L`b_t%?OwHpPdhWRQaM64`)}qYSwfUQlTn1WLe-fvS!X-h_$wx)~#!gZ}I0+>#re%>3jwVKd6W7mCa6L<9dKewFlf~wu@Eu$qRN@`{UJDM?wg0fEd>(y|0j8V1E_@h$m3WYY&4&TiyPiPw3?!>aRVllHBMu^D<n4gP95V7q-US+xHv2%tOW6SNjE?D3F1?Xfc#0~n3te}Ho^rEz{`F3UuX2Y)H{jE{#i+aH1@h$VjRa!QqNz0t_M3f*YRp$KcQ9#nOlyjD#>O(Hz<2BJ$G4>5K_YLCZj3uc9kH`IeU93l2VT>N9$TqSBxS9$61d?SMAX}7TSAB$J-ATG*gQt%c)cwcI8>CHuyqCd6a&iNX8;$2kj$cyLXu+-_SgKr0o*+fv#TwvP#6WtQ2yhbkqjOBqdk-yIcPChMRG(wlaeTD0WEzFBw`@hK0}<&j+gyJ-Iw!`1T*LUV&v!xwj6yvY%&GesnAt@1n@HgA$;2{aW3?Yi$apl;hL!N2^y~XO0lIHpNVoD+?~lPCR{gX7s*ZjU4>uKv23`8ExI%7$oTNFJ?P8aw}8HFBKWI8dm|>3-3@+oEZ#w>*G~eCsf?Jv&ayI+)ae%r3ZD7sQ6A#3g4}cGlmq;a1&G;0$${*I&bttL!%R;9_4K!=e>(kVOj1-=@z19ppZ@;zlhcpj@xPz`3;dmaE>jY(qKV$<54+<}>{Lc`xD3vFN;!TE`X#%?!#P{taN*P~t1{ucW6>yFYY@7~to(!#+?6r<C?`zCV{G>?COdOiYIp0k*J6w-ikB9#*4GV0JGQC|`q$up2}bINT)4`~%hx$0up0Cj6FyT=+|P`D?*{9*5?C6<jn-v?`b78h*|9o0bmZbip9eoQ`|iqKhIkpRxtZ33Lt;&_$ERNQt$F%wyjk&ZuoadS<Z%gDQbC~}m*6K%y6-R$U%}>Jg}jg>U7-@)#jGdhn3w+QAO=1OE@C*`B+oLhJCQXT{yj4%>o{dzACb`PWE`K-?Ge>!l#^*GHK#|=OJGX*7;AxqJxDEr_!?Z1Yj`WF*XoP)wb%2rl5Hi(F4d8-zNIhj7-<XMiZd!Zi;EiwStc<>mEDk%NZ5KVn4X&YfwzRjXxQF-JKkoaSJ?$I431AhIL)nU{We0-+<>XCrZVlJxtquj53Cq(jUN9u0m8XEi0^+xuHRN;Q~p6I@~~K-{Se%{MxFoOcGBMh0U8e?qR2A~AXt|PcHajk)CujRgAVk|(fEiC7#JJWgkZ=uhT9<2qM(ToG!r!8ZR2{{g<IUGul(M<q=%*Ly2+NFrX*#PFoW~q<nG|IpxU<@?cE^^h(C4dC;36O=ZBZSr@v3V5o5(NJ&fDjwB83*rT0*xudu=PaO>XoW;{DRbL-k%z!oCJi|3MYbCm8KSVjBhf=$#(pb6F9;_syB7YW0|PG?Kgr78O-P)|FZeGsh=@aLJSnX41i(-Tuy=7PJ+OYuxR9Zv_zL1PP+br()PE@1!!xJSq*4+AXFEn5P&d=ugMbeZW%T;3+A1`kOsi-OBHu3@Nb0Q2uXRFjyEH;2s@)e*RR8I6?j;?ip?)mt$^DJ@T@+7XT$HYQ6SZ+K@%o6)0y*Kq<xu%I?|WwJIk8^K?uDT)@AsVGRcI_=K>!{7}N69a~(ZqvI0`plV7y{$bABT}FNbMB$Tz;goj#~23GaM8k;wC^{2oi;jQN@m{H^531sFXW%P!(JE8Cw&W=WgU}ha4C32mn66p3?6hQ2F;{ryC&f1;K})(0)mB(uGs&M_nYlb51!8c&EfwCL)@cZ"
def notebook_from_embedded_source():
    import base64, zlib, nbformat
    raw=zlib.decompress(base64.b85decode(EMBEDDED_SOURCE_B85.encode('ascii'))).decode('utf-8')
    nb=nbformat.reads(raw,as_version=4)
    needle='EMBEDDED_SOURCE_B85 = "ML5_V6_EMBEDDED_NOTEBOOK_SNAPSHOT_7a3c2e"'
    replacement='EMBEDDED_SOURCE_B85 = "'+EMBEDDED_SOURCE_B85+'"'
    located=[c for c in nb.cells if c.cell_type=='code' and c.source.startswith('# V6 · неизменяемая копия исходного кода')]
    if len(located)!=1 or located[0].source.splitlines()[2]!=needle:
        raise RuntimeError('V6: внутренний снимок исходника повреждён')
    located[0].source=located[0].source.replace(needle,replacement,1)
    nbformat.validate(nb)
    expected='stage2-ml5-v6.0.0-full-replacement'
    if not any(expected in c.source for c in nb.cells if c.cell_type=='code'):
        raise RuntimeError('V6: снимок другой версии')
    return nb


## 2. Среда Colab, зависимости и статус
CPU достаточен для первичного запуска. Библиотеки устанавливаются только при отсутствии. Результаты SMOKE не публикуются.

In [ ]:
import os, sys, time, json, re, hashlib, base64, io, math, gc, copy, traceback, platform, shutil, subprocess, importlib.util
from datetime import datetime, timezone
from collections import Counter, defaultdict
from urllib.parse import quote

DEPENDENCIES={'psutil':'psutil','numpy':'numpy','pandas':'pandas','pyarrow':'pyarrow','duckdb':'duckdb',
 'sklearn':'scikit-learn','scipy':'scipy','catboost':'catboost','lightgbm':'lightgbm','statsmodels':'statsmodels',
 'networkx':'networkx','xlsxwriter':'XlsxWriter','nbformat':'nbformat','requests':'requests',
 'tqdm':'tqdm','joblib':'joblib','python_calamine':'python-calamine','matplotlib':'matplotlib'}
missing=[pkg for module,pkg in DEPENDENCIES.items() if importlib.util.find_spec(module) is None]
print('Среда: Python',platform.python_version(),'· проверка зависимостей завершена;',
      'отсутствуют:', ', '.join(missing) if missing else 'нет',flush=True)
if missing:
    print('Установка зависимостей; дальнейшее выполнение начнётся только после её завершения',flush=True)
    subprocess.check_call([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing])
    print('Зависимости установлены',flush=True)
import numpy as np, pandas as pd, duckdb, pyarrow as pa, pyarrow.parquet as pq
import requests, joblib, networkx as nx, nbformat, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, PoissonRegressor, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# Ограниченный DuckDB: промежуточные сортировки/агрегации — в локальные spill-файлы.
DB_TMP=ROOT/'duckdb_tmp';DB_TMP.mkdir(parents=True,exist_ok=True)
def duck_connect():
    con=duckdb.connect(database=':memory:')
    con.execute(f"SET memory_limit='{CFG.DUCKDB_MEMORY_LIMIT}'")
    con.execute(f'SET threads={int(CFG.DUCKDB_THREADS)}')
    con.execute("SET temp_directory='"+str(DB_TMP).replace("'","''")+"'")
    return con

def memory_point(stage):
    gc.collect()
    try:
        import psutil
        proc=psutil.Process(os.getpid())
        available=psutil.virtual_memory().available
        # Colab может использовать cgroup: psutil.virtual_memory() иногда отражает память хоста.
        try:
            cg=Path('/sys/fs/cgroup')
            if (cg/'memory.max').exists():
                cap=(cg/'memory.max').read_text().strip()
                current=int((cg/'memory.current').read_text().strip())
                if cap!='max':available=min(available,max(0,int(cap)-current))
            elif (cg/'memory/memory.limit_in_bytes').exists():
                cap=int((cg/'memory/memory.limit_in_bytes').read_text().strip())
                current=int((cg/'memory/memory.usage_in_bytes').read_text().strip())
                if cap<2**60:available=min(available,max(0,cap-current))
        except (OSError,ValueError):pass
        info={'stage':stage,'rss_gib':round(proc.memory_info().rss/2**30,2),
              'ram_available_gib':round(available/2**30,2),
              'disk_free_gib':round(shutil.disk_usage(str(ROOT)).free/2**30,2)}
        print('RAM',info,flush=True)
        if 'progress_event' in globals():progress_event('MEMORY',f'{stage}: доступно {info["ram_available_gib"]} GiB RAM, RSS {info["rss_gib"]} GiB',announce=True)
        if info['ram_available_gib']<CFG.MIN_FREE_RAM_GIB:
            raise MemoryError(f'Недостаточно RAM: свободно {info["ram_available_gib"]} GiB на стадии {stage}. Последняя диагностическая стадия записана в logs.')
    except ImportError:
        pass

START_TIME=time.monotonic()
STATE={'technical':'NOT_RUN','scientific':'NOT_RUN','publication':'NOT_RUN','overall':'ASSEMBLED_NOT_RUN'}
LOG=[];PANELS={};PANEL_FILES={};TABLE_QA=[];ISSUES=[];SOURCE_AUDIT=[];SEMANTICS=[];QUARANTINE=[]
PREDICTIONS=[];PRED_PARTS=[];PRED_TOTAL=0;METRICS=[];MODEL_RECORDS=[];MODEL_FILES={};FIGURES=[];SPLIT_AUDIT=[]
MODULES={f'ML-5.{i}':{'status':'NOT_STARTED','reason':'Ожидается DATA_GATE','n':0} for i in range(1,8)}

def stamp():return datetime.now(timezone.utc).isoformat()
def note(stage,message,status='INFO'):
    LOG.append({'utc':stamp(),'stage':stage,'status':status,'message':str(message)[:600]})
    if 'progress_event' in globals():progress_event(stage,str(message)[:150],announce=(status!='INFO'))
def fail(stage,why):
    note(stage,why,'FAIL')
    raise RuntimeError(f'{stage}: {why}')
def hash_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024**2),b''):h.update(b)
    return h.hexdigest()
def norm_code(s):
    return s.astype('string').str.strip().str.replace(r'\.0$','',regex=True)
def number(s):return pd.to_numeric(s,errors='coerce')
def safe_year(s):return pd.to_numeric(s,errors='coerce').astype('Int64')
def nonmissing_code(s):
    x=norm_code(s)
    return x.notna() & x.ne('') & ~x.str.fullmatch('0+').fillna(False) & ~x.str.lower().isin(['nan','none','неизвестно','unknown'])
def canonical_hash(o):return hashlib.sha256(json.dumps(o,sort_keys=True,ensure_ascii=False,default=str).encode('utf-8')).hexdigest()
def _disk_schema(df):
    # Примитивные наблюдаемые типы сохраняются, смешанные метаданные — строки.
    fields=[]
    for c in df.columns:
        d=df[c].dtype
        if pd.api.types.is_bool_dtype(d): t=pa.bool_()
        elif pd.api.types.is_integer_dtype(d): t=pa.int64()
        elif pd.api.types.is_float_dtype(d): t=pa.float64()
        else: t=pa.string()
        fields.append(pa.field(str(c),t))
    return pa.schema(fields)

def write_df_chunks(df,path,chunk_rows=None):
    chunk_rows=int(chunk_rows or CFG.EXPORT_PARQUET_CHUNK_ROWS)
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    schema=_disk_schema(df)
    writer=pq.ParquetWriter(str(path),schema=schema,compression='zstd',use_dictionary=True)
    try:
        for i in range(0,len(df),chunk_rows):
            part=df.iloc[i:i+chunk_rows].copy()
            for field in schema:
                c=field.name
                if pa.types.is_string(field.type):
                    part[c]=part[c].map(lambda v:None if v is None or v is pd.NA else
                        json.dumps(v,ensure_ascii=False,default=str,sort_keys=True)
                        if isinstance(v,(dict,list,tuple,set)) else str(v))
                elif pa.types.is_integer(field.type):part[c]=pd.to_numeric(part[c],errors='coerce').astype('Int64')
                elif pa.types.is_floating(field.type):part[c]=pd.to_numeric(part[c],errors='coerce').astype(float)
                elif pa.types.is_boolean(field.type):part[c]=part[c].astype('boolean')
            writer.write_table(pa.Table.from_pandas(part,schema=schema,preserve_index=False,safe=True))
            del part
            if i and i%(chunk_rows*10)==0:progress_event('DISK',f'Parquet {path.name}',min(i+chunk_rows,len(df)),len(df),announce=False)
    finally:writer.close()
    return path

def qa_table(name,df,keys=None,allow_empty=False):
    if df is None:df=pd.DataFrame()
    if not isinstance(df,pd.DataFrame):fail('DATA_GATE',name+': ожидается DataFrame')
    if keys and not all(x in df for x in keys):fail('DATA_GATE',f'{name}: отсутствуют колонки ключа {keys}')
    duplicate_count=int(df.duplicated(subset=keys).sum()) if keys else None
    if duplicate_count:fail('DATA_GATE',f'{name}: дубли ключа {keys}: {duplicate_count}')
    if df.empty and not allow_empty:note('DATA_GATE',f'{name}: панель пуста','WARN')
    TABLE_QA.append({'table':name,'rows':int(len(df)),'columns':int(len(df.columns)),
                     'duplicate_keys':duplicate_count,'status':'PASS' if len(df)>0 else 'EMPTY',
                     'unique_keys':','.join(keys or []),
                     'storage':'PARQUET_FULL_PLUS_PREVIEW' if len(df)>CFG.PANEL_IN_MEMORY_MAX_ROWS else 'MEMORY'})
    if len(df)>CFG.PANEL_IN_MEMORY_MAX_ROWS:
        fp=write_df_chunks(df,ROOT/'derived'/'panels_full'/(name+'.parquet'))
        PANEL_FILES[name]=fp
        PANELS[name]=df.head(CFG.PANEL_PREVIEW_ROWS).copy()
        progress_event('DISK',f'{name}: {len(df):,} строк сохранено на диск; в ОЗУ только preview',announce=True)
    else:PANELS[name]=df
    return df

def limitation(module,reason,required=''):
    MODULES[module].update(status='NOT_FEASIBLE_FOR_PREDICTION',reason=reason)
    ISSUES.append({'module':module,'status':'NOT_FEASIBLE_FOR_PREDICTION','reason':reason,'required':required})

def diagnostic():
    mem='unknown'
    try:
        import psutil
        mem=round(psutil.virtual_memory().total/2**30,1)
    except Exception:pass
    info={'python':platform.python_version(),'cpu':os.cpu_count(),'ram_gib':mem,
          'disk_free_gib':round(shutil.disk_usage(str(ROOT)).free/2**30,1),
          'gpu_available': bool(shutil.which('nvidia-smi')),
          'versions':{}}
    note('ENV',json.dumps(info,ensure_ascii=False))
    print('Colab environment:',{k:v for k,v in info.items() if k!='versions'})
    return info
ENV=diagnostic()

# Реальный прогресс: завершённые этапы + текущая операция + heartbeat во время блокирующих запросов.
# Проценты показывают ТОЛЬКО завершённые этапы, а не вымышленную долю расчётов.
import threading, html
from IPython.display import display, HTML
PIPELINE_STAGES=('INPUT','CATALOG_52','SEMANTICS','PANELS','FEATURES','CROSS_SOURCE','MODEL_DESIGN','UNIT_TESTS',
                 'SEVEN_MODULES','EXTENDED_MODULES','STATISTICS','VISUALS','EXPORT','QA_AND_PUBLICATION','DOWNLOAD_RESULTS')
_PROGRESS_LOCK=threading.RLock(); _PROGRESS_STOP=threading.Event()
_PROGRESS={'completed':[],'active':'BOOT','detail':'Инициализация среды',
           'stage_started':time.monotonic(),'units':None,'last_render':0.,'last_log':0.}
try:
    _PROGRESS_HANDLE=display(HTML('<b>ML-05 V6.0.0 — запуск</b>'),display_id=True)
except Exception:
    _PROGRESS_HANDLE=None

def _progress_html():
    with _PROGRESS_LOCK:
        state={k:(list(v) if isinstance(v,list) else v) for k,v in _PROGRESS.items()}
    completed=len(state['completed']);total=len(PIPELINE_STAGES);pct=round(100*completed/total,1)
    elapsed=int(time.monotonic()-START_TIME);stage_age=int(time.monotonic()-state['stage_started'])
    units=state['units']
    progress_units='' if not units else f' · {html.escape(str(units))}'
    detail=html.escape(state['detail'] or '')
    return (f'<div style="font-family:Arial,sans-serif;max-width:1100px;padding:8px 0">'
      f'<div><b>ML-05 V6.0.0</b> · завершено этапов {completed}/{total} ({pct:.1f}%) · '
      f'время {elapsed//60:02d}:{elapsed%60:02d}</div>'
      f'<div style="margin:7px 0;height:16px;background:#e1e5eb;border-radius:8px;overflow:hidden">'
      f'<div style="width:{pct}%;height:16px;background:#2176b5"></div></div>'
      f'<div><b>{html.escape(state["active"])}</b>: {detail}{progress_units}</div>'
      f'<div style="font-size:12px;color:#666">Текущая операция: {stage_age} с · '
      f'показатель прогресса считает только действительно завершённые этапы</div></div>')

def progress_refresh(force=False):
    now=time.monotonic()
    with _PROGRESS_LOCK:
        if not force and now-_PROGRESS['last_render']<2: return
        _PROGRESS['last_render']=now
    if _PROGRESS_HANDLE is not None:
        try:_PROGRESS_HANDLE.update(HTML(_progress_html()))
        except Exception:pass

def progress_event(stage=None,detail=None,current=None,total=None,announce=True):
    now=time.monotonic()
    with _PROGRESS_LOCK:
        if stage and stage!=_PROGRESS['active']:
            _PROGRESS['active']=stage;_PROGRESS['stage_started']=now
        if detail is not None:_PROGRESS['detail']=str(detail)[:240]
        _PROGRESS['units']=(f'{current:,}/{total:,}' if total else f'{current:,}' if current is not None else None)
        line=f'[{datetime.now().strftime("%H:%M:%S")}] {_PROGRESS["active"]}: {_PROGRESS["detail"]}'
        if announce and line and (now-_PROGRESS['last_log']>.5):
            _PROGRESS['last_log']=now
            print(line,flush=True)
            try:
                log_path=ROOT/'logs'/'live_progress.jsonl'
                with log_path.open('a',encoding='utf-8') as f:
                    f.write(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),
                       'stage':_PROGRESS['active'],'detail':_PROGRESS['detail'],
                       'completed':len(_PROGRESS['completed'])},ensure_ascii=False)+'\n')
            except (OSError,NameError):pass
    progress_refresh(force=announce)

def mark_stage(stage):
    if stage not in PIPELINE_STAGES:fail('PROGRESS','Неизвестный этап '+stage)
    with _PROGRESS_LOCK:
        if stage in _PROGRESS['completed']:
            print('Повторное выполнение завершённого этапа:',stage,flush=True)
            return
        expected=PIPELINE_STAGES[len(_PROGRESS['completed'])]
        if stage!=expected:fail('PROGRESS',f'Нарушен порядок: ожидался {expected}, получено {stage}')
        _PROGRESS['completed'].append(stage)
    progress_event(stage,'Завершён успешно',announce=True)

class _ProgressBarFacade:
    def close(self):
        _PROGRESS_STOP.set()
        progress_refresh(force=True)
MAIN_PROGRESS=_ProgressBarFacade()

def _progress_heartbeat():
    # Пульс лишь сообщает об активности ядра; он НЕ изменяет процент готовности.
    while not _PROGRESS_STOP.wait(8):
        progress_refresh(force=True)
        with _PROGRESS_LOCK:
            stage=_PROGRESS['active'];age=int(time.monotonic()-_PROGRESS['stage_started'])
        print(f'[Пульс {datetime.now().strftime("%H:%M:%S")}] {stage}, операция продолжается {age} с',flush=True)
_HEARTBEAT_THREAD=threading.Thread(target=_progress_heartbeat,daemon=True,name='ml05_progress')
_HEARTBEAT_THREAD.start()
progress_event('BOOT','Среда и зависимости подготовлены')
# Лёгкая машинная проверка интеграционных инвариантов в самом ноутбуке.
def validate_table_contract(name,frame,mandatory=(),key=()):
    if not isinstance(frame,pd.DataFrame):fail('CONTRACT',f'{name} is not DataFrame')
    lost=set(mandatory)-set(frame.columns)
    if lost:fail('CONTRACT',f'{name}: отсутствуют обязательные поля {sorted(lost)}')
    if len(frame) and key and frame.duplicated(list(key)).any():fail('CONTRACT',f'{name}: дубли натурального ключа')
    return True

# Проверка дискового ParquetWriter до начала тяжёлой загрузки GitHub.
_probe_path=ROOT/'derived'/'_memory_writer_probe.parquet'
_probe_frame=pd.DataFrame({'region_code':pd.Series(['01000000','02000000'],dtype='string'),
    'test_year':pd.Series([2024,2025],dtype='Int64'),
    'measure':pd.Series([1.5,2.5],dtype=float),
    'period':pd.Series(['historical',2025],dtype=object)})
write_df_chunks(_probe_frame,_probe_path,chunk_rows=1)
_probe_pf=pq.ParquetFile(_probe_path)
assert _probe_pf.metadata.num_rows==2 and _probe_pf.schema_arrow.names==list(_probe_frame.columns)
_probe_read=pd.read_parquet(_probe_path)
assert _probe_read['region_code'].tolist()==['01000000','02000000']
assert _probe_read['test_year'].tolist()==[2024,2025]
assert _probe_read['period'].tolist()==['historical','2025']
del _probe_frame,_probe_pf,_probe_read
_probe_path.unlink(missing_ok=True)
progress_event('MEMORY','Тест потоковой записи Parquet и типов: PASS',announce=True)


## 3. Приватный GitHub: доступ, неизменяемый вход и SHA-256
Токен вводится через Secrets или скрытый ввод. Источники считываются из закреплённого Stage I; чужие сегменты не изменяются.

In [ ]:
import getpass
progress_event('INPUT','Проверка токена и GitHub без изменения репозитория')

def github_token():
    token=(os.environ.get('GITHUB_TOKEN') or '').strip()
    if not token:
        try:
            from google.colab import userdata
            token=(userdata.get('GITHUB_TOKEN') or '').strip()
        except Exception:pass
    if not token:token=getpass.getpass('GITHUB_TOKEN (скрытый ввод): ').strip()
    if not token:fail('AUTH','Необходим доступ к приватному репозиторию GitHub')
    return token

_TOKEN=github_token()
BASE='https://api.github.com/repos/'+CFG.REPO
SESS=requests.Session()
SESS.headers.update({'Authorization':'Bearer '+_TOKEN,'Accept':'application/vnd.github+json',
                     'X-GitHub-Api-Version':'2022-11-28'})

def api_call(method,url,*,headers=None,stream=False,timeout=(30,600),**kwargs):
    last=None
    for attempt in range(4):
        try:
            if attempt:progress_event('INPUT',f'Повтор GitHub API, попытка {attempt+1}/4',announce=True)
            r=SESS.request(method,url,headers=headers,stream=stream,timeout=timeout,**kwargs)
            if r.status_code in (429,500,502,503,504):
                last=r.status_code;r.close();time.sleep(2**attempt);continue
            if r.status_code in (401,403,404):
                status=r.status_code;r.close()
                raise PermissionError(f'GitHub HTTP {status}: права, доступ или объект отсутствуют; URL скрыт')
            if not r.ok:
                status=r.status_code;r.close()
                raise RuntimeError(f'GitHub HTTP {status}: отклонён запрос; секреты и ответ не выводятся')
            return r
        except PermissionError:raise
        except requests.RequestException:
            if attempt==3:fail('NETWORK','Ошибка запроса GitHub после повторных попыток')
            time.sleep(2**attempt)
    fail('NETWORK',f'GitHub ответил временной ошибкой {last}')

def api_json(method,url,**kw):
    r=api_call(method,url,**kw)
    try:return r.json()
    finally:r.close()

def repo_head(branch):
    p=api_json('GET',BASE+'/git/ref/heads/'+quote(branch,safe='/'))
    return p['object']['sha']

if CFG.PUBLISH_RESULTS_TO_PRIVATE_GITHUB:
    _preflight_repo=api_json('GET',BASE)
    if not _preflight_repo.get('private'):
        raise RuntimeError('WRITE_PREFLIGHT: репозиторий должен быть приватным')
    _rights=_preflight_repo.get('permissions') or {}
    if not (_rights.get('push') or _rights.get('admin') or _rights.get('maintain')):
        raise PermissionError('WRITE_PREFLIGHT: GITHUB_TOKEN должен иметь Contents:write для приватного репозитория')
    progress_event('INPUT','GitHub: приватность и права на публикацию проверены ДО обучения',announce=True)
progress_event('INPUT','Фиксация HEAD исходного и целевого репозиториев')
SOURCE_COMMIT=repo_head(CFG.SOURCE_BRANCH)
TARGET_HEAD_AT_PREFLIGHT=repo_head(CFG.TARGET_BRANCH)
# Точная фиксация одного исходного снапшота Stage I.
def read_git_blob(path,ref=SOURCE_COMMIT,with_metadata=False):
    p=api_json('GET',BASE+'/contents/'+quote(path,safe='/'),params={'ref':ref})
    if p.get('type')!='file':fail('INPUT','Ожидался Git файл '+path)
    if p.get('content'):
        content=base64.b64decode(p['content'])
    else:
        # GitHub Contents для >1 MiB может не возвращать base64; /git/blobs гарантирует исходный blob.
        blob=api_json('GET',BASE+'/git/blobs/'+p['sha'])
        if blob.get('encoding')!='base64':fail('INPUT','Неизвестный тип Git blob')
        content=base64.b64decode(blob['content'])
    return (content,p) if with_metadata else content

progress_event('INPUT','Чтение dataset_index.json и проверка Git blob SHA')
index_bytes,index_info=read_git_blob(CFG.INPUT_INDEX,with_metadata=True)
if CFG.EXPECTED_INDEX_BLOB_SHA and index_info['sha']!=CFG.EXPECTED_INDEX_BLOB_SHA:
    fail('INPUT','Изменился Git blob каталога Stage I: требуется предметная проверка версии')
INDEX=json.loads(index_bytes)
if INDEX.get('repo')!=CFG.REPO or INDEX.get('version')!='7.2':
    fail('INPUT','Несовместимый каталог Stage I')
ASSETS=INDEX['assets']; ASSET_BY_ID={a['asset_id']:a for a in ASSETS}
if len(ASSET_BY_ID)!=len(ASSETS):fail('INPUT','Повторяющиеся asset_id')
for a in ASSETS:
    if not a.get('sha256_verified') or len(a.get('sha256',''))!=64:
        fail('INPUT','Каталог содержит непроверенный исходник '+a['asset_id'])
SOURCE_MANIFESTS={}
for k,v in INDEX['source_manifests'].items():
    progress_event('INPUT','Чтение манифеста '+str(k),announce=True)
    SOURCE_MANIFESTS[k]=json.loads(read_git_blob(v))
EXPECTED_COUNT=len(ASSETS)
INPUT_SHA={}

def download_stream(r,path):
    part=Path(str(path)+'.part');part.unlink(missing_ok=True)
    try:
        expected=int(r.headers.get('content-length') or 0)
        received=0;last_report=0.
        with open(part,'wb') as f:
            for block in r.iter_content(2*1024**2):
                if block:
                    f.write(block);received+=len(block)
                    if time.monotonic()-last_report>2:
                        progress_event('DOWNLOAD','Получение Release-asset',received,expected,announce=False)
                        last_report=time.monotonic()
        progress_event('DOWNLOAD','Release-asset получен',received,expected,announce=True)
        if part.stat().st_size==0:fail('INPUT','Получен пустой файл')
        part.replace(path)
    finally:
        r.close();part.unlink(missing_ok=True)

def fetch_release_asset(asset_id,path):
    u=BASE+'/releases/assets/'+str(asset_id)
    r=api_call('GET',u,headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,900))
    content_type=r.headers.get('content-type','').lower()
    if 'json' in content_type:
        obj=r.json();r.close()
        if not obj.get('browser_download_url'):fail('INPUT','Не найден адрес закрытого Release-asset')
        r=api_call('GET',obj['browser_download_url'],stream=True,timeout=(30,900))
        if 'json' in r.headers.get('content-type','').lower():
            r.close();fail('INPUT','Private release-asset возвращает JSON вместо бинарного файла')
    download_stream(r,path)
    if path.stat().st_size==0:fail('INPUT','Скачан пустой Release-asset')
    return path

def fetch_asset(a):
    name=hashlib.sha256(a['asset_id'].encode()).hexdigest()[:20]+'.parquet'
    path=ROOT/'inputs'/name
    progress_event('INPUT',f'Исходный набор {a["asset_id"]}: проверка кэша и SHA',announce=True)
    if path.exists() and hash_file(path)==a['sha256']:
        INPUT_SHA[a['asset_id']]=a['sha256'];return path
    path.unlink(missing_ok=True)
    if shutil.disk_usage(str(ROOT)).free < CFG.STAGE1_DISK_RESERVE_GIB*2**30:
        fail('INPUT',f'Недостаточно свободного диска: требуется резерв {CFG.STAGE1_DISK_RESERVE_GIB} GiB; можно подключить дополнительный диск или увеличить доступное место')
    loc=a['locator']
    if loc['type']=='git':
        # Вся загрузка закреплена SOURCE_COMMIT, а не плавающим main.
        progress_event('DOWNLOAD',f'Git blob {a["asset_id"]}',announce=True)
        data=read_git_blob(loc['path'])
        path.write_bytes(data)
    elif loc['type']=='release':
        progress_event('DOWNLOAD',f'Release asset {a["asset_id"]}',announce=True)
        fetch_release_asset(loc['asset_id'],path)
    else:fail('INPUT',f"Неподдерживаемый locator {loc['type']}")
    if hash_file(path)!=a['sha256']:
        path.unlink(missing_ok=True);fail('INPUT','SHA-256 не совпал '+a['asset_id'])
    INPUT_SHA[a['asset_id']]=a['sha256']
    progress_event('INPUT',f'Проверено {a["asset_id"]} · {path.stat().st_size/2**20:.1f} МиБ',announce=True)
    total=sum((ROOT/'inputs'/f).stat().st_size for f in os.listdir(ROOT/'inputs') if (ROOT/'inputs'/f).is_file())
    if total/2**30>CFG.MAX_REMOTE_INPUT_GIB:fail('INPUT','Лимит локального кэша Stage I превышен')
    return path

def original_spo_workbook():
    x=[a for a in INDEX.get('extra_originals',[]) if a.get('role')=='migration_matrix_original_xlsx']
    if len(x)!=1:return None
    path=ROOT/'inputs'/'spo_original_e01.xlsx'
    if not path.exists():fetch_release_asset(x[0]['locator']['asset_id'],path)
    # В Stage I для оригинала нет SHA в dataset_index: это не равно подтверждению контрольной суммы.
    return path

def source_asset(segment=None,role=None,source_id=None,prefix=None):
    return [a for a in ASSETS if (segment is None or a['segment']==segment)
            and (role is None or a.get('role')==role)
            and (source_id is None or a.get('source_id')==source_id)
            and (prefix is None or a['asset_id'].startswith(prefix))]

def required_one(items,label):
    if len(items)!=1:fail('INPUT',f'{label}: ожидается 1 источник, найдено {len(items)}')
    return items[0]

def read_table(a,columns=None,where=None,limit=None):
    progress_event('SQL',f'Чтение {a["asset_id"]}, фильтр={bool(where)}, limit={limit}',announce=True)
    path=fetch_asset(a)
    cols=a.get('columns_manifest') or list(pq.ParquetFile(path).schema_arrow.names)
    if columns:
        missing=set(columns)-set(cols)
        if missing:fail('SCHEMA',f'{a["asset_id"]}: отсутствуют {sorted(missing)}')
        selected=','.join('"'+c.replace('"','""')+'"' for c in columns)
    else:selected='*'
    sql=f"SELECT {selected} FROM read_parquet(?)"
    if where:sql+=' WHERE '+where
    if limit:sql+=f' LIMIT {int(limit)}'
    con=duck_connect()
    try:
        if limit is None and int(a.get('rows_actual') or 0)>CFG.MAX_PANDAS_ROWS:
            rows=int(con.execute('SELECT COUNT(*) FROM ('+sql+') AS _guard',[str(path)]).fetchone()[0])
            if rows>CFG.MAX_PANDAS_ROWS:
                fail('RAM_GUARD',f'{a["asset_id"]}: запрос даёт {rows:,} строк; запрещено .df() в FULL, используйте DuckDB COPY/агрегацию')
        result=con.execute(sql,[str(path)]).df()
        progress_event('SQL',f'{a["asset_id"]}: получено {len(result):,} строк',announce=True)
        return result
    finally:con.close()

def sql_query(a,query):
    progress_event('SQL',f'Агрегация {a["asset_id"]}',announce=True)
    path=fetch_asset(a)
    con=duck_connect()
    try:
        result=con.execute(query,[str(path)]).df()
        progress_event('SQL',f'Агрегация {a["asset_id"]}: {len(result):,} строк',announce=True)
        return result
    finally:con.close()

stage1_sources=pd.DataFrame([{'asset_id':a['asset_id'],'segment':a['segment'],'source_id':a['source_id'],
 'role':a.get('role',''),'registered_rows':a.get('rows_actual'), 'locator':a['locator']['type'],
 'sha256':a['sha256'], 'columns':','.join(a.get('columns_manifest',[]))[:500]} for a in ASSETS])
qa_table('stage1_sources',stage1_sources,['asset_id'])
source_ids=set(a.get('source_id') for a in ASSETS)
for missing_id in ('D03','D06'):
    if missing_id not in source_ids:
        ISSUES.append({'module':'STAGE1','status':'SOURCE_NOT_INGESTED',
                       'reason':f'В каталоге нет {missing_id}; нельзя подменять внешними данными','required':missing_id})
note('INPUT',f'pin {SOURCE_COMMIT}; catalog {index_info["sha"]}; {EXPECTED_COUNT} assets')
print('INPUT preflight:', len(ASSETS),'objects; git commit',SOURCE_COMMIT[:12],'; no writes')

mark_stage('INPUT')


## 3А. Полная инвентаризация Stage I — 52/52

В режиме FULL каждое зарегистрированное Parquet скачивается, проверяется по SHA-256, схеме и числу строк, затем построчно оценивается годовой охват без переноса гигантских таблиц в Pandas. Единица анализа при этом сохраняется: **проверка файла не приписывает его значения образовательной миграции**. Годовые shards СПО рассматриваются как альтернативные представления, а не добавочные наблюдения.

In [ ]:
# Stage I FULL52 — strict SHA+Parquet+content profiles, disk-conscious, resumable download cache.
progress_event('CATALOG_52','Проверка ВСЕХ файлов Stage I, SHA, схем, лет и строк',announce=True)
FULL52_INVENTORY=[];FULL52_YEAR_STATS=[];FULL52_THEME_STATS=[];FULL52_ORIGINALS=[]

def _catalog_family(a):
    x=a['asset_id'];seg=a.get('segment','')
    if seg=='01':return 'MUNICIPAL_DEMOGRAPHY'
    if seg=='02':return 'MUNICIPAL_ECONOMY_LABOUR_EDUCATION_TAX'
    if seg=='03':
        if a.get('role')=='education_od_matrix_raw_cells':return 'SPO_ORIGINAL_OD_CELLS_NOT_ADDITIVE'
        if a.get('role')=='organization_indicators':return 'SPO_ALL_YEARS_OR_YEAR_SHARD_NOT_ADDITIVE'
        return 'SPO_REGISTRY_OD_REGIONAL'
    if seg=='04':return 'HIGHER_EDUCATION_VPO'
    if seg=='05':return 'GRADUATE_LABOUR_OUTCOMES'
    if seg=='06':return 'MUNICIPAL_SPATIAL_BDMO_TRANSPORT'
    return 'UNKNOWN'

def _source_year_scan(a,p,cols):
    key=a.get('time_key')
    if key not in cols:key=next((c for c in ('year','observation_year','graduation_year') if c in cols),None)
    if not key:return [],'NO_TIME_KEY'
    column='"'+key.replace('"','""')+'"'
    try:
        with duck_connect() as con:
            years=con.execute(f'''SELECT CAST({column} AS VARCHAR) observed_year,COUNT(*)::BIGINT n
                FROM read_parquet(?) GROUP BY 1 ORDER BY 1''',[str(p)]).fetchall()
        return [{'asset_id':a['asset_id'],'year_label':str(y),'rows':int(n),'time_column':key} for y,n in years],'SCANNED_FULL'
    except (duckdb.OutOfMemoryException,duckdb.IOException) as ex:
        ISSUES.append({'module':'SOURCE_AUDIT','status':'TIME_DISTRIBUTION_NOT_AVAILABLE',
                       'reason':a['asset_id']+' '+type(ex).__name__,'required':'more spill memory for time scan'})
        return [],'YEAR_SCAN_UNAVAILABLE'

for i,a in enumerate(ASSETS,1):
    aid=a['asset_id']
    if CFG.MODE!='FULL' and aid not in {'01::0','03::42f47d3cf2159992::000','03::89505d5993e6f4ad::000'}:
        FULL52_INVENTORY.append({'asset_id':aid,'segment':a.get('segment'),'source_id':a.get('source_id'),
           'research_family':_catalog_family(a),'observed_rows':None,'expected_rows':a.get('rows_actual'),
           'physical_read_ok':False,'footer_rows_ok':False,'sha256_ok':False,
           'years_scanned':0,'year_scan_status':'NOT_SCANNED_IN_PILOT',
           'source_sha256':a['sha256'],'level_gate':'PILOT_NOT_FULL'})
        continue
    progress_event('CATALOG_52',f'[{i:02d}/52] {aid}: SHA + контроль Parquet',i,len(ASSETS),announce=True)
    source=fetch_asset(a)  # verifies SHA both fresh and cached; never silently skipped
    footer=pq.ParquetFile(source)
    cols=list(footer.schema_arrow.names);registered=int(a.get('rows_actual') or 0)
    found=int(footer.metadata.num_rows)
    if found!=registered:fail('FULL52',f'{aid}: rows {found:,} != Stage I {registered:,}')
    manifest_cols=list(a.get('columns_manifest') or [])
    if manifest_cols and set(cols)!=set(manifest_cols):fail('FULL52',f'{aid}: Parquet fields != Stage I catalog')
    # Content touch: inspect a physical row through a LIMIT, not whole table in pandas.
    with duck_connect() as con:
        touched=int(con.execute('SELECT COUNT(*) FROM (SELECT 1 FROM read_parquet(?) LIMIT 1)',[str(source)]).fetchone()[0])
    if touched!=int(found>0):fail('FULL52',f'{aid}: нельзя прочитать содержимое Parquet')
    yr,yr_status=_source_year_scan(a,source,cols) if CFG.CATALOG_SCAN_ALL_YEARS else ([], 'SCAN_DISABLED')
    if yr and sum(r['rows'] for r in yr)!=found:fail('FULL52',f'{aid}: сумма годового охвата не совпадает с footer')
    FULL52_YEAR_STATS.extend(yr)
    family=_catalog_family(a)
    FULL52_INVENTORY.append({'asset_id':aid,'segment':a.get('segment'),'source_id':a.get('source_id'),
      'data_role':a.get('role') or a.get('data_role'),'research_family':family,'observed_rows':found,
      'expected_rows':registered,'columns':len(cols),'bytes':source.stat().st_size,
      'physical_read_ok':True,'footer_rows_ok':found==registered,'sha256_ok':INPUT_SHA.get(aid)==a['sha256'],
      'years_scanned':len(yr),'year_scan_status':yr_status,'first_year':yr[0]['year_label'] if yr else None,
      'last_year':yr[-1]['year_label'] if yr else None,'source_sha256':a['sha256'],
      'original_grain_status':a.get('grain_status'),'level_gate':'ORIGINAL_GRAIN_NOT_ASSUMED_ADDITIVE'})
    del footer
    if i%4==0:gc.collect();memory_point(f'catalog_{i}_of_{len(ASSETS)}')

if CFG.MODE=='FULL' and (len(FULL52_INVENTORY)!=len(ASSETS) or len(INPUT_SHA)!=len(ASSETS)):
    fail('FULL52','Неполное чтение/контроль входных файлов')
# Role-specific research trail: every source has a declared analytic role, with no double-counting.
for fam,grp in pd.DataFrame(FULL52_INVENTORY).groupby('research_family'):
    FULL52_THEME_STATS.append({'family':fam,'n_assets':len(grp), 'source_rows_scanned_by_footer':int(grp.observed_rows.sum()),
      'interpretation':'ALT_REPRESENTATIONS_NOT_ADDITIVE' if 'SHARD' in fam or 'CELLS' in fam else 'OWN_GRAIN',
      'sha_verified':int(grp.sha256_ok.sum())})
qa_table('full52_source_inventory',pd.DataFrame(FULL52_INVENTORY),['asset_id'])
qa_table('full52_year_coverage',pd.DataFrame(FULL52_YEAR_STATS),allow_empty=True)
qa_table('full52_theme_coverage',pd.DataFrame(FULL52_THEME_STATS),['family'])
# Original SPO workbook is outside the 52 Parquet assets. Record accessibility separately.
try:
    ori=original_spo_workbook()
    if ori:
        FULL52_ORIGINALS=[{'name':ori.name,'bytes':ori.stat().st_size,'sha256_locally_calculated':hash_file(ori),
          'registered_in_52':False,'status':'CONTENT_AVAILABLE_METHODOLOGY_NOT_YET_IDENTIFIED'}]
except Exception as ex:
    FULL52_ORIGINALS=[{'name':'original_spo_workbook','status':'NOT_AVAILABLE','reason':type(ex).__name__}]
qa_table('full52_raw_originals',pd.DataFrame(FULL52_ORIGINALS),allow_empty=True)
print('CATALOG 52:',len(INPUT_SHA),'SHA verified;',sum(t['observed_rows'] or 0 for t in FULL52_INVENTORY),
      'Parquet rows across files (NOT unique observations);',len(FULL52_YEAR_STATS),'asset-year groups')
mark_stage('CATALOG_52')


## 4. Измерение и определения исходных величин
Для E01 сохраняются фактические заголовки оригинального XLSX и исходных raw-таблиц. Нельзя автоматически называть показателем миграции населения или числом зачисленных наблюдение об образовании и текущем обучении.

In [ ]:
memory_point('before_semantics')
progress_event('SEMANTICS','Аудит определений E01/E02/ВПО/СПО')
# Семантика извлекается из значений исходных таблиц, а не из догадки по названию файла.
E01=required_one(source_asset(role='education_od_flows'),'E01 OD')
SPO_ORG=required_one(source_asset(role='education_organizations'),'E01 organizations')
SPO_IND=sorted(source_asset(segment='03',role='organization_indicators'),
               key=lambda a:0 if 'all_years' in a.get('member','') else 1)
SPO_MASTER=next((a for a in SPO_IND if 'all_years' in a.get('member','')),None)
SPO_REG=required_one(source_asset(role='regional_spo'),'SPO regional indicators')
E02=required_one(source_asset(prefix='04::E02'),'E02 universities')
VPO_211=required_one(source_asset(prefix='04::VPO_2_1_1'),'VPO-1 admissions table')
VPO_213=required_one(source_asset(prefix='04::VPO_2_1_3'),'VPO-1 graduates table')
GRAD_ORG=required_one(source_asset(prefix='05::E04::organization_total'),'E04 graduates')
GRAD_FIELD=required_one(source_asset(prefix='05::E05::field_specialty'),'E05 specialties')
MUN=required_one(source_asset(segment='01'),'Municipal demography')
TRANSPORT=required_one(source_asset(prefix='06::TRANSPORT'),'transport graph')
SPATIAL=required_one(source_asset(prefix='06::D04::D04_member00::spatial_reference'),'geography reference')

# Для FULL/PILOT10 производим сканирование всех строк выбранных источников колонночным движком.
def indicator_catalog(a,fields,cap=None):
    cols=[c for c in fields if c in a.get('columns_manifest',[])]
    if 'indicator_name' not in cols:fail('SEMANTICS','Нет колонки indicator_name: '+a['asset_id'])
    progress_event('SEMANTICS',f'Сканирование словаря {a["asset_id"]}',announce=True)
    path=fetch_asset(a)
    with duck_connect() as con:
        # В SMOKE ограничиваем ввод — это не окончательный словарь.
        sample=f' LIMIT {CFG.MAX_READ_ROWS_SMOKE}' if CFG.MODE=='SMOKE' else ''
        group=','.join('"'+c+'"' for c in cols)
        sql=f'SELECT {group}, count(*)::BIGINT as n FROM (SELECT {group} FROM read_parquet(?) {sample}) GROUP BY {group} ORDER BY n DESC'
        d=con.execute(sql,[str(path)]).df()
    d['asset_id']=a['asset_id'];d['catalog_exhaustive']=CFG.MODE!='SMOKE'
    progress_event('SEMANTICS',f'Словарь готов: {a["asset_id"]} · {len(d):,} записей',announce=True)
    return d

CAT_E02=indicator_catalog(E02,['indicator_name','indicator_unit','area'])
CAT_VPO=indicator_catalog(VPO_211,['indicator_name','admission_type','education_level','education_form'])
CAT_SPO=indicator_catalog(SPO_MASTER,['indicator_name','indicator_unit','indicator_section']) if SPO_MASTER else pd.DataFrame()
CAT_SPO_REG=indicator_catalog(SPO_REG,['indicator_name','indicator_unit','indicator_section'])
CAT_VPO_GRAD=indicator_catalog(VPO_213,['indicator_name','education_level','education_form'])
for name,tab in [('dictionary_e02',CAT_E02),('dictionary_vpo_admission',CAT_VPO),
                 ('dictionary_spo',CAT_SPO),('dictionary_spo_regional',CAT_SPO_REG),
                 ('dictionary_vpo_graduation',CAT_VPO_GRAD)]:qa_table(name,tab,allow_empty=True)

POS_ADMISSION=re.compile(r'(?i)(?:^|\W)(?:при[её]м|принят|зачисл|поступил|поступивш|принятых)(?:\w|\W|$)')
NEG_ADMISSION=re.compile(r'(?i)(?:заявлен|подавш|конкурс|при[её]мн\w* комисс|при[её]м на работу|планируем|места приема|контрольн\w+ цифр|ваканс)')
POS_GRAD=re.compile(r'(?i)(выпуск|окончивш|завершивш)')
NEG_TOTAL=re.compile(r'(?i)(процент|доля|удельн|на 100|отношение|коэффициент|темп|индекс|стоимост|тыс\. руб|млн)')

# Утверждение требует положительного семантического определения, единицы и отсутствия разрезов-двойников.
def classify_indicator(name,unit=None,category=None):
    value='' if pd.isna(name) else str(name).strip()
    u='' if pd.isna(unit) else str(unit).strip().lower()
    cat='' if pd.isna(category) else str(category)
    if POS_ADMISSION.search(value) and not NEG_ADMISSION.search(value) and not NEG_TOTAL.search(value):
        role='ADMISSION_CANDIDATE'
    elif POS_GRAD.search(value) and not NEG_TOTAL.search(value):role='GRADUATION_CANDIDATE'
    else:role='OTHER'
    unit_ok=not u or bool(re.search(r'чел|человек|единиц|численност|шт',u))
    # Автоматическая классификация создаёт кандидатов, но НЕ утверждает научную семантику.
    grade='ELIGIBLE_FOR_ROW_CHECK' if role!='OTHER' and unit_ok else 'QUARANTINE'
    if role!='OTHER' and not unit_ok:grade='UNEXPECTED_UNIT'
    return {'semantic_role':role,'semantic_gate':grade,
            'name_exact':value,'unit_exact':u,'category_exact':cat}

CANDIDATES=[]
for source,frame in [('E02',CAT_E02),('VPO_2_1_1',CAT_VPO),('SPO_ORG',CAT_SPO),('SPO_REG',CAT_SPO_REG),('VPO_GRAD',CAT_VPO_GRAD)]:
    for _,r in frame.iterrows():
        d=classify_indicator(r.get('indicator_name'),r.get('indicator_unit'),r.get('admission_type'))
        CANDIDATES.append({'source':source,'indicator_name':r.get('indicator_name'),'rows':int(r['n']),
                           'level':r.get('education_level'), 'form':r.get('education_form'),
                           'admission_type':r.get('admission_type'), 'unit':r.get('indicator_unit'),
                           'indicator_section':r.get('indicator_section'),'area':r.get('area'),**d})
CANDIDATE_CATALOG=qa_table('semantic_indicator_catalog',pd.DataFrame(CANDIDATES),allow_empty=True)

OD_SEMANTIC={
 'source_id':'E01','origin_definition':'region_of_previous_general_education',
 'destination_definition':'region_of_current_vocational_study',
 'label':'observed_E01_schooling_to_study_value',
 'admissions_confirmed':False,'permanent_migration_confirmed':False,
 'yearly_count_unit_confirmed':False,'unit':'reported_indicator_value',
 'methodological_status':'SOURCE_DESCRIPTION_REQUIRES_PRIMARY_CELL_VERIFICATION',
 'statistics_available_asof':'2026_SOURCE_VINTAGE_ONLY',
 'admission_target_gate':'NOT_CONFIRMED',
 'source_page':'https://tochno.st/datasets/monitoring_spo',
 'primary_workbook':'Stage I extra_originals/migration_matrix_original_xlsx'
}
ORIGINAL_SHEETS=[]
try:
    progress_event('SEMANTICS','Чтение первичного XLSX E01',announce=True)
    p=original_spo_workbook()
    if p:
        xf=pd.ExcelFile(p,engine='calamine')
        for sheet in xf.sheet_names:
            raw=pd.read_excel(p,sheet_name=sheet,header=None,nrows=25,engine='calamine')
            cells=' | '.join(str(v)[:100] for v in raw.to_numpy().reshape(-1)
                           if not pd.isna(v) and str(v).strip())[:2500]
            ORIGINAL_SHEETS.append({'sheet':sheet,'first_cells':cells,'file_sha256':hash_file(p)})
        OD_SEMANTIC['original_opened']=True
except Exception as ex:
    OD_SEMANTIC['original_opened']=False
    ISSUES.append({'module':'ML-5.1','status':'PRIMARY_SOURCE_REVIEW_PENDING',
                   'reason':'Оригинальный XLSX E01 не открыт; ошибка '+type(ex).__name__,
                   'required':'original E01 workbook access'})
qa_table('e01_original_headers',pd.DataFrame(ORIGINAL_SHEETS),allow_empty=True)
# Даже наличие слова «миграция» в заголовке не доказывает, что показатель — поток зачисления.
if not OD_SEMANTIC.get('original_opened'):
    note('SEM-E01','Оригинал не открыт: допустим ограниченный анализ наблюдаемой величины','WARN')
else:
    note('SEM-E01','Оригинальные заголовки записаны; семантика admission пока не подтверждена')

RAW_SPO_CELL_AUDIT=[]
for a in source_asset(role='education_od_matrix_raw_cells'):
    try:
        p=fetch_asset(a);file=pq.ParquetFile(p)
        RAW_SPO_CELL_AUDIT.append({'asset_id':a['asset_id'],'rows':file.metadata.num_rows,
                                   'columns':','.join(file.schema_arrow.names),'sha256':a['sha256']})
    except Exception as ex:
        RAW_SPO_CELL_AUDIT.append({'asset_id':a['asset_id'],'error':type(ex).__name__})
qa_table('e01_raw_cell_sources',pd.DataFrame(RAW_SPO_CELL_AUDIT),allow_empty=True)

# Проверка ручного списка после чтения фактических словарей; не считать пустоту списка отсутствием исходных данных.
def approved_names(source):
    manually={'E02':CFG.APPROVED_VO_ADMISSION_NAMES,'SPO_ORG':CFG.APPROVED_SPO_ADMISSION_NAMES}
    names=tuple(manually.get(source,()))
    existing=set(CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.source.eq(source),'indicator_name'].dropna().astype(str))
    bad=set(names)-existing
    if bad:fail('SEMANTICS',f'{source}: утверждённые имена не найдены в данных: {sorted(bad)}')
    return sorted(set(names))

SEMANTIC_APPROVAL=pd.DataFrame([{'source':src,'indicator_name':n,'approved_for_admission':n in approved_names(src),
    'semantic_role':r,'unit':u} for src in ('E02','SPO_ORG')
    for n,r,u in CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.source.eq(src),
        ['indicator_name','semantic_role','unit']].drop_duplicates().itertuples(index=False,name=None)])
qa_table('admission_approval_register',SEMANTIC_APPROVAL,allow_empty=True)
print('Semantic candidates:',len(CANDIDATE_CATALOG),'| E01 original sheets:',len(ORIGINAL_SHEETS),
      '| SPO raw-cell files:',len(RAW_SPO_CELL_AUDIT))

mark_stage('SEMANTICS')


## 5. Исследовательские панели и контролируемые сопоставления
Статистические уровни ВО, СПО, организация, филиал, регион и муниципалитет не смешиваются. Спорные организационные ID исключаются в карантин, ноль отличается от отсутствия.

In [ ]:
memory_point('before_panels')
progress_event('PANELS','Исходные панели: E01, организации, ВПО, E04/E05, муниципалитеты')
# Шесть исходных исследовательских панелей и независимые таблицы ошибок сопоставления.
OD_RAW=read_table(E01,['year','origin_region_code','origin_region_name',
                       'destination_region_code','destination_region_name','indicator_value'])
OD_RAW=OD_RAW.rename(columns={'origin_region_code':'origin_region','destination_region_code':'destination_region',
                              'indicator_value':'observed_value','year':'observation_year'})
OD_RAW['origin_region']=norm_code(OD_RAW['origin_region'])
OD_RAW['destination_region']=norm_code(OD_RAW['destination_region'])
OD_RAW['observation_year']=safe_year(OD_RAW['observation_year'])
OD_RAW['observed_value']=number(OD_RAW['observed_value'])
OD_RAW['sector']='SPO';OD_RAW['origin_definition']='previous_general_education_region'
OD_RAW['destination_definition']='current_study_region'
OD_RAW['target_definition']=OD_SEMANTIC['label']
OD_RAW['known_origin']=nonmissing_code(OD_RAW.origin_region)
OD_RAW['known_destination']=nonmissing_code(OD_RAW.destination_region)
OD_RAW['code_status']=np.select([(~OD_RAW.known_origin|~OD_RAW.known_destination).fillna(True).to_numpy(dtype=bool),
                                OD_RAW.origin_region.eq(OD_RAW.destination_region).fillna(False).to_numpy(dtype=bool)],
                                ['UNKNOWN_CODE','INTRA_REGION'],default='INTER_REGION')
if OD_RAW['observed_value'].lt(0).any():fail('SEM-E01','Отрицательные величины E01: нужна особая интерпретация')
if OD_RAW['observed_value'].isna().any():
    QUARANTINE.extend([{'table':'edu_od_year','reason':'missing_value','row':int(i)}
                       for i in OD_RAW.index[OD_RAW.observed_value.isna()].tolist()[:100]])
OD_RAW['explicit_zero']=OD_RAW.observed_value.eq(0)&OD_RAW.observed_value.notna()
OD_RAW['missing_edge']=False;OD_RAW['suppressed']=False
OD_RAW['source_asset_id']=E01['asset_id'];OD_RAW['source_vintage']=INDEX['created_at']
# Год может быть статистическим периодом учёбы; не переименовывать в admission_year.
OD=qa_table('edu_od_year',OD_RAW,
            ['observation_year','origin_region','destination_region'])
OD_INTER=qa_table('edu_od_interregional',OD[OD.code_status=='INTER_REGION'].copy(),
              ['observation_year','origin_region','destination_region'],allow_empty=True)
OD_INTRA=qa_table('edu_od_intraregional',OD[OD.code_status=='INTRA_REGION'].copy(),allow_empty=True)
OD_UNKNOWN=qa_table('edu_od_unknown_code',OD[OD.code_status=='UNKNOWN_CODE'].copy(),allow_empty=True)
if sum(map(len,[OD_INTER,OD_INTRA,OD_UNKNOWN]))!=len(OD):fail('DATA_GATE','OD: категории не исчерпывают данные')

counts=OD.groupby(['observation_year','code_status'],dropna=False).agg(
    rows=('observed_value','size'),total_value=('observed_value','sum')).reset_index()
qa_table('od_composition',counts,allow_empty=True)
EDGES=OD_INTER[['observation_year','origin_region','destination_region','observed_value']].copy()
EDGES['edge_id']=EDGES.origin_region.astype(str)+'|'+EDGES.destination_region.astype(str)
edge_years={int(y):set(g.edge_id) for y,g in EDGES.groupby('observation_year')}
coverage=[];transitions=[]
for y in sorted(edge_years):
    keys=edge_years[y];past=edge_years.get(y-1,set())
    coverage.append({'year':y,'observed_edges':len(keys),'new_vs_previous':len(keys-past) if past else pd.NA,
                     'lost_vs_previous':len(past-keys) if past else pd.NA,
                     'intersection_vs_previous':len(keys&past) if past else pd.NA,
                     'jaccard_vs_previous':len(keys&past)/len(keys|past) if past and (keys|past) else np.nan,
                     'total_observed_value':float(EDGES.loc[EDGES.observation_year==y,'observed_value'].sum())})
    if past:
        transitions.extend([{'year':y,'edge_id':e,'transition':'NEW'} for e in sorted(keys-past)])
        transitions.extend([{'year':y,'edge_id':e,'transition':'NOT_REPORTED'} for e in sorted(past-keys)])
coverage_df=qa_table('od_edge_coverage',pd.DataFrame(coverage),['year'])
qa_table('od_edge_transitions',pd.DataFrame(transitions),allow_empty=True)
if len(coverage_df)>1:
    coverage_df['edge_growth_ratio']=coverage_df.observed_edges/coverage_df.observed_edges.shift(1)
    coverage_df['potential_break']=coverage_df.edge_growth_ratio.gt(1.5) | coverage_df.edge_growth_ratio.lt(0.67)
    if coverage_df.potential_break.any():
        ISSUES.append({'module':'SEM-E01','status':'COVERAGE_BREAK_UNRESOLVED',
         'reason':'Изменение регистрации OD-направлений: прямое межгодовое сравнение требует аудита полноты',
         'required':'Единые определения и покрытие матрицы по годам'})

# Реестр ВО из E02: комбинация исторического года, организационного id и географического положения.
# E02 длинная таблица показателей (2.25 млн строк): реестр читаем DISTINCT в DuckDB,
# не загружая миллионы повторяющихся идентификаторов в оперативную память.
with duck_connect() as con:
    progress_event('PANELS','E02: DISTINCT организаций из полной таблицы, без переноса 2.25 млн строк в pandas',announce=True)
    vo=con.execute('''SELECT DISTINCT id,name,inn,ogrn,region,oktmo,institution,parent_id,parent_name,year
                     FROM read_parquet(?)''',[str(fetch_asset(E02))]).df()
if CFG.MODE=='SMOKE':vo=vo.head(CFG.MAX_READ_ROWS_SMOKE).copy()
for c in ['id','parent_id','oktmo','inn','ogrn']:vo[c]=norm_code(vo[c])
vo['observation_year']=safe_year(vo['year'])
vo=vo.drop(columns=['year'])
vo=vo.loc[vo.id.notna()&vo.observation_year.notna()].drop_duplicates().copy()
vo['sector']='VO';vo['organization_id']=vo['id']
vo['campus_id']=vo['id'] # Только при явном собственном id в исходнике; головной id не переносить на филиал.
vo['parent_group_id']=vo.parent_id.fillna(vo.organization_id)
vo['entity_id']='VO|'+vo.organization_id.astype(str)
vo['campus_geo_status']=np.where(nonmissing_code(vo.oktmo),'REPORTING_OKTMO_NOT_VERIFIED_CAMPUS','UNKNOWN')
vo['entity_match_confidence']=1.0

sporeg=read_table(SPO_ORG,['year','region_name','region_oktmo','id','name','city','lat','lon','type','head','address','branches'])
sporeg['observation_year']=safe_year(sporeg.year);sporeg=sporeg.drop(columns=['year'])
sporeg['id']=norm_code(sporeg.id);sporeg['region_oktmo']=norm_code(sporeg.region_oktmo)
sporeg['sector']='SPO';sporeg['organization_id']=sporeg.id
sporeg['entity_id']='SPO|'+sporeg.id.fillna('UNRESOLVED')
sporeg['campus_geo_status']='POINT_WITHOUT_MUNICIPAL_OKTMO'
sporeg['entity_match_confidence']=np.where(sporeg.id.notna(),1.0,0.0)
sporeg['parent_group_id']=sporeg['head'].fillna(sporeg['id']).astype('string')
sporeg['name_normalized']=sporeg.name.astype('string').str.lower().str.replace(r'\s+',' ',regex=True).str.strip()

# Проверять уникальность натурального ключа после исключения полных дублей.
# Исторические конфликты не удаляются молча: полные дубли снимаются, остальные в карантин.
def quarantine_conflicting_registry(df,sector):
    key=['entity_id','observation_year']
    if df.empty:return df
    df=df.drop_duplicates().copy()
    unknown=df.entity_id.astype(str).str.contains('UNRESOLVED',regex=False) | df.entity_id.isna()
    if unknown.any():
        QUARANTINE.extend(df.loc[unknown].head(500).assign(source=sector+'_MISSING_ID').to_dict('records'))
        df=df.loc[~unknown].copy()
    counts=df.groupby(key,dropna=False).size()
    badkeys=set(counts[counts>1].index)
    if badkeys:
        badmask=pd.MultiIndex.from_frame(df[key]).isin(pd.MultiIndex.from_tuples(list(badkeys),names=key))
        QUARANTINE.extend(df.loc[badmask].head(1000).assign(source=sector+'_CONFLICTING_ID_YEAR').to_dict('records'))
        df=df.loc[~badmask].copy()
    return df
vo=quarantine_conflicting_registry(vo,'VO')
sporeg=quarantine_conflicting_registry(sporeg,'SPO')
for x,tab in [('VO',vo),('SPO',sporeg)]:
    if tab.empty:ISSUES.append({'module':'ML-5.2','status':'NO_REGISTRY',
          'reason':f'Исторический реестр {x} пуст после контроля конфликтных идентификаторов','required':'unique stable IDs'})
registry_columns=['sector','observation_year','entity_id','organization_id','parent_group_id','name','region','oktmo',
                  'city','address','campus_geo_status','entity_match_confidence']
vo2=vo.rename(columns={'region':'region','oktmo':'oktmo'})
spo2=sporeg.rename(columns={'region_name':'region','region_oktmo':'region_oktmo_source'})
reg_parts=[]
for frame in (vo2,spo2):
    part=frame.copy()
    for col in registry_columns:
        if col not in part:part[col]=pd.NA
    reg_parts.append(part[registry_columns])
REGISTRY=qa_table('edu_org_registry',pd.concat(reg_parts,ignore_index=True),
                  ['sector','entity_id','observation_year'],allow_empty=True)
REGISTRY_AUDIT=pd.DataFrame([{'sector':s,'records':len(g),'unique_org_ids':g.entity_id.nunique(),
    'years':g.observation_year.nunique(),'confirmed_campuses':int(g.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS').sum())}
    for s,g in REGISTRY.groupby('sector')])
qa_table('registry_coverage_audit',REGISTRY_AUDIT,allow_empty=True)

# Словари по учреждениям: ни один показатель не принимается по непустоте конфигурационного кортежа.
def selected_names(source,role='ADMISSION_CANDIDATE'):
    # В словарях сохраняются кандидаты, но в прогноз попадают только утверждённые точные имена.
    sub=CANDIDATE_CATALOG[(CANDIDATE_CATALOG.source==source)&
         (CANDIDATE_CATALOG.semantic_role==role)&
         (CANDIDATE_CATALOG.semantic_gate=='ELIGIBLE_FOR_ROW_CHECK')]
    candidates=set(sub.indicator_name.dropna().astype(str).unique())
    if source in ('E02','SPO_ORG'):
        approved=set(approved_names(source));unapproved=sorted(candidates-approved)
        if unapproved:ISSUES.append({'module':'ML-5.2','status':'UNAPPROVED_ADMISSION_CANDIDATES',
          'reason':f'{source}: {len(unapproved)} неутверждённых исходных имён','required':'утвердить точные имена в конфигурации'})
        return sorted(candidates&approved)
    return sorted(candidates)

# Панель приема ВО на уровне E02: все положительные candidates + точный натуральный ключ + единица.
# При неоднозначности исходных area/specialty/indicator нужна quarantine, а не сложение строк.
def selective_records(a,cols,names):
    if not names:return pd.DataFrame(columns=cols)
    esc=lambda t:"'"+t.replace("'","''")+"'"
    name_list=','.join(esc(t) for t in names)
    return read_table(a,cols,where=f'indicator_name IN ({name_list})',
        limit=CFG.MAX_READ_ROWS_SMOKE if CFG.MODE=='SMOKE' else None)

vo_admission=selective_records(E02,['id','parent_id','year','name','region','oktmo',
 'area','specialty_code','specialty_section','indicator_name','indicator_unit','indicator_value'],
 selected_names('E02'))
if len(vo_admission):
    vo_admission['entity_id']='VO|'+norm_code(vo_admission.id)
    vo_admission['observation_year']=safe_year(vo_admission.year)
    vo_admission['value']=number(vo_admission.indicator_value)
    vo_admission['indicator_grain']='E02_ID_YEAR_AREA_SPECIALTY_INDICATOR'
    # Сохраняем полный programme-разрез до отбора только общих строк для орг-панели.
    vo_program_candidates=vo_admission.copy()
    # area — раздел мониторинга, а не доказанный подитог. Ограничиваем только specialty_code.
    for categorical in ('specialty_code',):
        v=vo_admission[categorical].astype('string').str.strip().str.lower()
        keep=v.isna() | v.eq('') | v.isin(['всего','итого','total','0','00'])
        rejected=vo_admission.loc[~keep].copy()
        if len(rejected):QUARANTINE.extend(rejected.head(100).assign(source='VO_NON_TOTAL_'+categorical).to_dict('records'))
        vo_admission=vo_admission.loc[keep].copy()

# SPO данные организации сопрягаются только по уникальному совпадению региона/названия/года.
spo_admission=selective_records(SPO_MASTER,['year','region_name','name','property','indicator_section',
 'indicator_name','indicator_unit','org_value'],selected_names('SPO_ORG')) if SPO_MASTER else pd.DataFrame()
if len(spo_admission):
    spo_admission['name_normalized']=spo_admission.name.astype('string').str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
    spo_admission['observation_year']=safe_year(spo_admission.year)
    spo_admission['region_name']=spo_admission.region_name.astype('string').str.strip()
    lookup=sporeg[['region_name','name_normalized','observation_year','entity_id']].drop_duplicates()
    count=lookup.groupby(['region_name','name_normalized','observation_year'],dropna=False).size().rename('matches').reset_index()
    lookup=lookup.merge(count,on=['region_name','name_normalized','observation_year'],validate='many_to_one')
    ambiguous=lookup[lookup.matches!=1].copy()
    if len(ambiguous):QUARANTINE.extend(ambiguous.head(500).assign(source='SPO_MATCH').to_dict('records'))
    lookup=lookup[lookup.matches==1].drop(columns=['matches'])
    spo_admission=spo_admission.merge(lookup,on=['region_name','name_normalized','observation_year'],
                                     how='left',validate='many_to_one')
    spo_admission['value']=number(spo_admission.org_value)
    spo_admission['indicator_grain']='SPO_ORG_YEAR_PROPERTY_INDICATOR'
    # property в мониторинге СПО может обозначать форму собственности организации,
    # а не разрез значения. Не удаляем реальные организации по этому полю.

# ВПО-1: полная точная агрегация, частями на диск. Лимит 512 МБ используется только для лёгких запросов.
VPO_FULL_PROGRAM_DESCRIPTION=pd.DataFrame()
VPO_FULL_PROGRAM_PATH=None
_vpo_names=selected_names('VPO_2_1_1')
if _vpo_names and CFG.MODE=='FULL':
    _vpo_path=Path(fetch_asset(VPO_211))
    _base=ROOT/'derived'; _base.mkdir(parents=True,exist_ok=True)
    _shards=_base/'vpo_year_shards'; _shards.mkdir(parents=True,exist_ok=True)
    VPO_FULL_PROGRAM_PATH=_base/'vpo_full_program_description.parquet'
    _esc=lambda value:"'"+str(value).replace("'","''")+"'"
    _where_names='indicator_name IN ('+','.join(_esc(x) for x in _vpo_names)+')'
    _src='read_parquet('+_esc(str(_vpo_path))+')'
    # Каждый год рассчитывается независимо; результаты объединяются без повторной агрегации.
    # Группы год×индикатор×специальность нельзя дробить между выходными файлами.
    _select='''SELECT TRY_CAST(year AS INTEGER) AS observation_year, specialty_code, indicator_name,
        object_level, education_form, education_level, admission_type,
        COUNT(*)::BIGINT AS observed_records,
        MEDIAN(TRY_CAST(indicator_value AS DOUBLE)) AS median_reported_value,
        COUNT(DISTINCT object_name)::BIGINT AS distinct_geographic_names,
        FALSE AS totals_additive_confirmed'''
    _start_filter=f'({_where_names}) AND TRY_CAST(year AS INTEGER) IS NOT NULL'
    memory_point('before_vpo_full_aggregate')
    progress_event('PANELS','ВПО-1: проверка годов и полной численности',announce=True)
    with duck_connect() as _con:
        _con.execute("SET preserve_insertion_order=false")
        _years=[int(r[0]) for r in _con.execute(
            f'SELECT DISTINCT TRY_CAST(year AS INTEGER) AS y FROM {_src} WHERE {_start_filter} ORDER BY 1'
        ).fetchall()]
        _expected=int(_con.execute(f'SELECT COUNT(*) FROM {_src} WHERE {_start_filter}').fetchone()[0])
        _bad_years=int(_con.execute(f'SELECT COUNT(*) FROM {_src} WHERE ({_where_names}) AND TRY_CAST(year AS INTEGER) IS NULL').fetchone()[0])
    if _bad_years:
        ISSUES.append({'module':'ML-5.4','status':'VPO_UNDATED_EXCLUDED',
            'reason':f'ВПО-1: {_bad_years} строк без корректного года исключены из динамики',
            'required':'проверка первичных дат'})
    # Очистка только технических файлов этой конкретной агрегации.
    for _old in _shards.glob('vpo_*.parquet'): _old.unlink()
    _parts=[]

    def _vpo_copy(where,output):
        # Точный MEDIAN и COUNT(DISTINCT) без замены статистик приближёнными.
        _sql=f'''COPY ({_select} FROM {_src} WHERE ({where}) GROUP BY 1,2,3,4,5,6,7)
                    TO {_esc(str(output))} (FORMAT PARQUET, COMPRESSION ZSTD)'''
        output.unlink(missing_ok=True)
        try:
            with duck_connect() as _con:
                _con.execute("SET memory_limit='3GB'")
                _con.execute('SET threads=1')
                _con.execute('SET preserve_insertion_order=false')
                _con.execute("SET max_temp_directory_size='40GB'")
                _con.execute(_sql)
            return output
        except Exception:
            output.unlink(missing_ok=True)
            raise

    for _y in _years:
        _filter=f'{_start_filter} AND TRY_CAST(year AS INTEGER)={_y}'
        _outfile=_shards/f'vpo_year_{_y}.parquet'
        progress_event('PANELS',f'ВПО-1: полный расчёт за {_y} год',announce=True)
        try:
            _parts.append(_vpo_copy(_filter,_outfile))
        except duckdb.OutOfMemoryException:
            progress_event('PANELS',f'ВПО-1 {_y}: дробление по индикаторам и специальностям',announce=True)
            for _i,_indicator in enumerate(_vpo_names):
                _ifilter=f'{_filter} AND indicator_name={_esc(_indicator)}'
                _ipath=_shards/f'vpo_year_{_y}_indicator_{_i}.parquet'
                try:
                    _parts.append(_vpo_copy(_ifilter,_ipath))
                except duckdb.OutOfMemoryException:
                    # hash(specialty_code) фиксирован для каждой группы; 16 частей не пересекаются.
                    for _bucket in range(16):
                        _bfilter=f'''{_ifilter} AND hash(COALESCE(CAST(specialty_code AS VARCHAR),'__NULL__')) % 16={_bucket}'''
                        _bpath=_shards/f'vpo_year_{_y}_indicator_{_i}_part_{_bucket:02}.parquet'
                        _parts.append(_vpo_copy(_bfilter,_bpath))
    if not _parts:
        ISSUES.append({'module':'ML-5.4','status':'VPO_NO_MATCHES','reason':'Нет строк по утверждённым именам индикаторов'})
        VPO_FULL_PROGRAM_PATH=None
    else:
        _filelist='['+','.join(_esc(p) for p in _parts)+']'
        with duck_connect() as _con:
            _con.execute("SET preserve_insertion_order=false")
            _con.execute(f'''COPY (SELECT * FROM read_parquet({_filelist})) TO
                          {_esc(VPO_FULL_PROGRAM_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)''')
            _actual=int(_con.execute('SELECT SUM(observed_records) FROM read_parquet(?)',
                                     [str(VPO_FULL_PROGRAM_PATH)]).fetchone()[0] or 0)
            VPO_FULL_PROGRAM_DESCRIPTION=_con.execute(
                'SELECT * FROM read_parquet(?) LIMIT 2000',[str(VPO_FULL_PROGRAM_PATH)]).df()
        if _actual!=_expected:
            fail('VPO_DATA_GATE',f'Потеря или дубль строк ВПО-1: источник {_expected}, агрегация {_actual}')
        _pf=pq.ParquetFile(VPO_FULL_PROGRAM_PATH)
        PANEL_FILES['vpo_program_description']=VPO_FULL_PROGRAM_PATH
        PANELS['vpo_program_description']=VPO_FULL_PROGRAM_DESCRIPTION
        TABLE_QA.append({'table':'vpo_program_description','rows':_pf.metadata.num_rows,
            'columns':len(_pf.schema_arrow.names),'duplicate_keys':None,'status':'PASS',
            'unique_keys':'','storage':'DUCKDB_YEAR_SHARDED_EXACT_AGGREGATE'})
        progress_event('PANELS',f'ВПО-1: {_expected:,} исходных строк, {_pf.metadata.num_rows:,} групп; контроль PASS',announce=True)
        del _pf
    _parts.clear()
    gc.collect()
# Полная агрегация выше; ниже только просмотр небольшой выборки, который не является полным источником.
vpo_admission=selective_records(VPO_211,['year','object_level','object_name','object_oktmo','university_type',
 'education_form','education_level','specialty_name','specialty_code','indicator_name','indicator_value','admission_type'],
 _vpo_names) if CFG.MODE!='FULL' else (read_table(VPO_211,['year','object_level','object_name','object_oktmo','university_type',
 'education_form','education_level','specialty_name','specialty_code','indicator_name','indicator_value','admission_type'],
 where='indicator_name IN ('+','.join("'"+x.replace("'","''")+"'" for x in _vpo_names)+')',limit=2000)
 if _vpo_names else pd.DataFrame())
if len(vpo_admission):
    vpo_admission['observation_year']=safe_year(vpo_admission.year)
    vpo_admission['value']=number(vpo_admission.indicator_value)
    vpo_admission['geography_grain']='REPORTED_OBJECT_LEVEL_'+vpo_admission.object_level.astype(str)

# Автоматический агрегат для прогнозирования строится только когда один уникальный показатель
# на ID/year, и источник действительно указывает итог, а не несопоставимые категории.
def unambiguous_org_admission(raw,sector):
    if raw.empty or not {'entity_id','observation_year','indicator_name','value'}.issubset(raw):return pd.DataFrame()
    x=raw.dropna(subset=['entity_id','observation_year','value']).copy()
    integerish=np.isclose(x.value.to_numpy(float),np.rint(x.value.to_numpy(float)),atol=1e-8,rtol=0)
    noncount=x.loc[~integerish].copy()
    if len(noncount):QUARANTINE.extend(noncount.head(200).assign(source=sector+'_NONINTEGER_ADMISSION').to_dict('records'))
    x=x.loc[x.value.ge(0)&integerish].copy()
    if x.empty:return pd.DataFrame()
    keys=['entity_id','observation_year']
    counts=x.groupby(keys,dropna=False).agg(n=('value','size'),n_indicators=('indicator_name','nunique')).reset_index()
    usable=counts[(counts.n==1)&(counts.n_indicators==1)][keys]
    good=x.merge(usable,on=keys,validate='many_to_one')
    if good.duplicated(keys).any():fail('SEMANTICS',f'{sector}: неоднозначный показатель приёма после фильтра')
    good=good.assign(sector=sector,target_definition='source_approved_count_of_admitted')
    bad=x.merge(usable.assign(used=True),on=keys,how='left')
    bad=bad[bad.used.isna()].drop(columns=['used'])
    if len(bad):QUARANTINE.extend(bad.head(100).assign(source=f'{sector}_MULTIPLE_ADMISSION_VALUES').to_dict('records'))
    return good[['sector','entity_id','observation_year','value','indicator_name','target_definition']].rename(columns={'value':'actual'})

org_parts=[unambiguous_org_admission(vo_admission,'VO'),unambiguous_org_admission(spo_admission,'SPO')]
ORG_YEAR=qa_table('edu_org_year',pd.concat(org_parts,ignore_index=True) if any(len(p) for p in org_parts)
                  else pd.DataFrame(columns=['sector','entity_id','observation_year','actual','indicator_name','target_definition']),
                  ['sector','entity_id','observation_year'],allow_empty=True)
qa_table('vo_admission_raw_candidates',vo_admission,allow_empty=True)
qa_table('spo_admission_raw_candidates',spo_admission,allow_empty=True)
qa_table('vpo_admission_audit',vpo_admission,allow_empty=True)

# Образовательные программы: источники E02 и ВПО-1 сохраняются без географической реконструкции OD×programme.
parts=[]
if 'vo_program_candidates' in globals() and len(vo_program_candidates):
    v=vo_program_candidates[['entity_id','observation_year','specialty_code','specialty_section','indicator_name','value']].copy()
    v['sector']='VO';v['geography_grain']='ORGANIZATION';parts.append(v)
if len(vpo_admission):
    v=vpo_admission[['observation_year','specialty_code','indicator_name','value','geography_grain']].copy()
    v['specialty_section']=pd.NA;v['entity_id']=pd.NA;v['sector']='VO_VPO';parts.append(v)
PROGRAM=qa_table('edu_program_year',pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(),allow_empty=True)

# Выпускники: FULL обрабатывает каждую строку E04 и E05 через DuckDB, в RAM — только итоги.
# Оригинальные E04/E05 остаются в Stage I (SHA фиксируются). Отчёт об исходных записях явный.
GRADUATE_GROUP_SUMMARY=pd.DataFrame();GRADUATE_FIELD_SUMMARY=pd.DataFrame()
GRAD_E04_VALID_ROWS=0;GRAD_E05_VALID_ROWS=0
GRAD_COLS=['object_level','object_name','oktmo','gender','education_level','year','university',
           'count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']
E05_COLS=['object_level','object_name','year','education_level','study_area','specialty_section',
          'specialty','specialty_code','count_graduate','percent_employed','average_salary_fact_med']
if CFG.MODE=='FULL':
    progress_event('PANELS','E04/E05: загрузка оригинальных файлов, агрегация на диске',announce=True)
    path_e04=fetch_asset(GRAD_ORG);path_e05=fetch_asset(GRAD_FIELD)
    with duck_connect() as con:
        progress_event('PANELS','E04: полная агрегатная обработка DuckDB',announce=True)
        GRADUATE_GROUP_SUMMARY=con.execute("""SELECT TRY_CAST(year AS INTEGER) AS graduation_year,
                 object_level,gender,education_level,COUNT(*)::BIGINT AS n_source_records,
                 MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) AS median_reported_graduates,
                 MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) AS median_observed_salary,
                 MEDIAN(TRY_CAST(percent_employed AS DOUBLE)) AS median_employment_percentage
          FROM read_parquet(?) WHERE TRY_CAST(count_graduate AS DOUBLE)>=0
          GROUP BY 1,2,3,4""",[str(path_e04)]).df()
        GRAD_E04_VALID_ROWS=int(GRADUATE_GROUP_SUMMARY.n_source_records.sum())
        progress_event('PANELS','E05: полная агрегатная обработка DuckDB',announce=True)
        GRADUATE_FIELD_SUMMARY=con.execute("""SELECT TRY_CAST(year AS INTEGER) AS graduation_year,
                 object_level,education_level,CAST(specialty_code AS VARCHAR) AS specialty_code,
                 COUNT(*)::BIGINT AS observed_records,
                 MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) AS median_reported_graduates,
                 MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) AS median_wage
          FROM read_parquet(?) WHERE TRY_CAST(count_graduate AS DOUBLE)>=0
            AND specialty_code IS NOT NULL AND TRY_CAST(year AS INTEGER) IS NOT NULL
          GROUP BY 1,2,3,4""",[str(path_e05)]).df()
        GRAD_E05_VALID_ROWS=int(GRADUATE_FIELD_SUMMARY.observed_records.sum()) if len(GRADUATE_FIELD_SUMMARY) else 0
    progress_event('PANELS',f'E04/E05: обработано {GRAD_E04_VALID_ROWS:,}+{GRAD_E05_VALID_ROWS:,} допустимых строк',announce=True)
    GRADUATE_GROUP_SUMMARY['aggregate_totals_identified']=False
    GRADUATE_GROUP_SUMMARY['employment_asof']='2024-12-31'
    GRADUATE_FIELD_SUMMARY['aggregate_totals_identified']=False
    GRADUATE_FIELD_SUMMARY['employment_asof']='2024-12-31'
    qa_table('graduate_descriptive_statistics',GRADUATE_GROUP_SUMMARY,allow_empty=True)
    qa_table('graduate_specialty_description',GRADUATE_FIELD_SUMMARY,allow_empty=True)
    qa_table('graduate_full_scan_audit',pd.DataFrame([
       {'asset_id':GRAD_ORG['asset_id'],'source_rows':int(GRAD_ORG['rows_actual']),
        'valid_rows_in_statistics':GRAD_E04_VALID_ROWS,'method':'DUCKDB_FULL_SCAN_MEDIAN','raw_stage1_sha256':GRAD_ORG['sha256']},
       {'asset_id':GRAD_FIELD['asset_id'],'source_rows':int(GRAD_FIELD['rows_actual']),
        'valid_rows_in_statistics':GRAD_E05_VALID_ROWS,'method':'DUCKDB_FULL_SCAN_MEDIAN','raw_stage1_sha256':GRAD_FIELD['sha256']}]),allow_empty=True)
    # Доступен только явно обозначенный preview, не выдаваемый за полную базу.
    grad=read_table(GRAD_ORG,GRAD_COLS,limit=1000)
    grad['graduation_year']=safe_year(grad['year'])
    grad['employment_asof']=pd.Timestamp('2024-12-31')
    grad['salary_reference_year']=2024;grad['employment_location']=pd.NA
    for c in ['count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']:
        grad[c]=number(grad[c])
    grad['source_id']=GRAD_ORG['asset_id']
    GRADUATES=qa_table('graduate_outcomes_preview',grad,allow_empty=True)
    del grad;gc.collect()
    note('MEMORY','E04/E05 FULL: все строки агрегированы в DuckDB; в Pandas только сводные таблицы и 1000 исходных строк')
else:
    grad=read_table(GRAD_ORG,GRAD_COLS,limit=CFG.MAX_READ_ROWS_SMOKE)
    grad['graduation_year']=safe_year(grad['year']);grad['employment_asof']=pd.Timestamp('2024-12-31')
    grad['salary_reference_year']=2024;grad['employment_location']=pd.NA
    for c in ['count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']:
        grad[c]=number(grad[c])
    grad['source_id']=GRAD_ORG['asset_id']
    GRADUATES=qa_table('graduate_outcomes',grad,allow_empty=True)
    gradf=read_table(GRAD_FIELD,E05_COLS,limit=CFG.MAX_READ_ROWS_SMOKE)
    gradf['graduation_year']=safe_year(gradf['year']);gradf['employment_asof']=pd.Timestamp('2024-12-31')
    gradf['specialty_code']=norm_code(gradf['specialty_code'])
    for c in ['count_graduate','percent_employed','average_salary_fact_med']:gradf[c]=number(gradf[c])
    gradf['source_asset_id']=GRAD_FIELD['asset_id']
    qa_table('graduate_specialty_outcomes',gradf,allow_empty=True)

# Освободить полный первичный E04/E05 сразу после расчёта сводных таблиц.
for _unused in ('gradf',):
    if _unused in globals():del globals()[_unused]
gc.collect();memory_point('panels_after_graduates')

# Муниципальная демография: точные исходные ОКТМО и исторические года; значения не размножаются по вузам.
mun=read_table(MUN,['oktmo','region','municipality','year','population','migration','migration_rate','births','deaths'])
mun['oktmo']=norm_code(mun.oktmo);mun['observation_year']=safe_year(mun.year)
for c in ['population','migration','migration_rate','births','deaths']:mun[c]=number(mun[c])
mun=mun.drop(columns=['year'])
mun['geography_grain']='MUNICIPALITY_FROM_D01'
mun=mun.dropna(subset=['oktmo','observation_year']).copy()
mun_unique=mun.groupby(['oktmo','observation_year']).size()
if (mun_unique>1).any():
    ISSUES.append({'module':'ML-5.7','status':'CONFLICTING_MUNICIPAL_KEYS',
      'reason':f'D01 дублирует ОКТМО×год: {int((mun_unique>1).sum())} ключей; исключены',
      'required':'reconciliation of historical municipality borders'})
    ok=mun_unique[mun_unique==1].index
    mun=mun.set_index(['oktmo','observation_year']).loc[ok].reset_index()
MUNI=qa_table('municipal_demography',mun,['oktmo','observation_year'],allow_empty=True)

# Только verified campus география; наличие без ОКТМО никогда не трактуется как отсутствие.
conf=vo.loc[vo.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS'),
  ['entity_id','observation_year','oktmo','parent_group_id']].dropna(subset=['oktmo']).copy()
conf=conf[nonmissing_code(conf.oktmo)]
conf=conf.drop_duplicates(['entity_id','observation_year'])
conf=conf.groupby(['oktmo','observation_year'],as_index=False).agg(
    confirmed_vo_sites=('entity_id','nunique'))
MUNI_EDU=MUNI.merge(conf,on=['oktmo','observation_year'],how='left',validate='one_to_one')
MUNI_EDU['education_coverage_known']=MUNI_EDU.confirmed_vo_sites.notna()
# Missing is unknown, NOT zero campuses.
MUNI_EDU['confirmed_spo_sites']=pd.Series(pd.NA,index=MUNI_EDU.index,dtype='Int64')
qa_table('municipality_education_context',MUNI_EDU,['oktmo','observation_year'],allow_empty=True)

qa_table('entity_match_quarantine',pd.DataFrame(QUARANTINE),allow_empty=True)
print('Panels: OD',len(OD),'inter',len(OD_INTER),'VO+SPO registry',len(REGISTRY),
      'org_year',len(ORG_YEAR),'program',len(PROGRAM),'graduates',len(GRADUATES),
      'municipal',len(MUNI_EDU))

mark_stage('PANELS')


## 6. Временные признаки, экономика и транспорт
Все признаки имеют исторический период и осторожный лаг доступности. Дистанции между городами остаются прокси, не истинным расстоянием регионов.

In [ ]:
memory_point('before_features')
progress_event('FEATURES','География, экономический контекст и исторические признаки')
# Экономический и пространственный контекст: только фактические данные, с пометкой уровня агрегации.
NAME_MAP=pd.concat([
    OD_INTER[['origin_region','origin_region_name']].rename(columns={'origin_region':'region_code','origin_region_name':'region_name'}),
    OD_INTER[['destination_region','destination_region_name']].rename(columns={'destination_region':'region_code','destination_region_name':'region_name'})
],ignore_index=True).drop_duplicates()
NAME_MAP['region_name_norm']=NAME_MAP.region_name.astype('string').str.lower().str.replace('ё','е').str.strip()
conflicts=NAME_MAP.groupby('region_code',dropna=False).region_name_norm.nunique(dropna=True)
if (conflicts>1).any():
    QUARANTINE.extend(NAME_MAP[NAME_MAP.region_code.isin(conflicts[conflicts>1].index)].head(150)
                      .assign(source='REGION_CODE_NAME_CONFLICT').to_dict('records'))
    NAME_MAP=NAME_MAP[~NAME_MAP.region_code.isin(conflicts[conflicts>1].index)].copy()
NAME_MAP=NAME_MAP.drop_duplicates(subset=['region_code'])
if NAME_MAP.region_code.duplicated().any():fail('GEOGRAPHY','Конфликтная карта кодов субъектов')
# Название региона должно задавать единственный код. Иначе исключаем сомнительные соединения.
_name_counts=NAME_MAP.groupby('region_name_norm',dropna=False).region_code.nunique()
_ambiguous=set(_name_counts[_name_counts>1].index)
if _ambiguous:
    QUARANTINE.extend(NAME_MAP[NAME_MAP.region_name_norm.isin(_ambiguous)].head(100).assign(source='AMBIGUOUS_REGION_NAME').to_dict('records'))
    ISSUES.append({'module':'GEOGRAPHY','status':'AMBIGUOUS_REGION_NAME','reason':f'{len(_ambiguous)} названий соответствуют нескольким кодам','required':'territorial crosswalk'})
NAME_LOOKUP=NAME_MAP[NAME_MAP.region_name_norm.notna() & ~NAME_MAP.region_name_norm.isin(_ambiguous)].copy()
REGION_CONTEXT=[];DISTANCE=pd.DataFrame()

try:
    raw=read_table(TRANSPORT,['region_from','region_to','distance_km','transport_type'],
                   limit=CFG.MAX_READ_ROWS_SMOKE if CFG.MODE=='SMOKE' else None)
    raw['origin_region_name_norm']=raw.region_from.astype('string').str.lower().str.replace('ё','е').str.strip()
    raw['destination_region_name_norm']=raw.region_to.astype('string').str.lower().str.replace('ё','е').str.strip()
    raw['distance_proxy_km']=number(raw.distance_km)
    raw=raw[(raw.distance_proxy_km>0)&raw.distance_proxy_km.notna()]
    d=raw.groupby(['origin_region_name_norm','destination_region_name_norm'],as_index=False).agg(
        distance_proxy_km=('distance_proxy_km','median'),distance_measurements=('distance_proxy_km','size'))
    o=NAME_LOOKUP[['region_code','region_name_norm']].rename(columns={'region_code':'origin_region',
        'region_name_norm':'origin_region_name_norm'})
    t=NAME_LOOKUP[['region_code','region_name_norm']].rename(columns={'region_code':'destination_region',
        'region_name_norm':'destination_region_name_norm'})
    DISTANCE=d.merge(o,on='origin_region_name_norm',validate='many_to_one').merge(
        t,on='destination_region_name_norm',validate='many_to_one')[['origin_region','destination_region',
        'distance_proxy_km','distance_measurements']]
    DISTANCE=DISTANCE.drop_duplicates(['origin_region','destination_region'])
    note('GEOGRAPHY','Расстояние — медиана транспортных пар городов; НЕ истинное расстояние субъект-субъект')
except Exception as ex:
    note('GEOGRAPHY','Транспортный контекст недоступен: '+type(ex).__name__,'WARN')
qa_table('region_distance_proxy',DISTANCE,allow_empty=True)
# Карантин дополняется при географическом сопоставлении, исходная панель обновляется после этапа.
PANELS['entity_match_quarantine']=pd.DataFrame(QUARANTINE)

# Региональный контекст собирается только из источников с явно указанным субъектным grain.
D05_LABOR=source_asset(source_id='D05')
for a in D05_LABOR:
    if 'object_level' not in a.get('columns_manifest',[]) or 'indicator_name' not in a.get('columns_manifest',[]):continue
    try:
        path=fetch_asset(a)
        with duck_connect() as con:
            sql='''SELECT object_name, object_level, year, indicator_name, indicator_unit,
                         TRY_CAST(indicator_value AS DOUBLE) AS indicator_value,
                         COUNT(*) as duplicates
                    FROM read_parquet(?)
                    WHERE regexp_matches(lower(indicator_name), 'безработ|заработ|доход|зарплат|численность рабочей силы')
                    GROUP BY ALL'''
            v=con.execute(sql,[str(path)]).df()
        v['source_asset_id']=a['asset_id']
        v['observation_year']=safe_year(v.year)
        # Нельзя считать объект регионом только по совпадению названия; требуется документированный уровень.
        v=v[v.object_level.astype(str).str.contains(r'регион|субъект|region',case=False,regex=True,na=False)].copy()
        if len(v):REGION_CONTEXT.append(v)
    except Exception as ex:
        ISSUES.append({'module':'ML-5.1','status':'ECONOMIC_SOURCE_GATED',
            'reason':a['asset_id']+': '+type(ex).__name__,'required':'correct unit and regional grain'})
ECON_RAW=pd.concat(REGION_CONTEXT,ignore_index=True) if REGION_CONTEXT else pd.DataFrame()
qa_table('economic_indicator_candidates',ECON_RAW,allow_empty=True)

# Точное сопоставление названий; отсутствие точной пары = NA, не произвольное присвоение.
ECON_PANEL=pd.DataFrame()
if len(ECON_RAW):
    e=ECON_RAW.copy()
    e['region_name_norm']=e.object_name.astype('string').str.lower().str.replace('ё','е').str.strip()
    e=e.merge(NAME_LOOKUP[['region_code','region_name_norm']],on='region_name_norm',how='inner',
              validate='many_to_one')
    e=e.dropna(subset=['indicator_value','observation_year'])
    e['indicator_key']=e.indicator_name.astype(str)+'|'+e.indicator_unit.astype(str)
    # Только единичное значение: похожие категории и единицы не складываются.
    natural=['region_code','observation_year','indicator_key']
    multiplicity=e.groupby(natural,dropna=False)['indicator_value'].transform('size')
    if 'duplicates' in e:
        multiplicity=multiplicity.where(e['duplicates'].fillna(1).le(1),other=2)
    rejected=e.loc[multiplicity.ne(1)].copy()
    if len(rejected):
        ISSUES.append({'module':'ML-5.1','status':'ECONOMIC_GRAIN_CONFLICT',
           'reason':f'Исключено {len(rejected)} неоднозначных строк регион×год×индикатор',
           'required':'verified regional stratum and indicator definition'})
        QUARANTINE.extend(rejected.head(200).assign(source='ECON_CONFLICTING_REGION_YEAR_INDICATOR').to_dict('records'))
    e=e.loc[multiplicity.eq(1)].copy()
    if len(e) and e.duplicated(natural).any():fail('ECONOMY','Конфликт натурального ключа регион×год×индикатор')
    if not e.empty:
        top=e.groupby('indicator_key').region_code.nunique().sort_values(ascending=False).head(3).index
        e=e[e.indicator_key.isin(top)]
        ECON_PANEL=e[['region_code','observation_year','indicator_key','indicator_value','source_asset_id']]
qa_table('regional_economy',ECON_PANEL,allow_empty=True)

# Подготовка лага/скользящих сумм без доступа к целевому году.
def historical_pair_panel(observed,entity_fields,target_col,feature_lag=None,horizon=None):
    fl=CFG.FEATURE_PUBLICATION_LAG_YEARS if feature_lag is None else feature_lag
    h=CFG.HORIZON if horizon is None else horizon
    if observed.empty:return pd.DataFrame()
    o=observed.copy()
    if 'observation_year' not in o:fail('TIME','Нет observation_year')
    o['observation_year']=safe_year(o.observation_year)
    if o.duplicated(entity_fields+['observation_year']).any():fail('TIME','Дубли натурального временного ключа')
    out=o.copy()
    out['target_year']=out.observation_year.astype('Int64')
    out['origin_year']=out.target_year-h
    out['information_cutoff_year']=out.origin_year
    out['historical_year']=out.origin_year-fl
    out['historical_year_2']=out.historical_year-1
    out=out.rename(columns={target_col:'actual'})
    hist=o[entity_fields+['observation_year',target_col]].rename(columns={
        'observation_year':'historical_year',target_col:'prior_value'})
    hist2=o[entity_fields+['observation_year',target_col]].rename(columns={
        'observation_year':'historical_year_2',target_col:'prior2_value'})
    out=out.merge(hist,on=entity_fields+['historical_year'],how='left',validate='many_to_one')
    out=out.merge(hist2,on=entity_fields+['historical_year_2'],how='left',validate='many_to_one')
    out=out[out.prior_value.notna()&out.actual.notna()].copy()
    out['feature_available_year']=out.historical_year+fl
    out['target_available_year']=out.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
    out['horizon']=h
    out['source_vintage']=INDEX['created_at']
    out['vintage_warning']='RETROSPECTIVE_ONLY_HISTORICAL_VINTAGES_NOT_ARCHIVED'
    out['split_id']='UNASSIGNED'
    if not (out.feature_available_year<=out.information_cutoff_year).all():
        fail('TIME','Будущие признаки попали в historical panel')
    return out

OD_MODEL=historical_pair_panel(
    OD_INTER[['observation_year','origin_region','destination_region','observed_value']].dropna(subset=['observed_value']).copy(),
    ['origin_region','destination_region'],'observed_value')
if not OD_MODEL.empty:
    OD_MODEL['entity_id']=OD_MODEL.origin_region.astype(str)+'|'+OD_MODEL.destination_region.astype(str)
    OD_MODEL['sector']='SPO'
    # Маргинали/число партнёров строго за доступный прошлый год.
    counts=OD_INTER.groupby(['observation_year','origin_region']).agg(
        origin_mass=('observed_value','sum'),origin_destinations=('destination_region','nunique')).reset_index()
    dests=OD_INTER.groupby(['observation_year','destination_region']).agg(
        destination_mass=('observed_value','sum'),destination_origins=('origin_region','nunique')).reset_index()
    outm=counts.rename(columns={'observation_year':'historical_year'})
    inm=dests.rename(columns={'observation_year':'historical_year'})
    OD_MODEL=OD_MODEL.merge(outm,on=['historical_year','origin_region'],validate='many_to_one',how='left')
    OD_MODEL=OD_MODEL.merge(inm,on=['historical_year','destination_region'],validate='many_to_one',how='left')
    if len(DISTANCE):OD_MODEL=OD_MODEL.merge(DISTANCE[['origin_region','destination_region','distance_proxy_km']],
                                          on=['origin_region','destination_region'],validate='many_to_one',how='left')
    # Лагирование экономических факторов проверяется датой доступности. Никаких future margins.
    if len(ECON_PANEL):
        e=ECON_PANEL.pivot(index=['region_code','observation_year'],columns='indicator_key',
                           values='indicator_value').reset_index()
        keys=[k for k in e.columns if k not in ('region_code','observation_year')]
        for direction in ('origin','destination'):
            f=e.rename(columns={'region_code':f'{direction}_region','observation_year':'historical_year',
                                **{k:f'{direction}_econ_{i}' for i,k in enumerate(keys)}})
            OD_MODEL=OD_MODEL.merge(f,on=[f'{direction}_region','historical_year'],
                                    how='left',validate='many_to_one')
    OD_MODEL['target_id']=OD_SEMANTIC['label']
    flagged=coverage_df.loc[coverage_df.potential_break.eq(True),'year'] if 'potential_break' in coverage_df else pd.Series(dtype=int)
    OD_MODEL['coverage_break_target_year']=OD_MODEL.target_year.isin(flagged).astype(bool)
    OD_MODEL['coverage_break_historical_year']=OD_MODEL.historical_year.isin(flagged).astype(bool)
qa_table('forecast_od_candidates',OD_MODEL,allow_empty=True)

# Отток: СУММА известных межрегиональных образовательных маршрутов, не migration_residence.
OUTFLOW=OD_INTER.dropna(subset=['observed_value']).groupby(['observation_year','origin_region'],as_index=False).agg(
    observed_interregional_routes=('observed_value','sum'),observed_destinations=('destination_region','nunique'))
OUTFLOW['sector']='SPO';OUTFLOW['target_id']='E01_observed_interregional_routes_from_school_region'
qa_table('outflow_observed',OUTFLOW,['observation_year','origin_region'],allow_empty=True)
OUTFLOW_MODEL=historical_pair_panel(OUTFLOW[['observation_year','origin_region','observed_interregional_routes']],
                                  ['origin_region'],'observed_interregional_routes')
if len(OUTFLOW_MODEL):
    OUTFLOW_MODEL['entity_id']=OUTFLOW_MODEL.origin_region.astype(str)
    OUTFLOW_MODEL['sector']='SPO';OUTFLOW_MODEL['target_id']='E01_observed_interregional_routes_from_school_region'
qa_table('forecast_outflow_candidates',OUTFLOW_MODEL,allow_empty=True)

ORG_MODEL=pd.DataFrame()
if len(ORG_YEAR):
    for sector,g in ORG_YEAR.groupby('sector'):
        m=historical_pair_panel(g[['entity_id','observation_year','actual']],['entity_id'],'actual')
        if len(m):
            m['sector']=sector
            m['target_id']='verified_organizational_admission_count'
            if ORG_MODEL.empty:ORG_MODEL=m
            else:ORG_MODEL=pd.concat([ORG_MODEL,m],ignore_index=True)
    if len(ORG_MODEL):
        rg=REGISTRY[['sector','entity_id','observation_year','parent_group_id','region']].rename(
            columns={'observation_year':'target_year'})
        ORG_MODEL=ORG_MODEL.merge(rg,on=['sector','entity_id','target_year'],how='left',validate='many_to_one')
qa_table('forecast_org_candidates',ORG_MODEL,allow_empty=True)

# Профиль программ по одному источнику; несопоставимые классификаторы не агрегируем.
PROGRAM_PROFILE=pd.DataFrame()
if len(PROGRAM) and {'specialty_code','value'}.issubset(PROGRAM):
    pp=PROGRAM[(PROGRAM.sector=='VO')&PROGRAM.specialty_code.notna()].copy()
    pp=pp.dropna(subset=['value','observation_year'])
    pp['specialty_code']=norm_code(pp.specialty_code)
    pp=pp[pp.value.ge(0)]
    # Не складывать разные индикаторы в одном направлении/году: оставляем строгую согласованную совокупность.
    natural=['entity_id','observation_year','specialty_code','indicator_name']
    multiplicity=pp.groupby(natural,dropna=False).size().rename('records').reset_index()
    ok=multiplicity.loc[multiplicity.records.eq(1),natural]
    excluded=int(len(pp)-len(ok))
    if excluded:ISSUES.append({'module':'ML-5.4','status':'AMBIGUOUS_PROGRAM_GRAIN',
      'reason':f'{excluded} строк нельзя однозначно агрегировать по программе; карантин без произвольного сложения',
      'required':'programme/level/form/classifier disaggregation'})
    pp=pp.merge(ok,on=natural,how='inner',validate='one_to_one')
    # Суммы на одном и том же идентификаторе индикатора допустимы лишь как суммы строк с непересекающимися организациями.
    grp=pp.groupby(['observation_year','specialty_code','indicator_name'],as_index=False).agg(
        observed_value=('value','sum'),organizations=('entity_id','nunique'))
    if not grp.empty:
        grp['sector']='VO';grp['source_grain_gate']='UNIQUE_ORG_YEAR_SPECIALTY_INDICATOR_NOT_POPULATION_TOTAL';PROGRAM_PROFILE=grp
qa_table('program_profile',PROGRAM_PROFILE,allow_empty=True)
# ВПО-1 может содержать реальную специализацию при отсутствии утверждённой организации-приёма: не отключать описание.
VPO_PROGRAM_DESCRIPTION=pd.DataFrame()
if len(vpo_admission):
    z=vpo_admission.dropna(subset=['observation_year','specialty_code','value']).copy()
    z['specialty_code']=norm_code(z.specialty_code)
    if len(z):
        VPO_PROGRAM_DESCRIPTION=z.groupby(['observation_year','specialty_code','indicator_name','object_level',
          'education_form','education_level','admission_type'],dropna=False,as_index=False).agg(
          observed_records=('value','size'),median_reported_value=('value','median'),
          distinct_geographic_names=('object_name','nunique'))
        VPO_PROGRAM_DESCRIPTION['totals_additive_confirmed']=False
if VPO_FULL_PROGRAM_PATH is None:
    qa_table('vpo_program_description',VPO_PROGRAM_DESCRIPTION,allow_empty=True)
else:
    VPO_PROGRAM_DESCRIPTION=VPO_FULL_PROGRAM_DESCRIPTION
    note('VPO','ВПО-1: полная агрегированная панель находится в evidence-Parquet; Pandas содержит только preview')

progress_event('FEATURES',f'Готовы исторические панели: OD {len(OD_MODEL):,}, отток {len(OUTFLOW_MODEL):,}',announce=True)
print('Feature coverage:',len(OD_MODEL),'OD historical rows,',len(OUTFLOW_MODEL),'outflow,',
      len(ORG_MODEL),'organization; economic indicators',len(ECON_PANEL),
      'distance pairs',len(DISTANCE))

memory_point('after_features')
mark_stage('FEATURES')


## 6А. Исследовательское использование всех групп источников

Этот блок выводит дополнительные контекстные показатели без смешения пространственных уровней: экономические и образовательные индикаторы БДМО, муниципальную историю ОКТМО, географические справочники, все E04/E05 и все разделы ВПО. Крупные группировки записываются непосредственно в Parquet. Математически корректное сопоставление сохраняет исходный `indicator_code` и ограничения на сложение пересекающихся разрезов.

In [ ]:
# FULL52 second pass: thematic *derived* panels at the original statistical grain.
# Each of 52 datasets has already passed a physical SHA + row-coverage audit above.
memory_point('before_cross_source');progress_event('CROSS_SOURCE','БДМО, ВПО, E04/E05, СПО и история ОКТМО',announce=True)
CONTEXT_PARQUET_FILES=[];D02_PROFILE=[];VPO_ALL_PROFILE=[];SPO_YEAR_CHECK=[];FAMILY_E04E05=[];SPATIAL_CROSS=[]
BDMO_EDU_PROXY=pd.DataFrame();BDMO_EDU_PROXY_CANDIDATES=pd.DataFrame()
BDMO_SPATIAL=pd.DataFrame();D04_SPATIAL=pd.DataFrame();HISTORY_MUNICIPAL=pd.DataFrame();BDMO_PROXY_FULL_VALID=False
GRAD_ALL_ASSETS=[]

def _sql_literal(v):return "'"+str(v).replace("'","''")+"'"

def thematic_sql_disk(asset,query,out_stem,preview=1500,mem='2GB'):
    # SELECT query may contain __INPUT__ placeholder; never use SELECT * to pandas.
    p=fetch_asset(asset);out=ROOT/'derived'/'full52_context'/(out_stem+'.parquet')
    out.parent.mkdir(parents=True,exist_ok=True)
    sql=query.replace('__INPUT__',_sql_literal(p))
    tmp=out.with_suffix('.partial.parquet');tmp.unlink(missing_ok=True)
    with duckdb.connect(database=':memory:') as con:
        con.execute(f"SET memory_limit='{mem}'")
        con.execute('SET threads=1');con.execute('SET preserve_insertion_order=false')
        con.execute('SET temp_directory='+_sql_literal(DB_TMP))
        con.execute('COPY ('+sql+') TO '+_sql_literal(tmp)+' (FORMAT PARQUET, COMPRESSION ZSTD)')
        data=con.execute('SELECT * FROM read_parquet(?) LIMIT '+str(int(preview)),[str(tmp)]).df()
    tmp.replace(out);pf=pq.ParquetFile(out)
    CONTEXT_PARQUET_FILES.append(out)
    return out,data,int(pf.metadata.num_rows)

# A. Every economic/BDMO source: content-level year/indicator/region coverage (but no false additive sums).
# 5-NDFL/7-NDFL cut across multiple tax forms; count/coverage ≠ observed number of residents.
for i,a in enumerate(source_asset(segment='02') if CFG.MODE=='FULL' else source_asset(prefix='02::labor'),1):
    aid=a['asset_id'];cols=set(a['columns_manifest']);progress_event('CROSS_SOURCE',f'D02 [{i}/11]: {aid}',i,11,announce=True)
    ind=next((c for c in ('indicator_code','indicator_name') if c in cols),None)
    yr='year' if 'year' in cols else None
    geo=next((c for c in ('region_name','report_region_name','object_name') if c in cols),None)
    if not ind or not yr:continue
    select_geo='CAST('+_sql_literal('UNSPECIFIED_LEVEL')+' AS VARCHAR)' if not geo else 'CAST("'+geo+'" AS VARCHAR)'
    level_where="AND regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект|region')" if geo=='object_name' and 'object_level' in cols else ''
    query=f'''SELECT CAST("{yr}" AS VARCHAR) observed_year,
      {select_geo} AS reported_geography,CAST("{ind}" AS VARCHAR) indicator_key,
      COUNT(*)::BIGINT AS reported_records,
      COUNT(DISTINCT CAST("{geo}" AS VARCHAR)) AS n_locations
      FROM read_parquet(__INPUT__) WHERE "{ind}" IS NOT NULL {level_where}
      GROUP BY 1,2,3'''
    if geo is None:query=query.replace('COUNT(DISTINCT CAST("None" AS VARCHAR))','COUNT(*)::BIGINT')
    try:
        p,preview,rows=thematic_sql_disk(a,query,'d02_'+str(i).zfill(2),preview=800,mem='2GB')
        D02_PROFILE.append({'asset_id':aid,'rows_in_source':a['rows_actual'],'year_indicator_geography_groups':rows,
            'profile_parquet':p.name,'status':'GROUPED_ALL_SOURCE_ROWS_NO_DOUBLE_COUNT'})
    except Exception as ex:
        ISSUES.append({'module':'D02','status':'SOURCE_PROFILE_FAILURE','reason':aid+' '+type(ex).__name__+': '+str(ex)[:130],
                       'required':'repair dedicated source query; full SHA inventory still valid'})
        D02_PROFILE.append({'asset_id':aid,'rows_in_source':a['rows_actual'],'year_indicator_geography_groups':None,
                            'status':'GROUP_PROFILE_FAILED'})
qa_table('d02_indicator_coverage',pd.DataFrame(D02_PROFILE),['asset_id'])

# B. All 11 E02/VPO tables: count by publication year and exact indicator name, each input processed independently.
for i,a in enumerate(source_asset(segment='04') if CFG.MODE=='FULL' else source_asset(prefix='04::E02'),1):
    cols=set(a.get('columns_manifest') or [])
    if 'indicator_name' not in cols:continue
    query='''SELECT TRY_CAST(year AS INTEGER) observation_year,CAST(indicator_name AS VARCHAR) indicator_name,
      COUNT(*)::BIGINT observed_rows
      FROM read_parquet(__INPUT__) WHERE indicator_name IS NOT NULL
      GROUP BY 1,2'''
    try:
        p,preview,rows=thematic_sql_disk(a,query,'vpo_'+str(i).zfill(2),preview=500,mem='2GB')
        VPO_ALL_PROFILE.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
                                'year_indicator_groups':rows,'parquet':p.name,'status':'PASS'})
    except Exception as ex:
        VPO_ALL_PROFILE.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
                                'status':'PROFILE_FAILED_'+type(ex).__name__})
        ISSUES.append({'module':'VPO-1','status':'SOURCE_PROFILE_FAILURE','reason':a['asset_id']+' '+type(ex).__name__,
                       'required':'retry annual indicator group scan'})
qa_table('vpo_all_sections_audit',pd.DataFrame(VPO_ALL_PROFILE),['asset_id'])

# C. SPO all-years and independent yearly shards; diagnose overlaps without summing them.
spo_assets=source_asset(segment='03',role='organization_indicators')
annual=pd.DataFrame(FULL52_YEAR_STATS)
master_id=SPO_MASTER['asset_id'] if SPO_MASTER else None
if len(annual) and master_id:
    ref=annual.loc[annual.asset_id.eq(master_id)].set_index('year_label')['rows']
    for a in spo_assets:
        cur=annual[annual.asset_id.eq(a['asset_id'])]
        for year,sub in cur.groupby('year_label'):
            n=int(sub['rows'].sum());v=ref.get(year,pd.NA)
            SPO_YEAR_CHECK.append({'asset_id':a['asset_id'],'year':year,'rows_in_representation':n,
               'rows_in_master':v,'same_count':bool(n==v) if pd.notna(v) else None,
               'treatment':'NOT_ADDITIVE_WITH_MASTER'})
qa_table('spo_annual_vs_master',pd.DataFrame(SPO_YEAR_CHECK),allow_empty=True)

# D. All four E04/E05 datasets: by actual year, educational level, gender or area (never sum intersecting cross-sections).
for i,a in enumerate(source_asset(segment='05') if CFG.MODE=='FULL' else source_asset(segment='05')[:1],1):
    cols=set(a.get('columns_manifest') or [])
    group=['year','education_level']+[q for q in ('gender','study_area') if q in cols]
    keys=','.join('"'+k+'"' for k in group)
    q=f'''SELECT {keys},COUNT(*)::BIGINT source_records,
          MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) median_reported_graduates,
          MEDIAN(TRY_CAST(percent_employed AS DOUBLE)) median_employed_percent,
          MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) median_wage
          FROM read_parquet(__INPUT__) GROUP BY {keys}'''
    try:
        p,d,n=thematic_sql_disk(a,q,'graduates_source_'+str(i),preview=2000,mem='2GB')
        FAMILY_E04E05.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
           'group_rows':n,'group_columns':','.join(group),'status':'PASS_NOT_ADDITIVE',
           'employment_asof':'2024-12-31','geography_of_work_known':False})
        qa_table('graduate_family_'+str(i),d,allow_empty=True)
        PANEL_FILES['graduate_family_'+str(i)]=p
    except Exception as ex:
        FAMILY_E04E05.append({'asset_id':a['asset_id'],'status':'FAILED_'+type(ex).__name__})
        ISSUES.append({'module':'ML-5.5','status':'FAMILY_PROFILE_FAILED','reason':a['asset_id']+' '+type(ex).__name__,
           'required':'recheck exact statistical strata'})
qa_table('e04_e05_full_family_audit',pd.DataFrame(FAMILY_E04E05),['asset_id'])

# E. Historical municipal reference: full SQL join, no 431k+ rows in Pandas.
for key,asset_prefix in [('D04','06::D04::D04_member00::spatial_reference'),
                         ('BDMO_REG','06::BDMO6::BDMO6_member00::municipality_registry_observed'),
                         ('BDMO_HIST','06::BDMO6::BDMO6_member00::municipality_history_observed')]:
    a=required_one(source_asset(prefix=asset_prefix),key)
    cols=a.get('columns_manifest') or []
    chosen=[x for x in ('source_oktmo','region_name','municipality_name','settlement_name','observation_year',
      'coordinates_valid','latitude','longitude','census_year','observed_name_change','observed_type_change',
      'boundary_change_not_verified') if x in cols]
    if chosen:
        frame=read_table(a,chosen,limit=2000)  # explicitly labelled preview; full source profiled above
        if key=='D04':D04_SPATIAL=frame
        elif key=='BDMO_REG':BDMO_SPATIAL=frame
        else:HISTORY_MUNICIPAL=frame
        qa_table('spatial_'+key+'_preview',frame,allow_empty=True)

registry_asset=required_one(source_asset(prefix='06::BDMO6::BDMO6_member00::municipality_registry_observed'),'BDMO REG')
municipal_path=fetch_asset(MUN); registry_path=fetch_asset(registry_asset)
query=f'''SELECT CAST(d.oktmo AS VARCHAR) oktmo, TRY_CAST(d.year AS INTEGER) observation_year,
    MIN(CAST(d.region AS VARCHAR)) region, MIN(CAST(d.municipality AS VARCHAR)) municipality,
    MIN(CAST(b.municipality_name AS VARCHAR)) municipality_name,
    COUNT(DISTINCT CAST(b.municipality_name AS VARCHAR)) n_registry_names
    FROM read_parquet({_sql_literal(municipal_path)}) d
    INNER JOIN read_parquet({_sql_literal(registry_path)}) b
      ON CAST(d.oktmo AS VARCHAR)=CAST(b.source_oktmo AS VARCHAR)
     AND TRY_CAST(d.year AS INTEGER)=TRY_CAST(b.observation_year AS INTEGER)
    GROUP BY 1,2'''
try:
    with duckdb.connect(':memory:') as con:
        con.execute("SET memory_limit='2GB'");con.execute('SET threads=1')
        joined=con.execute(query).df()  # upper bound: 31,445 D01 rows
    joined['municipality_names_equal']=joined['municipality'].astype(str).str.lower().str.replace('ё','е').str.strip().eq(
       joined['municipality_name'].astype(str).str.lower().str.replace('ё','е').str.strip())
    joined['spatial_gate']='EXACT_OKTMO_YEAR_NAME_COMPARE_ONLY_NOT_CAMPUS'
    SPATIAL_CROSS=joined
    qa_table('spatial_exact_crosswalk',joined,['oktmo','observation_year'],allow_empty=True)
except Exception as ex:
    ISSUES.append({'module':'GEOGRAPHY','status':'CROSSWALK_FAILED','reason':type(ex).__name__+': '+str(ex)[:140],
       'required':'inspect municipality-year and historical OKTMO definitions'})
    qa_table('spatial_exact_crosswalk',pd.DataFrame(),allow_empty=True)

# F. Educational infrastructure D02::education at municipality×year×indicator_code, conservative uniqueness.
edu_source=required_one(source_asset(prefix='02::education'),'BDMO educational indicators')
edu_cols=edu_source['columns_manifest']
# Selection is a *hypothesis candidate*, not an automatic campus register.
pattern='(профессиональн|высш[еиг]|колледж|техникум|университет|образовательн.*организац)'
q=f'''SELECT CAST(oktmo AS VARCHAR) oktmo,TRY_CAST(year AS INTEGER) observation_year,
  CAST(indicator_code AS VARCHAR) indicator_code,CAST(indicator_name AS VARCHAR) indicator_name,
  CAST(indicator_unit AS VARCHAR) indicator_unit,
  COUNT(*)::BIGINT n_raw_rows,
  MIN(TRY_CAST(indicator_value AS DOUBLE)) AS min_reported,
  MAX(TRY_CAST(indicator_value AS DOUBLE)) AS max_reported,
  MEDIAN(TRY_CAST(indicator_value AS DOUBLE)) AS median_reported
  FROM read_parquet(__INPUT__)
  WHERE regexp_matches(lower(CAST(indicator_name AS VARCHAR)), {_sql_literal(pattern)})
  AND oktmo IS NOT NULL AND year IS NOT NULL
  GROUP BY 1,2,3,4,5'''
try:
    if CFG.MODE!='FULL':raise ValueError('MUNICIPAL_ECONOMIC_FULL_SCAN_ONLY')
    fp,preview,n=thematic_sql_disk(edu_source,q,'bdmo_education_municipal',preview=1000,mem='2GB')
    # Bounded filtered result only. If high-cardinality, use SQL to filter exact one-row groups first.
    with duckdb.connect(':memory:') as con:
        con.execute("SET memory_limit='2GB'");con.execute('SET threads=1')
        nuniq=int(con.execute('SELECT COUNT(*) FROM read_parquet(?) WHERE n_raw_rows=1 AND min_reported=max_reported',[str(fp)]).fetchone()[0])
        if nuniq<=CFG.MAX_PANDAS_ROWS:
            BDMO_PROXY_FULL_VALID=True
            BDMO_EDU_PROXY_CANDIDATES=con.execute('''SELECT * FROM read_parquet(?)
              WHERE n_raw_rows=1 AND min_reported=max_reported AND median_reported>=0''',[str(fp)]).df()
        else:
            BDMO_EDU_PROXY_CANDIDATES=preview.copy()
            ISSUES.append({'module':'ML-5.7','status':'PROXY_PREVIEW_ONLY',
               'reason':f'{nuniq} unique municipal education rows exceed RAM cap','required':'SQL-only E0/E1'})
    BDMO_EDU_PROXY_CANDIDATES['source_definition']='BDMO_MUNICIPAL_EDUCATION_INDICATOR_NOT_CAMPUS'
    qa_table('bdmo_education_candidates',BDMO_EDU_PROXY_CANDIDATES,allow_empty=True)
except Exception as ex:
    ISSUES.append({'module':'ML-5.7','status':'BDMO_EDUCATION_REVIEW_SKIPPED_OR_FAILED',
       'reason':type(ex).__name__+': '+str(ex)[:150],'required':'review BDMO indicator units/levels'})
    qa_table('bdmo_education_candidates',pd.DataFrame(),allow_empty=True)

# Human-readable quality conditions, all sourced from actually processed files.
qa_table('municipal_infrastructure_candidates',BDMO_EDU_PROXY_CANDIDATES,allow_empty=True)
print('CROSS SOURCE:',len(D02_PROFILE),'D02 members;',len(VPO_ALL_PROFILE),'E02/VPO;',
      len(FAMILY_E04E05),'E04/E05;',len(SPATIAL_CROSS),'exact municipal code-year matches;',
      len(BDMO_EDU_PROXY_CANDIDATES),'municipal infrastructure candidates')
memory_point('after_cross_source')
mark_stage('CROSS_SOURCE')


## 6Б. Содержательная проверка 52/52 источников
Пофайловое содержание без подмены единиц измерения; годовые СПО и raw-матрицы не суммируются повторно.

In [ ]:
# ML-05 V4 · independent full-content utilization register (52/52).
# These are measurements of the SOURCE itself, not automatically legal predictive features.
progress_event('CROSS_SOURCE','V4: физическое обследование значений всех наборов Stage I',announce=True)
V4_SOURCE_PROFILES=[];V4_SOURCE_ERRORS=[]
for _i,_a in enumerate(ASSETS,1):
    _id=_a['asset_id'];_p=fetch_asset(_a);_cols=list(pq.ParquetFile(_p).schema_arrow.names)
    _time=next((x for x in ('year','observation_year','graduation_year') if x in _cols),None)
    _measure=next((x for x in ('indicator_value','observed_value','count_graduate','population','value','migration') if x in _cols),None)
    _y=f'TRY_CAST("{_time}" AS INTEGER)' if _time else 'NULL::INTEGER'
    _v=f'TRY_CAST("{_measure}" AS DOUBLE)' if _measure else 'NULL::DOUBLE'
    _q=f'''SELECT {_y} year_label,COUNT(*)::BIGINT source_rows,
      COUNT({_v})::BIGINT numeric_values,
      SUM(CASE WHEN {_v}<0 THEN 1 ELSE 0 END)::BIGINT negative_values,
      SUM(CASE WHEN {_v}=0 THEN 1 ELSE 0 END)::BIGINT zero_values,
      SUM(CASE WHEN {_v} IS NULL THEN 1 ELSE 0 END)::BIGINT null_or_non_numeric
      FROM read_parquet({_sql_literal(_p)}) GROUP BY 1'''
    _out=ROOT/'derived'/'v4_source_values'/f'source_{_i:02d}.parquet';_out.parent.mkdir(parents=True,exist_ok=True)
    _tmp=_out.with_suffix('.partial.parquet');_tmp.unlink(missing_ok=True)
    progress_event('CROSS_SOURCE',f'V4 [{_i}/52] число, пропуски, нули: {_id}',_i,len(ASSETS),announce=(_i==1 or _i%5==0 or _i==len(ASSETS)))
    try:
        with duckdb.connect(':memory:') as _db:
            _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
            _db.execute('SET preserve_insertion_order=false')
            _db.execute('SET temp_directory='+_sql_literal(DB_TMP))
            _db.execute('COPY ('+_q+') TO '+_sql_literal(_tmp)+' (FORMAT PARQUET, COMPRESSION ZSTD)')
            _r=_db.execute('SELECT COALESCE(SUM(source_rows),0),COUNT(*) FROM read_parquet(?)',[str(_tmp)]).fetchone()
        if int(_r[0])!=int(_a['rows_actual']):raise AssertionError(f'Content count {_r[0]} != {_a["rows_actual"]}')
        _tmp.replace(_out);CONTEXT_PARQUET_FILES.append(_out)
        V4_SOURCE_PROFILES.append({'asset_id':_id,'segment':_a['segment'],'source_id':_a.get('source_id'),
           'rows_actual':int(_r[0]),'time_field':_time,'numeric_field':_measure,
           'groups_by_year':int(_r[1]),'sha_verified':_id in INPUT_SHA,'content_status':'FULL_VALUE_COVERAGE_PASS',
           'model_eligibility':'REQUIRES_TARGET_SEMANTICS_AND_GRAIN','derived_parquet':_out.name})
    except Exception as _ex:
        _tmp.unlink(missing_ok=True)
        V4_SOURCE_ERRORS.append({'asset_id':_id,'error':type(_ex).__name__,'details':str(_ex)[:280]})
        V4_SOURCE_PROFILES.append({'asset_id':_id,'segment':_a['segment'],'source_id':_a.get('source_id'),
           'rows_actual':None,'time_field':_time,'numeric_field':_measure,'groups_by_year':None,
           'sha_verified':_id in INPUT_SHA,'content_status':'FAIL_CONTENT_SCAN','derived_parquet':''})
_v4_used=pd.DataFrame(V4_SOURCE_PROFILES)
qa_table('v4_all52_source_value_audit',_v4_used,['asset_id'])
qa_table('v4_source_scan_failures',pd.DataFrame(V4_SOURCE_ERRORS),allow_empty=True)
if CFG.MODE=='FULL' and (len(_v4_used)!=52 or len(V4_SOURCE_ERRORS)):
    fail('ALL52_CONTENT','One or more Stage I members failed full numeric content scan: '+str(V4_SOURCE_ERRORS[:2]))

# Source -> defensible research use. Each stage is recorded independently, no claim of 52 model predictors.
_v4_roles=[]
for _a in ASSETS:
    _seg=_a['segment'];_r=_a.get('role','') or ''
    _topics={'01':'Municipal demography and ML-5.7','02':'Economic/education context and ML-5.1/ML-5.7',
      '03':'SPO OD, organizational panels and network ML-5.1/2/3/6',
      '04':'VO monitoring, admissions and specialties ML-5.2/5.4',
      '05':'Graduate employability and outcomes ML-5.5',
      '06':'Historic OKTMO, places and geographic accessibility ML-5.1/5.7'}
    _direct=_a['asset_id'] in {E01['asset_id'],E02['asset_id'],VPO_211['asset_id'],
              GRAD_ORG['asset_id'],GRAD_FIELD['asset_id'],MUN['asset_id'],TRANSPORT['asset_id']}
    _v4_roles.append({'asset_id':_a['asset_id'],'segment':_seg,'role_in_program':_topics.get(_seg),
       'content_scanned':True,'scientific_content_used':'YES_SOURCE_SPECIFIC_PROFILE',
       'main_prediction_input':bool(_direct),
       'not_main_input_reason':('Derived/alternate reporting representation or incompatible key/grain;' 
                                'not pooled into target by assumption' if not _direct else ''),
       'prevent_double_counting':_seg=='03' and (_r=='organization_indicators' or _r=='education_od_matrix_raw_cells'),
       'source_sha':INPUT_SHA.get(_a['asset_id'],'')})
qa_table('v4_all52_research_use',pd.DataFrame(_v4_roles),['asset_id'])
print('V4 / source use:',len(V4_SOURCE_PROFILES),'complete value-audits;',len(V4_SOURCE_ERRORS),'failures')


## 7. Предрегистрация, временные разбиения, модели
Сопоставляются persistence, среднее двух наблюдений, затухающий тренд, Ridge, CatBoost; PPML только для доказанных неотрицательных счётных величин. TEST не выбирает победителя.

In [ ]:
progress_event('MODEL_DESIGN','Разбиения, метрики, базовые методы, Ridge/CatBoost')
# ML-05 · план эксперимента и функции, одинаковые для всех доступных модулей.
EXPERIMENT_PLAN={
 'version':CFG.VERSION,'mode':CFG.MODE,'horizon':CFG.HORIZON,
 'source_commit':SOURCE_COMMIT,'source_index_blob':index_info['sha'],
 'feature_lag':CFG.FEATURE_PUBLICATION_LAG_YEARS,
 'label_lag':CFG.LABEL_PUBLICATION_LAG_YEARS,
 'candidate_models':['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','catboost','lightgbm','ppml_if_observed_integer_counts','negbin_fixed_dispersion_if_integer_counts'],
 'admission_approval_required':True,'unknown_edges_never_zero':True,
 'E01_is_not_certified_admissions':True,
 'validation':'DEV only if its label is available by TEST origin; otherwise fixed preregistered comparison, no selection',
 'seeds_dev':CFG.DEV_SEEDS,'seeds_final':CFG.FINAL_SEEDS,
 'test_keys_policy':'strict common support; no synthetic missing OD edges',
 'source_vintage':INDEX['created_at'],
 'publication_policy':'only FULL + local checks + explicit opt-in + science gate',
}
PLAN_HASH=canonical_hash(EXPERIMENT_PLAN)
PRED_KEYS=['module','target_id','sector','entity_id','origin_year','horizon','target_year',
           'split_id','model','seed','information_cutoff_year','source_vintage',
           'geography_grain','observation_definition','actual','predicted']

# Предварительно заданные признаки; семантика источника не попадает в X как oracle.
CAND_NUM=['prior_value','prior2_value','origin_mass','destination_mass',
          'origin_destinations','destination_origins','distance_proxy_km',
          'confirmed_vo_sites','prev_population','prev_migration_rate',
          'origin_year']
CAND_CAT=['origin_region','destination_region','region','sector','parent_group_id']

def feature_spec(frame):
    nums=[c for c in CAND_NUM if c in frame.columns and number(frame[c]).notna().any()]
    cats=[c for c in CAND_CAT if c in frame.columns and frame[c].notna().any()]
    for c in frame.columns:
        if c.startswith(('origin_econ_','destination_econ_')) and number(frame[c]).notna().any():nums.append(c)
    if 'prior_value' not in nums:fail('MODEL','Нет prior_value: нельзя оценить историческую инерцию')
    return nums,cats

def x_prepare(frame,nums,cats):
    x=frame[nums+cats].copy()
    for c in nums:x[c]=number(x[c]).astype(float)
    for c in cats:x[c]=x[c].fillna('__MISSING__').astype(str)
    return x

def effective_mae(y,yp):
    y=np.asarray(y,dtype=float);p=np.asarray(yp,dtype=float)
    mask=np.isfinite(y)&np.isfinite(p)
    if not mask.any():return {'MAE':np.nan,'RMSE':np.nan,'WAPE':np.nan,'n':0}
    y,p=y[mask],p[mask]
    return {'MAE':float(np.mean(np.abs(y-p))),'RMSE':float(np.sqrt(np.mean((y-p)**2))),
            'WAPE':float(np.abs(y-p).sum()/np.abs(y).sum()) if np.abs(y).sum()>0 else np.nan,'n':int(len(y))}

# Проверка, что один и тот же target не оказался одновременно в train/dev/test.
def temporal_partition(frame,min_train=None,min_test=None):
    if frame.empty:return None,'EMPTY_FRAME'
    f=frame.sort_values('target_year').copy()
    years=sorted(int(x) for x in f.target_year.dropna().unique())
    if len(years)<2:return None,'INSUFFICIENT_TARGET_YEARS'
    test_year=years[-1];test_origin=test_year-CFG.HORIZON
    tr=f[(f.target_year<test_year)&
         (f.target_available_year<=test_origin)].copy()
    test=f[f.target_year==test_year].copy()
    min_train=int(CFG.MIN_TRAIN_ROWS if min_train is None else min_train)
    min_test=int(CFG.MIN_TEST_ROWS if min_test is None else min_test)
    if len(tr)<min_train or len(test)<min_test:
        return None,f'LOW_SUPPORT: train={len(tr)},test={len(test)}'
    # DEV может существовать только если доступен к моменту финального решения.
    available_dev=sorted(int(y) for y in tr.target_year.unique())
    dev=None
    if len(available_dev)>=2:
        dv=available_dev[-1];dev_origin=dv-CFG.HORIZON
        early=tr[(tr.target_year<dv)&(tr.target_available_year<=dev_origin)].copy()
        potential=tr[tr.target_year==dv].copy()
        if len(early)>=min_train and len(potential)>=CFG.MIN_DEV_ROWS:
            dev=potential;tr=early
    if tr.duplicated(['entity_id','target_year']).any():fail('SPLIT','Неверная кратность train')
    if test.duplicated(['entity_id','target_year']).any():fail('SPLIT','Неверная кратность test')
    if (tr.target_available_year>test_origin).any():fail('SPLIT','train label published after test origin')
    if (test.feature_available_year>test.information_cutoff_year).any():fail('SPLIT','future feature in test')
    if dev is not None and (dev.target_available_year>test_origin).any():fail('SPLIT','dev label leaked to test')
    out={'train':tr,'dev':dev,'test':test,'test_year':test_year,'test_origin':test_origin,
         'protocol':'DEV_SELECTED' if dev is not None else 'FIXED_NO_DEV',
         'retrospective_vintage':True}
    SPLIT_AUDIT.append({'module':f.get('module',pd.Series(['UNSET'])).iloc[0],
       'protocol':out['protocol'],'train_rows':len(tr),'dev_rows':0 if dev is None else len(dev),
       'test_rows':len(test),'test_year':test_year,'test_origin':test_origin,
       'labels_known_asof_test':True,'source_vintage_at_test':'NOT_ARCHIVED'})
    return out,None


# V4: fixed-dispersion NB GLM, not mislabeled as an estimated NB2 dispersion model.
from sklearn.base import BaseEstimator, RegressorMixin
class NegativeBinomialFixedDispersion(RegressorMixin, BaseEstimator):
    """NB2 (alpha fixed), penalized MLE with stable log-link and declared prediction-support bound.

    Count targets must be observed nonnegative integers. Fits a low-dimensional, numeric-only
    design with L2 penalty; never densifies the high-cardinality categorical one-hot matrix.
    The predeclared upper bound is 10×max(train count): clipping is reported as diagnostic,
    so predictions cannot silently explode into 1e68 while still receiving PASS.
    """
    def __init__(self, alpha=1.0, penalty=0.5, support_multiplier=10.0):
        self.alpha=float(alpha);self.penalty=float(penalty);self.support_multiplier=float(support_multiplier)
    def fit(self,X,y):
        from scipy.optimize import minimize
        from scipy.special import expit
        a=np.asarray(X,dtype=np.float64); y=np.asarray(y,dtype=np.float64).reshape(-1)
        if a.ndim==1:a=a.reshape(-1,1)
        if a.shape[0]!=len(y) or not len(y) or a.shape[1]>64:raise ValueError('NB2 incompatible design')
        if not np.isfinite(y).all() or (y<0).any() or not np.allclose(y,np.round(y),atol=1e-8,rtol=0):
            raise ValueError('NB2 requires observed integer nonnegative counts')
        a=np.clip(np.nan_to_num(a,nan=0.0,posinf=0.0,neginf=0.0),-8.,8.)
        a=np.column_stack([np.ones(len(y)),a]); r=1.0/self.alpha
        # Stable NB log-likelihood: (y+r)*softplus(eta-log(r))-y*(eta-log(r)), up to constants.
        scale=max(1.,float(np.mean(y+r)));x0=np.zeros(a.shape[1]);x0[0]=np.log(max(.001,float(np.mean(y))))
        def fun(beta):
            e=a@beta-np.log(r)
            loss=np.mean(((y+r)*np.logaddexp(0.,e)-y*e))/scale+0.5*self.penalty*np.dot(beta[1:],beta[1:])
            g=(a.T@((y+r)*expit(e)-y))/(len(y)*scale)
            g[1:]+=self.penalty*beta[1:]
            return float(loss),g
        bounds=[(-22.,22.)]+[(-6.,6.)]*(a.shape[1]-1)
        opt=minimize(fun,x0,method='L-BFGS-B',jac=True,bounds=bounds,options={'maxiter':350,'ftol':1e-10,'gtol':1e-7})
        if not opt.success or not np.isfinite(opt.fun):
            raise RuntimeError('NB2 regularized likelihood did not converge: '+str(opt.message))
        self.coef_=opt.x;self.fit_converged_=True;self.fit_objective_=float(opt.fun)
        self.train_max_=float(np.max(y));self.train_mean_=float(np.mean(y))
        self.n_fit_=len(y);self.count_truncated_=0;self.max_raw_prediction_=np.nan
        return self
    def predict(self,X):
        a=np.asarray(X,dtype=np.float64)
        if a.ndim==1:a=a.reshape(-1,1)
        a=np.clip(np.nan_to_num(a,nan=0.,posinf=0.,neginf=0.),-8.,8.)
        eta=np.column_stack([np.ones(len(a)),a])@self.coef_
        # The model's response space is constrained by a *predeclared* historical support bound.
        # This is not replacement of missing observations. Truncation frequency is an output QC.
        max_mean=max(1.,self.train_max_)*self.support_multiplier
        clipped=np.clip(eta,-30.,np.log(max_mean))
        self.count_truncated_=int(np.count_nonzero(clipped!=eta))
        self.truncated_fraction_=float(self.count_truncated_/max(1,len(a)))
        self.max_raw_logmu_=float(np.max(eta)) if len(a) else np.nan
        pred=np.exp(clipped)
        if not np.isfinite(pred).all():raise FloatingPointError('NB2 unstable response')
        return pred

def pipeline_fit_predict(model,train,test,seed=11,allow_ppml=False,nonnegative=True):
    nums,cats=feature_spec(train)
    if set(nums+cats)&set(['actual','target_year','target_available_year','observed_value']):
        fail('LEAKAGE','В модель попала фактическая целевая переменная или период метки')
    Xtr=x_prepare(train,nums,cats);Xte=x_prepare(test,nums,cats)
    y=number(train.actual).to_numpy(float)
    if not np.isfinite(y).all():fail('MODEL','Невалидные training labels')
    if model=='ridge':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.5)),
                      ('model',Ridge(alpha=30.,solver='lsqr'))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='elasticnet':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.7)),
             ('model',ElasticNet(alpha=1.0,l1_ratio=.35,max_iter=4000,random_state=(None if int(seed)<0 else int(seed)),selection='cyclic'))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='lightgbm':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median'))]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.6)),
             ('model',LGBMRegressor(n_estimators=220,num_leaves=15,max_depth=5,learning_rate=.05,
                       reg_lambda=8.,min_child_samples=15,verbosity=-1,n_jobs=1,random_state=(None if int(seed)<0 else int(seed))))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='ppml':
        if not allow_ppml or np.any(y<0):raise ValueError('PPML requires verified nonnegative counts')
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.5)),
                      ('model',PoissonRegressor(alpha=3.,max_iter=200))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='negbin':
        if not allow_ppml or np.any(y<0) or not np.isfinite(y).all():
            raise ValueError('Negative binomial GLM requires observed nonnegative finite counts')
        # NB2: numeric-only design; avoids high-cardinality one-hot IRLS explosion.
        # All numeric inputs are observed past information, not current target.
        obj=Pipeline([('prep',ColumnTransformer([
                 ('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                          ('scale',StandardScaler())]),nums)],sparse_threshold=0.)),
            ('model',NegativeBinomialFixedDispersion(alpha=1.,penalty=0.5,
                          support_multiplier=CFG.MAX_COUNT_PREDICTION_MULTIPLIER))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        cats=[];artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats,
                  'dispersion':'FIXED_ALPHA_1_PENALIZED_NB2',
                  'extrapolation_rule':'PREDICT_MEAN_AT_MOST_10X_MAX_OBSERVED_TRAIN_COUNT',
                  'prediction_guard_events':int(obj.named_steps['model'].count_truncated_),
                  'prediction_guard_fraction':float(obj.named_steps['model'].truncated_fraction_)}
    elif model=='catboost':
        med={c:float(Xtr[c].median()) if Xtr[c].notna().any() else 0. for c in nums}
        for c in nums:
            Xtr[c]=Xtr[c].fillna(med[c]);Xte[c]=Xte[c].fillna(med[c])
        obj=CatBoostRegressor(iterations=60 if CFG.MODE=='SMOKE' else 260,
                  depth=5,learning_rate=.05,l2_leaf_reg=8.,loss_function='RMSE',
                  random_seed=int(seed),verbose=False,thread_count=1,used_ram_limit='2gb',allow_writing_files=False)
        obj.fit(Xtr,y,cat_features=cats);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats,'medians':med}
    else:fail('MODEL','Неизвестная модель '+model)
    yp=np.asarray(yp,dtype=np.float64)
    if len(yp)!=len(test) or not np.isfinite(yp).all():raise FloatingPointError(model+' invalid prediction array')
    if nonnegative and (yp<0).any():yp=np.maximum(yp,0)
    return yp,artifact

def reload_predict(artifact,test,nonnegative=True):
    X=x_prepare(test,artifact['nums'],artifact['cats'])
    if artifact['kind']=='catboost':
        for c,v in artifact['medians'].items():X[c]=X[c].fillna(v)
    p=artifact['pipeline'].predict(X)
    return np.maximum(p,0) if nonnegative else p

def baseline_prediction(test,kind,nonnegative=True):
    a=number(test.prior_value).to_numpy(float)
    if kind=='persistence':b=a
    elif kind in ('rolling_mean_2','damped_trend'):
        other=number(test.prior2_value).to_numpy(float)
        b=np.where(np.isfinite(other),(a+other)/2 if kind=='rolling_mean_2' else a+.35*(a-other),a)
    else:fail('BASELINE','Неизвестный baseline '+kind)
    return np.maximum(b,0) if nonnegative else b

def predictions_frame(frame,module,model,yp,seed=-1,protocol='ROLLING_TIME'):
    out=pd.DataFrame({'module':module,'target_id':frame.get('target_id',pd.Series(['UNDEFINED']*len(frame))).to_numpy(),
     'sector':frame.get('sector',pd.Series(['UNKNOWN']*len(frame))).to_numpy(),
     'entity_id':frame.entity_id.astype(str).to_numpy(),
     'origin_year':frame.origin_year.to_numpy(),'horizon':CFG.HORIZON,'target_year':frame.target_year.to_numpy(),
     'split_id':protocol,'model':model,'seed':int(seed),
     'information_cutoff_year':frame.information_cutoff_year.to_numpy(),
     'feature_available_year':frame.feature_available_year.to_numpy(),
     'target_available_year':frame.target_available_year.to_numpy(),
     'source_vintage':INDEX['created_at'],
     'geography_grain':'ORIGIN_DESTINATION' if module=='ML-5.1' else ('REGION' if module=='ML-5.3' else 'ORGANIZATION'),
     'observation_definition':frame.get('target_id',pd.Series(['observed_value']*len(frame))).to_numpy(),
     'actual':number(frame.actual).to_numpy(float),'predicted':np.asarray(yp,dtype=float)})
    for c in ('origin_region','destination_region','parent_group_id','region'):
        if c in frame:out[c]=frame[c].astype('string').to_numpy()
    return out

def record_prediction(frame):
    # Прогнозы всех сидов сохраняются немедленно; массивы не остаются в RAM.
    global PRED_TOTAL
    out=ROOT/'derived'/'prediction_parts';out.mkdir(parents=True,exist_ok=True)
    path=out/f'part_{len(PRED_PARTS):05d}.parquet'
    write_df_chunks(frame,path)
    PRED_PARTS.append(str(path));PRED_TOTAL+=int(len(frame))
    del frame
    if len(PRED_PARTS)%10==0:
        gc.collect();memory_point(f'predictions_saved_{len(PRED_PARTS)}')

def fit_and_record(module,model,train,test,seed=-1,nonnegative=True,allow_ppml=False,protocol='ROLLING_TIME'):
    sector=str(test.sector.iloc[0]) if 'sector' in test and test.sector.nunique()==1 else 'MIXED'
    if model in ('persistence','rolling_mean_2','damped_trend'):
        pred=baseline_prediction(test,model,nonnegative);artifact=None
    else:
        pred,artifact=pipeline_fit_predict(model,train,test,seed=seed,allow_ppml=allow_ppml,
                                         nonnegative=nonnegative)
        ident={'module':module,'sector':sector,'protocol':protocol,
               'target_id':sorted(test.target_id.astype(str).unique().tolist()) if 'target_id' in test else [],
               'model':model,'seed':int(seed)}
        short=hashlib.sha256(json.dumps(ident,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:16]
        name=f'{module.replace("-","").replace(".","_")}_{re.sub(r"[^A-Za-z0-9_]+","_",sector)[:28]}_{re.sub(r"[^A-Za-z0-9_]+","_",protocol)[:32]}_{model}_{seed}_{short}.joblib'
        path=ROOT/'models'/name
        if name in MODEL_FILES or path.exists():
            raise RuntimeError('DUPLICATE_MODEL_ARTIFACT: '+name)
        joblib.dump(artifact,path,compress=3)
        loaded=joblib.load(path)
        rp=reload_predict(loaded,test,nonnegative=nonnegative)
        if not np.allclose(rp,pred,rtol=1e-7,atol=1e-7):fail('MODEL_RELOAD',name+' discrepancy')
        MODEL_FILES[name]=str(path)
        MODEL_RECORDS.append({'module':module,'sector':sector,'protocol':protocol,'target_id':ident['target_id'],'model':model,'seed':seed,'path':name,
            'file_sha256':hash_file(path),'feature_num':artifact['nums'],'feature_cat':artifact['cats'],
            'training_rows':len(train),'test_rows':len(test),'reloaded':True,
            'prediction_guard_events':artifact.get('prediction_guard_events',0),
            'prediction_guard_fraction':artifact.get('prediction_guard_fraction',0.)})
    if len(pred)!=len(test):fail('SUPPORT','Модель изменила число тестовых ключей')
    record_prediction(predictions_frame(test,module,model,pred,seed,protocol))
    met=effective_mae(test.actual.to_numpy(float),pred)
    METRICS.append({'module':module,'sector':sector,'model':model,'protocol':protocol,'seed':seed,
       'target_year':int(test.target_year.max()),'source_vintage':INDEX['created_at'],
       **met})
    return pred

mark_stage('MODEL_DESIGN')


## 8. Встроенные интеграционные проверки без GitHub-публикации
11 автоматических тестов на малом искусственном наборе проверяют чистоту времени, восстановление моделей и карантин. Их численные значения не используются как научные результаты.

In [ ]:
progress_event('UNIT_TESTS','Проверка моделей, утечек и повторного чтения')
# Локальные встроенные тесты математической и организационной логики; не являются научными наблюдениями.
SELF_TESTS=[]
def unit_result(name,condition):
    SELF_TESTS.append({'test':name,'result':'PASS' if bool(condition) else 'FAIL'})
    if not bool(condition):fail('UNIT',name)

# Временной split — метки всегда доступны к соответствующему моменту принятия решения.
_fixture=pd.DataFrame([{'entity_id':f'TEST_{e:02d}','observation_year':y,
                        'response':float(e+2*(y-2010)), 'sector':'UNIT'}
                       for e in range(35) for y in range(2012,2026)])
_panel=historical_pair_panel(_fixture[['entity_id','observation_year','response']],['entity_id'],'response')
_panel['sector']='UNIT';_panel['module']='UNIT';_panel['region']='UNIT_REGION'
_panel['target_id']='ARTIFICIAL_TEST_ONLY'
_split,_reason=temporal_partition(_panel)
unit_result('TIME_PANEL_EXISTS',len(_panel)>100)
unit_result('NO_FUTURE_FEATURES',bool((_panel.feature_available_year<=_panel.information_cutoff_year).all()))
unit_result('SPLIT_AVAILABLE',_split is not None)
unit_result('TRAIN_LABELS_KNOWN_AS_OF_TEST',bool((_split['train'].target_available_year<=_split['test_origin']).all()))
unit_result('TEST_KEYS_UNIQUE',not _split['test'].duplicated(['entity_id','target_year']).any())
if _split['dev'] is not None:
    unit_result('DEV_LABELS_KNOWN_AS_OF_TEST',bool((_split['dev'].target_available_year<=_split['test_origin']).all()))

# Модельный прогноз должен повторяться без дообучения после восстановления состояния.
_train=_split['train'];_test=_split['test'];_y_ridge,_ridge_art=pipeline_fit_predict('ridge',_train,_test)
unit_result('RIDGE_RELOAD_EQUAL',np.allclose(_y_ridge,reload_predict(_ridge_art,_test),atol=1e-8,rtol=1e-8))
_cb_y,_cb_art=pipeline_fit_predict('catboost',_train,_test,seed=CFG.DEV_SEEDS[0])
unit_result('CATBOOST_RELOAD_EQUAL',np.allclose(_cb_y,reload_predict(_cb_art,_test),atol=1e-7,rtol=1e-7))
unit_result('MODEL_OUTPUT_FINITE',np.isfinite(_y_ridge).all() and np.isfinite(_cb_y).all())
_en_y,_en_art=pipeline_fit_predict('elasticnet',_train,_test,seed=-1)
unit_result('ELASTICNET_SENTINEL_NEGATIVE_SEED_SAFE',np.isfinite(_en_y).all() and
             np.allclose(_en_y,reload_predict(_en_art,_test)))
_nb_fixture=_train.copy();_nb_test=_test.copy()
for _q in (_nb_fixture,_nb_test):_q['actual']=np.maximum(np.rint(_q.actual),0)
_po_y,_po_art=pipeline_fit_predict('ppml',_nb_fixture,_nb_test,seed=-1,allow_ppml=True)
unit_result('PPML_OBSERVED_INTEGER_FIT_AND_RELOAD',np.isfinite(_po_y).all() and
             np.allclose(_po_y,reload_predict(_po_art,_nb_test)))
_nb_y,_nb_art=pipeline_fit_predict('negbin',_nb_fixture,_nb_test,seed=-1,allow_ppml=True)
unit_result('NEGBIN_FIXED_DISPERSION_RELOAD',np.isfinite(_nb_y).all() and
             np.allclose(_nb_y,reload_predict(_nb_art,_nb_test)))
unit_result('BASELINE_NOT_FUTURE',np.allclose(baseline_prediction(_test,'persistence'),_test.prior_value))

# Спорный id×год отбрасывается с содержательным карантином, не объединяется по имени.
_quarantine_n=len(QUARANTINE)
_bad_registry=pd.DataFrame([{'entity_id':'X','observation_year':2024,'name':'Вуз 1'},
                            {'entity_id':'X','observation_year':2024,'name':'Вуз 2'},
                            {'entity_id':'Y','observation_year':2024,'name':'Колледж'}])
_clean=quarantine_conflicting_registry(_bad_registry,'TEST_ONLY')
unit_result('AMBIGUOUS_REGISTRY_QUARANTINED',len(_clean)==1 and _clean.entity_id.iloc[0]=='Y')
del QUARANTINE[_quarantine_n:] # тестовые записи не являются реальными спорными организациями

qa_table('embedded_unit_tests',pd.DataFrame(SELF_TESTS),['test'])
assert all(r['result']=='PASS' for r in SELF_TESTS)
print('ML-05 embedded tests:',len(SELF_TESTS),'PASS (контрольные данные; не научные результаты)')
del _fixture,_panel,_split,_reason,_train,_test,_ridge_art,_cb_art,_en_art,_en_y,_po_art,_po_y,_nb_art,_nb_y,_nb_fixture,_nb_test,_y_ridge,_cb_y,_quarantine_n,_bad_registry,_clean

mark_stage('UNIT_TESTS')


## 9. Семь предметных блоков ML-5.1–ML-5.7
Научные блоки выполняются на фактически наблюдаемых источниках. При отсутствии target или подтверждённого grain предусмотрен описательный результат или мотивированный отказ от прогноза.

In [ ]:
memory_point('before_model_training')
progress_event('SEVEN_MODULES','Расчёт семи модулей; расширенные статусы',announce=True)
# Семь предметных экспериментов. Наличие модели не является условием выполнения описательной части.
MODULE_DIAGNOSTICS=[];CONFIRMATORY=[];HYPOTHESES=[];MODULE_SUBTASKS=[]

def blocked(module,why,required=''):
    limitation(module,why,required)
    MODULE_DIAGNOSTICS.append({'module':module,'prediction_feasible':False,
        'n_observed':0,'reason':why})

def two_way_mae_bootstrap(base,model,groups_a,groups_b,reps=399,seed=20261104):
    # Two-way pigeonhole bootstrap — строки получают произведение кратностей независимых блоков.
    a=pd.Series(groups_a).fillna('__UNKNOWN__').astype(str).to_numpy()
    b=pd.Series(groups_b).fillna('__UNKNOWN__').astype(str).to_numpy()
    unique_a=np.unique(a);unique_b=np.unique(b)
    if len(unique_a)<8 or len(unique_b)<8:return {'delta_MAE':np.nan,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,'clusters_a':len(unique_a),'clusters_b':len(unique_b)}
    v=np.asarray(model,dtype=float);u=np.asarray(base,dtype=float)
    d=v-u;delta=float(d.mean());rng=np.random.default_rng(seed)
    boot=[]
    for _ in range(reps):
        ca=Counter(rng.choice(unique_a,len(unique_a),replace=True))
        cb=Counter(rng.choice(unique_b,len(unique_b),replace=True))
        w=np.array([ca[x]*cb[y] for x,y in zip(a,b)],dtype=float)
        if w.sum():boot.append(float(np.average(d,weights=w)))
    if len(boot)<100:return {'delta_MAE':delta,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,'clusters_a':len(unique_a),'clusters_b':len(unique_b)}
    arr=np.asarray(boot)
    p=min(1.,2*min((np.sum(arr<=0)+1)/(len(arr)+1),(np.sum(arr>=0)+1)/(len(arr)+1)))
    return {'delta_MAE':delta,'ci_low':float(np.quantile(arr,.025)),
        'ci_high':float(np.quantile(arr,.975)),'p_value':float(p),
        'clusters_a':len(unique_a),'clusters_b':len(unique_b)}

def test_predictive(module,raw,nonnegative=True,count_confirmed=False):
    if raw.empty:return blocked(module,'Нет валидной целевой панели после семантического контроля',
       'entity_id, observed target years, known historical features')
    progress_event('MODELS',f'{module}: подготовка временного теста ({len(raw):,} строк)',announce=True)
    frame=raw.copy();frame['module']=module
    split,reason=temporal_partition(frame,min_train=CFG.REGION_MIN_TRAIN_ROWS if module=='ML-5.3' else None, min_test=CFG.REGION_MIN_TEST_ROWS if module=='ML-5.3' else None)
    if split is None:return blocked(module,reason,'train + independent test + as-of labels')
    train,dev,test=split['train'],split['dev'],split['test']
    cand=['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','catboost','lightgbm']
    if count_confirmed:cand+=['ppml','negbin']
    if CFG.MODE=='SMOKE':cand=['persistence','rolling_mean_2','ridge']
    if CFG.MODE=='PILOT10' and len(train)>10000:
        train=train.sample(n=10000,random_state=CFG.RANDOM_SEED)
    primary='persistence';dev_scores={};dev_selection=False
    if dev is not None:
        # DEV labels действительно доступны к времени TEST-предсказания.
        for model in cand:
            try:
                if model in ('persistence','rolling_mean_2','damped_trend'):
                    p=baseline_prediction(dev,model,nonnegative)
                else:
                    p,_=pipeline_fit_predict(model,train,dev,seed=CFG.DEV_SEEDS[0],
                                             allow_ppml=count_confirmed,nonnegative=nonnegative)
                dev_scores[model]=effective_mae(dev.actual,p)['MAE']
            except Exception as ex:
                note('DEV',module+' '+model+': '+type(ex).__name__,'WARN')
        if dev_scores:
            priority={m:i for i,m in enumerate(cand)}
            primary=min(dev_scores,key=lambda m:(round(dev_scores[m],8),priority[m]))
            dev_selection=True
            # dev labels допустимо добавить в обучение только при as-of совпадении.
            train=pd.concat([train,dev],ignore_index=True)
    else:
        note('TEMPORAL',module+': DEV не опубликован к дате прогноза; модели предзаданы, без выбора по TEST','WARN')
    pred_by_model={}
    for model in cand:
        progress_event('MODELS',f'{module}: расчёт {model}',announce=True)
        seeds=[-1]
        if model=='catboost':
            if CFG.MODE=='SMOKE':seeds=[CFG.DEV_SEEDS[0]]
            elif CFG.MODE=='PILOT10' and dev_selection and primary=='catboost':seeds=list(CFG.DEV_SEEDS)
            elif CFG.MODE=='FULL':seeds=list(CFG.FINAL_SEEDS)[:CFG.ML54_FINAL_SEEDS] if module=='ML-5.4' else list(CFG.FINAL_SEEDS)  # 50 seeds for all FULL temporal tasks
            else:seeds=[CFG.FINAL_SEEDS[0]]
        preds=[]
        for i_seed,seed in enumerate(seeds,1):
            progress_event('MODELS',f'{module}: {model}, сид {i_seed}/{len(seeds)}',i_seed,len(seeds),announce=(i_seed==1 or i_seed%5==0 or i_seed==len(seeds)))
            try:
                p=fit_and_record(module,model,train,test,seed=seed,
                    nonnegative=nonnegative,allow_ppml=count_confirmed)
                preds.append(p)
            except Exception as ex:
                ISSUES.append({'module':module,'status':'MODEL_FAILED',
                     'reason':model+': '+type(ex).__name__+' — '+str(ex)[:180],
                     'required':'Inspect modeling frame; no fabricated substitute'})
                if CFG.MODE=='FULL' and model in ('negbin','ppml','ridge','elasticnet','lightgbm'):
                    note('MODEL_QA',module+' '+model+' failed, recorded as scientific FAIL','FAIL')
                if model==primary:raise
        if not preds:continue
        pred_by_model[model]=np.mean(preds,axis=0)
        if model=='catboost' and len(preds)>1:
            record_prediction(predictions_frame(test,module,'catboost_ensemble',pred_by_model[model],-1))
            METRICS.append({'module':module,'sector':str(test.sector.iloc[0]) if 'sector' in test else 'UNKNOWN','model':'catboost_ensemble','protocol':'ROLLING_TIME',
                 'seed':-1,'target_year':split['test_year'],
                 **effective_mae(test.actual,pred_by_model[model])})
    if 'persistence' not in pred_by_model:fail('MODEL','Нет сопоставимого исторического baseline')
    y=test.actual.to_numpy(float)
    base_err=np.abs(y-pred_by_model['persistence'])
    primary_err=np.abs(y-pred_by_model.get(primary,pred_by_model['persistence']))
    if module=='ML-5.1':
        ga=test.origin_region;gb=test.destination_region
    elif module=='ML-5.3':
        ga=test.origin_region;gb=test.target_year.astype(str)
    else:
        ga=test.parent_group_id.fillna(test.entity_id) if 'parent_group_id' in test else test.entity_id
        gb=test.target_year.astype(str)
    if dev_selection and primary!='persistence':
        ci=two_way_mae_bootstrap(base_err,primary_err,ga,gb,
               reps=99 if CFG.MODE=='SMOKE' else CFG.BOOTSTRAP_REPS,seed=CFG.RANDOM_SEED)
    else:
        ci={'delta_MAE':np.nan,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,
            'clusters_a':len(pd.Series(ga).dropna().unique()),
            'clusters_b':len(pd.Series(gb).dropna().unique())}
    chosen=effective_mae(y,pred_by_model.get(primary,pred_by_model['persistence']))
    original=effective_mae(y,pred_by_model['persistence'])
    CONFIRMATORY.append({'module':module,'sector':str(test.sector.iloc[0]) if 'sector' in test else 'UNKNOWN','test_year':split['test_year'],'test_n':len(test),
        'choice_basis':'DEV_ASOF' if dev_selection else 'NO_CONFIRMATORY_COMPARISON_NO_DEV',
        'selected_model':primary,'baseline_mae':original['MAE'],'selected_mae':chosen['MAE'],
        'delta_mae':chosen['MAE']-original['MAE'],**ci,'source_vintage_warning':True})
    progress_event('MODELS',f'{module}: тест завершён, инерция MAE={original["MAE"]:.3g}, выбранная={chosen["MAE"]:.3g}',announce=True)
    MODULES[module].update(status='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT',
        reason=('Ретроспективный тест с неподтверждёнными историческими версиями источника; '
                'DEV-проверка доступности соблюдена; улучшение не гарантировано'),
        n=len(test),test_year=split['test_year'],selected=primary)
    MODULE_DIAGNOSTICS.append({'module':module,'prediction_feasible':True,
        'n_observed':len(raw),'n_test':len(test),'test_year':split['test_year'],
        'dev_available_asof':dev_selection,'selected_model':primary,
        'mae_baseline':original['MAE'],'mae_selected':chosen['MAE']})
    if not dev_selection:ISSUES.append({'module':module,'status':'SEVERE_TEMPORAL_LIMIT',
       'reason':'Нет DEV-меток, опубликованных до cutoff TEST; сравнения альтернатив только разведочные',
       'required':'older archived vintages or longer observations'})
    return split

# ML-5.1: истинная единица E01 пока не подтверждена как приём; регрессируем именно E01_observed_value.
if len(OD_INTER):
    MODULE_DIAGNOSTICS.append({'module':'ML-5.1','prediction_feasible':None,
        'n_observed':len(OD_INTER),'target_definition':OD_SEMANTIC['label'],
        'unknown_codes':len(OD_UNKNOWN),'diagonal_excluded':len(OD_INTRA),
        'coverage_breaks':int(coverage_df.potential_break.fillna(False).sum())})
    test_predictive('ML-5.1',OD_MODEL,nonnegative=True,count_confirmed=bool(OD_MODEL.actual.ge(0).all() and np.isclose(OD_MODEL.actual,np.rint(OD_MODEL.actual),atol=1e-8,rtol=0).all()))
    if coverage_df.potential_break.fillna(False).any():
        ISSUES.append({'module':'ML-5.1','status':'UNRESOLVED_YEAR_COVERAGE_BREAK',
          'reason':'Смена числа OD-пар в 2023 г. не доказана как изменение реальной мобильности; модельные результаты — условно разведочные',
          'required':'semantics/coverage harmonization across 2022–2023'})
else:blocked('ML-5.1','Нет истинно наблюдаемых межрегиональных E01-рёбер')

# ML-5.2: отдельно ВО/СПО; не моделировать учреждения, если не удалось установить уникальную величину.
if len(ORG_YEAR):
    for sector,group in ORG_MODEL.groupby('sector') if len(ORG_MODEL) else []:
        if len(group):
            result={'status':'NOT_STARTED','reason':'source not evaluated','n':0}
            try:
                test_predictive('ML-5.2',group,nonnegative=True,count_confirmed=True)
                result=MODULES['ML-5.2'].copy()
            except Exception as ex:
                result={'status':'FAILED','reason':f'{sector}: {type(ex).__name__}: {str(ex)[:100]}','n':0}
                raise
            finally:
                MODULE_SUBTASKS.append({'module':'ML-5.2','sector':sector,**result})
            MODULE_DIAGNOSTICS.append({'module':'ML-5.2','sector':sector,'n_organizations':group.entity_id.nunique(),
                    'n_observed':len(group)})
    if MODULES['ML-5.2']['status']=='NOT_STARTED':
        limitation('ML-5.2','Установлены потенциальные индикаторы, но недостаточно исторических пар для модели',
                   'admissions×organization×year with unambiguous units')
else:
    limitation('ML-5.2','Предметный каталог E02/VPO/SPO прочитан, но однозначное зачисление организация×год не подтверждено',
               'source-admission indicator and organization with unique key')

if MODULE_SUBTASKS:
    qa_table('organization_sector_subtasks',pd.DataFrame(MODULE_SUBTASKS),allow_empty=True)
    statuses=[r['status'] for r in MODULE_SUBTASKS]
    if any(x.startswith('TESTED') for x in statuses):
        MODULES['ML-5.2']['status']='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT'
        MODULES['ML-5.2']['reason']='Отдельные сектора: '+', '.join(f"{r['sector']}={r['status']}" for r in MODULE_SUBTASKS)

# ML-5.3: только межрегиональные маршруты из региона получения общего образования.
if len(OUTFLOW_MODEL):test_predictive('ML-5.3',OUTFLOW_MODEL,nonnegative=True,count_confirmed=bool(OUTFLOW_MODEL.actual.ge(0).all() and np.isclose(OUTFLOW_MODEL.actual,np.rint(OUTFLOW_MODEL.actual),atol=1e-8,rtol=0).all()))
else:blocked('ML-5.3','Недостаточно наблюдений агрегированного межрегионального E01',
             'origin schooling region, destination study region, comparable years')

# ML-5.4: реальная программа, не фиктивное происхождение по специальности.
if len(PROGRAM_PROFILE):
    pp=PROGRAM_PROFILE.copy()
    pp['denominator']=pp.groupby(['observation_year','indicator_name']).observed_value.transform('sum')
    pp['relative_share_qualifier']='CONDITIONAL_ON_EXACT_INDICATOR_AND_OBSERVED_PROGRAM_RECORDS'
    pp['share_observed']=np.where(pp.denominator>0,pp.observed_value/pp.denominator,np.nan)
    pp['herfindahl_component']=pp.share_observed**2
    concentration=pp.groupby(['observation_year','indicator_name'],as_index=False).agg(
        reported_programs=('specialty_code','nunique'),reported_value=('observed_value','sum'),
        HHI=('herfindahl_component','sum'))
    qa_table('program_concentration',concentration,allow_empty=True)
    qa_table('program_distribution',pp,allow_empty=True)
    MODULES['ML-5.4'].update(status='DESCRIPTIVE_VALID',reason='Программы — условный срез конкретных индикаторов и кодов; OD×programme отсутствует',n=len(pp))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.4','prediction_feasible':False,
       'n_observed':len(pp),'reason':'Нельзя прогнозировать выбор программы по OD без OD×programme'})
elif len(VPO_PROGRAM_DESCRIPTION):
    MODULES['ML-5.4'].update(status='DESCRIPTIVE_VALID',
        reason='Реальные направления ВПО-1 по категориям и статистическим уровням, без двойного суммирования и ненаблюдаемого OD×programme',
        n=len(VPO_PROGRAM_DESCRIPTION))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.4','prediction_feasible':False,
      'n_observed':len(VPO_PROGRAM_DESCRIPTION),'reason':'Представлены медианы исходных разрезов; точные когорты программы не установлены'})
else:limitation('ML-5.4','Доступные словари программ не дали согласованного временного показателя',
                'stable specialty codes, verified target, level and year')

# ML-5.5: полная E04/E05 обрабатывается потоково в DuckDB; не сравнивать preview со всей базой.
if CFG.MODE=='FULL':
    if GRAD_E04_VALID_ROWS>0 and not GRADUATE_GROUP_SUMMARY.empty:
        MODULES['ML-5.5'].update(status='DESCRIPTIVE_VALID',
            reason='Полные E04/E05: статистические группировки всех наблюдений в DuckDB; employment_asof 2024-12-31; RETENTION_NOT_OBSERVABLE',
            n=GRAD_E04_VALID_ROWS)
        MODULE_DIAGNOSTICS.append({'module':'ML-5.5','prediction_feasible':False,
            'n_observed_e04':GRAD_E04_VALID_ROWS,'n_observed_e05':GRAD_E05_VALID_ROWS,
            'employment_asof':'2024-12-31','reason':'Срез на одной референсной дате; место работы не подтверждено'})
    else:
        limitation('ML-5.5','Агрегат E04 по всем строкам пуст после валидации показателя')
elif len(GRADUATES):
    gr=GRADUATES.dropna(subset=['count_graduate']).copy()
    gr=gr[gr.count_graduate.ge(0)]
    gr_summary=gr.groupby(['graduation_year','object_level','gender','education_level'],dropna=False).agg(
       n_source_records=('university','size'),median_reported_graduates=('count_graduate','median'),
       median_observed_salary=('average_salary_fact_med','median'),
       median_employment_percentage=('percent_employed','median')).reset_index()
    gr_summary['aggregate_totals_identified']=False
    qa_table('graduate_descriptive_statistics',gr_summary,allow_empty=True)
    if 'graduate_specialty_outcomes' in PANELS:
        f=PANELS['graduate_specialty_outcomes'].dropna(subset=['graduation_year','specialty_code'])
        f=f[f.count_graduate.notna() & f.count_graduate.ge(0)]
        source_groups=f.groupby(['graduation_year','object_level','education_level','specialty_code'],dropna=False).agg(
          observed_records=('specialty','size'),median_reported_graduates=('count_graduate','median'),
          median_wage=('average_salary_fact_med','median')).reset_index()
        source_groups['aggregate_totals_identified']=False
        qa_table('graduate_specialty_description',source_groups,allow_empty=True)
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_VALID',
      reason='SMOKE/PILOT: описательная статистика по ограниченным данным; retention неизвестен',n=len(gr))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.5','prediction_feasible':False,'n_observed':len(gr),
      'employment_asof':'2024-12-31','reason':'Одна дата занятости; нет географии работы'})
else:limitation('ML-5.5','E04/E05: нет валидной описательной статистики')

# ML-5.6: динамическая сеть, петли исключены; сетевые изменения с учетом неодинакового покрытия.
NETWORK=[];NETWORK_EDGES=[];COMMUNITIES=[];NET_SETS={}
for year,sub in OD_INTER.dropna(subset=['observed_value']).groupby('observation_year'):
    progress_event('NETWORK',f'Год {year}: расчёт наблюдаемой сети',announce=True)
    g=nx.DiGraph()
    for r in sub.itertuples(index=False):
        if pd.notna(r.observed_value) and r.observed_value>0:
            g.add_edge(str(r.origin_region),str(r.destination_region),weight=float(r.observed_value))
    if nx.number_of_selfloops(g):fail('NETWORK','Обнаружена диагональ в межрегиональном графе')
    if not g.number_of_edges():continue
    pager=nx.pagerank(g,weight='weight')
    undirected=g.to_undirected()
    try:comms=nx.community.louvain_communities(undirected,weight='weight',seed=CFG.RANDOM_SEED)
    except Exception:comms=list(nx.community.greedy_modularity_communities(undirected,weight='weight'))
    membership={node:i for i,community in enumerate(comms) for node in community}
    NET_SETS[int(year)]=set((a,b) for a,b in g.edges())
    for node in g.nodes():
        incoming=g.in_degree(node,weight='weight');outgoing=g.out_degree(node,weight='weight')
        vals=np.asarray([float(d['weight']) for _,_,d in g.out_edges(node,data=True)],float)
        sh=vals/vals.sum() if vals.sum()>0 else np.asarray([],float)
        out_hhi=float((sh*sh).sum()) if len(sh) else np.nan
        out_entropy=float(-(sh*np.log(sh)).sum()) if len(sh) else np.nan
        NETWORK.append({'year':int(year),'region_code':node,'in_strength':incoming,'out_strength':outgoing,
          'outbound_HHI_observed':out_hhi,'outbound_entropy_observed':out_entropy,
          'pagerank':pager.get(node,np.nan),'community_id':membership.get(node,-1),
          'n_in':g.in_degree(node),'n_out':g.out_degree(node),'observed_edges':g.number_of_edges()})
    for a,b,d in g.edges(data=True):NETWORK_EDGES.append({'year':int(year),'origin':a,'destination':b,
        'value':d['weight']})
    COMMUNITIES.append({'year':int(year),'communities':len(comms),
        'n_regions':len(g.nodes()),'n_edges':g.number_of_edges(),
        'coverage_break_warning':bool(coverage_df.loc[coverage_df.year==int(year),'potential_break'].fillna(False).any())})
qa_table('education_network_year',pd.DataFrame(NETWORK),allow_empty=True)
qa_table('education_network_edges',pd.DataFrame(NETWORK_EDGES),allow_empty=True)
qa_table('education_network_communities',pd.DataFrame(COMMUNITIES),allow_empty=True)
if len(NET_SETS)>=2:
    stability=[]
    for a,b in zip(sorted(NET_SETS)[:-1],sorted(NET_SETS)[1:]):
        x,y=NET_SETS[a],NET_SETS[b]
        stability.append({'year_from':a,'year_to':b,'edge_jaccard':len(x&y)/len(x|y) if x|y else np.nan,
                          'n_edges_from':len(x),'n_edges_to':len(y),
                          'coverage_break_warning':bool(coverage_df.loc[coverage_df.year==b,'potential_break'].fillna(False).any())})
    qa_table('network_stability',pd.DataFrame(stability),allow_empty=True)
if len(NETWORK):
    MODULES['ML-5.6'].update(status='DESCRIPTIVE_VALID',reason='Сетевые показатели наблюдаемой E01 без самопетель; покрытие лет различается',n=len(NETWORK))
else:limitation('ML-5.6','Нет валидных взвешенных межрегиональных ребер E01')

# ML-5.7: одинаковые муниципалитеты для E0/E1; пропущенный факт нахождения кампуса нельзя считать нулём.
context=MUNI_EDU[MUNI_EDU.education_coverage_known].copy()
if (context.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1 and
        context.observation_year.nunique()>=CFG.MIN_YEARS_FOR_TEMPORAL and
        context.confirmed_vo_sites.nunique()>1):
    x=context[['oktmo','observation_year','migration_rate','population','confirmed_vo_sites']].rename(
      columns={'oktmo':'entity_id','migration_rate':'actual','population':'prev_population'})
    # Не подтягивать число университетов из целевого года: lag, as-of.
    m0=historical_pair_panel(x[['entity_id','observation_year','actual']],['entity_id'],'actual')
    lag=context.rename(columns={'oktmo':'entity_id','observation_year':'historical_year',
                                'confirmed_vo_sites':'confirmed_vo_sites',
                                'population':'prev_population'})
    m0=m0.merge(lag[['entity_id','historical_year','confirmed_vo_sites','prev_population']],
        on=['entity_id','historical_year'],validate='many_to_one',how='left')
    m0['target_id']='municipal_migration_rate';m0['sector']='MUNICIPAL_CONTEXT'
    qa_table('municipal_e0e1_model_frame',m0,allow_empty=True)
    # Прогнозная процедура E0/E1 — одна и та же модель и поддержка; отключаем инфра в E0.
    split,reason=temporal_partition(m0)
    if split:
        tr=split['train'];te=split['test'];features0=['prior_value','prev_population','origin_year']
        features1=features0+['confirmed_vo_sites']
        for version,features in [('E0',features0),('E1',features1)]:
            model=Pipeline([('imputer',SimpleImputer(strategy='median')),
                            ('scaler',StandardScaler()),('reg',Ridge(alpha=20.))])
            model.fit(tr[features],tr.actual)
            predictions=model.predict(te[features])
            record_prediction(predictions_frame(te,'ML-5.7',version,predictions))
            METRICS.append({'module':'ML-5.7','sector':'MUNICIPAL_CONTEXT','model':version,'protocol':'COMMON_SUPPORT_MUNICIPAL',
              'seed':-1,'target_year':split['test_year'],**effective_mae(te.actual,predictions)})
            fp=ROOT/'models'/f'ML_5_7_{version}.joblib';joblib.dump({'model':model,'features':features},fp)
            reload=joblib.load(fp)
            assert np.allclose(reload['model'].predict(te[features]),predictions)
            MODEL_FILES[fp.name]=str(fp)
        MODULES['ML-5.7'].update(status='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT',
            reason='E0/E1 на одинаковом подтверждённом составе ОКТМО',n=len(te))
    else:limitation('ML-5.7',reason,'more confirmed municipality-years')
else:
    limitation('ML-5.7',f'Недостаточно подтвержденного кампусного покрытия: {context.oktmo.nunique()} ОКТМО; неизвестно != 0',
               'verified municipality campus registry with historical validity')

# Все семь модулей получают явный предметный статус; ни один не теряется.
for key,result in MODULES.items():
    if result['status']=='NOT_STARTED':
        limitation(key,'Данные просканированы, но применимый протокол не смог сформировать результат')
qa_table('seven_module_status',pd.DataFrame([{'module':k,**v} for k,v in MODULES.items()]),['module'])
qa_table('module_diagnostics',pd.DataFrame(MODULE_DIAGNOSTICS),allow_empty=True)
print('MODULES:',{k:v['status'] for k,v in MODULES.items()})

memory_point('after_seven_modules')
mark_stage('SEVEN_MODULES')


## 9А. Дополнительные исследования ML-5.2–ML-5.7

Строго отделяются проверенный приём и неутверждённые кандидаты, описательная структура программ и индивидуальный выбор, занятость выпускников и территориальное закрепление. Региональные и муниципальные экспериментальные протоколы запускаются только на проверенной общей области наблюдения. Сетевые показатели проверяются на устойчивость. Отрицательные результаты фиксируются без замены тестовой модели задним числом.

In [ ]:
# ML-05: substantive completion of all seven modules with explicit evidence gates.
progress_event('EXTENDED_MODULES','Дополнительные исследования ML-5.2–ML-5.7 и сетевой стресс-тест',announce=True)
EXT_MODULES=[]

# ML-5.2: distinguish candidates, verified admission, and non-additive section splits.
if 'CANDIDATE_CATALOG' in globals() and len(CANDIDATE_CATALOG):
    cand=CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.semantic_role.eq('ADMISSION_CANDIDATE')].copy()
    sector_rows=cand.groupby('source',dropna=False).agg(indicator_names=('indicator_name','nunique'),
                         rows_in_source=('rows','sum')).reset_index() if len(cand) else pd.DataFrame()
    if len(sector_rows):
        sector_rows['approved_entity_year_targets']=len(ORG_YEAR)
        sector_rows['interpretation']='SOURCE_CANDIDATES_ONLY_NOT_VERIFIED_ADMISSIONS'
    qa_table('ml52_admission_candidates',sector_rows,allow_empty=True)
    if len(sector_rows) and not len(ORG_YEAR):
        MODULES['ML-5.2'].update(status='DESCRIPTIVE_ONLY_TARGET_UNAPPROVED',n=int(sector_rows.rows_in_source.sum()),
          reason='Семантические кандидаты действительно найдены, но утверждённой численности приёма организация×год нет')
        EXT_MODULES.append({'module':'ML-5.2','evidence':'SEMANTIC_CANDIDATES_NOT_COUNTS',
          'n_candidates':int(sector_rows.indicator_names.sum()),'prediction_status':'NOT_FEASIBLE'})

# ML-5.4: whole-dataset specialty histories, not a preview of the first 2,000 aggregate rows.
ML54_HISTORY=pd.DataFrame()
if VPO_FULL_PROGRAM_PATH and Path(VPO_FULL_PROGRAM_PATH).exists():
    p=Path(VPO_FULL_PROGRAM_PATH)
    q='''SELECT observation_year,CAST(specialty_code AS VARCHAR) specialty_code,
      CAST(indicator_name AS VARCHAR) indicator_name,
      SUM(observed_records)::BIGINT rows_in_source,
      COUNT(*)::BIGINT independently_reported_strata,
      MEDIAN(median_reported_value) median_of_source_stratum_medians
      FROM read_parquet(?)
      WHERE specialty_code IS NOT NULL AND TRY_CAST(observation_year AS INT) IS NOT NULL
      GROUP BY 1,2,3'''
    with duck_connect() as con:
        n=int(con.execute('SELECT COUNT(*) FROM ('+q+')',[str(p)]).fetchone()[0])
        if n<=CFG.MAX_PANDAS_ROWS:
            ML54_HISTORY=con.execute(q,[str(p)]).df()
        else:
            out=ROOT/'derived'/'ml54_full_history.parquet'
            con.execute('COPY ('+q.replace('read_parquet(?)','read_parquet('+_sql_literal(p)+')')+
              ') TO '+_sql_literal(out)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
            PANEL_FILES['ml54_specialty_history']=out
            ML54_HISTORY=con.execute('SELECT * FROM read_parquet(?) LIMIT 2000',[str(out)]).df()
    qa_table('ml54_specialty_history',ML54_HISTORY,allow_empty=True)
    if n:
        MODULES['ML-5.4'].update(status='DESCRIPTIVE_FULL_SOURCE_NOT_OD_PROGRAMME',n=n,
          reason='Всё содержимое VPO-1 сгруппировано по годам/кодам/показателям; составное медианное значение не равно численности зачисленных')
    EXT_MODULES.append({'module':'ML-5.4','full_specialty_year_indicator_groups':n,
          'forecast_valid':'NO_VERIFIED_ADMISSIONS_TARGET','unit':'source_records_and_medians_not_additive'})

# Exploratory specialty continuity forecast for stable time series (not a verified admission forecast).
# Predictions are directly row counts of source records in a published statistical strata grid.
ML54_TEST=pd.DataFrame()
if len(ML54_HISTORY) and 'rows_in_source' in ML54_HISTORY and 'observation_year' in ML54_HISTORY:
    z=ML54_HISTORY.copy();z=z.dropna(subset=['observation_year','specialty_code','indicator_name'])
    z['observation_year']=pd.to_numeric(z.observation_year,errors='coerce')
    z=z.sort_values(['specialty_code','indicator_name','observation_year'])
    g=z.groupby(['specialty_code','indicator_name'],sort=False)
    z['lag1_rows']=g.rows_in_source.shift(1)
    z['lag1_year']=g.observation_year.shift(1)
    last=int(z.observation_year.max()) if len(z) else 0
    ML54_TEST=z[(z.observation_year.eq(last))&(z.lag1_year.eq(last-1))&z.lag1_rows.notna()].copy()
    if len(ML54_TEST):
        ML54_TEST['abs_error_persistence']=(ML54_TEST.rows_in_source-ML54_TEST.lag1_rows).abs()
        ML54_TEST['measure_definition']='VPO_REPORTING_STRATA_COVERAGE_ONLY'
        qa_table('ml54_history_persistence_diagnostic',ML54_TEST,allow_empty=True)
        EXT_MODULES.append({'module':'ML-5.4','coverage_persistence_year':last,
          'test_cells':len(ML54_TEST),'MAE_source_row_coverage':float(ML54_TEST.abs_error_persistence.mean()),
          'scientific_gate':'NOT_ADMISSIONS_OR_STUDENT_FLOW'})

# ML-5.5: all 4 E04/E05 sources, male/female strata kept separate, no artificial employment location.
GRAD_DISPARITIES=pd.DataFrame()
family_frames=[]
for i in range(1,5):
    name='graduate_family_'+str(i)
    path=PANEL_FILES.get(name)
    if path and Path(path).exists():
        with duck_connect() as con:
            f=con.execute('SELECT * FROM read_parquet(?) LIMIT 30000',[str(path)]).df()
        f['asset_id']=source_asset(segment='05')[i-1]['asset_id']
        family_frames.append(f)
if family_frames:
    GRAD_DISPARITIES=pd.concat(family_frames,ignore_index=True,sort=False)
    GRAD_DISPARITIES['employment_reference']='2024-12-31'
    GRAD_DISPARITIES['outcome_grain']='MEDIAN_OF_PUBLISHED_AGGREGATE_RECORDS'
    qa_table('ml55_outcome_disparities',GRAD_DISPARITIES,allow_empty=True)
    EXT_MODULES.append({'module':'ML-5.5','source_families_analysed':len(family_frames),
        'output_strata':len(GRAD_DISPARITIES),'time_reference':'2024-12-31',
        'retention':'NOT_OBSERVABLE_WITHOUT_WORK_LOCATION'})
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_ALL_FOUR_FAMILIES',
       reason='E04/E05 всех четырёх типов изучены по исходным разрезам, без двойного суммирования; закрепление неизвестно',n=len(GRAD_DISPARITIES))

# ML-5.6: reciprocity, removal robustness, SCC/WCC and stable network coverage.
ML56_STRESS=[]
for yr,sub in pd.DataFrame(NETWORK_EDGES).groupby('year') if len(NETWORK_EDGES) else []:
    g=nx.DiGraph()
    for r in sub.itertuples(index=False):
        if r.value>0 and r.origin!=r.destination:g.add_edge(str(r.origin),str(r.destination),weight=float(r.value))
    if not g.number_of_nodes():continue
    pg=nx.pagerank(g,weight='weight');leading=max(pg,key=pg.get)
    reciprocal=sum(1 for u,v in g.edges() if g.has_edge(v,u))
    w0=len(max(nx.weakly_connected_components(g),key=len))
    s0=len(max(nx.strongly_connected_components(g),key=len))
    h=g.copy();h.remove_node(leading)
    w1=len(max(nx.weakly_connected_components(h),key=len)) if h.number_of_nodes() else 0
    edges_before=g.number_of_edges();edges_after=h.number_of_edges()
    ML56_STRESS.append({'year':int(yr),'n_nodes':g.number_of_nodes(),'n_edges':edges_before,
       'reciprocity_directed_edge_share':reciprocal/edges_before if edges_before else np.nan,
       'largest_weak_component':w0,'largest_strong_component':s0,'removed_top_pagerank_region':leading,
       'weak_component_after_removal':w1,'edges_after_removal':edges_after,
       'lost_edges_percent':100*(edges_before-edges_after)/edges_before if edges_before else np.nan,
       'interpretation':'STRUCTURAL_STRESS_NOT_EMPIRICAL_MIGRATION_DISRUPTION'})
qa_table('ml56_network_stress',pd.DataFrame(ML56_STRESS),allow_empty=True)
if len(ML56_STRESS):
    MODULES['ML-5.6'].update(status='DESCRIPTIVE_FULL_NETWORK_STRESS',n=len(NETWORK),
      reason='Динамические сети, PageRank, Louvain, концентрация, reciprocity и устойчивость удаления центрального узла; сопоставимость лет ограничена')
    EXT_MODULES.append({'module':'ML-5.6','n_years':len(ML56_STRESS),'validation':'OBSERVED_EDGES_ONLY'})

# ML-5.7: BDMO education-infrastructure proxy E0/E1, same OKTMO-year rows, no fabricated university locations.
ML57_SUPPORT=pd.DataFrame();ML57_RESULT={}
if BDMO_PROXY_FULL_VALID and len(BDMO_EDU_PROXY_CANDIDATES) and len(MUNI):
    e=BDMO_EDU_PROXY_CANDIDATES.dropna(subset=['oktmo','observation_year','indicator_code','median_reported']).copy()
    e['oktmo']=norm_code(e.oktmo);e['observation_year']=safe_year(e.observation_year)
    e=e[e.n_raw_rows.eq(1)&e.median_reported.ge(0)].copy()
    if len(e):
        mig=MUNI[['oktmo','observation_year','migration_rate','population']].drop_duplicates(['oktmo','observation_year']).copy()
        mig['observation_year']=safe_year(mig.observation_year)
        # only same historically observed municipality, lookback=one year at origin.
        mig=mig.sort_values(['oktmo','observation_year'])
        mig['prev_year']=mig.observation_year-1
        past=mig[['oktmo','observation_year','migration_rate','population']].rename(
             columns={'observation_year':'prev_year','migration_rate':'lag_migration_rate','population':'lag_population'})
        mig=mig.merge(past,on=['oktmo','prev_year'],how='inner',validate='many_to_one')
        for c in ('migration_rate','lag_migration_rate','lag_population'):mig[c]=number(mig[c])
        last=int(mig.observation_year.max()) if len(mig) else None
        # Candidate selection entirely on PRE-TEST historical years; never on test labels.
        prior=e[e.observation_year.lt(last)] if last is not None else e.iloc[:0]
        stats=prior.groupby('indicator_code',dropna=False).agg(n_muni=('oktmo','nunique'),
           n_years=('observation_year','nunique'),n_rows=('median_reported','size'),
           n_distinct=('median_reported','nunique')).reset_index()
        stats=stats[(stats.n_muni>=CFG.MIN_MUNICIPALITIES_E0E1)&
                    (stats.n_years>=CFG.MIN_YEARS_FOR_TEMPORAL)&(stats.n_distinct>1)]
        if len(stats):
            stats=stats.sort_values(['n_muni','n_years','n_rows','indicator_code'],ascending=[False,False,False,True])
            selected=str(stats.iloc[0].indicator_code)
            x=e[e.indicator_code.astype(str).eq(selected)][['oktmo','observation_year','median_reported']].copy()
            x=x.rename(columns={'observation_year':'prev_year','median_reported':'lag_education_proxy'})
            # municipality's published education indicator is used at t-1 only, not t.
            ML57_SUPPORT=mig.merge(x,on=['oktmo','prev_year'],how='inner',validate='many_to_one')
            ML57_SUPPORT=ML57_SUPPORT.dropna(subset=['migration_rate','lag_migration_rate',
                               'lag_population','lag_education_proxy'])
            ML57_SUPPORT['indicator_code']=selected
            ML57_SUPPORT['source_definition']='BDMO_PUBLISHED_EDUCATION_PROXY_NOT_VERIFIED_CAMPUS'
            qa_table('ml57_common_support',ML57_SUPPORT,allow_empty=True)
            tr=ML57_SUPPORT[ML57_SUPPORT.observation_year.lt(last)].copy()
            te=ML57_SUPPORT[ML57_SUPPORT.observation_year.eq(last)].copy()
            if (len(tr)>=max(50,CFG.REGION_MIN_TRAIN_ROWS) and
                len(te)>=CFG.MIN_MUNICIPALITIES_E0E1 and tr.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1):
                # Identical E0/E1 keys and model, change ONLY historic education covariate.
                estimates={};baselines=te.lag_migration_rate.to_numpy(float)
                baseline_score=effective_mae(te.migration_rate,baselines)
                for vers,features in [('E0',['lag_migration_rate','lag_population']),
                                      ('E1',['lag_migration_rate','lag_population','lag_education_proxy'])]:
                    model=Pipeline([('imputer',SimpleImputer(strategy='median')),
                                   ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                    model.fit(tr[features].astype(float),tr.migration_rate.to_numpy(float))
                    pred=model.predict(te[features].astype(float));estimates[vers]=effective_mae(te.migration_rate,pred)
                    fp=ROOT/'models'/('ML_5_7_proxy_'+vers+'.joblib')
                    artifact={'pipeline':model,'features':features,'proxy_indicator_code':selected}
                    joblib.dump(artifact,fp,compress=3);rp=joblib.load(fp)['pipeline'].predict(te[features].astype(float))
                    if not np.allclose(rp,pred):fail('ML-5.7','Model serialization discrepancy')
                    MODEL_FILES[fp.name]=str(fp)
                    MODEL_RECORDS.append({'module':'ML-5.7','sector':'MUNICIPAL_BDMO_PROXY','model':vers,
                      'seed':-1,'path':fp.name,'reloaded':True,'training_rows':len(tr),'test_rows':len(te),
                      'feature_num':features,'feature_cat':[],'file_sha256':hash_file(fp)})
                    out=te[['oktmo','observation_year','migration_rate','prev_year']].copy()
                    out=out.rename(columns={'oktmo':'entity_id','observation_year':'target_year',
                         'migration_rate':'actual','prev_year':'origin_year'})
                    out['information_cutoff_year']=out.origin_year
                    out['feature_available_year']=out.origin_year
                    out['target_available_year']=out.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
                    out['target_id']='D01_MUNICIPAL_MIGRATION_RATE'
                    out['sector']='MUNICIPAL_BDMO_PROXY'
                    pf=predictions_frame(out,'ML-5.7',vers,pred,seed=-1,protocol='RETROSPECTIVE_COMMON_SUPPORT')
                    pf['geography_grain']='MUNICIPALITY';record_prediction(pf)
                    METRICS.append({'module':'ML-5.7','sector':'MUNICIPAL_BDMO_PROXY','model':vers,
                       'protocol':'RETROSPECTIVE_COMMON_SUPPORT','seed':-1,'target_year':last,
                       'source_vintage':INDEX['created_at'],**estimates[vers]})
                ML57_RESULT={'status':'RETROSPECTIVE_PROXY_E0E1','year':last,'n_train':len(tr),
                  'n_test':len(te),'n_muni_test':te.oktmo.nunique(),'indicator_code':selected,
                  'inertia_MAE':baseline_score['MAE'],'e0_MAE':estimates['E0']['MAE'],
                  'e1_MAE':estimates['E1']['MAE'],'delta_MAE_E1_minus_E0':estimates['E1']['MAE']-estimates['E0']['MAE']}
                MODULES['ML-5.7'].update(status='RETROSPECTIVE_MUNICIPAL_PROXY_TEST',n=len(te),
                    reason='E0/E1 на общих ОКТМО с опубликованным BDMO-показателем: не доказана фактическая образовательная площадка; причинные выводы запрещены')
if not ML57_RESULT:
    ML57_RESULT={'status':'NOT_FEASIBLE_WITH_VERIFIED_GEOGRAPHY',
      'reason':'Недостаточная муниципальная область наблюдения или отсутствующий сопоставимый показатель инфраструктуры'}
EXT_MODULES.append({'module':'ML-5.7',**ML57_RESULT})
# ML-5.1 generalization checks: both time and territory blocked, no test-based hyperparameter selection.
SPATIAL_MODEL_RESULTS=[]
if len(OD_MODEL):
    frame=OD_MODEL.copy()
    frame['module']='ML-5.1'
    years=sorted(int(y) for y in frame.target_year.dropna().unique())
    if len(years)>=2:
        ty=years[-1];origin=ty-CFG.HORIZON
        origins=sorted(frame.origin_region.dropna().astype(str).unique())
        destinations=sorted(frame.destination_region.dropna().astype(str).unique())
        def stable_holdout(code,salt):
            return int(hashlib.sha256((salt+'|'+str(code)).encode()).hexdigest()[:8],16)%5==0
        oo={r for r in origins if stable_holdout(r,'ML5_ORIGIN_HOLDOUT')}
        dd={r for r in destinations if stable_holdout(r,'ML5_DEST_HOLDOUT')}
        protocols=[('TERRITORY_ONLY',frame.origin_region.astype(str).isin(oo),
                    ~frame.origin_region.astype(str).isin(oo)),
                   ('CROSSED',frame.origin_region.astype(str).isin(oo)&frame.destination_region.astype(str).isin(dd),
                    ~frame.origin_region.astype(str).isin(oo)&~frame.destination_region.astype(str).isin(dd))]
        for protocol,is_test,is_train in protocols:
            tr=frame[frame.target_year.lt(ty)&frame.target_available_year.le(origin)&is_train].copy()
            te=frame[frame.target_year.eq(ty)&is_test].copy()
            meta={'module':'ML-5.1','protocol':protocol,'train_rows':len(tr),'test_rows':len(te),
                  'n_origins':te.origin_region.nunique(),'n_destinations':te.destination_region.nunique(),
                  'date_origin':origin,'test_year':ty,'selection_basis':'PREDECLARED_GEOGRAPHIC_HASH'}
            if len(tr)<CFG.MIN_TRAIN_ROWS or len(te)<CFG.MIN_TEST_ROWS:
                meta['status']='NOT_FEASIBLE_SUPPORT';SPATIAL_MODEL_RESULTS.append(meta);continue
            # Persist the baseline on precisely the same test keys as each spatial model.
            pbase=fit_and_record('ML-5.1','persistence',tr,te,seed=-1,
                                 nonnegative=True,protocol=protocol)
            reference=effective_mae(te.actual,pbase)
            meta['baseline_mae']=reference['MAE'];meta['status']='TESTED_RETROSPECTIVE_EXPLORATORY'
            # Single fixed seed; no TEST choice; same held-out keys for all models.
            for model in ('ridge','elasticnet','lightgbm'):
                try:
                    yp=fit_and_record('ML-5.1',model,tr,te,seed=CFG.RANDOM_SEED,
                                     nonnegative=True,allow_ppml=False,protocol=protocol)
                    mt=effective_mae(te.actual,yp)
                    meta[model+'_MAE']=mt['MAE'];meta[model+'_minus_baseline']=mt['MAE']-reference['MAE']
                except (ValueError,RuntimeError) as ex:
                    ISSUES.append({'module':'ML-5.1','status':'SPATIAL_MODEL_FAILURE',
                        'reason':protocol+' '+model+': '+type(ex).__name__,'required':'inspect spatial support'})
            SPATIAL_MODEL_RESULTS.append(meta)
qa_table('ml51_spatial_protocols',pd.DataFrame(SPATIAL_MODEL_RESULTS),allow_empty=True)
if len(SPATIAL_MODEL_RESULTS):
    EXT_MODULES.append({'module':'ML-5.1','protocols':len(SPATIAL_MODEL_RESULTS),
      'tested':sum(x['status'].startswith('TESTED') for x in SPATIAL_MODEL_RESULTS),
      'interpretation':'SPATIAL_GENERALIZATION_RETROSPECTIVE_CAVEATS'})

qa_table('module_52_extension',pd.DataFrame(EXT_MODULES),allow_empty=True)
qa_table('seven_module_status_final',pd.DataFrame([{'module':k,**v} for k,v in MODULES.items()]),['module'])
print('Extended modules:',[(k,v['status']) for k,v in MODULES.items()])
mark_stage('EXTENDED_MODULES')


## 9Б. Региональный приём по специальностям и вузовский контингент
Подтверждённые количества по ВПО-1, отличные от отсутствующего приёма организации.

In [ ]:
# V4: measured organization outcomes and complete regional specialty admissions, with separate scientific targets.
progress_event('EXTENDED_MODULES','V4: реальный приём по специальностям ВПО-1; организация — отдельная величина',announce=True)
V4_SUBEXPERIMENTS=[]

def v4_block(name,reason,required='',rows=0):
    V4_SUBEXPERIMENTS.append({'subexperiment':name,'status':'NOT_FEASIBLE_FOR_TARGET',
       'reason':reason,'required':required,'n':int(rows)})
    note('V4_DATA_GATE',name+': '+reason,'WARN')

# Source VPO_2_1_1 has exact textual 'Принято, всего'. This is NOT a campus/branch observation.
_v4_vpo_p=fetch_asset(VPO_211)
V4_ADMISSION_PATH=ROOT/'derived'/'v4_regional_specialty_admissions.parquet'
_v4_s=f'''SELECT TRY_CAST(year AS INTEGER) observation_year, CAST(object_level AS VARCHAR) object_level,
    CAST(object_name AS VARCHAR) object_name, CAST(specialty_code AS VARCHAR) specialty_code,
    CAST(education_level AS VARCHAR) education_level,CAST(education_form AS VARCHAR) education_form,
    CAST(admission_type AS VARCHAR) admission_type,CAST(university_type AS VARCHAR) university_type,
    COUNT(*)::BIGINT records_in_cell,MIN(TRY_CAST(indicator_value AS DOUBLE)) minimum,
    MAX(TRY_CAST(indicator_value AS DOUBLE)) maximum
  FROM read_parquet({_sql_literal(_v4_vpo_p)})
  WHERE TRIM(CAST(indicator_name AS VARCHAR))='Принято, всего'
    AND regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект')
    AND specialty_code IS NOT NULL AND year IS NOT NULL
  GROUP BY 1,2,3,4,5,6,7,8'''
_v4_tmp=V4_ADMISSION_PATH.with_suffix('.partial.parquet');_v4_tmp.unlink(missing_ok=True)
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='2048MB'");_db.execute('SET threads=1')
    _db.execute('SET temp_directory='+_sql_literal(DB_TMP));_db.execute('SET preserve_insertion_order=false')
    _db.execute('COPY ('+_v4_s+') TO '+_sql_literal(_v4_tmp)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
    _v4_counts=_db.execute('''SELECT COUNT(*),COUNT(*) FILTER (WHERE records_in_cell=1 AND minimum=maximum
      AND minimum>=0 AND ABS(minimum-ROUND(minimum))<1e-8),COUNT(DISTINCT observation_year)
      FROM read_parquet(?)''',[str(_v4_tmp)]).fetchone()
_v4_tmp.replace(V4_ADMISSION_PATH);CONTEXT_PARQUET_FILES.append(V4_ADMISSION_PATH)
qa_table('v4_specialty_admission_gate',pd.DataFrame([{'source_id':VPO_211['asset_id'],
   'full_source_rows':int(VPO_211['rows_actual']),'exact_name':'Принято, всего',
   'regional_specialty_strata':int(_v4_counts[0]),'valid_singleton_nonnegative_integer_strata':int(_v4_counts[1]),
   'years':int(_v4_counts[2]),'source_grain':'REGION_X_SPECIALTY_X_EDUCATION_LEVEL_X_FORM_X_ADMISSION_TYPE_X_UNIVERSITY_TYPE',
   'warning':'NOT_ORGANIZATION_ADMISSIONS_AND_NOT_OD_BY_SPECIALTY'}]))
V4_PROGRAM_FRAME=pd.DataFrame()
if int(_v4_counts[1])>0:
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _v4_q='''SELECT * FROM read_parquet(?) WHERE records_in_cell=1 AND minimum=maximum
             AND minimum>=0 AND ABS(minimum-ROUND(minimum))<1e-8
             AND observation_year IS NOT NULL AND specialty_code!='' '''
        _v4_n=int(_db.execute('SELECT COUNT(*) FROM ('+_v4_q+')',[str(V4_ADMISSION_PATH)]).fetchone()[0])
        if CFG.MODE=='FULL' and CFG.ML54_FULL_NO_STRATA_CAP:
            # ALL verified strata (not 'longest 15000'); 237k observed in V4.
            # Pandas materialization is restricted to verified regional/specialty cells,
            # not the 7.5 million source rows. Never truncate study population.
            V4_PROGRAM_FRAME=_db.execute(_v4_q,[str(V4_ADMISSION_PATH)]).df()
            V4_PROGRAM_FULL_VALID_ROWS=int(len(V4_PROGRAM_FRAME))
        elif _v4_n<=CFG.MAX_PANDAS_ROWS:
            V4_PROGRAM_FRAME=_db.execute(_v4_q,[str(V4_ADMISSION_PATH)]).df()
            V4_PROGRAM_FULL_VALID_ROWS=int(len(V4_PROGRAM_FRAME))
        else:
            raise RuntimeError('ML54_FULL_STRATA_REQUIRED: no discretionary sample allowed')
    if int(_v4_n)!=int(len(V4_PROGRAM_FRAME)):
        raise RuntimeError(f'ML54_COVERAGE_LOSS: eligible={_v4_n}, materialized={len(V4_PROGRAM_FRAME)}')
    qa_table('v5_ml54_full_strata_coverage',pd.DataFrame([{
        'verified_rows':int(_v4_n),'materialized_rows':int(len(V4_PROGRAM_FRAME)),
        'fraction':1.0,'bounded_sample_used':False}]))
if len(V4_PROGRAM_FRAME):
    _f=V4_PROGRAM_FRAME.copy()
    for _c in ['object_name','specialty_code','education_level','education_form','admission_type','university_type']:
        _f[_c]=_f[_c].fillna('__MISSING__').astype(str)
    _fields=['object_name','specialty_code','education_level','education_form','admission_type','university_type']
    _f['entity_id']=_f[_fields].agg('|'.join,axis=1)
    _f['value']=_f['minimum'].astype(float)
    # A unique reporting stratum must not be interpreted as the sum of its components.
    _f=_f.drop_duplicates(['entity_id','observation_year'],keep=False)
    _f=_f[['entity_id','observation_year','value','object_name','specialty_code','education_level']]
    _rows=historical_pair_panel(_f[['entity_id','observation_year','value']],['entity_id'],'value')
    if len(_rows):
        _rows['sector']='VO_REGIONAL_SPECIALTY'
        _rows['target_id']='VPO_2_1_1_REPORTED_ADMITTED_EXACT_STRATUM'
        _rows['geography_grain']='REPORTED_REGION_SPECIALTY'
        qa_table('v4_specialty_model_frame',_rows,allow_empty=True)
        test_predictive('ML-5.4',_rows,nonnegative=True,count_confirmed=True)
        if MODULES['ML-5.4']['status'].startswith('TESTED'):
            MODULES['ML-5.4']['reason']='Exact VPO admitted value regional specialty/level/form/admission_type stratum; not OD and not institution-level'
            V4_SUBEXPERIMENTS.append({'subexperiment':'ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS',
               'status':'MODEL_TESTED','n':len(_rows),'source':VPO_211['asset_id']})
        else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','No historically available train/test for exact cells',rows=len(_rows))
    else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','Too few historical observations under publication lag',rows=len(_f))
else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','No unique regional speciality admission strata',rows=int(_v4_counts[0]))
del V4_PROGRAM_FRAME;gc.collect()

# ML-5.2: do not auto-approve a number of admitted students from name-matched E02.
# A separate observed 'Приведенный контингент студентов' is a permissible alternative target,
# but it CANNOT be reported as organization admissions or interregional entrants.
_v4_e02_p=fetch_asset(E02)
_v4_org_target='Приведенный контингент студентов'
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='1024MB'");_db.execute('SET threads=1')
    _org=_db.execute('''SELECT CAST(id AS VARCHAR) id,TRY_CAST(year AS INTEGER) observation_year,
        COUNT(*)::BIGINT n_source_records,COUNT(DISTINCT CAST(area AS VARCHAR)) n_area,
        MIN(TRY_CAST(indicator_value AS DOUBLE)) low_value,MAX(TRY_CAST(indicator_value AS DOUBLE)) high_value
        FROM read_parquet(?) WHERE indicator_name=? AND id IS NOT NULL AND year IS NOT NULL
        GROUP BY 1,2''',[str(_v4_e02_p),_v4_org_target]).df()
qa_table('v4_organization_contingent_candidates',_org,allow_empty=True)
_org=_org[_org.n_source_records.eq(1)&_org.low_value.notna()&_org.low_value.eq(_org.high_value)&_org.low_value.ge(0)].copy()
if len(_org):
    _org['entity_id']='VO|'+norm_code(_org.id)
    _observed=_org[['entity_id','observation_year','low_value']].rename(columns={'low_value':'value'})
    _observed=_observed.drop_duplicates(['entity_id','observation_year'],keep=False)
    _panel=historical_pair_panel(_observed,['entity_id'],'value')
    if len(_panel):
        _panel['sector']='VO';_panel['target_id']='E02_REPORTED_STUDENT_CONTINGENT_NOT_ADMISSIONS'
        _panel['parent_group_id']=_panel['entity_id'];_panel['geography_grain']='ORGANIZATION'
        _split,_why=temporal_partition(_panel,min_train=CFG.MIN_TRAIN_ROWS,min_test=CFG.MIN_TEST_ROWS)
        if _split is not None:
            _tr=_split['train'];_te=_split['test']
            for _model in ('persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm'):
                fit_and_record('ML-5.2',_model,_tr,_te,seed=CFG.RANDOM_SEED,nonnegative=True,
                  protocol='ALT_CONTINGENT_RETROSPECTIVE')
            qa_table('v4_organization_contingent_model_frame',_panel,allow_empty=True)
            V4_SUBEXPERIMENTS.append({'subexperiment':'ML-5.2_ALTERNATIVE_CONTINGENT','status':'MODEL_TESTED_NOT_ADMISSIONS',
                                     'n':len(_te),'target':_v4_org_target})
            MODULES['ML-5.2'].update(status='ALTERNATIVE_TARGET_TESTED_ADMISSIONS_NOT_FEASIBLE',
                  reason='E02 student contingent model evaluated; original number admitted still unavailable for organization × year',n=len(_te))
        else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT',str(_why),rows=len(_panel))
    else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT','Insufficient stable organization/year contingent panel',rows=len(_org))
else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT','No unique organization/year source value; cannot sum area splits',rows=len(_org))
v4_block('ML-5.2_ORGANIZATION_ADMISSIONS','No certified admitted-count × organization × year in Stage I; student contingent is not admissions',
         'exact admission count, organization/branch identifiers, year and verified indicator/units') if len(ORG_YEAR)==0 else None
qa_table('v4_additional_subexperiments',pd.DataFrame(V4_SUBEXPERIMENTS),allow_empty=True)
progress_event('EXTENDED_MODULES','V4: проверка фактических величин ВПО/ВО завершена',announce=True)


## 9Г. ML-5.2 · углублённый временной эксперимент по организации и образовательной страте

Целевая величина — реально опубликованный контингент Е02. Это **не** число зачисленных; наблюдения разных страт не суммируются и не выдаются за организационный итог.

In [ ]:
# V5 · дополнительный строгий эксперимент ML-5.2: контингент организация × образовательная страта.
# Не равно приёму, не суммируется по перекрывающимся направлениям, не подменяет ML-5.2 admission target.
progress_event('EXTENDED_MODULES','V5 ML-5.2: проверка 106 тыс. исходных наблюдений по образовательным стратам',announce=True)
V5_ORG_AREA_STATUS={'status':'NOT_FEASIBLE_SOURCE_GRAIN','source':E02['asset_id']}
_v5_52_file=fetch_asset(E02)
_v5_52_sql='''SELECT CAST(id AS VARCHAR) id,TRY_CAST(year AS INTEGER) observation_year,
    COALESCE(CAST(area AS VARCHAR),'__NA__') area,
    COALESCE(CAST(specialty_code AS VARCHAR),'__NA__') specialty_code,
    COALESCE(CAST(specialty_section AS VARCHAR),'__NA__') specialty_section,
    COALESCE(CAST(indicator_unit AS VARCHAR),'__NA__') indicator_unit,
    COUNT(*)::BIGINT n_original_records,
    MIN(TRY_CAST(indicator_value AS DOUBLE)) min_value,
    MAX(TRY_CAST(indicator_value AS DOUBLE)) max_value
    FROM read_parquet(?) WHERE TRIM(CAST(indicator_name AS VARCHAR))='Приведенный контингент студентов'
      AND id IS NOT NULL AND year IS NOT NULL GROUP BY 1,2,3,4,5,6'''
with duckdb.connect(':memory:') as _con:
    _con.execute("SET memory_limit='1536MB'");_con.execute('SET threads=1')
    _v5_52_raw=_con.execute(_v5_52_sql,[str(_v5_52_file)]).df()
_v5_52_eligible=_v5_52_raw.loc[
    _v5_52_raw.n_original_records.eq(1)&_v5_52_raw.min_value.notna()&
    _v5_52_raw.min_value.eq(_v5_52_raw.max_value)&_v5_52_raw.min_value.ge(0)].copy()
V5_ORG_AREA_STATUS.update(raw_strata=len(_v5_52_raw),uniquely_reported_strata=len(_v5_52_eligible),
    duplicate_or_ambiguous_strata=int(len(_v5_52_raw)-len(_v5_52_eligible)))
if len(_v5_52_eligible):
    # The full combination, including area and specialty, is the statistical entity.
    _v5_52_eligible['entity_id']='VOAREA|'+_v5_52_eligible[
        ['id','area','specialty_code','specialty_section','indicator_unit']].astype(str).agg(
        lambda row:hashlib.sha256(json.dumps(row.tolist(),ensure_ascii=False).encode()).hexdigest()[:24],axis=1)
    _v5_52_eligible=_v5_52_eligible.drop_duplicates(['entity_id','observation_year'],keep=False)
    _v5_52_obs=_v5_52_eligible[['entity_id','observation_year','min_value']].rename(columns={'min_value':'value'})
    _v5_52_panel=historical_pair_panel(_v5_52_obs,['entity_id'],'value')
    V5_ORG_AREA_STATUS['complete_source_entity_year_rows']=len(_v5_52_obs)
    V5_ORG_AREA_STATUS['eligible_lagged_rows']=len(_v5_52_panel)
    qa_table('v5_ml52_organization_area_source_strata',_v5_52_eligible,allow_empty=True)
    if len(_v5_52_panel):
        _v5_52_panel['sector']='VO_CONTINGENT_AREA';_v5_52_panel['target_id']='E02_CONTINGENT_ORGANIZATION_AREA_NOT_ADMISSIONS'
        _v5_52_panel['parent_group_id']=_v5_52_panel.entity_id
        _v5_52_panel['geography_grain']='ORGANIZATION_AREA_SPECIALTY'
        _v5_52_split,_v5_52_reason=temporal_partition(_v5_52_panel)
        if _v5_52_split is not None:
            progress_event('EXTENDED_MODULES','ML-5.2: полный разведочный временной эксперимент по контингенту',announce=True)
            test_predictive('ML-5.2',_v5_52_panel,nonnegative=True,count_confirmed=False)
            if MODULES['ML-5.2']['status'].startswith('TESTED'):
                MODULES['ML-5.2'].update(status='ALTERNATIVE_ORG_AREA_CONTINGENT_TESTED_NOT_ADMISSIONS',
                    reason='Observed E02 student contingent by organization/area/specialty; NOT admitted cohorts, not institutional total')
                V5_ORG_AREA_STATUS.update(status='ALTERNATIVE_RETROSPECTIVE_TESTED_NOT_ADMISSIONS',
                    n_test=int(len(_v5_52_split['test'])),target_year=int(_v5_52_split['test_year']))
        else:
            V5_ORG_AREA_STATUS.update(status='NOT_FEASIBLE_HISTORICAL_SUPPORT',reason=str(_v5_52_reason))
else:
    V5_ORG_AREA_STATUS['reason']='No uniquely reported organization-area-year source cell, no admissible target'
qa_table('v5_ml52_area_model_execution',pd.DataFrame([V5_ORG_AREA_STATUS]),allow_empty=True)
print('ML-5.2 alternative:',V5_ORG_AREA_STATUS['status'],'| scientific admission target still gated')
del _v5_52_raw,_v5_52_eligible
import gc;gc.collect()


## 9В. E0/E1 по муниципальной образовательной инфраструктуре и выпускники
Полный дисковый SQL по БДМО; отдельный групповой тест занятости без выводов о закреплении.

In [ ]:
# V4. ML-5.7 disk-backed E0/E1; ML-5.5 independently held-out geography test.
progress_event('EXTENDED_MODULES','V4: E0/E1 полный муниципальный SQL и выпускники E04',announce=True)
V4_MUNICIPAL_STATUS={};V4_GRADUATE_STATUS={}
_v4_bdmo_file=ROOT/'derived'/'full52_context'/'bdmo_education_municipal.parquet'
if CFG.MODE=='FULL' and _v4_bdmo_file.is_file():
    _mi=MUNI[['oktmo','observation_year','migration_rate','population']].copy()
    _mi['oktmo']=norm_code(_mi.oktmo)
    for _c in ('migration_rate','population'):_mi[_c]=number(_mi[_c])
    _mi=_mi.dropna(subset=['oktmo','observation_year','migration_rate'])
    _mi=_mi.drop_duplicates(['oktmo','observation_year'],keep=False)
    _mi['target_year']=_mi.observation_year.astype(int)
    _mi['origin_year']=_mi.target_year-CFG.HORIZON
    _mi['feature_year']=_mi.origin_year-CFG.FEATURE_PUBLICATION_LAG_YEARS
    _hist=_mi[['oktmo','observation_year','migration_rate','population']].rename(columns={
        'observation_year':'feature_year','migration_rate':'lag_migration_rate','population':'lag_population'})
    _mi=_mi.merge(_hist,on=['oktmo','feature_year'],validate='many_to_one',how='inner')
    _mi=_mi.dropna(subset=['lag_migration_rate','lag_population'])
    _latest=int(_mi.target_year.max()) if len(_mi) else 0
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _db.execute('SET temp_directory='+_sql_literal(DB_TMP))
        _db.register('m',_mi)
        _q='''SELECT m.oktmo,m.target_year,m.origin_year,m.feature_year,m.migration_rate actual,
            m.lag_migration_rate,m.lag_population,CAST(e.indicator_code AS VARCHAR) indicator_code,
            CAST(e.indicator_name AS VARCHAR) indicator_name,CAST(e.indicator_unit AS VARCHAR) indicator_unit,
            e.median_reported education_historic_value
            FROM m JOIN read_parquet(?) e ON CAST(m.oktmo AS VARCHAR)=CAST(e.oktmo AS VARCHAR)
               AND m.feature_year=e.observation_year
            WHERE e.n_raw_rows=1 AND e.min_reported=e.max_reported AND e.median_reported>=0
              AND e.indicator_code IS NOT NULL'''
        _matched=ROOT/'derived'/'v4_municipality_education_exact_support.parquet'
        _db.execute('COPY ('+_q.replace('read_parquet(?)','read_parquet('+_sql_literal(_v4_bdmo_file)+')')+
                    ') TO '+_sql_literal(_matched)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
        # Indicators are selected by training coverage, unit constancy and values, not TEST errors.
        _stats=_db.execute('''SELECT indicator_code,indicator_unit,MIN(indicator_name) AS example_indicator_name,
            COUNT(*) FILTER (WHERE target_year<=?) n_train_candidates,
            COUNT(DISTINCT oktmo) FILTER (WHERE target_year<=?) n_municipalities_train,
            COUNT(DISTINCT target_year) FILTER (WHERE target_year<=?) n_years_train,
            COUNT(DISTINCT education_historic_value) FILTER (WHERE target_year<=?) n_distinct_train,
            COUNT(*) FILTER (WHERE target_year=?) n_test_coverage,
            COUNT(DISTINCT oktmo) FILTER (WHERE target_year=?) n_municipalities_test,
            COUNT(DISTINCT indicator_name) n_indicator_names,
            COUNT(*) as rows_all
            FROM read_parquet(?) GROUP BY 1,2''',
          [_latest-2]*4+[_latest]*2+[str(_matched)]).df()
        _stats=_stats.sort_values(['n_municipalities_train','n_test_coverage','indicator_code'],ascending=[False,False,True])
        qa_table('v4_ml57_indicator_support',_stats,allow_empty=True)
        _eligible=_stats.loc[(_stats.n_train_candidates>=50)&(_stats.n_municipalities_train>=CFG.MIN_MUNICIPALITIES_E0E1)&
            (_stats.n_years_train>=2)&(_stats.n_distinct_train>1)&
            (_stats.n_municipalities_test>=CFG.MIN_MUNICIPALITIES_E0E1)&(_stats.n_indicator_names==1)]
        if len(_eligible):
            _eligible=_eligible.copy()
            _eligible['is_professional_education']=_eligible.example_indicator_name.astype(str).str.contains(
                r'профессиональн|высш|колледж|техникум|университет',case=False,regex=True,na=False)
            _eligible=_eligible.sort_values(['is_professional_education','n_municipalities_train','n_test_coverage','indicator_code'],
                ascending=[False,False,False,True])
            _one=_eligible.iloc[0];_selected=str(_one.indicator_code);_unit=str(_one.indicator_unit)
            V4_MUNICIPAL_INDICATOR_SEMANTIC='PROFESSIONAL_EDUCATION_CONTEXT' if bool(_one.is_professional_education) else 'GENERAL_EDUCATION_PROXY_ONLY'
            _rows=_db.execute('''SELECT * FROM read_parquet(?) WHERE indicator_code=?
                  AND COALESCE(indicator_unit,'')=COALESCE(?,'')''',[str(_matched),_selected,_unit]).df()
        else:_rows=pd.DataFrame();_selected=None
    CONTEXT_PARQUET_FILES.append(_matched)
    V4_MUNICIPAL_STATUS={'candidate_indicators':len(_stats),'matched_municipality_indicator_year_rows':int(pq.ParquetFile(_matched).metadata.num_rows),
       'target_year':_latest,'feasible_indicators':len(_eligible),'selected_code':_selected,'selected_semantic':locals().get('V4_MUNICIPAL_INDICATOR_SEMANTIC','UNAVAILABLE')}
    if len(_rows):
        _rows=_rows.sort_values(['oktmo','target_year','indicator_code'])
        _rows=_rows.drop_duplicates(['oktmo','target_year'],keep=False)
        _tr=_rows[_rows.target_year.le(_latest-2)].dropna(subset=['actual','lag_migration_rate','lag_population','education_historic_value']).copy()
        _te=_rows[_rows.target_year.eq(_latest)].dropna(subset=['actual','lag_migration_rate','lag_population','education_historic_value']).copy()
        if len(_tr)>=50 and len(_te)>=CFG.MIN_MUNICIPALITIES_E0E1:
            # Historical features are at target_year - 1 - publication lag; not current-year infrastructure.
            assert (_tr.feature_year<_tr.target_year).all() and (_te.feature_year<=_te.origin_year).all()
            assert (_tr.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS<=_te.origin_year.min()).all()
            qa_table('v4_ml57_exact_common_support',_rows,allow_empty=True)
            _p0=['lag_migration_rate','lag_population'];_p1=_p0+['education_historic_value']
            _results=[]
            for _label,_features in [('E0',_p0),('E1',_p1)]:
                _mod=Pipeline([('fill',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                _mod.fit(_tr[_features].to_numpy(float),_tr.actual.to_numpy(float))
                _yp=_mod.predict(_te[_features].to_numpy(float))
                _f=ROOT/'models'/f'v4_ml57_{_label}.joblib'
                joblib.dump({'model':_mod,'features':_features,'indicator_code':_selected},_f,compress=3)
                _loaded=joblib.load(_f)
                assert np.allclose(_loaded['model'].predict(_te[_features].to_numpy(float)),_yp)
                MODEL_FILES[_f.name]=str(_f)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'BDMO_EDUCATION_PROXY','model':_label,'seed':-1,
                  'path':_f.name,'file_sha256':hash_file(_f),'feature_num':_features,'feature_cat':[],
                  'training_rows':len(_tr),'test_rows':len(_te),'reloaded':True})
                _p=pd.DataFrame({'entity_id':_te.oktmo.astype(str).to_numpy(),
                  'origin_year':_te.origin_year.to_numpy(),'target_year':_te.target_year.to_numpy(),
                  'information_cutoff_year':_te.origin_year.to_numpy(),
                  'feature_available_year':_te.origin_year.to_numpy(),
                  'target_available_year':_te.target_year.to_numpy()+CFG.LABEL_PUBLICATION_LAG_YEARS,
                  'sector':'BDMO_EDUCATION_PROXY','target_id':'D01_MUNICIPAL_MIGRATION_RATE',
                  'actual':_te.actual.to_numpy()})
                _pred=predictions_frame(_p,'ML-5.7',_label,_yp,protocol='RETROSPECTIVE_PROXY_E0E1')
                _pred['geography_grain']='MUNICIPALITY';record_prediction(_pred)
                _mt=effective_mae(_te.actual,_yp)
                METRICS.append({'module':'ML-5.7','sector':'BDMO_EDUCATION_PROXY','model':_label,
                    'protocol':'RETROSPECTIVE_PROXY_E0E1','seed':-1,'target_year':_latest,
                    'source_vintage':INDEX['created_at'],**_mt})
                _results.append({'variant':_label,'n_test':len(_te),'n_train':len(_tr),'municipalities_test':_te.oktmo.nunique(),**_mt})
            qa_table('v4_ml57_results',pd.DataFrame(_results),allow_empty=True)
            V4_MUNICIPAL_STATUS.update(status='E0_E1_TESTED_PROXY_NOT_VERIFIED_CAMPUS',
                 n_train=len(_tr),n_test=len(_te),MAE_E0=_results[0]['MAE'],MAE_E1=_results[1]['MAE'])
            MODULES['ML-5.7'].update(status='RETROSPECTIVE_MUNICIPAL_PROXY_TEST',
                  reason='BDMO lagged '+V4_MUNICIPAL_INDICATOR_SEMANTIC+' E0/E1; same OKTMO-year support, DOES NOT verify campus or causal effect',n=len(_te))
        else:V4_MUNICIPAL_STATUS.update(status='NOT_FEASIBLE_TEMPORAL_SUPPORT',train=len(_tr),test=len(_te))
    else:V4_MUNICIPAL_STATUS['status']='NOT_FEASIBLE_EXACT_SHARED_SUPPORT'
else:V4_MUNICIPAL_STATUS={'status':'NOT_FEASIBLE_NO_BDMO_FULL_FILE'}
qa_table('v4_ml57_execution',pd.DataFrame([V4_MUNICIPAL_STATUS]),allow_empty=True)

# ML-5.5: out-of-region validation of employment rate on E04's original reporting rows.
# This is deliberately cross-sectional, not a forecast of retention or postgraduation residence.
progress_event('EXTENDED_MODULES','V4: трудоустройство E04, проверка вне регионов обучения',announce=True)
_gpath=fetch_asset(GRAD_ORG)
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
    _gg=_db.execute('''SELECT CAST(object_name AS VARCHAR) region_name,
      CAST(object_level AS VARCHAR) object_level,CAST(university AS VARCHAR) university,
      CAST(gender AS VARCHAR) gender,CAST(education_level AS VARCHAR) education_level,
      TRY_CAST(year AS INTEGER) graduation_year,
      TRY_CAST(count_graduate AS DOUBLE) count_graduate,
      TRY_CAST(percent_employed AS DOUBLE) employed_pct
      FROM read_parquet(?) WHERE regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект')
      AND TRY_CAST(percent_employed AS DOUBLE) BETWEEN 0 AND 100
      AND TRY_CAST(count_graduate AS DOUBLE)>=0''',[str(_gpath)]).df()
_gkeys=['region_name','university','gender','education_level','graduation_year']
_gduplicates=_gg.duplicated(_gkeys,keep=False).sum()
_gg=_gg.drop_duplicates(_gkeys,keep=False).copy()
_gcount=_gg.region_name.nunique()
V4_GRADUATE_STATUS={'valid_individual_reporting_rows':len(_gg),'ambiguous_reporting_rows_dropped':int(_gduplicates),
                    'n_geographies':int(_gcount),'employment_reference':'2024-12-31','is_longitudinal':False}
if len(_gg)>=100 and _gcount>=8 and _gg.employed_pct.nunique()>1:
    from sklearn.model_selection import GroupShuffleSplit
    _sp=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=CFG.RANDOM_SEED)
    _a,_b=next(_sp.split(_gg,groups=_gg.region_name))
    _gtr=_gg.iloc[_a].copy();_gte=_gg.iloc[_b].copy()
    assert set(_gtr.region_name).isdisjoint(_gte.region_name)
    _cn=['count_graduate','graduation_year'];_cc=['education_level','gender']
    _Xtr=_gtr[_cn+_cc].copy();_Xte=_gte[_cn+_cc].copy()
    for _k in _cc:
        _Xtr[_k]=_Xtr[_k].fillna('UNKNOWN').astype(str);_Xte[_k]=_Xte[_k].fillna('UNKNOWN').astype(str)
    _gpred=[];_gy=_gte.employed_pct.to_numpy(float)
    _gbase=np.full(len(_gte),float(_gtr.employed_pct.mean()))
    _gpred.append(('region_blind_train_mean',_gbase))
    for _kind in ('ridge','lightgbm','catboost'):
        _num=Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())])
        if _kind=='catboost':
            _med={k:float(_Xtr[k].median()) for k in _cn}
            _A=_Xtr.copy();_B=_Xte.copy()
            for _k in _cn:
                _A[_k]=_A[_k].fillna(_med[_k]);_B[_k]=_B[_k].fillna(_med[_k])
            _mod=CatBoostRegressor(iterations=180,depth=4,learning_rate=.06,verbose=False,
                random_seed=CFG.RANDOM_SEED,thread_count=1,allow_writing_files=False)
            _mod.fit(_A,_gtr.employed_pct.to_numpy(float),cat_features=_cc)
            _gp=_mod.predict(_B)
            _art={'model':_mod,'medians':_med,'categoricals':_cc,'features':_cn+_cc}
        else:
            _prep=ColumnTransformer([('n',_num,_cn),('c',OneHotEncoder(handle_unknown='ignore'),_cc)])
            if _kind=='ridge':_est=Ridge(alpha=30.,solver='lsqr')
            else:_est=LGBMRegressor(n_estimators=170,max_depth=4,num_leaves=12,
                   n_jobs=1,verbosity=-1,random_state=CFG.RANDOM_SEED)
            _mod=Pipeline([('prep',_prep),('reg',_est)])
            _mod.fit(_Xtr,_gtr.employed_pct.to_numpy(float));_gp=_mod.predict(_Xte)
            _art={'model':_mod,'features':_cn+_cc}
        _gp=np.clip(_gp,0,100)
        _gpred.append((_kind,_gp))
        _mp=ROOT/'models'/f'v4_ml55_region_heldout_{_kind}.joblib'
        joblib.dump(_art,_mp,compress=3)
        _rt=joblib.load(_mp)
        if _kind=='catboost':
            _verify=_Xte.copy()
            for _k in _cn:_verify[_k]=_verify[_k].fillna(_rt['medians'][_k])
        else:_verify=_Xte
        assert np.allclose(np.clip(_rt['model'].predict(_verify),0,100),_gp)
        MODEL_FILES[_mp.name]=str(_mp)
        MODEL_RECORDS.append({'module':'ML-5.5','sector':'GRAD_E04_CROSS_SECTION',
            'model':_kind,'seed':CFG.RANDOM_SEED,'path':_mp.name,'file_sha256':hash_file(_mp),
            'training_rows':len(_gtr),'test_rows':len(_gte),'reloaded':True,'feature_num':_cn,'feature_cat':_cc})
    _gm=[];_gpframe=_gte[['region_name','university','graduation_year','employed_pct']].reset_index(drop=True).copy()
    for _kind,_yp in _gpred:
        _gpframe['pred_'+_kind]=_yp
        _gm.append({'module':'ML-5.5','model':_kind,'protocol':'REGION_HELD_OUT_CROSS_SECTION_2024',
            'n_test':len(_gte),'n_regions_test':_gte.region_name.nunique(),**effective_mae(_gy,_yp)})
    qa_table('v4_ml55_crosssection_predictions',_gpframe,allow_empty=True)
    qa_table('v4_ml55_crosssection_metrics',pd.DataFrame(_gm),allow_empty=True)
    V4_GRADUATE_STATUS.update(status='REGIONAL_HOLDOUT_EMPLOYMENT_TEST_NOT_RETENTION',n_test=len(_gte),n_train=len(_gtr),
       baseline_mae=_gm[0]['MAE'],tested_methods=len(_gpred))
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_AND_CROSS_SECTIONAL_TEST',
       reason='All four E04/E05 descriptive; E04 regional OOS employment test does not imply retention/temporal forecast',n=len(_gte))
else:V4_GRADUATE_STATUS['status']='NOT_FEASIBLE_REGION_HELDOUT_AFTER_EXACT_STRATA_GATE'
qa_table('v4_ml55_execution',pd.DataFrame([V4_GRADUATE_STATUS]),allow_empty=True)

# ML-5.6: historical node centrality is tested on common regions only; coverage break is an explicit caveat.
_v4_net=pd.DataFrame(NETWORK)
_v4_nresults=[]
if len(_v4_net) and _v4_net.year.nunique()>1:
    _v4_years=sorted(_v4_net.year.unique())
    for _prev,_now in zip(_v4_years[:-1],_v4_years[1:]):
        _p=_v4_net[_v4_net.year.eq(_prev)][['region_code','pagerank']].rename(columns={'pagerank':'prior_pagerank'})
        _n=_v4_net[_v4_net.year.eq(_now)][['region_code','pagerank']].rename(columns={'pagerank':'actual_pagerank'})
        _joined=_p.merge(_n,on='region_code',validate='one_to_one')
        _rho,_pv=(spearmanr(_joined.prior_pagerank,_joined.actual_pagerank) if len(_joined)>=3 else (np.nan,np.nan))
        _v4_nresults.append({'origin_year':int(_prev),'target_year':int(_now),'common_regions':len(_joined),
            'pagerank_persistence_mae':float(np.abs(_joined.prior_pagerank-_joined.actual_pagerank).mean()) if len(_joined) else np.nan,
            'pagerank_rank_spearman':float(_rho) if np.isfinite(_rho) else np.nan,
            'nominal_p_value_NOT_CONFIRMATORY':float(_pv) if np.isfinite(_pv) else np.nan,
            'coverage_break':'UNRESOLVED_2023_OR_OTHER_YEAR_COVERAGE'})
qa_table('v4_ml56_node_forecast',pd.DataFrame(_v4_nresults),allow_empty=True)


## 9Д. ML-5.7 · независимое географическое испытание образовательного муниципального контекста

Дополнительный тест E0/E1 по пространственно отложенным муниципалитетам. Образовательный признак берётся из БДМО в тот же наблюдаемый год и поэтому **не** является историческим прогнозным признаком. Это только диагностическая ассоциация, а не эффект университета или доказательство миграции.

In [ ]:
# V5 · ML-5.7: независимый *пространственный*, не временной E0/E1 diagnostic.
# General education proxy from BDMO; no academic-campus location, no residence effect inferred.
progress_event('EXTENDED_MODULES','V5 ML-5.7: пространственный E0/E1 на наблюдаемом образовательном контексте',announce=True)
V5_MUNICIPAL_CROSS_STATUS={'status':'NOT_IDENTIFIABLE_ON_AVAILABLE_CROSS_SECTION'}
if CFG.MODE=='FULL' and 'BDMO_EDU_PROXY_CANDIDATES' in globals() and len(BDMO_EDU_PROXY_CANDIDATES):
    _v5_e=BDMO_EDU_PROXY_CANDIDATES.copy()
    _v5_e['oktmo']=norm_code(_v5_e.oktmo);_v5_e['observation_year']=safe_year(_v5_e.observation_year)
    _v5_e=_v5_e.loc[_v5_e.n_raw_rows.eq(1)&_v5_e.min_reported.eq(_v5_e.max_reported)&
        _v5_e.median_reported.notna()&_v5_e.median_reported.ge(0)].copy()
    _v5_m=MUNI[['oktmo','observation_year','migration_rate','population']].copy()
    _v5_m['oktmo']=norm_code(_v5_m.oktmo);_v5_m['observation_year']=safe_year(_v5_m.observation_year)
    for _k in ('migration_rate','population'):_v5_m[_k]=number(_v5_m[_k])
    _v5_m=_v5_m.dropna(subset=['oktmo','observation_year','migration_rate','population'])
    _v5_m=_v5_m.drop_duplicates(['oktmo','observation_year'],keep=False)
    _v5_join=_v5_m.merge(_v5_e,on=['oktmo','observation_year'],how='inner',validate='one_to_many')
    _v5_join['is_heldout']=_v5_join.oktmo.astype(str).map(lambda x:int(
         hashlib.sha256(('V5_MUNICIPAL_HOLDOUT|'+x).encode()).hexdigest()[:8],16)%5==0)
    _v5_join['indicator_code']=_v5_join.indicator_code.astype(str)
    _v5_join['indicator_unit']=_v5_join.indicator_unit.astype(str)
    _v5_stats=_v5_join.loc[~_v5_join.is_heldout].groupby(['indicator_code','indicator_unit'],dropna=False).agg(
        n_train_muni=('oktmo','nunique'),n_train_rows=('migration_rate','size'),
        n_train_years=('observation_year','nunique'),value_variation=('median_reported','nunique')).reset_index()
    _v5_stats=_v5_stats.sort_values(['n_train_muni','n_train_years','indicator_code'],ascending=[False,False,True])
    qa_table('v5_ml57_spatial_proxy_eligibility',_v5_stats,allow_empty=True)
    _v5_best=_v5_stats.loc[(_v5_stats.n_train_muni>=CFG.MIN_MUNICIPALITIES_E0E1)&
        (_v5_stats.value_variation>1)]
    if len(_v5_best):
        _v5_code=str(_v5_best.iloc[0].indicator_code);_v5_unit=str(_v5_best.iloc[0].indicator_unit)
        _v5_k=_v5_join.loc[_v5_join.indicator_code.eq(_v5_code)&_v5_join.indicator_unit.eq(_v5_unit)].copy()
        _v5_k=_v5_k.drop_duplicates(['oktmo','observation_year'],keep=False)
        _v5_train=_v5_k.loc[~_v5_k.is_heldout].copy();_v5_test=_v5_k.loc[_v5_k.is_heldout].copy()
        if len(_v5_train)>=100 and _v5_test.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1:
            _v5_features={'E0':['population','observation_year'],
                          'E1':['population','observation_year','median_reported']}
            _v5_out=[]
            for _variant,_cols in _v5_features.items():
                _v5_mod=Pipeline([('fill',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler()),('ridge',Ridge(alpha=30.0))])
                _v5_mod.fit(_v5_train[_cols].to_numpy(float),_v5_train.migration_rate.to_numpy(float))
                _v5_pred=_v5_mod.predict(_v5_test[_cols].to_numpy(float))
                _v5_f=ROOT/'models'/('v5_ml57_crosssection_'+_variant+'_'+hashlib.sha256(
                    (_v5_code+'|'+_v5_unit).encode()).hexdigest()[:10]+'.joblib')
                if _v5_f.exists() or _v5_f.name in MODEL_FILES:raise RuntimeError('V5 ML57 model artifact collision')
                joblib.dump({'model':_v5_mod,'features':_cols,'indicator_code':_v5_code,
                   'protocol':'CROSS_SECTION_NOT_TEMPORAL'},_v5_f,compress=3)
                _v5_loaded=joblib.load(_v5_f)
                if not np.allclose(_v5_pred,_v5_loaded['model'].predict(_v5_test[_cols].to_numpy(float))):
                    raise RuntimeError('V5 ML57 reloaded cross-sectional model mismatch')
                MODEL_FILES[_v5_f.name]=str(_v5_f)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'protocol':'SPATIAL_CROSS_SECTION_NOT_FORECAST','model':_variant,'seed':CFG.RANDOM_SEED,
                  'path':_v5_f.name,'file_sha256':hash_file(_v5_f),'feature_num':_cols,'feature_cat':[],
                  'training_rows':len(_v5_train),'test_rows':len(_v5_test),'reloaded':True})
                _v5_pf=pd.DataFrame({'entity_id':_v5_test.oktmo.astype(str).to_numpy(),
                  'origin_year':_v5_test.observation_year.to_numpy(),'target_year':_v5_test.observation_year.to_numpy(),
                  'information_cutoff_year':_v5_test.observation_year.to_numpy(),
                  'feature_available_year':_v5_test.observation_year.to_numpy(),
                  'target_available_year':_v5_test.observation_year.to_numpy(),
                  'sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'target_id':'D01_MUNICIPAL_MIGRATION_RATE_CONTEMPORANEOUS_ASSOCIATION',
                  'actual':_v5_test.migration_rate.to_numpy(float)})
                _v5_pred_rows=predictions_frame(_v5_pf,'ML-5.7',_variant,_v5_pred,
                                   seed=CFG.RANDOM_SEED,protocol='SPATIAL_CROSS_SECTION_NOT_FORECAST')
                _v5_pred_rows['geography_grain']='MUNICIPALITY';record_prediction(_v5_pred_rows)
                _v5_met=effective_mae(_v5_test.migration_rate,_v5_pred)
                METRICS.append({'module':'ML-5.7','sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'model':_variant,'protocol':'SPATIAL_CROSS_SECTION_NOT_FORECAST',
                  'seed':CFG.RANDOM_SEED,'target_year':int(_v5_test.observation_year.max()),**_v5_met})
                _v5_out.append({'variant':_variant,'n_train':len(_v5_train),'n_test':len(_v5_test),
                       'heldout_municipalities':_v5_test.oktmo.nunique(),**_v5_met})
            qa_table('v5_ml57_spatial_E0_E1_results',pd.DataFrame(_v5_out),allow_empty=True)
            V5_MUNICIPAL_CROSS_STATUS={'status':'CROSS_SECTION_ASSOCIATION_TESTED_NOT_TEMPORAL',
                'selected_source_indicator':_v5_code,'unit':_v5_unit,
                'years':int(_v5_k.observation_year.nunique()),'n_train':len(_v5_train),
                'n_test':len(_v5_test),'MAE_E0':_v5_out[0]['MAE'],'MAE_E1':_v5_out[1]['MAE'],
                'target':'municipal_migration_rate','not_campus_metric':True}
            if MODULES['ML-5.7'].get('status','').startswith('NOT_FEASIBLE'):
                MODULES['ML-5.7'].update(status='SPATIAL_ASSOCIATION_ONLY_NOT_E0E1_TEMPORAL',
                    reason='Municipal held-out E0/E1 general education context, no temporal as-of claim and no verified VO/SPO campuses')
        else:V5_MUNICIPAL_CROSS_STATUS['reason']='Insufficient held-out municipality count for spatial test'
    else:V5_MUNICIPAL_CROSS_STATUS['reason']='No valid unique municipal indicator across enough training municipalities'
qa_table('v5_ml57_spatial_E0_E1_execution',pd.DataFrame([V5_MUNICIPAL_CROSS_STATUS]),allow_empty=True)
print('ML-5.7 supplemental:',V5_MUNICIPAL_CROSS_STATUS['status'],'| temporal E0/E1 remains a separate gate')


## 10. Независимые метрики, canary, статистика
Метрики пересчитываются по прогнозным строкам. При отсутствии законного DEV сравнения не становятся подтверждающими. Отрицательные результаты сохраняются.

In [ ]:
progress_event('STATISTICS','Независимые метрики, канарейки, проверки гипотез')
# Независимые сквозные проверки: источники/будущее/статистика/содержательные выводы.
CANARIES=[]

def test_future_mutation(raw,keys,value_col,module):
    if raw.empty or 'observation_year' not in raw or value_col not in raw:
        return {'module':module,'status':'NOT_APPLICABLE','reason':'Нет валидной временной панели'}
    r=raw[keys+['observation_year',value_col]].dropna(subset=['observation_year']).copy()
    if r.empty or r.observation_year.nunique()<2:return {'module':module,'status':'NOT_APPLICABLE','reason':'Мало лет'}
    r=r.drop_duplicates(keys+['observation_year'])
    if r.empty:return {'module':module,'status':'NOT_APPLICABLE','reason':'Нет уникальных ключей'}
    p1=historical_pair_panel(r,keys,value_col)
    mx=int(r.observation_year.max());r2=r.copy()
    r2.loc[r2.observation_year.eq(mx),value_col]=number(r2.loc[r2.observation_year.eq(mx),value_col]).fillna(0)+10**8
    p2=historical_pair_panel(r2,keys,value_col)
    # Проверяется строгое отсутствие будущих лагов; отдельный тест экономических и сетевых join — ниже.
    if len(p1) and (p1.feature_available_year>p1.information_cutoff_year).any():
        fail('CANARY','Исходная панель содержит будущие признаки: '+module)
    # До мутации будущего итоговые X прежних origins должны быть идентичны.
    common=[c for c in keys+['target_year','prior_value','prior2_value','historical_year'] if c in p1 and c in p2]
    early1=p1[p1.target_year<mx][common].sort_values(keys+['target_year']).reset_index(drop=True)
    early2=p2[p2.target_year<mx][common].sort_values(keys+['target_year']).reset_index(drop=True)
    if not early1.equals(early2):fail('CANARY','Мутация будущего изменила прошлые признаки: '+module)
    return {'module':module,'status':'PASS','n_untouched':len(early1),'mutated_year':mx}

CANARIES.append(test_future_mutation(OD_INTER,['origin_region','destination_region'],'observed_value','ML-5.1'))
CANARIES.append(test_future_mutation(OUTFLOW,['origin_region'],'observed_interregional_routes','ML-5.3'))
if len(ORG_YEAR):CANARIES.append(test_future_mutation(ORG_YEAR,['sector','entity_id'],'actual','ML-5.2'))
if len(PROGRAM_PROFILE):
    prog_test=PROGRAM_PROFILE.drop_duplicates(['specialty_code','indicator_name','observation_year'])
    CANARIES.append(test_future_mutation(prog_test,['specialty_code','indicator_name'],'observed_value','ML-5.4'))
if len(GRADUATES):
    CANARIES.append({'module':'ML-5.5','status':'DESCRIPTIVE_ONLY',
      'reason':'Единственная дата измерения занятости; модель предсказаний не запускалась'})
if len(MUNI):CANARIES.append(test_future_mutation(MUNI,['oktmo'],'migration_rate','ML-5.7'))
CANARIES.append({'module':'ECONOMIC_CONTEXT','status':'STATIC_OR_ASOF_AUDIT',
 'reason':'Использована только экономическая панель observation_year=historical_year; future rows not joined'})
if len(OD_MODEL):
    # Отдельный тест исходных маргиналей: признаки прошлого года должны совпадать с исходными агрегатами.
    src_mass=OD_INTER.groupby(['observation_year','origin_region'],as_index=False).observed_value.sum()
    tmp=OD_MODEL[['origin_region','historical_year','origin_mass']].dropna().drop_duplicates()
    exp=src_mass.rename(columns={'observation_year':'historical_year','observed_value':'expected_mass'})
    comp=tmp.merge(exp,on=['historical_year','origin_region'],how='left',validate='many_to_one')
    if len(comp) and not np.allclose(comp.origin_mass,comp.expected_mass,equal_nan=True):
        fail('CANARY','Региональные маргинали не совпали с историческим исходным годом')
    CANARIES.append({'module':'OD_MARGINS','status':'PASS' if len(comp) else 'NOT_APPLICABLE',
       'verified_origin_year_margins':int(len(comp))})
qa_table('future_mutation_canaries',pd.DataFrame(CANARIES),allow_empty=True)

# Весь стек прогнозов, включая 50 сидов, объединяется НА ДИСКЕ.
PRED=pd.DataFrame();RECALCULATED=[];PRED_STORAGE_FILE=None
if PRED_PARTS:
    progress_event('STATISTICS',f'Независимая проверка {PRED_TOTAL:,} строк из {len(PRED_PARTS)} файлов',announce=True)
    PRED_STORAGE_FILE=ROOT/'derived'/'education_predictions.parquet'
    # read_parquet(list) обрабатывается потоком в DuckDB, не через pandas.concat.
    with duck_connect() as con:
        _files='['+','.join("'"+str(p).replace("'","''")+"'" for p in PRED_PARTS)+']'
        _q=f"SELECT * FROM read_parquet({_files},union_by_name=true)"
        _fp=str(PRED_STORAGE_FILE).replace("'","''")
        con.execute(f"COPY ({_q}) TO '{_fp}' (FORMAT PARQUET, COMPRESSION ZSTD)")
        _num=con.execute('SELECT COUNT(*) FROM read_parquet(?)',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _num!=PRED_TOTAL:fail('REPRO',f'Прогнозов в Parquet {_num} вместо {PRED_TOTAL}')
        _invalid=con.execute('''SELECT COUNT(*) FROM read_parquet(?)
            WHERE predicted IS NULL OR actual IS NULL OR NOT isfinite(predicted) OR NOT isfinite(actual)
            OR feature_available_year>information_cutoff_year''',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _invalid:fail('REPRO',f'Невалидных прогнозов: {_invalid}')
        _duplicates=con.execute('''SELECT COUNT(*) FROM (
            SELECT module,sector,entity_id,target_year,model,seed,split_id,COUNT(*) AS n
            FROM read_parquet(?) GROUP BY ALL HAVING COUNT(*)>1)''',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _duplicates:fail('SUPPORT',f'Повторные натуральные ключи прогнозов: {_duplicates}')
        # Точные MAE/RMSE/WAPE по ВСЕМ строкам всех файлов.
        RECALCULATED=con.execute('''SELECT module,sector,model,seed,target_year,split_id,
            AVG(ABS(actual-predicted)) AS MAE,
            SQRT(AVG(POWER(actual-predicted,2))) AS RMSE,
            SUM(ABS(actual-predicted))/NULLIF(SUM(ABS(actual)),0) AS WAPE,
            COUNT(*)::BIGINT AS n
            FROM read_parquet(?) GROUP BY 1,2,3,4,5,6''',[str(PRED_STORAGE_FILE)]).df().to_dict('records')
        PRED=con.execute('SELECT * FROM read_parquet(?) LIMIT 2000',[str(PRED_STORAGE_FILE)]).df()
    PANEL_FILES['education_predictions']=PRED_STORAGE_FILE
    final=pd.DataFrame(RECALCULATED)
    for r in final.itertuples(index=False):
        original=[m for m in METRICS if m['module']==r.module and m.get('sector')==r.sector
                  and m['model']==r.model and m['seed']==r.seed and m['target_year']==r.target_year and m.get('protocol')==r.split_id]
        if not original:fail('REPRO','Нет строки метрики для '+r.module+' '+r.model)
        for m in original:
            if not np.isclose(m['MAE'],r.MAE,atol=1e-7,rtol=1e-7):fail('REPRO','MAE не совпал: '+r.module+' '+r.model)
qa_table('metrics_recomputed_from_predictions',pd.DataFrame(RECALCULATED),allow_empty=True)

# Предрегистрация Holm: только подтверждающие сравнения, для которых есть валидный p.
CONFIRM=pd.DataFrame(CONFIRMATORY)
if not CONFIRM.empty:
    valid=CONFIRM.p_value.notna()&np.isfinite(CONFIRM.p_value)
    CONFIRM['p_holm']=np.nan
    if valid.any():
        CONFIRM.loc[valid,'p_holm']=multipletests(CONFIRM.loc[valid,'p_value'].to_numpy(float),
                                                method='holm')[1]
    CONFIRM['interpretation']=np.where(CONFIRM.delta_mae.lt(0),'LOWER_MAE_ON_SELECTED_COMMON_SUPPORT',
                                      np.where(CONFIRM.delta_mae.gt(0),'HIGHER_MAE_ON_SELECTED_COMMON_SUPPORT','EQUAL_MAE'))
    CONFIRM['historical_vintage_validated']=False
    CONFIRM['claim_level']='EXPLORATORY_RETROSPECTIVE_SOURCE_2026'

qa_table('confirmatory_comparisons',CONFIRM,allow_empty=True)

# Доказательный реестр 7 гипотез: отдельно результат и предел идентификации.
for i in range(1,8):
    module=f'ML-5.{i}'
    status=MODULES[module]
    subset=CONFIRM[CONFIRM.module.eq(module)] if len(CONFIRM) else pd.DataFrame()
    if len(subset):
        r=subset.iloc[0]
        if r.choice_basis!='DEV_ASOF':
            conclusion='Подтверждающий отбор невозможен: DEV-метки недоступны на момент TEST; сравнения моделей разведочные'
            metric=f'baseline_MAE={r.baseline_mae:.5g}; confirmatory_delta=NOT_APPLICABLE'
        else:
            conclusion=('выбранная на DEV модель не превзошла инерцию на ретроспективном тесте' if r.delta_mae>=0
                        else 'в ретроспективном тесте MAE ниже инерции; исторические версии данных не подтверждены')
            metric=f'baseline_MAE={r.baseline_mae:.5g}; selected_MAE={r.selected_mae:.5g}; delta={r.delta_mae:+.5g}'
    else:conclusion=status['reason'];metric='Нет корректного независимого прогнозного теста'
    HYPOTHESES.append({'hypothesis':f'H5.{i}','module':module,'status':status['status'],
        'metric_or_basis':metric,'scientific_conclusion':conclusion,
        'source_observability':('E01_schooling_to_study_NOT_residence' if i in (1,3,6)
             else 'source_specific_unverified_retention' if i==5 else 'source_specific'),
        'causal_identification':'NOT_ESTABLISHED'})
qa_table('seven_hypotheses',pd.DataFrame(HYPOTHESES),['hypothesis'])
qa_table('split_audit',pd.DataFrame(SPLIT_AUDIT),allow_empty=True)
qa_table('model_metrics',pd.DataFrame(METRICS),allow_empty=True)
qa_table('model_registry',pd.DataFrame(MODEL_RECORDS),allow_empty=True)

# Статусы: техническая полнота != научное подтверждение семи гипотез.
executed_all=all(m['status']!='NOT_STARTED' for m in MODULES.values())
failed_modules=[k for k,m in MODULES.items() if m['status']=='FAILED']
research_full=all(m['status']=='TESTED_FULL' for m in MODULES.values()) and OD_SEMANTIC['yearly_count_unit_confirmed']
STATE['technical']='PASS_ASSEMBLED' if executed_all and not failed_modules else 'FAIL'
STATE['scientific']='FULL_RESEARCH_PASS' if research_full and CFG.MODE=='FULL' else (
    'FULL_EXECUTION_WITH_LIMITATIONS' if CFG.MODE=='FULL' and executed_all and not failed_modules else 'PILOT_OR_PARTIAL')
STATE['overall']=STATE['scientific']

print('Technical / scientific statuses:',STATE['technical'],STATE['scientific'])
print('Confirmed comparisons:',len(CONFIRM),'| exported predictions:',PRED_TOTAL)

mark_stage('STATISTICS')


## 10А. Парные сравнения и проверка завершённости программы
Разделение вычислительной полноты и научной идентифицируемости; отсутствующие измерения не фабрикуются.

In [ ]:
# V4 · audit model completion independently of the notebook's technical PASS.
progress_event('STATISTICS','V4: численные сравнения на общих ключах, матрица научной полноты',announce=True)
V4_PAIRED_COMPARISONS=[]
if PRED_STORAGE_FILE and Path(PRED_STORAGE_FILE).is_file():
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _unique=_db.execute('''SELECT DISTINCT module,sector,split_id,model FROM read_parquet(?) ORDER BY 1,2,3,4''',
                           [str(PRED_STORAGE_FILE)]).df()
        _combos=_unique[~_unique.model.isin(['persistence','rolling_mean_2','damped_trend','catboost'])].copy()
        for _rec in _combos.itertuples(index=False):
            _pred=_db.execute('''WITH b AS (SELECT entity_id,target_year,actual,predicted baseline,
                     origin_region,destination_region,parent_group_id FROM read_parquet(?)
                  WHERE module=? AND sector=? AND split_id=? AND model='persistence' AND seed=-1),
                 v AS (SELECT entity_id,target_year,actual,predicted as candidate FROM read_parquet(?)
                  WHERE module=? AND sector=? AND split_id=? AND model=?)
                  SELECT b.*,v.candidate FROM b JOIN v USING(entity_id,target_year)
                  WHERE ABS(b.actual-v.actual)<1e-7''',
               [str(PRED_STORAGE_FILE),_rec.module,_rec.sector,_rec.split_id,
                str(PRED_STORAGE_FILE),_rec.module,_rec.sector,_rec.split_id,_rec.model]).df()
            if not len(_pred):continue
            _a=np.abs(_pred.actual.to_numpy(float)-_pred.baseline.to_numpy(float))
            _v=np.abs(_pred.actual.to_numpy(float)-_pred.candidate.to_numpy(float))
            if _rec.module=='ML-5.1' and {'origin_region','destination_region'}.issubset(_pred):
                _g1=_pred.origin_region.fillna('__UNK__').astype(str)
                _g2=_pred.destination_region.fillna('__UNK__').astype(str)
                _ci=two_way_mae_bootstrap(_a,_v,_g1,_g2,reps=CFG.BOOTSTRAP_REPS,seed=CFG.RANDOM_SEED)
            else:
                _g1=(_pred.parent_group_id.fillna(_pred.entity_id).astype(str) if
                     'parent_group_id' in _pred else _pred.entity_id.astype(str))
                # Only one test target_year; two-way geographic-year bootstrap not identified.
                _unique_groups=_g1.nunique()
                if _unique_groups>=8:
                    _rng=np.random.default_rng(CFG.RANDOM_SEED)
                    _groups=_g1.to_numpy();_base=np.asarray(_v-_a,float)
                    _levels=np.unique(_groups);_boot=[]
                    for _ in range(CFG.BOOTSTRAP_REPS):
                        _mult=Counter(_rng.choice(_levels,len(_levels),replace=True))
                        _weights=np.fromiter((_mult[v] for v in _groups),dtype=float,count=len(_groups))
                        if _weights.sum()>0:_boot.append(float(np.average(_base,weights=_weights)))
                    _vals=np.asarray(_boot)
                    _ci={'delta_MAE':float(np.mean(_base)),'ci_low':float(np.quantile(_vals,.025)) if len(_vals)>20 else np.nan,
                         'ci_high':float(np.quantile(_vals,.975)) if len(_vals)>20 else np.nan,
                         'p_value':min(1.0,2*min((np.count_nonzero(_vals<=0)+1)/(len(_vals)+1),
                            (np.count_nonzero(_vals>=0)+1)/(len(_vals)+1))) if len(_vals)>20 else np.nan,
                         'clusters_a':_unique_groups,'clusters_b':None}
                else:_ci={'delta_MAE':float(np.mean(_v-_a)),'ci_low':np.nan,'ci_high':np.nan,
                           'p_value':np.nan,'clusters_a':_unique_groups,'clusters_b':None}
            V4_PAIRED_COMPARISONS.append({'module':_rec.module,'sector':_rec.sector,
                'split_id':_rec.split_id,'model':_rec.model,'n_common_keys':len(_pred),
                'persistence_MAE':float(_a.mean()),'candidate_MAE':float(_v.mean()),**_ci,
                'inference_type':'EXPLORATORY_PAIRED_RETROSPECTIVE_NOT_CONFIRMATORY',
                'historical_source_vintage_at_prediction':'NOT_ARCHIVED'})
_V4_BOOT=pd.DataFrame(V4_PAIRED_COMPARISONS)
if len(_V4_BOOT):
    _mask=_V4_BOOT['p_value'].notna()&np.isfinite(_V4_BOOT['p_value'])
    _V4_BOOT['p_holm_exploratory']=np.nan
    if _mask.any():_V4_BOOT.loc[_mask,'p_holm_exploratory']=multipletests(
        _V4_BOOT.loc[_mask,'p_value'].to_numpy(float),method='holm')[1]
qa_table('v4_exploratory_paired_comparisons',_V4_BOOT,allow_empty=True)

V4_METHOD_EXECUTIONS=[]
# Check every required model separately, not merely that a module produced one metric.
_v4_contract_base={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm','catboost_ensemble'}
_v4_expected={}
for _m in ('ML-5.1','ML-5.3','ML-5.4'):
    _actual={z.get('model') for z in METRICS if z.get('module')==_m and z.get('protocol')=='ROLLING_TIME'}
    if _actual:
        _req=_v4_contract_base.copy()
        # PPML/NB only if their observed-source integer semantics passed the count gate.
        _frame=OD_MODEL if _m=='ML-5.1' else OUTFLOW_MODEL if _m=='ML-5.3' else None
        _count_eligible=bool(_frame is not None and len(_frame) and
             number(_frame.actual).ge(0).all() and
             np.isclose(number(_frame.actual),np.rint(number(_frame.actual)),atol=1e-8,rtol=0).all())
        if _m=='ML-5.4':
            _count_eligible=MODULES[_m]['status'].startswith('TESTED')
        if _count_eligible:_req.update({'ppml','negbin'})
        _v4_expected[_m]=_req
if any(z.get('module')=='ML-5.2' and z.get('protocol')=='ALT_CONTINGENT_RETROSPECTIVE' for z in METRICS):
    _v4_expected['ML-5.2']={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm'}
if any(z.get('module')=='ML-5.7' and z.get('protocol')=='RETROSPECTIVE_PROXY_E0E1' for z in METRICS):
    _v4_expected['ML-5.7']={'E0','E1'}
for _module in ['ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7']:
    _r=[m for m in METRICS if m.get('module')==_module]
    _failed=[x for x in ISSUES if x.get('module')==_module and x.get('status') in
            ('MODEL_FAILED','FAMILY_PROFILE_FAILED','SOURCE_PROFILE_FAILURE','SPATIAL_MODEL_FAILURE')]
    _expected=_v4_expected.get(_module,set())
    _actually={z.get('model') for z in _r}
    _missing=sorted(_expected-_actually)
    if _missing:_failed.append({'reason':'Feasible models without metrics: '+','.join(_missing),
                               'status':'METHOD_NOT_EXECUTED'})
    if _module=='ML-5.5' and V4_GRADUATE_STATUS.get('status')=='REGIONAL_HOLDOUT_EMPLOYMENT_TEST_NOT_RETENTION':
        if len(PANELS.get('v4_ml55_crosssection_metrics',pd.DataFrame()))<4:
            _failed.append({'reason':'E04 cross-sectional baseline/Ridge/LightGBM/CatBoost incomplete'})
    if _module=='ML-5.6' and len(NETWORK) and not len(PANELS.get('v4_ml56_node_forecast',pd.DataFrame())):
        _failed.append({'reason':'Node centrality persistence test missing'})
    _feasible=bool(_r) or MODULES[_module].get('status','').startswith('DESCRIPTIVE')
    V4_METHOD_EXECUTIONS.append({'module':_module,'module_status':MODULES[_module].get('status'),
        'models_with_metrics':len(set(x.get('model') for x in _r)),
        'metric_rows':len(_r),'saved_model_artifacts':sum(m.get('module')==_module for m in MODEL_RECORDS),
        'prediction_rows_validated':bool(any(x.get('module')==_module for x in RECALCULATED)),
        'required_models':','.join(sorted(_expected)), 'missing_required_models':','.join(_missing),
        'computational_failures':len(_failed),
        'failure_reason':' | '.join(str(x.get('reason',''))[:120] for x in _failed[:3]),
        'target_is_scientifically_identified':bool(_module in ('ML-5.4','ML-5.6')),
        'historical_vintage_validated':False,
        'scientific_completion':'RETROSPECTIVE_OR_DESCRIPTIVE_WITH_GATES'})
qa_table('v4_methods_execution_ledger',pd.DataFrame(V4_METHOD_EXECUTIONS),['module'])
V4_COMPUTATIONAL_COMPLETE=bool(not V4_SOURCE_ERRORS and not any(x['computational_failures'] for x in V4_METHOD_EXECUTIONS))
V4_SCIENTIFIC_COMPLETE=bool(V4_COMPUTATIONAL_COMPLETE and
                           all(MODULES[m]['status']=='TESTED_FULL' for m in MODULES) and
                           OD_SEMANTIC.get('yearly_count_unit_confirmed',False) and
                           len(ORG_YEAR)>0)
STATE['scientific']='FULL_RESEARCH_PASS' if V4_SCIENTIFIC_COMPLETE else (
    'FULL_COMPUTATIONS_WITH_DOCUMENTED_DATA_LIMITS' if V4_COMPUTATIONAL_COMPLETE and CFG.MODE=='FULL'
     else 'FULL_PARTIAL_COMPUTATIONS_OR_MODEL_FAILURES')
STATE['overall']=STATE['scientific']
qa_table('v4_scientific_acceptance',pd.DataFrame([{'version':CFG.VERSION,'source_assets':len(ASSETS),
    'value_scans_passed':sum(x['content_status']=='FULL_VALUE_COVERAGE_PASS' for x in V4_SOURCE_PROFILES),
    'source_assets_not_in_model':sum(not x['main_prediction_input'] for x in _v4_roles),
    'seven_modules_reported':len(MODULES),'computational_failures':sum(x['computational_failures'] for x in V4_METHOD_EXECUTIONS),
    'full_computations_without_failures':V4_COMPUTATIONAL_COMPLETE,'all_seven_scientifically_identified':V4_SCIENTIFIC_COMPLETE,
    'scientific_status':STATE['scientific'],'human_approval_needed_for_admissions':len(ORG_YEAR)==0,
    'OD_2023_coverage_break_unresolved':bool(coverage_df.potential_break.fillna(False).any()),
    'historical_vintages_not_archived':True,'retention_unobservable_without_work_region':True}]))
print('V4 acceptability:',STATE['scientific'],'| calculations with actual errors:',not V4_COMPUTATIONAL_COMPLETE)

# Rolling-origin feasibility is measured year by year. Lack of archived source vintages forbids a confirmatory claim.
_v4_roll=[]
for _m,_f in [('ML-5.1',OD_MODEL),('ML-5.3',OUTFLOW_MODEL)]:
    if _f is None or not len(_f):continue
    _years=sorted(int(y) for y in _f.target_year.dropna().unique())
    for _y in _years[1:]:
        _cut=_y-CFG.HORIZON
        _train=_f[_f.target_year.lt(_y)&_f.target_available_year.le(_cut)]
        _test=_f[_f.target_year.eq(_y)]
        _gate=len(_train)>=(CFG.REGION_MIN_TRAIN_ROWS if _m=='ML-5.3' else CFG.MIN_TRAIN_ROWS) and \
              len(_test)>=(CFG.REGION_MIN_TEST_ROWS if _m=='ML-5.3' else CFG.MIN_TEST_ROWS)
        _v4_roll.append({'module':_m,'target_year':_y,'n_asof_train':len(_train),
             'n_test_observed':len(_test),'legally_feasible_given_publication_lag':bool(_gate),
             'source_vintage_asof_origin':'NOT_ARCHIVED',
             'scientific_status':'RETROSPECTIVE_ONLY' if _gate else 'NOT_FEASIBLE_SMALL_HISTORICAL_TRAIN'})
qa_table('v4_rolling_origin_feasibility',pd.DataFrame(_v4_roll),allow_empty=True)


# 10Б. Контракт V5: полнота моделей, численная состоятельность и доказательная база

Этот блок проверяет результаты независимо от общей отметки выполнения ноутбука. Модели со сбоями, отсутствующими прогнозами или нестабильными значениями не засчитываются. Нет исходных показателей — `NOT_IDENTIFIABLE`, а не вымышленный результат.

In [ ]:
# ML-05 V5 · fail-closed model registry, numerical diagnostics and evidence matrix.
progress_event('STATISTICS','V5: полный контракт моделей, протоколов и проверка числа сидов',announce=True)
V5_MODEL_QA=[];V5_MODULE_ACCEPTANCE=[]
_artifact_paths=[m['path'] for m in MODEL_RECORDS]
if len(_artifact_paths)!=len(set(_artifact_paths)):
    raise RuntimeError('V5: different model experiments have duplicate output paths')
if len(MODEL_FILES)!=len(set(_artifact_paths)):
    raise RuntimeError('V5: model registry and actual model files differ')
for _m in MODEL_RECORDS:
    _fp=ROOT/'models'/_m['path']
    _ok=_fp.is_file() and _m.get('file_sha256')==hash_file(_fp) and _m.get('reloaded')
    V5_MODEL_QA.append({'module':_m['module'],'model':_m['model'],'protocol':_m.get('protocol','CUSTOM'),
        'seed':_m['seed'],'path':_m['path'],'sha_verified':bool(_ok),
        'guarded_predictions':_m.get('prediction_guard_events',0),
        'guard_fraction':_m.get('prediction_guard_fraction',0.)})
if V5_MODEL_QA and not all(x['sha_verified'] for x in V5_MODEL_QA):
    raise RuntimeError('V5: saved models changed or reload failed')
qa_table('v5_model_files_independently_verified',pd.DataFrame(V5_MODEL_QA),allow_empty=True)

# Reconcile required model names and their independent saved metrics, not a mere code branch.
for _m in ['ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7']:
    _mm=[r for r in METRICS if r.get('module')==_m]
    _issues=[r for r in ISSUES if r.get('module')==_m and r.get('status') in
      ('MODEL_FAILED','SPATIAL_MODEL_FAILURE','SOURCE_PROFILE_FAILURE','FAMILY_PROFILE_FAILED')]
    _status=MODULES[_m].get('status','UNKNOWN')
    _preds=sum(int(x.get('n',0)) for x in _mm)
    _time={x['model'] for x in _mm if x.get('protocol')=='ROLLING_TIME'}
    _required={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm','catboost_ensemble','ppml','negbin'}
    _targets_confirmed=bool(_status.startswith('TESTED'))
    if _m in ('ML-5.1','ML-5.3','ML-5.4') and _targets_confirmed:
        _missing=sorted(_required-_time)
        _seeds={int(x['seed']) for x in _mm if x.get('model')=='catboost' and x.get('protocol')=='ROLLING_TIME'}
        _missing_seeds=sorted(set(int(z) for z in CFG.FINAL_SEEDS)-_seeds) if CFG.MODE=='FULL' else []
    else:_missing=[];_missing_seeds=[]
    _numeric_invalid=[x['model'] for x in _mm if not np.isfinite(x.get('MAE',np.nan)) or
        not np.isfinite(x.get('RMSE',np.nan)) or float(x.get('MAE',0))<0]
    _failures=bool(_issues or _missing or _missing_seeds or _numeric_invalid)
    _scientific=('OBSERVED_TARGET_TESTED_RETROSPECTIVELY' if _targets_confirmed else
                 'DESCRIPTIVE_OR_PROXY_TEST_ONLY' if _mm or 'DESCRIPTIVE' in _status or 'PROXY' in _status else
                 'NOT_IDENTIFIABLE_WITH_AVAILABLE_STAGE_I')
    V5_MODULE_ACCEPTANCE.append({'module':_m,'status':_status,'metrics':len(_mm),
        'model_file_count':sum(x['module']==_m for x in MODEL_RECORDS),'failed_methods':len(_issues),
        'missing_methods':','.join(_missing),'missing_final_seeds':','.join(map(str,_missing_seeds)),
        'nonfinite_metrics':','.join(_numeric_invalid),
        'computational_pass':not _failures,'scientific_scope':_scientific,
        'original_target_fully_identified':bool(_m=='ML-5.4' and _targets_confirmed),
        'historical_source_vintage_validated':False})

V5_LEDGER=pd.DataFrame(V5_MODULE_ACCEPTANCE)
qa_table('v5_seven_module_acceptance',V5_LEDGER,['module'])

# Source evidence: 52 original datasets are all profiled but not uniformly identifiable as model features.
V5_SOURCE_EVIDENCE=pd.DataFrame(V4_SOURCE_PROFILES)
if len(V5_SOURCE_EVIDENCE)!=len(ASSETS) or not all(r.get('content_status')=='FULL_VALUE_COVERAGE_PASS' for r in V4_SOURCE_PROFILES):
    if CFG.MODE=='FULL':raise RuntimeError('V5 source coverage is not 52/52')
qa_table('v5_all52_source_evidence',V5_SOURCE_EVIDENCE,allow_empty=True)
V5_COMPUTATIONAL_PASS=bool(not V4_SOURCE_ERRORS and V5_LEDGER.computational_pass.all())
V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS=bool(V5_COMPUTATIONAL_PASS and all(V5_LEDGER.original_target_fully_identified))
V5_ACCEPTANCE_STATUS='FULL_COMPUTATIONS_WITH_SCIENTIFIC_LIMITS' if V5_COMPUTATIONAL_PASS else 'MODEL_OR_EVIDENCE_FAILURE'
STATE['scientific']=V5_ACCEPTANCE_STATUS
print('V5: 52-source integrity',len(V5_SOURCE_EVIDENCE),'/',len(ASSETS),
      '| all computational methods',V5_COMPUTATIONAL_PASS,'| scientifically identified 7/7',V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS)


## 11. Научные изображения
Диаграммы привязаны к исходному показателю и его периоду, без фиктивной геометрии и трактовки отсутствующей связи как нулевой.

In [ ]:
progress_event('VISUALS','Визуализация фактических E01, сетей и тестовых ошибок')
# Фигуры только на реально наблюдаемых исходных величинах; без псевдогеографических карт.
plt.rcParams.update({'font.family':'DejaVu Sans','figure.figsize':(10,5),'axes.grid':True})

def figure_save(stem):
    path=ROOT/'figures'/(stem+'.png')
    plt.tight_layout();plt.savefig(path,dpi=160,bbox_inches='tight');plt.close()
    FIGURES.append(str(path))
    return path

# Композиция берётся из неизменяемого PANELS, а не из локального counts (он изменяется в FEATURES).
od_comp=PANELS.get('od_composition',pd.DataFrame()).copy()
required={'observation_year','code_status','total_value'}
if not required.issubset(od_comp.columns):
    if {'observation_year','code_status','observed_value'}.issubset(OD.columns):
        od_comp=OD.groupby(['observation_year','code_status'],dropna=False,as_index=False).agg(total_value=('observed_value','sum'))
    else:fail('VISUALS','Потеряна первичная схема E01 / code_status')
if not od_comp.empty:
    od_comp['total_value']=pd.to_numeric(od_comp.total_value,errors='coerce')
    od_comp=od_comp.dropna(subset=['observation_year','code_status','total_value'])
    tab=od_comp.groupby(['observation_year','code_status']).total_value.sum().unstack(fill_value=0)
    tab=tab.reindex(columns=['INTER_REGION','INTRA_REGION','UNKNOWN_CODE'],fill_value=0).sort_index()
    if len(tab):
        fig,ax=plt.subplots(figsize=(10,5));tab.plot(kind='bar',stacked=True,ax=ax)
        ax.set(title='E01: состав зарегистрированного показателя (не постоянная миграция)',xlabel='Отчётный год',ylabel='Наблюдаемое значение E01')
        ax.legend(['Межрегиональные','Внутрирегиональные','Неопределённые коды'],fontsize=8)
        ax.tick_params(axis='x',rotation=0)
        figure_save('01_od_value_composition')
if len(coverage_df):
    cv=coverage_df.copy();cv['year']=pd.to_numeric(cv.year,errors='coerce');cv['observed_edges']=pd.to_numeric(cv.observed_edges,errors='coerce')
    cv=cv.dropna(subset=['year','observed_edges']).sort_values('year')
    if len(cv):
        fig,ax=plt.subplots(figsize=(10,4));ax.plot(cv.year,cv.observed_edges,'o-')
        flagged=cv.get('potential_break',pd.Series(False,index=cv.index)).fillna(False).astype(bool)
        for r in cv.loc[flagged].itertuples():
            ax.annotate('Изменение покрытия',(r.year,r.observed_edges),xytext=(0,12),textcoords='offset points',ha='center',fontsize=8)
        ax.set(title='E01: число зарегистрированных межрегиональных направлений',xlabel='Год',ylabel='Наблюдаемые пары регионов')
        ax.set_xticks(sorted(cv.year.astype(int).unique()))
        figure_save('02_edge_coverage')
if len(NETWORK):
    net=pd.DataFrame(NETWORK);year=int(net.year.max())
    x=net[net.year==year].nlargest(15,'in_strength').sort_values('in_strength')
    if len(x):
        fig,ax=plt.subplots(figsize=(11,6));ax.barh(x.region_code.astype(str),x.in_strength)
        ax.set(title=f'E01: крупнейшие регионы обучения, {year}',xlabel='Сумма исходного E01',ylabel='Код региона')
        figure_save('03_destination_centres')
if len(CONFIRM):
    q=CONFIRM[['module','baseline_mae','selected_mae']].copy()
    q['baseline_mae']=pd.to_numeric(q.baseline_mae,errors='coerce');q['selected_mae']=pd.to_numeric(q.selected_mae,errors='coerce')
    q=q.dropna();q=q[q.baseline_mae.gt(0)&q.selected_mae.ge(0)].copy()
    q['ratio']=q.selected_mae/q.baseline_mae;q=q[np.isfinite(q.ratio)]
    if len(q):
        fig,ax=plt.subplots(figsize=(10,5));at=np.arange(len(q));w=.34
        ax.bar(at-w/2,np.ones(len(q)),width=w,label='Инерция = 1,0')
        ax.bar(at+w/2,q.ratio,width=w,label='Выбранная до теста модель')
        ax.axhline(1,color='gray',linewidth=1,alpha=.7)
        ax.set_xticks(at);ax.set_xticklabels(q.module.astype(str).tolist())
        ax.set(ylabel='MAE / MAE инерции',title='Относительная ошибка на общих тестовых ключах')
        ax.legend();figure_save('04_mae_preselected_models')
print('Generated figures:',len(FIGURES))

mark_stage('VISUALS')


## 12. Экспорт: полный evidence в GitHub, аналитический Excel и воспроизводимый встроенный ноутбук

In [ ]:
memory_point('before_export')
progress_event('EXPORT','Excel, evidence-Parquet, модели, ноутбук и манифест')
import zipfile
from itertools import chain
EXCEL=ROOT/'exports'/'ML5_EDUCATION_MIGRATION_RESULTS.xlsx'
EVIDENCE=ROOT/'exports'/'ML5_EVIDENCE.zip'
NOTEBOOK=ROOT/'exports'/'ML5_EDUCATION_MIGRATION.ipynb'
MANIFEST=ROOT/'exports'/'run_manifest.json'

# Экспорт строго типизирован: смешанный int/string test_year не допускается.
STRING_HINTS={'entity_id','parent_group_id','origin_region','destination_region','region_code',
              'source_asset_id','oktmo','organization_id','indicator_name','test_period_label',
              'object_oktmo','specialty_code','area','campus_id','region_oktmo'}
INT_HINTS={'target_year','origin_year','observation_year','historical_year','historical_year_2',
           'information_cutoff_year','feature_available_year','target_available_year',
           'test_year','year','horizon','seed','graduation_year','salary_reference_year'}

def prepare_for_parquet(name,frame):
    f=frame.copy()
    f.columns=[str(c) for c in f.columns]
    if len(set(f.columns))!=len(f.columns):fail('EXPORT','Дубли названий столбцов '+name)
    if 'test_year' in f and f['test_year'].dtype in ('object','string'):
        original=f['test_year'].copy();numeric_year=number(original)
        descriptive=original.where(original.notna() & numeric_year.isna()).astype('string')
        if 'test_period_label' not in f:f['test_period_label']=descriptive.fillna('')
        else:f['test_period_label']=f['test_period_label'].astype('string').fillna(descriptive).fillna('')
        f['test_year']=numeric_year.astype('Int64')
    for c in f.columns:
        if c in INT_HINTS:
            z=number(f[c]);invalid=f[c].notna()&z.isna()
            if invalid.any():fail('PARQUET_SCHEMA',f'{name}.{c}: нечисловые годы/сиды')
            f[c]=z.astype('Int64')
        elif c in STRING_HINTS:f[c]=f[c].astype('string')
        elif pd.api.types.is_object_dtype(f[c]):
            sample=f[c].dropna()
            kinds={type(t) for t in sample.head(2000)}
            if not kinds or kinds<={str}:
                f[c]=f[c].astype('string')
            elif kinds<={int,float,np.int64,np.float64}:
                f[c]=number(f[c])
            elif any(isinstance(t,(dict,list,tuple,set)) for t in sample.head(2000)):
                f[c]=f[c].map(lambda v:json.dumps(v,ensure_ascii=False,default=str,sort_keys=True)
                                    if isinstance(v,(dict,list,tuple,set)) else None if v is None else str(v)).astype('string')
            elif all(isinstance(t,(str,bytes,int,float,complex,np.number)) for t in sample.head(2000)):
                # Смешанный административный код переводится в строку только как metadata,
                # строгие числовые признаки/метрики не трогаем.
                if re.search(r'(value|actual|predicted|MAE|RMSE|salary|rate|population)',c,re.I):
                    fail('PARQUET_SCHEMA',f'{name}.{c}: смешанный тип числового поля')
                f[c]=f[c].astype('string')
            else:
                f[c]=f[c].map(lambda v:str(v) if pd.notna(v) else None).astype('string')
    return f

# Регрессионный тест известной ArrowInvalid: числа и текстовый исторический период.
_schema_probe=prepare_for_parquet('group_protocols',pd.DataFrame({
    'test_year':[2025,'historical',pd.NA],
    'test_period_label':['',pd.NA,''],
    'entity_id':['001','002','003']}))
if str(_schema_probe.test_year.dtype)!='Int64' or _schema_probe.test_period_label.iloc[1]!='historical':
    fail('PARQUET_SCHEMA','Смешанные test_year снова повреждают метку исторического периода')
del _schema_probe

EVIDENCE_PARQUET={}
def parquet_pack(name,frame):
    progress_event('EXPORT',f'Parquet {name}: {len(frame):,} строк, порциями',announce=True)
    # Физически пишем по 10 тыс. строк; никакой pa.Table из полного DataFrame.
    clean=prepare_for_parquet(name,frame)
    path=ROOT/'derived'/(name+'.parquet')
    write_df_chunks(clean,path)
    pf=pq.ParquetFile(path)
    if pf.metadata.num_rows!=len(clean):fail('EXPORT','row count changed in '+name)
    if pf.schema_arrow.names!=list(clean.columns):fail('EXPORT','schema changed in '+name)
    sample=pf.read_row_group(0).slice(0,min(100,len(clean))).to_pandas()
    if len(sample)!=min(len(clean),100):fail('EXPORT','roundtrip failed '+name)
    EVIDENCE_PARQUET[name]={'rows':len(clean),'columns':len(clean.columns),
        'schema':str(pf.schema_arrow),'sha256':hash_file(path),'path':str(path)}
    del sample,clean,pf;gc.collect()
    return path

def excel_table(writer,source,sheet,cap=None):
    if not isinstance(source,pd.DataFrame) or source.empty:return False
    # XlsxWriter constant_memory + последовательная запись строк (pandas.to_excel с ним несовместим).
    cap=int(cap or CFG.EXCEL_PREVIEW_ROWS)
    x=source.head(cap)
    ws=writer.book.add_worksheet(sheet[:31]);ws.freeze_panes(1,0)
    header=writer.book.add_format({'bold':True,'bg_color':'#294862','font_color':'#FFFFFF','text_wrap':True,'valign':'vcenter'})
    ws.write_row(0,0,[str(c) for c in x.columns],header)
    ws.set_row(0,30)
    for j,c in enumerate(x.columns):ws.set_column(j,j,min(43,max(11,len(str(c))+2)))
    for idx,row in enumerate(x.itertuples(index=False,name=None),1):
        values=[]
        for v in row:
            if v is None or v is pd.NA:values.append(None)
            elif isinstance(v,(dict,list,tuple,set)):values.append(json.dumps(v,ensure_ascii=False,default=str))
            elif isinstance(v,(datetime,pd.Timestamp)):values.append(str(v))
            elif isinstance(v,(np.integer,)):values.append(int(v))
            elif isinstance(v,(np.floating,)):values.append(float(v) if np.isfinite(v) else None)
            elif isinstance(v,(int,float,bool,str)):values.append(v if not isinstance(v,float) or np.isfinite(v) else None)
            else:values.append(str(v))
        ws.write_row(idx,0,values)
    ws.autofilter(0,0,len(x),len(x.columns)-1)
    return True

EXCEL_TABLES=[
 ('v4_all52_source_value_audit','V4 данные 52 из 52'),
 ('v4_all52_research_use','V4 использование 52'),
 ('v4_source_scan_failures','V4 ошибки источников'),
 ('v4_specialty_admission_gate','V4 прием спец. данные'),
 ('v4_specialty_model_frame','V4 прогноз спец. база'),
 ('v4_organization_contingent_candidates','V4 контингент ВО'),
 ('v4_organization_contingent_model_frame','V4 прогноз контингента'),
 ('v4_additional_subexperiments','V4 дополнительные задачи'),
 ('v4_ml57_indicator_support','V4 охват БДМО'),
 ('v4_ml57_exact_common_support','V4 E0 E1 состав'),
 ('v4_ml57_results','V4 E0 E1 метрики'),
 ('v4_ml57_execution','V4 E0 E1 статус'),
 ('v4_ml55_crosssection_metrics','V4 занятость тест'),
 ('v4_ml55_execution','V4 выпуск тест'),
 ('v4_ml56_node_forecast','V4 сеть тест'),
 ('v4_exploratory_paired_comparisons','V4 парные сравнения'),
 ('v4_methods_execution_ledger','V4 выполнение методов'),
 ('v4_rolling_origin_feasibility','V4 скользящие годы'),
 ('v4_scientific_acceptance','V4 научная приемка'),
 ('full52_source_inventory', 'Все 52 источника'),
 ('seven_module_status_final', 'Семь модулей итог'),
 ('ml51_spatial_protocols', 'Пространственный тест OD'),
 ('ml54_history_persistence_diagnostic', 'Тренды спец. диагностика'),
 ('ml52_admission_candidates', 'Кандидаты приема'),
 ('full52_year_coverage', 'Все годы источников'),
 ('full52_theme_coverage', 'Охват тематик'),
 ('full52_raw_originals', 'Оригинальные XLSX'),
 ('bdmo_education_candidates', 'БДМО образование'),
 ('municipal_infrastructure_candidates', 'Инфраструктура ОКТМО'),
 ('spatial_exact_crosswalk', 'География ОКТМО'),
 ('e04_e05_full_family_audit', 'Все массивы выпускников'),
 ('vpo_all_sections_audit', 'Все разделы ВПО'),
 ('spo_annual_vs_master', 'СПО годовые сверки'),
 ('d02_indicator_coverage', 'Экономика 11 источников'),
 ('module_52_extension', 'Доп модули'),
 ('ml54_specialty_history', 'История специальностей'),
 ('ml55_outcome_disparities', 'Выпускники различия'),
 ('ml56_network_stress', 'Устойчивость сети'),
 ('ml57_common_support','БДМО E0 E1 исходные'),

 ('stage1_sources','Источники Stage I'),('semantic_indicator_catalog','Семантика индикаторов'),
 ('admission_approval_register','Утверждение индикаторов'),('registry_coverage_audit','Охват реестров'),
 ('organization_sector_subtasks','ВО СПО итоги модулей'),
 ('e01_original_headers','E01 первичные заголовки'),('e01_raw_cell_sources','E01 raw-проверка'),
 ('od_composition','Состав OD'),('od_edge_coverage','Покрытие OD'),
 ('od_edge_transitions','Новые и исчезнувшие OD'),('edu_org_registry','Реестр организаций'),
 ('edu_org_year','Приём ВО СПО'),('vpo_admission_audit','Приём ВПО-1'),
 ('program_profile','Программы'),('program_concentration','Концентрация программ'),
 ('vpo_program_description','ВПО программы отдельно'),
 ('graduate_full_scan_audit','E04 E05 полное сканирование'),('graduate_descriptive_statistics','Занятость выпускников'),('graduate_specialty_description','Выпуск по специальностям'),('municipality_education_context','Муниципалитеты E0 E1'),
 ('economic_indicator_candidates','Экономика источники'),('regional_economy','Региональные факторы'),
 ('region_distance_proxy','Транспорт и расстояния'),('education_network_year','Сетевые показатели'),
 ('education_network_communities','Сообщества'),('network_stability','Динамика сети'),
 ('seven_module_status','Семь модулей'),('module_diagnostics','Статусы модулей'),
 ('seven_hypotheses','Семь гипотез'),('model_metrics','Метрики'),
 ('confirmatory_comparisons','Подтверждающие тесты'),('split_audit','Временные разбиения'),
 ('future_mutation_canaries','Защита от утечек'),('metrics_recomputed_from_predictions','Независимый пересчёт'),
 ('entity_match_quarantine','Спорные совпадения'),('graduate_outcomes','Выпускники исходные'),
 ('edu_od_interregional','Рёбра межрегиональные')]

passport=pd.DataFrame([
  {'Параметр':'Проект','Значение':'Демографический советник · ML-05 Образовательная миграция'},
  {'Параметр':'Версия','Значение':CFG.VERSION},
  {'Параметр':'Режим','Значение':CFG.MODE},
  {'Параметр':'Снапшот Stage I','Значение':SOURCE_COMMIT},
  {'Параметр':'Каталог Stage I','Значение':index_info['sha']},
  {'Параметр':'Наборы в каталоге','Значение':len(ASSETS)},
  {'Параметр':'Наблюдаемая OD-связка','Значение':'Регион общего образования → регион СПО; не является фактом переезда'},
  {'Параметр':'E01 зачисления','Значение':'Не подтверждены первичным определением'},
  {'Параметр':'Статус научного исполнения','Значение':STATE['scientific']},
  {'Параметр':'Ограничение временного теста','Значение':'Исторические source vintages не доступны; ретроспективная проверка'},
  {'Параметр':'Публикация','Значение':'Только FULL при локальной приёмке и явно включённом разрешении'},
])

# При повторном запуске после ошибки NOTEBOOK_CAPTURE не пересоздаём огромный evidence.
def _completed_evidence_available():
    if not all(p.is_file() and p.stat().st_size>100 for p in (EXCEL,EVIDENCE)):return False
    try:
        with zipfile.ZipFile(EVIDENCE) as z:
            cfg=json.loads(z.read('metadata/config.json'))
            names=z.namelist()
            return (cfg.get('VERSION')==CFG.VERSION and 'metadata/v5_module_acceptance.json' in names
                    and 'predictions/education_predictions.parquet' in names
                    and any(p.startswith('models/') for p in names)
                    and any(p.startswith('panels/') for p in names))
    except Exception:return False

_REUSE_EXPORT=_completed_evidence_available()
if _REUSE_EXPORT:
    progress_event('EXPORT','Готовые Excel/evidence V5.1 обнаружены: повторная упаковка не требуется',announce=True)
    # Восстанавливаем реестр Parquet, сброшенный в начале этой ячейки.
    with zipfile.ZipFile(EVIDENCE) as z:
        for member in z.namelist():
            if not member.endswith('.parquet') or not member.startswith(('panels/','predictions/')):continue
            key=Path(member).stem
            p=Path(PRED_STORAGE_FILE) if key=='education_predictions' and PRED_STORAGE_FILE else Path(PANEL_FILES.get(key,ROOT/'derived'/(key+'.parquet')))
            if not p.is_file():
                p=ROOT/'derived'/(key+'.parquet');p.parent.mkdir(parents=True,exist_ok=True)
                with z.open(member) as source,p.open('wb') as target:shutil.copyfileobj(source,target,4*1024**2)
            pf=pq.ParquetFile(p)
            EVIDENCE_PARQUET[key]={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
              'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
    progress_event('EXPORT',f'Восстановлен реестр {len(EVIDENCE_PARQUET)} Parquet без пересчёта',announce=True)

if not _REUSE_EXPORT:
    progress_event('EXPORT','Сборка многолистового Excel',announce=True)
    with pd.ExcelWriter(EXCEL,engine='xlsxwriter',engine_kwargs={'options':{'constant_memory':True,'strings_to_formulas':False,'strings_to_urls':False}}) as writer:
        excel_table(writer,passport,'Паспорт')
        for key,title in EXCEL_TABLES:excel_table(writer,PANELS.get(key),title)
        excel_table(writer,pd.DataFrame(TABLE_QA),'Контроль таблиц')
        excel_table(writer,pd.DataFrame(ISSUES),'Ограничения')
        excel_table(writer,pd.DataFrame(LOG),'Журнал диагностики')

METHODOLOGY='''# ML-05 V6.0.0. Научный протокол

Восемь различных событий мобильности не отождествляются.
E01 region_of_previous_general_education × region_of_current_study НЕ является доказанным годовым приёмом.
Ретроспективные модели оценивают только измеренную E01 величину; нулевые отсутствующие рёбра не создаются.
Разбиения train/DEV/TEST подчинены доступности меток по консервативному лагу. Если DEV не был известен к test origin,
победителя из теста не выбирают; persistence остаётся предзаданной основной моделью.
Историческая версия источника 2026 не доказывает фактическую доступность значений в 2021–2025.
Сети описывают только наблюдаемые связи регионов предыдущего образования и обучения.
Выпускники E04/E05 — отдельный employment_reference 2024-12-31: место работы не известно, retention нельзя оценить.
Пересекающиеся географические/гендерные агрегаты E04/E05 нельзя суммировать, поэтому описаны медианы исходных строк по разрезам.
В пространственном E0/E1 отсутствие подтверждённой образовательной площадки не равно отсутствию площадки.
Экономические и транспортные признаки включаются только при доказанном grain и лаге.
Отрицательные результаты сохраняются, причинные эффекты по прогнозной важности не идентифицируются.
'''

if not _REUSE_EXPORT:
    progress_event('EXPORT','Сборка evidence-ZIP и контроль отдельных Parquet',announce=True)
    with zipfile.ZipFile(EVIDENCE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
        z.writestr('research_methods.md',METHODOLOGY)
        z.writestr('metadata/config.json',json.dumps(asdict(CFG),ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/plan.json',json.dumps(EXPERIMENT_PLAN,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/semantics_e01.json',json.dumps(OD_SEMANTIC,ensure_ascii=False,indent=2))
        z.writestr('metadata/qa.json',json.dumps(TABLE_QA,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/source_hashes.json',json.dumps(INPUT_SHA,ensure_ascii=False,indent=2))
        z.writestr('metadata/issues.json',json.dumps(ISSUES,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/module_status.json',json.dumps(MODULES,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/splits.json',json.dumps(SPLIT_AUDIT,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/catalog_52_sha.json',json.dumps(INPUT_SHA,ensure_ascii=False,indent=2))
        z.writestr('metadata/v5_module_acceptance.json',V5_LEDGER.to_json(orient='records',force_ascii=False,indent=2))
        z.writestr('metadata/v5_model_registry.json',json.dumps(MODEL_RECORDS,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/catalog_52_protocol.md', 'Каждый из 52 Parquet: SHA-256, число строк, schema, годовой охват. '+'Годовые shards СПО не суммируются с общей панелью. Другие уровни данных не приписываются образовательному OD.')
        for key,df in PANELS.items():
            if not isinstance(df,pd.DataFrame) or df.empty:continue
            if key in PANEL_FILES:
                p=Path(PANEL_FILES[key]);pf=pq.ParquetFile(p)
                EVIDENCE_PARQUET[key]={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
                    'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
            else:p=parquet_pack(key,df)
            z.write(p,'panels/'+key+'.parquet')
        if PRED_STORAGE_FILE is not None:
            p=Path(PRED_STORAGE_FILE);pf=pq.ParquetFile(p)
            EVIDENCE_PARQUET['education_predictions']={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
                'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
            z.write(p,'predictions/education_predictions.parquet')
        # Large source-grouped context: include every successfully built Parquet, not only Excel preview.
        for p in CONTEXT_PARQUET_FILES:
            if not Path(p).exists():fail('EXPORT','Отсутствует исходный контекстный Parquet '+str(p))
            z.write(p,'context/'+Path(p).name)
        for path in FIGURES:z.write(path,'figures/'+Path(path).name)
        for name,path in MODEL_FILES.items():z.write(path,'models/'+name)
        z.writestr('logs/run_events.json',json.dumps(LOG,ensure_ascii=False,indent=2))

# V6 SOURCE CAPTURE: восстанавливаем полный код из самого ноутбука; Colab get_ipynb НЕ используется.
progress_event('EXPORT','Сохранение встроенного неизменяемого V6 notebook',announce=True)
snap=notebook_from_embedded_source()
for _cell in snap.cells:
    if _cell.cell_type=='code':
        _cell.outputs=[]
        _cell.execution_count=None
nbformat.validate(snap)
clean_text=nbformat.writes(snap)
_token=globals().get('_TOKEN')
if (isinstance(_token,str) and len(_token)>15 and _token in clean_text) or re.search(r'(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}',clean_text):
    raise RuntimeError('V6 SECURITY: notebook contains GitHub token; export blocked')
_tmp_notebook=NOTEBOOK.with_suffix('.tmp.ipynb')
nbformat.write(snap,_tmp_notebook)
_tmp_notebook.replace(NOTEBOOK)
_saved_nb=nbformat.read(NOTEBOOK,as_version=4)
if [c.source for c in _saved_nb.cells]!=[c.source for c in snap.cells]:
    raise RuntimeError('V6: notebook roundtrip changed sources')
progress_event('EXPORT',f'V6 notebook saved, {len(snap.cells)} cells',announce=True)

# Manifest создаётся после локальных артефактов и содержит их реальные SHA; его собственный SHA нельзя включить в него же.
output_hashes={p.name:hash_file(p) for p in [EXCEL,EVIDENCE,NOTEBOOK] if p.exists()}
manifest={
 'version':CFG.VERSION,'mode':CFG.MODE,'created_utc':stamp(),
 'source_repo':CFG.REPO,'source_commit':SOURCE_COMMIT,'source_index_git_blob_sha':index_info['sha'],
 'source_index_version':INDEX['version'],'source_hashes':INPUT_SHA,
 'context_derived_files':{Path(p).name:{'rows':pq.ParquetFile(p).metadata.num_rows,'sha256':hash_file(p)} for p in CONTEXT_PARQUET_FILES},
 'catalog_52_coverage':{'registered':len(ASSETS),'sha_verified':len(INPUT_SHA),
 'verified_every_asset':len(INPUT_SHA)==len(ASSETS),
 'year_profiled':len(FULL52_YEAR_STATS)},
 'experiment_plan_sha256':PLAN_HASH,'v5_science_gates':{'all52_source_content_verified':not V4_SOURCE_ERRORS,'all_feasible_computations_succeeded':V5_COMPUTATIONAL_PASS,'all_seven_scientifically_identified':V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS,'model_failure_count':int((~V5_LEDGER.computational_pass).sum())},'status':STATE.copy(),'module_status':MODULES,
 'observed_od_counts':{'all':len(OD),'inter':len(OD_INTER),'intra':len(OD_INTRA),'unknown':len(OD_UNKNOWN)},
 'real_prediction_rows':PRED_TOTAL,'v5_model_file_count':len(MODEL_FILES),
 'v5_module_acceptance':V5_MODULE_ACCEPTANCE,
 'v5_original_seven_target_identifiable':V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS,'metric_rows':len(METRICS),'panels':{
      k:{'rows':v['rows'],'columns':v['columns'],'sha256':v['sha256']} for k,v in EVIDENCE_PARQUET.items()},
 'output_sha256':output_hashes,'runtime_seconds':round(time.monotonic()-START_TIME,1),
 'limitations':ISSUES,'publication_status':'NOT_PUBLISHED',
 'commit_self_reference':'Commit SHA can be reported after Git commit; cannot be embedded in the same hashed blob'
}
progress_event('EXPORT','Манифест с SHA-256 локальных результатов',announce=True)
MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
print('Local package:',EXCEL.name,round(EXCEL.stat().st_size/1e6,2),'MB;',
      EVIDENCE.name,round(EVIDENCE.stat().st_size/1e6,2),'MB; notebook:',NOTEBOOK.exists())

memory_point('after_export')
mark_stage('EXPORT')


## 13. Строгая локальная приёмка семи модулей, численных результатов и 52 источников

In [ ]:
progress_event('QA_AND_PUBLICATION','Локальная приёмка, воспроизводимость и безопасная публикация')
# Финальный QA запускается до первого GitHub-запроса на запись.
LOCAL_CHECKS=[]
def check(name,condition,why=''):
    ok=bool(condition)
    LOCAL_CHECKS.append({'check':name,'status':'PASS' if ok else 'FAIL','reason':why})
    if not ok:note('QA',name+': '+why,'FAIL')
    return ok

check('INPUT_SOURCE_PIN',bool(SOURCE_COMMIT and index_info['sha']==CFG.EXPECTED_INDEX_BLOB_SHA))
check('SEVEN_MODULES_REPORTED',len(MODULES)==7 and all(v['status']!='NOT_STARTED' for v in MODULES.values()))
check('EXPECTED_CATALOG',len(ASSETS)==52)
check('FULL52_SHA_COVERAGE',CFG.MODE!='FULL' or (len(INPUT_SHA)==len(ASSETS) and all(a['asset_id'] in INPUT_SHA for a in ASSETS)))
check('FULL52_ROW_COVERAGE',CFG.MODE!='FULL' or (len(FULL52_INVENTORY)==len(ASSETS) and all(r.get('footer_rows_ok') for r in FULL52_INVENTORY)))
check('FULL52_YEAR_DISTRIBUTION',CFG.MODE!='FULL' or all(r['year_scan_status'] in ('SCANNED_FULL','NO_TIME_KEY') for r in FULL52_INVENTORY))
check('FULL52_D02_PROFILE',CFG.MODE!='FULL' or (len(D02_PROFILE)==11 and all(r['status']=='GROUPED_ALL_SOURCE_ROWS_NO_DOUBLE_COUNT' for r in D02_PROFILE)))
check('FULL52_VPO_PROFILE',CFG.MODE!='FULL' or (len(VPO_ALL_PROFILE)==11 and all(r['status']=='PASS' for r in VPO_ALL_PROFILE)))
check('FULL52_E04_E05_PROFILE',CFG.MODE!='FULL' or (len(FAMILY_E04E05)==4 and all(r['status']=='PASS_NOT_ADDITIVE' for r in FAMILY_E04E05)))
check('FULL52_CONTEXT_EXPORT',CFG.MODE!='FULL' or len(CONTEXT_PARQUET_FILES)>=26)
check('V4_SOURCE_VALUES_52',CFG.MODE!='FULL' or (len(V4_SOURCE_PROFILES)==52 and not V4_SOURCE_ERRORS))
check('V4_MODEL_FAILURES_ZERO',V4_COMPUTATIONAL_COMPLETE,'Failed model or source value analysis is a real computational failure')
check('V5_COMPLETE_COMPUTATIONAL_CONTRACT',V5_COMPUTATIONAL_PASS,'All feasible models and checks must complete; no hidden failures')
check('V5_UNIQUE_MODEL_ARTIFACTS',len(MODEL_FILES)==len(MODEL_RECORDS),'No overwritten spatial model files')
check('V5_ALL_MODELS_VERIFIED',bool(V5_MODEL_QA) and all(x['sha_verified'] for x in V5_MODEL_QA))
check('V5_SEVEN_MODULE_LEDGER',len(V5_LEDGER)==7 and V5_LEDGER.module.nunique()==7)
check('V5_ALL52_SOURCE_EVIDENCE',CFG.MODE!='FULL' or len(V5_SOURCE_EVIDENCE)==52)
check('V5_ML54_NO_SAMPLE',CFG.MODE!='FULL' or (len(PANELS.get('v5_ml54_full_strata_coverage',pd.DataFrame()))==1))
check('V4_FINAL_LEDGER',len(V4_METHOD_EXECUTIONS)==7)
check('V4_SCIENTIFIC_TRUTHFUL',not V4_SCIENTIFIC_COMPLETE or (len(ORG_YEAR)>0 and OD_SEMANTIC.get('yearly_count_unit_confirmed',False)))
check('E01_PRIMARY_OPENED',CFG.MODE!='FULL' or bool(OD_SEMANTIC.get('original_opened')))
check('SEMANTIC_APPROVAL_GATED',not len(ORG_YEAR) or all(
    name in set(CFG.APPROVED_VO_ADMISSION_NAMES)|set(CFG.APPROVED_SPO_ADMISSION_NAMES)
    for name in ORG_YEAR.indicator_name.dropna().astype(str).unique()))
check('EDGES_PARTITION',len(OD)==len(OD_INTER)+len(OD_INTRA)+len(OD_UNKNOWN))
check('INTRA_EXCLUDED',OD_INTER.origin_region.ne(OD_INTER.destination_region).all())
check('NO_FUTURE_FEATURES',all(v.get('status')!='FAIL' for v in CANARIES))
check('CANARIES_EXECUTED',any(v.get('module')=='ML-5.1' and v.get('status')=='PASS' for v in CANARIES) and
      any(v.get('module')=='ML-5.3' and v.get('status')=='PASS' for v in CANARIES))
check('NO_TEST_SELECTED_MODEL',not len(CONFIRM) or
      (CONFIRM.choice_basis.isin(['DEV_ASOF','NO_CONFIRMATORY_COMPARISON_NO_DEV']).all() and
       CONFIRM.loc[CONFIRM.choice_basis.eq('NO_CONFIRMATORY_COMPARISON_NO_DEV'),
                   'selected_model'].eq('persistence').all()))
check('NEGATIVE_FINDINGS_PRESERVED',len(HYPOTHESES)==7)
check('ZIP_CRC',zipfile.ZipFile(EVIDENCE).testzip() is None)
with zipfile.ZipFile(EXCEL) as z:
    check('EXCEL_OPENS',z.testzip() is None and 'xl/workbook.xml' in z.namelist())
check('MODEL_RELOAD',all(m.get('reloaded') for m in MODEL_RECORDS))
check('NOTEBOOK_CAPTURED',NOTEBOOK.exists(),'Необходимо получить очищенный самостоятельный Colab notebook')
if NOTEBOOK.exists():
    try:
        nb=nbformat.read(NOTEBOOK,as_version=4)
        nbformat.validate(nb)
        content=NOTEBOOK.read_text(encoding='utf-8')
        suspicious=re.findall(r'\b(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}\b',content)
        check('NOTEBOOK_NO_TOKEN',not suspicious and _TOKEN not in content)
        check('NOTEBOOK_NO_OUTPUTS',sum(len(c.get('outputs',[])) for c in nb.cells if c.cell_type=='code')==0)
        check('NOTEBOOK_SELF_CONTAINED',not re.search(r'(?m)^\s*(?:%run|%%writefile)\b',content))
        check('NOTEBOOK_REQUIRED_SECTION_COUNT',sum(c.cell_type=='code' for c in nb.cells)>=12)
        check('NOTEBOOK_REASONABLE_SIZE',NOTEBOOK.stat().st_size<2_000_000)
    except Exception as ex:check('NOTEBOOK_VALIDATE',False,type(ex).__name__)

# Проверяем независимые Parquet исходя из физического ZIP, не доверяя только собственным логам.
with zipfile.ZipFile(EVIDENCE) as z:
    for name,info in EVIDENCE_PARQUET.items():
        target=('predictions/' if name=='education_predictions' else 'panels/')+name+'.parquet'
        if target not in z.namelist():
            check('PARQUET_'+name,False,'Нет файла в evidence');continue
        try:
            # Не z.read(): весь Parquet в bytes может мгновенно исчерпать RAM.
            p=Path(info['path']);pf=pq.ParquetFile(p)
            member=z.getinfo(target)
            check('PARQUET_'+name,int(pf.metadata.num_rows)==info['rows'] and
                  len(pf.schema_arrow.names)==info['columns'] and
                  int(member.file_size)==p.stat().st_size and hash_file(p)==info['sha256'])
        except Exception as ex:check('PARQUET_'+name,False,type(ex).__name__)

LOCAL_QA=pd.DataFrame(LOCAL_CHECKS)
progress_event('QA_AND_PUBLICATION',f'Проверено {len(LOCAL_QA):,} контролей; ошибок: {int(LOCAL_QA.status.eq("FAIL").sum())}',announce=True)
# Перезаписываем локальный manifest после QA, но Excel/ZIP уже фиксированы по SHA.
manifest['local_qa']=LOCAL_CHECKS
manifest['status']['technical']='PASS' if LOCAL_QA.status.eq('PASS').all() else 'FAIL'
manifest['status']['publication']='NOT_ATTEMPTED'
manifest['status']['overall']=(STATE['scientific'] if manifest['status']['technical']=='PASS'
                              else 'PARTIAL_LOCAL_QA')
MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
print('Independent QA:',int(LOCAL_QA.status.eq('PASS').sum()),'PASS /',len(LOCAL_QA),'checks')

# Публикация отсюда не выполняется, даже при FAIL локальной QA.
# Новый финальный этап публикует артефакты с честным статусом QA; сами данные не теряются.
PUBLICATION_RECEIPT={'status':'PENDING_FINAL_STAGE'}
STATE['publication']='PENDING_FINAL_STAGE'

print('ML-05 V5:',STATE['scientific'],'| technical:',manifest['status']['technical'],
      '| GitHub:',STATE['publication'])
print('Output paths:',EXCEL,EVIDENCE,NOTEBOOK,MANIFEST)

mark_stage('QA_AND_PUBLICATION')
# Heartbeat работает и во время сборки/загрузки итогового ZIP; закрываем после последней ячейки.


## Приёмочный контракт V6.0.0

- **TECHNICAL_PASS**: все 52 исходных набора проверены по SHA, 7 модулей имеют журналы исполнения, обязательные по данным расчёты завершены, модельные файлы сверены по SHA и локальная QA без ошибок.
- **SCIENTIFIC_LIMITED**: измеряемая цель не идентифицируется в первичных данных. Этот статус честно сохраняется в отчёте и не подменяется иной величиной.
- **REPLACE**: если локальные вычисления и QA успешны, публикуется новый Git-коммит, заменяющий только целевой ML-05 сегмент. При техническом FAIL полные результаты могут быть сохранены как черновой Release, но старая рабочая версия не удаляется.


## 14. Итог: компактная автозагрузка отчёта → проверенный GitHub Release → атомарная замена старого ML-05

In [ ]:
# ML-05 V6.0.0 · COMPLETE PIPELINE / VERIFIED GITHUB REPLACEMENT / COMPACT DOWNLOAD
# FULL -> 52 SHA verified -> seven modules -> QA -> small report -> draft Release -> atomic segment replace.
# Does not delete Stage I, unrelated projects or historical git commits. Every destructive change is prefix-scoped.
import json, os, math, hashlib, base64, zipfile, shutil, re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

progress_event('DOWNLOAD_RESULTS','V6: итоговый QA, компактная аналитика и безопасная замена ML-05',announce=True)
if CFG.MODE != 'FULL': raise RuntimeError('V6: для публикации необходим FULL')
if not CFG.PUBLISH_RESULTS_TO_PRIVATE_GITHUB: raise RuntimeError('V6: публикация в приватный GitHub выключена')
if not CFG.STORE_DERIVED_RESTRICTED_DATA: raise RuntimeError('V6: не разрешено публиковать производные результаты')
if not api_json('GET',BASE).get('private'): raise RuntimeError('V6: репозиторий не приватный')
if len(ASSETS)!=52 or len(INPUT_SHA)!=52 or len(V5_SOURCE_EVIDENCE)!=52:
    raise RuntimeError('V6: 52 источника не подтверждены SHA и профилями')
if len(V5_LEDGER)!=7 or V5_LEDGER.module.nunique()!=7:
    raise RuntimeError('V6: отсутствует одна или несколько приёмок семи модулей')

out=ROOT/'exports'; out.mkdir(parents=True,exist_ok=True)
needed=[Path(p) for p in (EXCEL,EVIDENCE,NOTEBOOK,MANIFEST)]
for p in needed:
    if not p.is_file() or p.stat().st_size==0: raise RuntimeError('V6: отсутствует файл '+str(p))
_saved=json.loads(Path(MANIFEST).read_text(encoding='utf-8'))
if _saved.get('version')!=CFG.VERSION or _saved.get('source_commit')!=SOURCE_COMMIT:
    raise RuntimeError('V6: манифест не соответствует активному запуску')
for p in (EXCEL,EVIDENCE,NOTEBOOK):
    if _saved.get('output_sha256',{}).get(Path(p).name)!=hash_file(p):
        raise RuntimeError('V6: нарушена контрольная сумма '+str(p))

# Source capture is a deterministic embedded copy of this notebook, not a stale Colab get_ipynb.
recovered=notebook_from_embedded_source()
if hash_file(NOTEBOOK)!=hashlib.sha256(nbformat.writes(recovered).encode('utf-8')).hexdigest():
    # nbformat serialization may differ in insignificant line breaks. Revalidate actual source instead.
    _check=nbformat.read(NOTEBOOK,as_version=4)
    if sum(c.cell_type=='code' for c in _check.cells)!=sum(c.cell_type=='code' for c in recovered.cells):
        raise RuntimeError('V6: сохранённый ноутбук не совпал с встроенным источником')
    _check_src=[c.source for c in _check.cells]
    if _check_src != [c.source for c in recovered.cells]:
        raise RuntimeError('V6: исходный код ноутбука отличается от эталона')

sha_evidence=hash_file(EVIDENCE)
run_key=hashlib.sha256((CFG.VERSION+'|'+SOURCE_COMMIT+'|'+sha_evidence).encode()).hexdigest()[:20]
release_tag='ml05-v6-full-'+run_key
prefix=CFG.OUTPUT_PREFIX.rstrip('/')+'/'
if prefix!='stage2/ml05_education/': raise RuntimeError('V6: удаление разрешено ТОЛЬКО в stage2/ml05_education/')
qa_ok=bool(len(LOCAL_QA) and LOCAL_QA.status.eq('PASS').all())
compute_ok=bool(V5_COMPUTATIONAL_PASS)
science_ok=bool(V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS)
can_replace=bool(qa_ok and compute_ok)

# Explicitly distinguish 52/52 source processing from direct model use and lack of identification.
source_ledger=[]
usage={str(x.get('asset_id')):x for x in V4_SOURCE_PROFILES}
for a in ASSETS:
    u=usage.get(a['asset_id'],{})
    source_ledger.append({'asset_id':a['asset_id'],'group':a['segment'],'registered_rows':a.get('rows_actual'),
        'sha256':a['sha256'],'sha_verified':a['asset_id'] in INPUT_SHA,
        'content_profile':u.get('content_status','NOT_PROFILED'),
        'direct_model_input':bool(u.get('main_prediction_input',False)),
        'role_note':'Источники с несовместимым измерением не объявляются признаками прогноза'})
source_csv=out/'ML5_V6_SOURCE_COVERAGE_52.csv';pd.DataFrame(source_ledger).to_csv(source_csv,index=False,encoding='utf-8-sig')
module_csv=out/'ML5_V6_SEVEN_MODULES.csv';V5_LEDGER.to_csv(module_csv,index=False,encoding='utf-8-sig')
metrics_csv=out/'ML5_V6_METRICS.csv';pd.DataFrame(METRICS).to_csv(metrics_csv,index=False,encoding='utf-8-sig')
models_csv=out/'ML5_V6_MODELS.csv';pd.DataFrame(V5_MODEL_QA).to_csv(models_csv,index=False,encoding='utf-8-sig')
qa_csv=out/'ML5_V6_TECHNICAL_QA.csv';LOCAL_QA.to_csv(qa_csv,index=False,encoding='utf-8-sig')
issues_csv=out/'ML5_V6_SCIENTIFIC_LIMITATIONS.csv';pd.DataFrame(ISSUES).to_csv(issues_csv,index=False,encoding='utf-8-sig')

summary={
    'version':CFG.VERSION,'run_key':run_key,'git_source_commit':SOURCE_COMMIT,
    'catalog_assets':len(ASSETS),'sha_verified':len(INPUT_SHA),
    'content_profiled':sum(x.get('content_status')=='FULL_VALUE_COVERAGE_PASS' for x in V4_SOURCE_PROFILES),
    'direct_model_sources':sum(x['direct_model_input'] for x in source_ledger),
    'seven_modules':V5_MODULE_ACCEPTANCE,'qa_pass':int(LOCAL_QA.status.eq('PASS').sum()),
    'qa_fail':int(LOCAL_QA.status.eq('FAIL').sum()),'computational_pass':can_replace,
    'original_seven_targets_scientifically_identified':science_ok,
    'prediction_rows':PRED_TOTAL,'metric_rows':len(METRICS),'model_count':len(MODEL_FILES),
    'evidence_sha256':sha_evidence,'evidence_bytes':EVIDENCE.stat().st_size,
    'scientific_caveat':'Невозможный по имеющимся данным прогноз не подменяется иным целевым показателем.'
}
summary_file=out/'ML5_V6_FINAL_ACCEPTANCE.json'
summary_file.write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

# Reports are downloaded before remote upload; complete evidence and heavy models NEVER enter this ZIP.
plots=sorted({Path(p) for p in FIGURES if Path(p).is_file()},key=lambda p:p.name)
reports=[Path(EXCEL),Path(NOTEBOOK),Path(MANIFEST),source_csv,module_csv,metrics_csv,models_csv,qa_csv,issues_csv,summary_file]+plots
small_zip=out/('ML5_V6_ANALYTICS_'+run_key+'.zip')
with zipfile.ZipFile(small_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
    for p in reports:
        z.write(p,('figures/' if p in plots else 'reports/')+p.name)
    z.writestr('reports/checksums.json',json.dumps({p.name:hash_file(p) for p in reports},indent=2))
with zipfile.ZipFile(small_zip) as z:
    if z.testzip() is not None: raise RuntimeError('V6: повреждён отчётный ZIP')
if small_zip.stat().st_size>100*2**20:raise RuntimeError('V6: отчётный архив >100 MiB, проверьте содержимое')
progress_event('DOWNLOAD_RESULTS',f'V6: готов отчётный архив {small_zip.stat().st_size/2**20:.1f} МиБ',announce=True)
print('АНАЛИТИКА:',small_zip,'|','EVIDENCE только в GitHub:',EVIDENCE.stat().st_size/2**20,'МиБ',flush=True)
if CFG.DOWNLOAD_COMPACT_REPORTS:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(small_zip))
    except ImportError:
        from IPython.display import FileLink,display
        display(FileLink(str(small_zip)))

# Precompute resumable chunks on disk; verify recombined SHA before GitHub upload.
size_part=int(CFG.GH_RELEASE_PART_MIB)*2**20
if not 16<=int(CFG.GH_RELEASE_PART_MIB)<=256:raise RuntimeError('V6: part size вне 16..256 MiB')
chunk_dir=out/('release_chunks_'+run_key);chunk_dir.mkdir(parents=True,exist_ok=True)
chunk_index={'version':CFG.VERSION,'source_evidence':Path(EVIDENCE).name,
    'evidence_sha256':sha_evidence,'evidence_bytes':EVIDENCE.stat().st_size,'parts':[]}
with Path(EVIDENCE).open('rb') as src:
    total=math.ceil(EVIDENCE.stat().st_size/size_part)
    for i in range(total):
        name=f'ML5_EVIDENCE.zip.part{i+1:03d}'
        p=chunk_dir/name
        n=min(size_part,EVIDENCE.stat().st_size-i*size_part)
        if p.exists() and p.stat().st_size==n:
            src.seek(n,1)
        else:
            tmp=p.with_suffix('.part.tmp')
            with tmp.open('wb') as dest:
                remaining=n
                while remaining:
                    buf=src.read(min(4*2**20,remaining))
                    if not buf:raise IOError('V6: преждевременный конец evidence')
                    dest.write(buf);remaining-=len(buf)
            tmp.replace(p)
        chunk_index['parts'].append({'name':name,'bytes':n,'sha256':hash_file(p)})
        progress_event('DOWNLOAD_RESULTS',f'Локальная часть evidence {i+1}/{total}',announce=True)
verify_hash=hashlib.sha256()
for part in chunk_index['parts']:
    with (chunk_dir/part['name']).open('rb') as f:
        for buf in iter(lambda:f.read(4*2**20),b''):verify_hash.update(buf)
if verify_hash.hexdigest()!=sha_evidence:raise RuntimeError('V6: разбитый evidence SHA не совпадает с исходным')
index_file=out/'ML5_V6_RELEASE_PARTS.json'
index_file.write_text(json.dumps(chunk_index,ensure_ascii=False,indent=2),encoding='utf-8')

# Helpers for GitHub. An existing asset is reused ONLY if GitHub's SHA-256 and size match.
def remote_release_assets(release_id):
    return {a['name']:a for a in api_json('GET',BASE+f'/releases/{release_id}/assets',params={'per_page':100})}

def upload_release_asset(release_id,p,mime):
    p=Path(p);sha=hash_file(p)
    already=remote_release_assets(release_id).get(p.name)
    if already and already.get('state')=='uploaded' and already.get('digest')=='sha256:'+sha and int(already.get('size',-1))==p.stat().st_size:
        progress_event('DOWNLOAD_RESULTS','GitHub SHA уже совпадает: '+p.name,announce=True)
        return already
    if already:
        resp=SESS.delete(BASE+'/releases/assets/'+str(already['id']),timeout=(30,120))
        status=resp.status_code;resp.close()
        if status!=204:raise RuntimeError('V6: не удалён повреждённый Release-asset '+p.name)
    upload_url=f'https://uploads.github.com/repos/{CFG.REPO}/releases/{release_id}/assets'
    progress_event('DOWNLOAD_RESULTS',f'GitHub: {p.name} · {p.stat().st_size/2**20:.1f} МиБ',announce=True)
    try:
        with p.open('rb') as f:
            resp=SESS.post(upload_url,params={'name':p.name},headers={'Content-Type':mime},data=f,timeout=(30,3600))
        status=resp.status_code
        if status==201:obj=resp.json()
        else:obj=None
        resp.close()
    except Exception:
        obj=None
    if obj is None:
        obj=remote_release_assets(release_id).get(p.name)
    if not obj or obj.get('state')!='uploaded' or obj.get('digest')!='sha256:'+sha or int(obj.get('size',-1))!=p.stat().st_size:
        raise RuntimeError('V6: ошибка GitHub asset '+p.name+'; можно повторить только финальную ячейку')
    return obj

branch=CFG.TARGET_BRANCH
remote_head=repo_head(branch)
if remote_head!=TARGET_HEAD_AT_PREFLIGHT:
    # A rerun after the successful commit is allowed, but unrelated changes must not be overwritten.
    try:
        existing=json.loads(read_git_blob(prefix+'publication_receipt.json',ref=remote_head))
        already_done=(existing.get('run_key')==run_key and existing.get('evidence_sha256')==sha_evidence)
    except Exception:
        already_done=False
    if not already_done:raise RuntimeError('V6: ветка изменилась после preflight. Не перезаписываю чужие изменения.')
else:already_done=False

# Atomic replacement is permitted ONLY if all computational checks and local QA passed.
# Otherwise publish a diagnostic draft Release; leave old active segment unchanged.
if not can_replace:
    print('V6: QA/вычисления не полностью пройдены. Отчёты скачаны. Старый рабочий сегмент НЕ удаляю.',flush=True)

publication_plan=out/'ML5_V6_PUBLICATION_PLAN.json'
planned=None
if publication_plan.is_file():
    try:
        j=json.loads(publication_plan.read_text('utf-8'))
        if j.get('run_key')==run_key and j.get('base_sha')==TARGET_HEAD_AT_PREFLIGHT:planned=j
    except Exception:pass

publication_receipt={
    'version':CFG.VERSION,'run_key':run_key,'release_tag':release_tag,
    'release_url':f'https://github.com/{CFG.REPO}/releases/tag/{release_tag}',
    'evidence_sha256':sha_evidence,'parts_manifest':index_file.name,
    'source_commit':SOURCE_COMMIT,'source_count':52,'computational_pass':can_replace,
    'scientific_targets_all_identified':science_ok,'target_prefix':prefix,
    'replacement_policy':'atomic current ML-05 tree only; preserve git history and Stage I'
}
receipt_path=out/'publication_receipt.json'
receipt_path.write_text(json.dumps(publication_receipt,ensure_ascii=False,indent=2),encoding='utf-8')
pubmanifest=out/'run_manifest_publication.json'
pubm=json.loads(Path(MANIFEST).read_text('utf-8'))
pubm['publication_status']='PUBLISHED_GITHUB' if can_replace else 'DRAFT_PARTIAL_FAIL'
pubm['publication_reference']={'release_tag':release_tag,'evidence_sha256':sha_evidence}
pubmanifest.write_text(json.dumps(pubm,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

# What remains in active stage2/ml05_education AFTER successful update.
target_files={
    prefix+'ML5_EDUCATION_MIGRATION.ipynb':Path(NOTEBOOK),
    prefix+'ML5_EDUCATION_MIGRATION_RESULTS.xlsx':Path(EXCEL),
    prefix+'run_manifest.json':pubmanifest,
    prefix+'publication_receipt.json':receipt_path,
    prefix+index_file.name:index_file,
    prefix+summary_file.name:summary_file,
    prefix+source_csv.name:source_csv,
    prefix+module_csv.name:module_csv,
    prefix+metrics_csv.name:metrics_csv,
    prefix+qa_csv.name:qa_csv,
}
if not all(k.startswith(prefix) for k in target_files): raise RuntimeError('V6: нарушена изоляция сегмента')

# Prepare a candidate Git commit *before* creating a release, so new tag points to V6 code, not old V2.
commit_sha=remote_head if already_done else None
if can_replace and not already_done:
    if planned:
        candidate=api_json('GET',BASE+'/git/commits/'+planned['candidate_sha'])
        commit_sha=planned['candidate_sha']
        if candidate['parents'][0]['sha']!=remote_head:raise RuntimeError('V6: кандидат не является потомком HEAD')
    else:
        old_tree=api_json('GET',BASE+'/git/trees/'+remote_head,params={'recursive':1})
        if old_tree.get('truncated'):raise RuntimeError('V6: неполное дерево Git, удаление запрещено')
        old_paths={x['path'] for x in old_tree.get('tree',[]) if x.get('type')=='blob' and x.get('path','').startswith(prefix)}
        obsolete=sorted(old_paths-set(target_files))
        changes=[]
        for filename,path in sorted(target_files.items()):
            binary=Path(path).read_bytes() # Small files only; evidence is Release-only.
            if len(binary)>45*2**20:raise RuntimeError('V6: слишком большой Git blob '+filename)
            blob=api_json('POST',BASE+'/git/blobs',json={'content':base64.b64encode(binary).decode(),'encoding':'base64'})
            changes.append({'path':filename,'mode':'100644','type':'blob','sha':blob['sha']})
        for filename in obsolete:
            changes.append({'path':filename,'mode':'100644','type':'blob','sha':None})
        old_commit=api_json('GET',BASE+'/git/commits/'+remote_head)
        tree=api_json('POST',BASE+'/git/trees',json={'base_tree':old_commit['tree']['sha'],'tree':changes})
        candidate=api_json('POST',BASE+'/git/commits',json={
            'message':f'ML-05 V6 FULL verified replacement: {run_key} (old segment files retired)',
            'tree':tree['sha'],'parents':[remote_head]})
        commit_sha=candidate['sha']
        plan={'run_key':run_key,'base_sha':remote_head,'candidate_sha':commit_sha,
              'obsolete_paths':obsolete,'new_files':sorted(target_files)}
        publication_plan.write_text(json.dumps(plan,ensure_ascii=False,indent=2),encoding='utf-8')
        progress_event('DOWNLOAD_RESULTS',f'Кандидат Git commit готов, старых файлов к замене {len(obsolete)}',announce=True)

# Create a draft Release. The tag MUST point to the prepared V6 commit.
all_releases=api_json('GET',BASE+'/releases',params={'per_page':100})
release=next((x for x in all_releases if x.get('tag_name')==release_tag),None)
if release is None:
    release=api_json('POST',BASE+'/releases',json={
        'tag_name':release_tag,'target_commitish':commit_sha or remote_head,
        'name':'ML-05 V6.0.0 FULL '+run_key,'draft':True,'prerelease':True,
        'body':f'Stage I 52/52. Computational PASS: {can_replace}. Scientific 7/7 identified: {science_ok}. '
               f'Evidence SHA-256 {sha_evidence}. Full results belong to this private repository.'})
release_id=release['id']
for p,mime in [(index_file,'application/json')]+[(chunk_dir/x['name'],'application/octet-stream') for x in chunk_index['parts']]+[(small_zip,'application/zip')]:
    upload_release_asset(release_id,p,mime)
assets=remote_release_assets(release_id)
needed_assets=[index_file,small_zip]+[chunk_dir/x['name'] for x in chunk_index['parts']]
for p in needed_assets:
    a=assets.get(p.name)
    if not a or a.get('state')!='uploaded' or a.get('digest')!='sha256:'+hash_file(p) or int(a.get('size',-1))!=p.stat().st_size:
        raise RuntimeError('V6: удалённый Release-asset не прошёл приёмку '+p.name)

if can_replace:
    if not already_done:
        if repo_head(branch)!=remote_head:raise RuntimeError('V6: конфликт обновления Git ветки, старый сегмент не изменён')
        api_json('PATCH',BASE+'/git/refs/heads/'+quote(branch,safe='/'),json={'sha':commit_sha,'force':False})
        if repo_head(branch)!=commit_sha:raise RuntimeError('V6: не подтверждён Git HEAD после коммита')
    # Check EVERY committed file by round-trip SHA256 from GitHub, then ensure old files are absent.
    for filename,path in target_files.items():
        remote=read_git_blob(filename,ref=commit_sha)
        if hashlib.sha256(remote).hexdigest()!=hash_file(path):
            raise RuntimeError('V6: GitHub roundtrip SHA mismatch '+filename)
    finaltree=api_json('GET',BASE+'/git/trees/'+commit_sha,params={'recursive':1})
    if finaltree.get('truncated'):raise RuntimeError('V6: Git tree truncated in final check')
    now={x['path'] for x in finaltree['tree'] if x['type']=='blob' and x['path'].startswith(prefix)}
    if now!=set(target_files):raise RuntimeError('V6: остались устаревшие файлы ML-05 или пропали новые')
    if release.get('draft'):
        release=api_json('PATCH',BASE+'/releases/'+str(release_id),json={'draft':False,'prerelease':True})
    published=api_json('GET',BASE+'/releases/'+str(release_id))
    if published.get('draft') or published.get('tag_name')!=release_tag:
        raise RuntimeError('V6: новый Release не подтвердился')
    # Confirm the tag points to the new commit, not the retired V2 commit.
    tagref=api_json('GET',BASE+'/git/ref/tags/'+quote(release_tag,safe=''))
    tagsha=tagref.get('object',{}).get('sha')
    if tagsha!=commit_sha:raise RuntimeError('V6: тег нового Release не указывает на проверенный commit')
    STATE['publication']='PASS_GITHUB_V6_REPLACED'
    print('ЗАМЕНА ML-05 ПОДТВЕРЖДЕНА:',commit_sha,flush=True)
    print('Только новый набор файлов в',prefix,'; старые файлы удалены из текущего дерева',flush=True)
else:
    STATE['publication']='DRAFT_REVIEW_ONLY_OLD_SEGMENT_PRESERVED'
    print('Новая доказательная база в черновом Release. Старый сегмент сохранён из-за FAIL.',flush=True)

receipt={**publication_receipt,'commit_sha':commit_sha,'release_id':release_id,
    'final_publication_status':STATE['publication'],
    'time_utc':datetime.now(timezone.utc).isoformat()}
(out/'ML5_V6_PUBLICATION_CONFIRMATION.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf-8')
print('GitHub Release:',publication_receipt['release_url'],flush=True)
print('Небольшой отчётный архив:',small_zip,flush=True)
print('Научная приёмка:', 'PASS' if science_ok else 'LIMITED_BY_OBSERVABILITY',flush=True)
if can_replace:mark_stage('DOWNLOAD_RESULTS')
else:progress_event('DOWNLOAD_RESULTS','Публикация частичного исследования; 100% исполнения не подтверждены',announce=True)
bar=globals().get('MAIN_PROGRESS')
if bar is not None: bar.close()
